In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

PROCESSED = BASE / "processed_data"
EXPERIMENTS = BASE / "experiments"
RESULTS = BASE / "results"

UKDALE_EXP = EXPERIMENTS / "EXP_UKDALE_Selected_Pipeline"
AMPDS2_EXP = EXPERIMENTS / "EXP_AMPDS2_Selected_Pipeline"
CROSS_MODEL_EXP = EXPERIMENTS / "EXP_UKDALE_AMPDS2_Cross_Model_Comparison"

print("BASE:", BASE)
print("BASE exists:", BASE.exists())
print("PROCESSED exists:", PROCESSED.exists())
print("UKDALE_EXP exists:", UKDALE_EXP.exists())
print("AMPDS2_EXP exists:", AMPDS2_EXP.exists())
print("CROSS_MODEL_EXP exists:", CROSS_MODEL_EXP.exists())

In [ ]:
patterns = [
    "*nab*60min*stride30*.csv",
    "*nab*60min*stride15*.csv",
    "*ukdale*selected23*valid_windows*.csv",
    "*ampds2*selected23*valid_windows*.csv",
]

for pattern in patterns:
    print(f"\n--- {pattern} ---")
    matches = list(BASE.rglob(pattern))
    for p in matches[:30]:
        print(p)
    print("Matches:", len(matches))

In [ ]:
keywords = [
    "if_scored",
    "evaluation_sample",
    "reference_sample",
    "cross_model",
    "predictions",
]

for keyword in keywords:
    print(f"\n--- keyword: {keyword} ---")
    matches = [
        p for p in BASE.rglob("*")
        if p.is_file() and keyword.lower() in p.name.lower()
    ]
    for p in matches[:50]:
        print(p)
    print("Matches:", len(matches))

In [ ]:
CORRECTION = EXPERIMENTS / "EXP_SPLIT_BOUNDARY_PURGE_CORRECTION"

CORRECTION.mkdir(parents=True, exist_ok=True)

print("Correction folder:", CORRECTION)
print("Exists:", CORRECTION.exists())

In [ ]:
# CELL 6 — Exact NAB 60/30 Split-Boundary Audit

import pandas as pd
import numpy as np
from pathlib import Path

NAB_6030_FILE = (
    BASE
    / "processed_data"
    / "PH4"
    / "EXP_NAB_60min_stride30"
    / "nab_windows_60min_stride30.csv"
)

print("Loading:", NAB_6030_FILE)
print("Exists:", NAB_6030_FILE.exists())

# Load only the columns needed for the leakage audit
nab6030 = pd.read_csv(
    NAB_6030_FILE,
    usecols=["series", "window_start", "window_end", "label"]
)

nab6030["window_start"] = pd.to_datetime(
    nab6030["window_start"],
    utc=True,
    errors="coerce"
)

nab6030["window_end"] = pd.to_datetime(
    nab6030["window_end"],
    utc=True,
    errors="coerce"
)

assert nab6030["window_start"].notna().all(), "Invalid window_start timestamps found."
assert nab6030["window_end"].notna().all(), "Invalid window_end timestamps found."

nab6030 = (
    nab6030
    .sort_values(["series", "window_start", "window_end"])
    .reset_index(drop=True)
)

print("\nLoaded windows:", len(nab6030))
print("Unique series:", nab6030["series"].nunique())
print("Positive windows:", int(nab6030["label"].sum()))

EXPECTED = {
    "train": 62096,
    "validation": 20705,
    "test": 20725,
    "total": 103526,
}

print("\nExpected historical split counts:")
print(EXPECTED)


# ------------------------------------------------------------
# Identify which exact integer split rule reproduces the
# historical saved split totals.
# ------------------------------------------------------------

def assign_candidate_splits(df, method):
    parts = []

    for series, g in df.groupby("series", sort=False):
        g = g.sort_values("window_start").copy()
        n = len(g)

        if method == "separate_floor":
            n_train = int(n * 0.60)
            n_val = int(n * 0.20)

            split = (
                ["train"] * n_train
                + ["validation"] * n_val
                + ["test"] * (n - n_train - n_val)
            )

        elif method == "cumulative_floor":
            train_end = int(n * 0.60)
            val_end = int(n * 0.80)

            split = (
                ["train"] * train_end
                + ["validation"] * (val_end - train_end)
                + ["test"] * (n - val_end)
            )

        else:
            raise ValueError(method)

        g["split_candidate"] = split
        parts.append(g)

    return pd.concat(parts, ignore_index=True)


candidate_results = {}

for method in ["separate_floor", "cumulative_floor"]:
    tmp = assign_candidate_splits(nab6030, method)

    counts = (
        tmp["split_candidate"]
        .value_counts()
        .reindex(["train", "validation", "test"])
        .fillna(0)
        .astype(int)
        .to_dict()
    )

    candidate_results[method] = counts

    print(f"\nCandidate method: {method}")
    print(counts)


matching_methods = []

for method, counts in candidate_results.items():
    if (
        counts["train"] == EXPECTED["train"]
        and counts["validation"] == EXPECTED["validation"]
        and counts["test"] == EXPECTED["test"]
    ):
        matching_methods.append(method)

print("\nExact historical split-rule matches:", matching_methods)

if len(matching_methods) != 1:
    raise RuntimeError(
        "Could not uniquely reproduce the historical split counts. "
        "STOP here and send me this output before continuing."
    )

SPLIT_METHOD = matching_methods[0]
print("\nConfirmed split rule:", SPLIT_METHOD)


# ------------------------------------------------------------
# Reconstruct the historical split
# ------------------------------------------------------------

nab6030_split = assign_candidate_splits(
    nab6030,
    SPLIT_METHOD
).rename(columns={"split_candidate": "split"})


# ------------------------------------------------------------
# Timestamp-level boundary audit
# ------------------------------------------------------------

audit_rows = []
purge_indices = []

for series, g in nab6030_split.groupby("series", sort=False):

    g = g.sort_values("window_start")

    train = g[g["split"] == "train"]
    val = g[g["split"] == "validation"]
    test = g[g["split"] == "test"]

    if len(train) == 0 or len(val) == 0 or len(test) == 0:
        audit_rows.append({
            "series": series,
            "train_windows": len(train),
            "validation_windows": len(val),
            "test_windows": len(test),
            "train_validation_conflicts": np.nan,
            "validation_test_conflicts": np.nan,
            "status": "missing_split"
        })
        continue

    # Treat windows as [start, end):
    # start == previous end is not overlapping.
    train_max_end = train["window_end"].max()

    val_conflict = val[
        val["window_start"] < train_max_end
    ].copy()

    # Purge validation conflicts conceptually for the audit
    val_retained = val[
        val["window_start"] >= train_max_end
    ].copy()

    if len(val_retained) == 0:
        raise RuntimeError(
            f"No validation rows remain after purge for {series}"
        )

    val_max_end = val_retained["window_end"].max()

    test_conflict = test[
        test["window_start"] < val_max_end
    ].copy()

    purge_indices.extend(val_conflict.index.tolist())
    purge_indices.extend(test_conflict.index.tolist())

    audit_rows.append({
        "series": series,
        "train_windows": len(train),
        "validation_windows": len(val),
        "test_windows": len(test),
        "train_max_end": train_max_end,
        "first_validation_start": val["window_start"].min(),
        "train_validation_conflicts": len(val_conflict),
        "retained_validation_max_end": val_max_end,
        "first_test_start": test["window_start"].min(),
        "validation_test_conflicts": len(test_conflict),
        "total_conflicts": len(val_conflict) + len(test_conflict),
        "status": "ok"
    })


nab6030_boundary_audit = pd.DataFrame(audit_rows)

purge_indices = sorted(set(purge_indices))

print("\n==============================")
print("NAB 60/30 AUDIT RESULT")
print("==============================")

print(
    "\nSeries with train-validation overlap:",
    int((nab6030_boundary_audit["train_validation_conflicts"] > 0).sum())
)

print(
    "Train-validation windows to purge:",
    int(nab6030_boundary_audit["train_validation_conflicts"].sum())
)

print(
    "\nSeries with validation-test overlap:",
    int((nab6030_boundary_audit["validation_test_conflicts"] > 0).sum())
)

print(
    "Validation-test windows to purge:",
    int(nab6030_boundary_audit["validation_test_conflicts"].sum())
)

print("\nTotal unique windows to purge:", len(purge_indices))

print(
    "Affected percentage:",
    f"{100 * len(purge_indices) / len(nab6030_split):.6f}%"
)


# ------------------------------------------------------------
# Calculate corrected counts WITHOUT modifying the source file
# ------------------------------------------------------------

purged_preview = nab6030_split.drop(index=purge_indices)

corrected_counts = (
    purged_preview["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .fillna(0)
    .astype(int)
)

print("\nHistorical split counts:")
print(
    nab6030_split["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nCorrected split counts:")
print(corrected_counts)

print("\nCorrected total:", len(purged_preview))


# ------------------------------------------------------------
# Show every affected series
# ------------------------------------------------------------

print("\nBoundary audit by affected series:")

affected = nab6030_boundary_audit[
    nab6030_boundary_audit["total_conflicts"] > 0
].copy()

print(
    affected[
        [
            "series",
            "train_validation_conflicts",
            "validation_test_conflicts",
            "total_conflicts"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Save ONLY the audit report — no data are changed
# ------------------------------------------------------------

AUDIT_OUT = CORRECTION / "nab_60_30_boundary_audit.csv"

nab6030_boundary_audit.to_csv(
    AUDIT_OUT,
    index=False
)

print("\nAudit report saved to:")
print(AUDIT_OUT)

print("\nIMPORTANT: Original NAB files were NOT modified.")

In [ ]:
# CELL 7 — Exact NAB 60/15 Split-Boundary Audit

import pandas as pd
import numpy as np

NAB_6015_FILE = (
    BASE
    / "processed_data"
    / "PH4"
    / "EXP_NAB_60min_stride15"
    / "nab_windows_60min_stride15.csv"
)

print("Loading:", NAB_6015_FILE)
print("Exists:", NAB_6015_FILE.exists())

# Load only columns required for the leakage audit
nab6015 = pd.read_csv(
    NAB_6015_FILE,
    usecols=["series", "window_start", "window_end", "label"]
)

nab6015["window_start"] = pd.to_datetime(
    nab6015["window_start"],
    utc=True,
    errors="coerce"
)

nab6015["window_end"] = pd.to_datetime(
    nab6015["window_end"],
    utc=True,
    errors="coerce"
)

assert nab6015["window_start"].notna().all(), "Invalid window_start timestamps found."
assert nab6015["window_end"].notna().all(), "Invalid window_end timestamps found."

nab6015 = (
    nab6015
    .sort_values(["series", "window_start", "window_end"])
    .reset_index(drop=True)
)

print("\nLoaded windows:", len(nab6015))
print("Unique series:", nab6015["series"].nunique())
print("Positive windows:", int(nab6015["label"].sum()))

EXPECTED_6015 = {
    "train": 124204,
    "validation": 41405,
    "test": 41424,
    "total": 207033,
}

print("\nExpected historical split counts:")
print(EXPECTED_6015)


# ------------------------------------------------------------
# Reconstruct the historical split rule
# ------------------------------------------------------------

def assign_candidate_splits_6015(df, method):
    parts = []

    for series, g in df.groupby("series", sort=False):
        g = g.sort_values("window_start").copy()
        n = len(g)

        if method == "separate_floor":
            n_train = int(n * 0.60)
            n_val = int(n * 0.20)

            split = (
                ["train"] * n_train
                + ["validation"] * n_val
                + ["test"] * (n - n_train - n_val)
            )

        elif method == "cumulative_floor":
            train_end = int(n * 0.60)
            val_end = int(n * 0.80)

            split = (
                ["train"] * train_end
                + ["validation"] * (val_end - train_end)
                + ["test"] * (n - val_end)
            )

        else:
            raise ValueError(method)

        g["split_candidate"] = split
        parts.append(g)

    return pd.concat(parts, ignore_index=True)


candidate_results_6015 = {}

for method in ["separate_floor", "cumulative_floor"]:
    tmp = assign_candidate_splits_6015(nab6015, method)

    counts = (
        tmp["split_candidate"]
        .value_counts()
        .reindex(["train", "validation", "test"])
        .fillna(0)
        .astype(int)
        .to_dict()
    )

    candidate_results_6015[method] = counts

    print(f"\nCandidate method: {method}")
    print(counts)


matching_methods_6015 = []

for method, counts in candidate_results_6015.items():
    if (
        counts["train"] == EXPECTED_6015["train"]
        and counts["validation"] == EXPECTED_6015["validation"]
        and counts["test"] == EXPECTED_6015["test"]
    ):
        matching_methods_6015.append(method)

print("\nExact historical split-rule matches:", matching_methods_6015)

if len(matching_methods_6015) != 1:
    raise RuntimeError(
        "Could not uniquely reproduce the historical NAB 60/15 split counts. "
        "STOP here and send me this output before continuing."
    )

SPLIT_METHOD_6015 = matching_methods_6015[0]
print("\nConfirmed split rule:", SPLIT_METHOD_6015)


# ------------------------------------------------------------
# Reconstruct historical split
# ------------------------------------------------------------

nab6015_split = assign_candidate_splits_6015(
    nab6015,
    SPLIT_METHOD_6015
).rename(columns={"split_candidate": "split"})


# ------------------------------------------------------------
# Timestamp-level boundary audit
#
# With a 60-minute window and 15-minute stride, up to
# three later-partition windows may overlap the preceding
# partition boundary.
# ------------------------------------------------------------

audit_rows_6015 = []
purge_indices_6015 = []

for series, g in nab6015_split.groupby("series", sort=False):

    g = g.sort_values("window_start")

    train = g[g["split"] == "train"]
    val = g[g["split"] == "validation"]
    test = g[g["split"] == "test"]

    if len(train) == 0 or len(val) == 0 or len(test) == 0:
        audit_rows_6015.append({
            "series": series,
            "train_windows": len(train),
            "validation_windows": len(val),
            "test_windows": len(test),
            "train_validation_conflicts": np.nan,
            "validation_test_conflicts": np.nan,
            "status": "missing_split"
        })
        continue

    # Treat windows as [start, end)
    train_max_end = train["window_end"].max()

    val_conflict = val[
        val["window_start"] < train_max_end
    ].copy()

    val_retained = val[
        val["window_start"] >= train_max_end
    ].copy()

    if len(val_retained) == 0:
        raise RuntimeError(
            f"No validation rows remain after purge for {series}"
        )

    val_max_end = val_retained["window_end"].max()

    test_conflict = test[
        test["window_start"] < val_max_end
    ].copy()

    purge_indices_6015.extend(val_conflict.index.tolist())
    purge_indices_6015.extend(test_conflict.index.tolist())

    audit_rows_6015.append({
        "series": series,
        "train_windows": len(train),
        "validation_windows": len(val),
        "test_windows": len(test),
        "train_max_end": train_max_end,
        "first_validation_start": val["window_start"].min(),
        "train_validation_conflicts": len(val_conflict),
        "retained_validation_max_end": val_max_end,
        "first_test_start": test["window_start"].min(),
        "validation_test_conflicts": len(test_conflict),
        "total_conflicts": len(val_conflict) + len(test_conflict),
        "status": "ok"
    })


nab6015_boundary_audit = pd.DataFrame(audit_rows_6015)

purge_indices_6015 = sorted(set(purge_indices_6015))

print("\n==============================")
print("NAB 60/15 AUDIT RESULT")
print("==============================")

print(
    "\nSeries with train-validation overlap:",
    int((nab6015_boundary_audit["train_validation_conflicts"] > 0).sum())
)

print(
    "Train-validation windows to purge:",
    int(nab6015_boundary_audit["train_validation_conflicts"].sum())
)

print(
    "\nSeries with validation-test overlap:",
    int((nab6015_boundary_audit["validation_test_conflicts"] > 0).sum())
)

print(
    "Validation-test windows to purge:",
    int(nab6015_boundary_audit["validation_test_conflicts"].sum())
)

print("\nTotal unique windows to purge:", len(purge_indices_6015))

print(
    "Affected percentage:",
    f"{100 * len(purge_indices_6015) / len(nab6015_split):.6f}%"
)


# ------------------------------------------------------------
# Calculate corrected counts WITHOUT modifying source data
# ------------------------------------------------------------

purged_preview_6015 = nab6015_split.drop(
    index=purge_indices_6015
)

corrected_counts_6015 = (
    purged_preview_6015["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .fillna(0)
    .astype(int)
)

print("\nHistorical split counts:")
print(
    nab6015_split["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nCorrected split counts:")
print(corrected_counts_6015)

print("\nCorrected total:", len(purged_preview_6015))


# ------------------------------------------------------------
# Distribution of conflict counts
# ------------------------------------------------------------

print("\nTrain-validation conflicts per series:")
print(
    nab6015_boundary_audit["train_validation_conflicts"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nValidation-test conflicts per series:")
print(
    nab6015_boundary_audit["validation_test_conflicts"]
    .value_counts(dropna=False)
    .sort_index()
)


# ------------------------------------------------------------
# Show exceptional series
# ------------------------------------------------------------

expected_conflicts_per_boundary = 3

exceptions = nab6015_boundary_audit[
    (nab6015_boundary_audit["train_validation_conflicts"] != expected_conflicts_per_boundary)
    |
    (nab6015_boundary_audit["validation_test_conflicts"] != expected_conflicts_per_boundary)
].copy()

print("\nSeries that do NOT have exactly 3 conflicts at both boundaries:")

if len(exceptions) == 0:
    print("None")
else:
    print(
        exceptions[
            [
                "series",
                "train_validation_conflicts",
                "validation_test_conflicts",
                "total_conflicts"
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# Save audit only
# ------------------------------------------------------------

AUDIT_OUT_6015 = CORRECTION / "nab_60_15_boundary_audit.csv"

nab6015_boundary_audit.to_csv(
    AUDIT_OUT_6015,
    index=False
)

print("\nAudit report saved to:")
print(AUDIT_OUT_6015)

print("\nIMPORTANT: Original NAB 60/15 files were NOT modified.")

In [ ]:
# CELL 8 — Exact UK-DALE Final Selected-Pipeline Split-Boundary Audit

import pandas as pd
import numpy as np

UKDALE_SCORED_FILE = (
    BASE
    / "processed_data"
    / "UKDALE_Selected_Pipeline"
    / "ukdale_clean_main_if_scored_no_injection.csv"
)

print("Loading:", UKDALE_SCORED_FILE)
print("Exists:", UKDALE_SCORED_FILE.exists())


# ------------------------------------------------------------
# Inspect available columns first
# ------------------------------------------------------------

uk_header = pd.read_csv(
    UKDALE_SCORED_FILE,
    nrows=0
).columns.tolist()

print("\nAvailable columns:")
print(uk_header)


required = [
    "window_start",
    "window_end",
    "split",
]

for col in required:
    assert col in uk_header, f"Required column missing: {col}"


# Prefer the preserved final series_key when available.
# Otherwise reconstruct the appliance-meter series identifier.
if "series_key" in uk_header:
    series_col = "series_key"
    usecols = [
        "series_key",
        "window_start",
        "window_end",
        "split",
    ]

elif all(c in uk_header for c in ["building", "meter", "primary_category"]):
    series_col = None
    usecols = [
        "building",
        "meter",
        "primary_category",
        "window_start",
        "window_end",
        "split",
    ]

else:
    raise RuntimeError(
        "Could not identify a reliable UK-DALE series identifier. "
        "STOP and send me the printed column list."
    )


# ------------------------------------------------------------
# Load only columns required for boundary auditing
# ------------------------------------------------------------

uk = pd.read_csv(
    UKDALE_SCORED_FILE,
    usecols=usecols
)

if series_col is None:
    uk["series_key"] = (
        "building"
        + uk["building"].astype(str)
        + "_meter"
        + uk["meter"].astype(str)
        + "_"
        + uk["primary_category"].astype(str)
    )
    series_col = "series_key"


uk["window_start"] = pd.to_datetime(
    uk["window_start"],
    utc=True,
    errors="coerce"
)

uk["window_end"] = pd.to_datetime(
    uk["window_end"],
    utc=True,
    errors="coerce"
)

assert uk["window_start"].notna().all(), \
    "Invalid UK-DALE window_start timestamps found."

assert uk["window_end"].notna().all(), \
    "Invalid UK-DALE window_end timestamps found."


uk = (
    uk
    .sort_values([series_col, "window_start", "window_end"])
    .reset_index(drop=True)
)


print("\n==============================")
print("UK-DALE HISTORICAL DATA CHECK")
print("==============================")

print("\nLoaded windows:", len(uk))
print("Unique appliance-meter series:", uk[series_col].nunique())

print("\nHistorical split counts:")
historical_counts_uk = (
    uk["split"]
    .value_counts()
    .reindex(["train", "val", "validation", "test"])
    .dropna()
)
print(historical_counts_uk)

print("\nUnique split labels:")
print(sorted(uk["split"].dropna().unique().tolist()))


# ------------------------------------------------------------
# Normalize validation label name
# ------------------------------------------------------------

uk["split_norm"] = uk["split"].replace({
    "val": "validation"
})

valid_split_names = {"train", "validation", "test"}

unexpected = set(
    uk["split_norm"].dropna().unique()
) - valid_split_names

if unexpected:
    raise RuntimeError(
        f"Unexpected split labels found: {unexpected}"
    )


print("\nNormalized split counts:")
print(
    uk["split_norm"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)


# ------------------------------------------------------------
# Exact timestamp-level split-boundary audit
#
# UK-DALE final windows are minute-level windows whose
# window_end is the timestamp of the final included observation
# (e.g. 22:28 through 23:27).
#
# Therefore:
#     later_start <= previous_max_end
# means at least one raw timestamp is shared.
# ------------------------------------------------------------

uk_audit_rows = []
uk_purge_indices = []
uk_conflict_rows = []

for series, g in uk.groupby(series_col, sort=False):

    g = g.sort_values("window_start")

    train = g[g["split_norm"] == "train"]
    val = g[g["split_norm"] == "validation"]
    test = g[g["split_norm"] == "test"]

    if len(train) == 0 or len(val) == 0 or len(test) == 0:
        uk_audit_rows.append({
            "series_key": series,
            "train_windows": len(train),
            "validation_windows": len(val),
            "test_windows": len(test),
            "train_validation_conflicts": np.nan,
            "validation_test_conflicts": np.nan,
            "status": "missing_split"
        })
        continue


    # --------------------------------------------------------
    # TRAIN -> VALIDATION
    # --------------------------------------------------------

    train_max_end = train["window_end"].max()

    val_conflict = val[
        val["window_start"] <= train_max_end
    ].copy()

    val_retained = val[
        val["window_start"] > train_max_end
    ].copy()

    if len(val_retained) == 0:
        raise RuntimeError(
            f"No validation rows remain after boundary purge for {series}"
        )


    # --------------------------------------------------------
    # VALIDATION -> TEST
    #
    # Use retained validation rows when defining the next
    # boundary, because those are the rows that would remain
    # in the corrected experiment.
    # --------------------------------------------------------

    val_max_end_retained = val_retained["window_end"].max()

    test_conflict = test[
        test["window_start"] <= val_max_end_retained
    ].copy()

    test_retained = test[
        test["window_start"] > val_max_end_retained
    ].copy()


    uk_purge_indices.extend(
        val_conflict.index.tolist()
    )

    uk_purge_indices.extend(
        test_conflict.index.tolist()
    )


    # Preserve exact conflicting rows for later verification
    for idx, row in val_conflict.iterrows():
        uk_conflict_rows.append({
            "series_key": series,
            "boundary": "train_validation",
            "row_index": idx,
            "window_start": row["window_start"],
            "window_end": row["window_end"]
        })

    for idx, row in test_conflict.iterrows():
        uk_conflict_rows.append({
            "series_key": series,
            "boundary": "validation_test",
            "row_index": idx,
            "window_start": row["window_start"],
            "window_end": row["window_end"]
        })


    uk_audit_rows.append({
        "series_key": series,

        "train_windows": len(train),
        "validation_windows": len(val),
        "test_windows": len(test),

        "last_train_start":
            train["window_start"].max(),

        "train_max_end":
            train_max_end,

        "first_validation_start":
            val["window_start"].min(),

        "train_validation_conflicts":
            len(val_conflict),

        "first_retained_validation_start":
            val_retained["window_start"].min(),

        "retained_validation_max_end":
            val_max_end_retained,

        "first_test_start":
            test["window_start"].min(),

        "validation_test_conflicts":
            len(test_conflict),

        "first_retained_test_start":
            (
                test_retained["window_start"].min()
                if len(test_retained) > 0
                else pd.NaT
            ),

        "total_conflicts":
            len(val_conflict) + len(test_conflict),

        "status":
            "ok"
    })


uk_boundary_audit = pd.DataFrame(
    uk_audit_rows
)

uk_conflicts = pd.DataFrame(
    uk_conflict_rows
)

uk_purge_indices = sorted(
    set(uk_purge_indices)
)


# ------------------------------------------------------------
# Final exact results
# ------------------------------------------------------------

print("\n==============================")
print("UK-DALE FINAL AUDIT RESULT")
print("==============================")

tv_series = int(
    (
        uk_boundary_audit[
            "train_validation_conflicts"
        ] > 0
    ).sum()
)

vt_series = int(
    (
        uk_boundary_audit[
            "validation_test_conflicts"
        ] > 0
    ).sum()
)

tv_windows = int(
    uk_boundary_audit[
        "train_validation_conflicts"
    ].sum()
)

vt_windows = int(
    uk_boundary_audit[
        "validation_test_conflicts"
    ].sum()
)

print(
    "\nSeries with train-validation overlap:",
    tv_series
)

print(
    "Train-validation windows to purge:",
    tv_windows
)

print(
    "\nSeries with validation-test overlap:",
    vt_series
)

print(
    "Validation-test windows to purge:",
    vt_windows
)

print(
    "\nTotal unique windows to purge:",
    len(uk_purge_indices)
)

print(
    "Affected percentage:",
    f"{100 * len(uk_purge_indices) / len(uk):.6f}%"
)


# ------------------------------------------------------------
# Corrected counts preview
# ------------------------------------------------------------

uk_purged_preview = uk.drop(
    index=uk_purge_indices
)

corrected_counts_uk = (
    uk_purged_preview["split_norm"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .fillna(0)
    .astype(int)
)

print("\nHistorical normalized counts:")
print(
    uk["split_norm"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nCorrected split counts:")
print(corrected_counts_uk)

print(
    "\nCorrected total:",
    len(uk_purged_preview)
)


# ------------------------------------------------------------
# Show all affected UK-DALE series
# ------------------------------------------------------------

affected_uk = uk_boundary_audit[
    uk_boundary_audit["total_conflicts"] > 0
].copy()

print("\nAffected UK-DALE appliance-meter series:")

if len(affected_uk) == 0:
    print("None")
else:
    print(
        affected_uk[
            [
                "series_key",
                "train_validation_conflicts",
                "validation_test_conflicts",
                "total_conflicts"
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# Show series with no boundary conflict
# ------------------------------------------------------------

unaffected_uk = uk_boundary_audit[
    uk_boundary_audit["total_conflicts"] == 0
].copy()

print("\nSeries with no boundary conflicts:")

if len(unaffected_uk) == 0:
    print("None")
else:
    print(
        unaffected_uk[
            ["series_key"]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# Save read-only audit artifacts
# ------------------------------------------------------------

UK_AUDIT_OUT = (
    CORRECTION
    / "ukdale_final_boundary_audit.csv"
)

UK_CONFLICT_OUT = (
    CORRECTION
    / "ukdale_final_boundary_conflicting_rows.csv"
)

uk_boundary_audit.to_csv(
    UK_AUDIT_OUT,
    index=False
)

uk_conflicts.to_csv(
    UK_CONFLICT_OUT,
    index=False
)

print("\nAudit report saved to:")
print(UK_AUDIT_OUT)

print("\nExact conflicting rows saved to:")
print(UK_CONFLICT_OUT)

print(
    "\nIMPORTANT: "
    "Original UK-DALE files were NOT modified."
)

In [ ]:
# CELL 9 — Exact AMPds2 Final Selected-Pipeline Split-Boundary Audit

import pandas as pd
import numpy as np

AMPDS2_SCORED_FILE = (
    BASE
    / "processed_data"
    / "AMPDS2_Selected_Pipeline"
    / "ampds2_p_if_scored_no_injection.csv"
)

print("Loading:", AMPDS2_SCORED_FILE)
print("Exists:", AMPDS2_SCORED_FILE.exists())


# ------------------------------------------------------------
# Inspect available columns
# ------------------------------------------------------------

amp_header = pd.read_csv(
    AMPDS2_SCORED_FILE,
    nrows=0
).columns.tolist()

print("\nAvailable columns:")
print(amp_header)


required = [
    "window_start",
    "window_end",
    "split",
]

for col in required:
    assert col in amp_header, f"Required column missing: {col}"


# Prefer series_key if it exists.
# Otherwise use channel_code, which uniquely identifies
# the evaluated AMPds2 electricity channel.
if "series_key" in amp_header:
    series_col_amp = "series_key"

    usecols_amp = [
        "series_key",
        "window_start",
        "window_end",
        "split",
    ]

elif "channel_code" in amp_header:
    series_col_amp = "channel_code"

    usecols_amp = [
        "channel_code",
        "window_start",
        "window_end",
        "split",
    ]

else:
    raise RuntimeError(
        "Could not identify a reliable AMPds2 series/channel identifier. "
        "STOP and send me the printed column list."
    )


# ------------------------------------------------------------
# Load only the columns needed for the boundary audit
# ------------------------------------------------------------

amp = pd.read_csv(
    AMPDS2_SCORED_FILE,
    usecols=usecols_amp
)

amp["window_start"] = pd.to_datetime(
    amp["window_start"],
    utc=True,
    errors="coerce"
)

amp["window_end"] = pd.to_datetime(
    amp["window_end"],
    utc=True,
    errors="coerce"
)

assert amp["window_start"].notna().all(), \
    "Invalid AMPds2 window_start timestamps found."

assert amp["window_end"].notna().all(), \
    "Invalid AMPds2 window_end timestamps found."


amp = (
    amp
    .sort_values([series_col_amp, "window_start", "window_end"])
    .reset_index(drop=True)
)


print("\n==============================")
print("AMPds2 HISTORICAL DATA CHECK")
print("==============================")

print("\nLoaded windows:", len(amp))
print("Unique channels:", amp[series_col_amp].nunique())

print("\nHistorical split counts:")
print(
    amp["split"]
    .value_counts()
)

print("\nUnique split labels:")
print(
    sorted(
        amp["split"]
        .dropna()
        .unique()
        .tolist()
    )
)


# ------------------------------------------------------------
# Normalize validation split naming
# ------------------------------------------------------------

amp["split_norm"] = amp["split"].replace({
    "val": "validation"
})

valid_split_names_amp = {
    "train",
    "validation",
    "test"
}

unexpected_amp = set(
    amp["split_norm"]
    .dropna()
    .unique()
) - valid_split_names_amp

if unexpected_amp:
    raise RuntimeError(
        f"Unexpected split labels found: {unexpected_amp}"
    )


print("\nNormalized split counts:")
print(
    amp["split_norm"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)


# ------------------------------------------------------------
# Exact timestamp-level boundary audit
#
# AMPds2 minute-level windows include the final recorded
# minute in window_end.
#
# Therefore:
#     later_start <= preceding_max_end
# indicates raw-observation overlap.
# ------------------------------------------------------------

amp_audit_rows = []
amp_purge_indices = []
amp_conflict_rows = []

for series, g in amp.groupby(series_col_amp, sort=False):

    g = g.sort_values("window_start")

    train = g[
        g["split_norm"] == "train"
    ]

    val = g[
        g["split_norm"] == "validation"
    ]

    test = g[
        g["split_norm"] == "test"
    ]


    if (
        len(train) == 0
        or len(val) == 0
        or len(test) == 0
    ):
        amp_audit_rows.append({
            "series_key": series,
            "train_windows": len(train),
            "validation_windows": len(val),
            "test_windows": len(test),
            "train_validation_conflicts": np.nan,
            "validation_test_conflicts": np.nan,
            "status": "missing_split"
        })
        continue


    # --------------------------------------------------------
    # TRAIN -> VALIDATION
    # --------------------------------------------------------

    train_max_end = train[
        "window_end"
    ].max()

    val_conflict = val[
        val["window_start"] <= train_max_end
    ].copy()

    val_retained = val[
        val["window_start"] > train_max_end
    ].copy()

    if len(val_retained) == 0:
        raise RuntimeError(
            f"No validation rows remain after purge for {series}"
        )


    # --------------------------------------------------------
    # VALIDATION -> TEST
    # --------------------------------------------------------

    val_max_end_retained = val_retained[
        "window_end"
    ].max()

    test_conflict = test[
        test["window_start"] <= val_max_end_retained
    ].copy()

    test_retained = test[
        test["window_start"] > val_max_end_retained
    ].copy()


    amp_purge_indices.extend(
        val_conflict.index.tolist()
    )

    amp_purge_indices.extend(
        test_conflict.index.tolist()
    )


    # Save exact conflicting rows for later correction
    for idx, row in val_conflict.iterrows():
        amp_conflict_rows.append({
            "series_key": series,
            "boundary": "train_validation",
            "row_index": idx,
            "window_start": row["window_start"],
            "window_end": row["window_end"]
        })

    for idx, row in test_conflict.iterrows():
        amp_conflict_rows.append({
            "series_key": series,
            "boundary": "validation_test",
            "row_index": idx,
            "window_start": row["window_start"],
            "window_end": row["window_end"]
        })


    amp_audit_rows.append({
        "series_key": series,

        "train_windows": len(train),
        "validation_windows": len(val),
        "test_windows": len(test),

        "last_train_start":
            train["window_start"].max(),

        "train_max_end":
            train_max_end,

        "first_validation_start":
            val["window_start"].min(),

        "train_validation_conflicts":
            len(val_conflict),

        "first_retained_validation_start":
            val_retained["window_start"].min(),

        "retained_validation_max_end":
            val_max_end_retained,

        "first_test_start":
            test["window_start"].min(),

        "validation_test_conflicts":
            len(test_conflict),

        "first_retained_test_start":
            (
                test_retained["window_start"].min()
                if len(test_retained) > 0
                else pd.NaT
            ),

        "total_conflicts":
            len(val_conflict)
            + len(test_conflict),

        "status":
            "ok"
    })


amp_boundary_audit = pd.DataFrame(
    amp_audit_rows
)

amp_conflicts = pd.DataFrame(
    amp_conflict_rows
)

amp_purge_indices = sorted(
    set(amp_purge_indices)
)


# ------------------------------------------------------------
# Exact results
# ------------------------------------------------------------

print("\n==============================")
print("AMPds2 FINAL AUDIT RESULT")
print("==============================")

amp_tv_series = int(
    (
        amp_boundary_audit[
            "train_validation_conflicts"
        ] > 0
    ).sum()
)

amp_vt_series = int(
    (
        amp_boundary_audit[
            "validation_test_conflicts"
        ] > 0
    ).sum()
)

amp_tv_windows = int(
    amp_boundary_audit[
        "train_validation_conflicts"
    ].sum()
)

amp_vt_windows = int(
    amp_boundary_audit[
        "validation_test_conflicts"
    ].sum()
)

print(
    "\nChannels with train-validation overlap:",
    amp_tv_series
)

print(
    "Train-validation windows to purge:",
    amp_tv_windows
)

print(
    "\nChannels with validation-test overlap:",
    amp_vt_series
)

print(
    "Validation-test windows to purge:",
    amp_vt_windows
)

print(
    "\nTotal unique windows to purge:",
    len(amp_purge_indices)
)

print(
    "Affected percentage:",
    f"{100 * len(amp_purge_indices) / len(amp):.6f}%"
)


# ------------------------------------------------------------
# Corrected counts preview
# ------------------------------------------------------------

amp_purged_preview = amp.drop(
    index=amp_purge_indices
)

corrected_counts_amp = (
    amp_purged_preview["split_norm"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .fillna(0)
    .astype(int)
)

print("\nHistorical normalized counts:")
print(
    amp["split_norm"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print("\nCorrected split counts:")
print(corrected_counts_amp)

print(
    "\nCorrected total:",
    len(amp_purged_preview)
)


# ------------------------------------------------------------
# Show all affected channels
# ------------------------------------------------------------

affected_amp = amp_boundary_audit[
    amp_boundary_audit["total_conflicts"] > 0
].copy()

print("\nAffected AMPds2 channels:")

print(
    affected_amp[
        [
            "series_key",
            "train_validation_conflicts",
            "validation_test_conflicts",
            "total_conflicts"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# Sanity check against the expected 21 + 21 = 42 result
# ------------------------------------------------------------

print("\nExpected from prior audit:")
print("21 train-validation + 21 validation-test = 42")

print("\nObserved:")
print(
    f"{amp_tv_windows} train-validation "
    f"+ {amp_vt_windows} validation-test "
    f"= {len(amp_purge_indices)}"
)

if (
    amp_tv_windows == 21
    and amp_vt_windows == 21
    and len(amp_purge_indices) == 42
):
    print("\nPASS: prior AMPds2 audit confirmed exactly.")
else:
    print(
        "\nNOTE: observed result differs from the prior audit. "
        "Use the exact full-table result printed above."
    )


# ------------------------------------------------------------
# Save read-only audit artifacts
# ------------------------------------------------------------

AMP_AUDIT_OUT = (
    CORRECTION
    / "ampds2_final_boundary_audit.csv"
)

AMP_CONFLICT_OUT = (
    CORRECTION
    / "ampds2_final_boundary_conflicting_rows.csv"
)

amp_boundary_audit.to_csv(
    AMP_AUDIT_OUT,
    index=False
)

amp_conflicts.to_csv(
    AMP_CONFLICT_OUT,
    index=False
)

print("\nAudit report saved to:")
print(AMP_AUDIT_OUT)

print("\nExact conflicting rows saved to:")
print(AMP_CONFLICT_OUT)

print(
    "\nIMPORTANT: "
    "Original AMPds2 files were NOT modified."
)

In [ ]:
# CELL 10 — Consolidated Full-Dataset Boundary Audit Summary

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Load the four saved audit reports
# ------------------------------------------------------------

AUDIT_FILES = {
    "NAB 60/30": CORRECTION / "nab_60_30_boundary_audit.csv",
    "NAB 60/15": CORRECTION / "nab_60_15_boundary_audit.csv",
    "UK-DALE": CORRECTION / "ukdale_final_boundary_audit.csv",
    "AMPds2": CORRECTION / "ampds2_final_boundary_audit.csv",
}

for name, path in AUDIT_FILES.items():
    print(f"{name}: {path.exists()} -> {path}")


# ------------------------------------------------------------
# Build summary directly from the saved audits
# ------------------------------------------------------------

summary_rows = []

for branch, path in AUDIT_FILES.items():

    df = pd.read_csv(path)

    train_windows = int(df["train_windows"].sum())
    validation_windows = int(df["validation_windows"].sum())
    test_windows = int(df["test_windows"].sum())

    tv_conflicts = int(
        df["train_validation_conflicts"]
        .fillna(0)
        .sum()
    )

    vt_conflicts = int(
        df["validation_test_conflicts"]
        .fillna(0)
        .sum()
    )

    total_removed = tv_conflicts + vt_conflicts

    original_total = (
        train_windows
        + validation_windows
        + test_windows
    )

    corrected_train = train_windows
    corrected_validation = validation_windows - tv_conflicts
    corrected_test = test_windows - vt_conflicts

    corrected_total = (
        corrected_train
        + corrected_validation
        + corrected_test
    )

    affected_percent = (
        100 * total_removed / original_total
    )

    summary_rows.append({
        "branch": branch,

        "original_train": train_windows,
        "original_validation": validation_windows,
        "original_test": test_windows,
        "original_total": original_total,

        "train_validation_removed": tv_conflicts,
        "validation_test_removed": vt_conflicts,
        "total_removed": total_removed,

        "corrected_train": corrected_train,
        "corrected_validation": corrected_validation,
        "corrected_test": corrected_test,
        "corrected_total": corrected_total,

        "affected_percent": affected_percent,
    })


boundary_summary = pd.DataFrame(summary_rows)


# ------------------------------------------------------------
# Assert the exact results already established in Cells 6–9
# ------------------------------------------------------------

expected = {
    "NAB 60/30": {
        "original_total": 103526,
        "total_removed": 114,
        "corrected_total": 103412,
    },

    "NAB 60/15": {
        "original_total": 207033,
        "total_removed": 340,
        "corrected_total": 206693,
    },

    "UK-DALE": {
        "original_total": 363940,
        "total_removed": 31,
        "corrected_total": 363909,
    },

    "AMPds2": {
        "original_total": 735819,
        "total_removed": 42,
        "corrected_total": 735777,
    },
}


for branch, exp in expected.items():

    row = boundary_summary[
        boundary_summary["branch"] == branch
    ].iloc[0]

    assert int(row["original_total"]) == exp["original_total"], \
        f"{branch}: original total mismatch"

    assert int(row["total_removed"]) == exp["total_removed"], \
        f"{branch}: removal count mismatch"

    assert int(row["corrected_total"]) == exp["corrected_total"], \
        f"{branch}: corrected total mismatch"


print("\nPASS: all four audits match the exact verified results.")


# ------------------------------------------------------------
# Display clean summary
# ------------------------------------------------------------

display_cols = [
    "branch",
    "original_train",
    "original_validation",
    "original_test",
    "train_validation_removed",
    "validation_test_removed",
    "total_removed",
    "corrected_train",
    "corrected_validation",
    "corrected_test",
    "corrected_total",
    "affected_percent",
]

print("\n==========================================")
print("CONSOLIDATED SPLIT-BOUNDARY AUDIT SUMMARY")
print("==========================================\n")

print(
    boundary_summary[
        display_cols
    ].to_string(
        index=False,
        formatters={
            "affected_percent": lambda x: f"{x:.6f}%"
        }
    )
)


# ------------------------------------------------------------
# Save permanent audit summary
# ------------------------------------------------------------

SUMMARY_CSV = (
    CORRECTION
    / "full_dataset_boundary_audit_summary.csv"
)

boundary_summary.to_csv(
    SUMMARY_CSV,
    index=False
)


# Also save a human-readable text report
SUMMARY_TXT = (
    CORRECTION
    / "full_dataset_boundary_audit_summary.txt"
)

with open(SUMMARY_TXT, "w", encoding="utf-8") as f:

    f.write(
        "Problem 2 — Full-Dataset Split-Boundary Audit\n"
    )

    f.write(
        "=============================================\n\n"
    )

    f.write(
        boundary_summary[
            display_cols
        ].to_string(
            index=False,
            formatters={
                "affected_percent":
                    lambda x: f"{x:.6f}%"
            }
        )
    )

    f.write(
        "\n\nAll values were obtained from timestamp-level "
        "audits of the saved experimental window assignments.\n"
    )

    f.write(
        "Training partitions remain unchanged. "
        "Only conflicting validation/test windows are candidates "
        "for removal.\n"
    )


print("\nSaved consolidated CSV:")
print(SUMMARY_CSV)

print("\nSaved human-readable summary:")
print(SUMMARY_TXT)

print(
    "\nIMPORTANT: "
    "No original experimental data or predictions were modified."
)

In [ ]:
# CELL 11 — Exact Controlled UK-DALE Reference/Evaluation Raw-Time Overlap Audit

import pandas as pd
import numpy as np

UK_REF_FILE = (
    CROSS_MODEL_EXP
    / "tables"
    / "ukdale_lof_reference_sample.csv"
)

UK_EVAL_FILE = (
    CROSS_MODEL_EXP
    / "tables"
    / "ukdale_cross_model_evaluation_sample.csv"
)

print("Reference file:", UK_REF_FILE)
print("Reference exists:", UK_REF_FILE.exists())

print("\nEvaluation file:", UK_EVAL_FILE)
print("Evaluation exists:", UK_EVAL_FILE.exists())


# ------------------------------------------------------------
# Inspect schemas
# ------------------------------------------------------------

ref_cols = pd.read_csv(
    UK_REF_FILE,
    nrows=0
).columns.tolist()

eval_cols = pd.read_csv(
    UK_EVAL_FILE,
    nrows=0
).columns.tolist()

print("\nReference columns:")
print(ref_cols)

print("\nEvaluation columns:")
print(eval_cols)


# ------------------------------------------------------------
# Required common columns
# ------------------------------------------------------------

required_common = [
    "window_start",
    "window_end",
]

for col in required_common:
    assert col in ref_cols, \
        f"Reference sample missing required column: {col}"

    assert col in eval_cols, \
        f"Evaluation sample missing required column: {col}"


assert "split" in eval_cols, \
    "Evaluation sample does not contain a split column."


# ------------------------------------------------------------
# Determine series identifier
# ------------------------------------------------------------

if (
    "series_key" in ref_cols
    and "series_key" in eval_cols
):
    SERIES_COL = "series_key"

elif all(
    c in ref_cols and c in eval_cols
    for c in [
        "building",
        "meter",
        "primary_category"
    ]
):
    SERIES_COL = None

else:
    raise RuntimeError(
        "Could not establish a common UK-DALE series identifier. "
        "STOP and send me the printed column lists."
    )


# ------------------------------------------------------------
# Load full saved samples
# ------------------------------------------------------------

uk_ref = pd.read_csv(UK_REF_FILE)
uk_eval = pd.read_csv(UK_EVAL_FILE)


if SERIES_COL is None:

    for df in [uk_ref, uk_eval]:

        df["series_key"] = (
            "building"
            + df["building"].astype(str)
            + "_meter"
            + df["meter"].astype(str)
            + "_"
            + df["primary_category"].astype(str)
        )

    SERIES_COL = "series_key"


# ------------------------------------------------------------
# Parse timestamps
# ------------------------------------------------------------

for df_name, df in [
    ("reference", uk_ref),
    ("evaluation", uk_eval)
]:

    df["window_start"] = pd.to_datetime(
        df["window_start"],
        utc=True,
        errors="coerce"
    )

    df["window_end"] = pd.to_datetime(
        df["window_end"],
        utc=True,
        errors="coerce"
    )

    assert df["window_start"].notna().all(), \
        f"{df_name}: invalid window_start found."

    assert df["window_end"].notna().all(), \
        f"{df_name}: invalid window_end found."


# ------------------------------------------------------------
# Normalize evaluation split names
# ------------------------------------------------------------

uk_eval["split_norm"] = uk_eval["split"].replace({
    "val": "validation"
})

unexpected = set(
    uk_eval["split_norm"].dropna().unique()
) - {
    "train",
    "validation",
    "test"
}

if unexpected:
    raise RuntimeError(
        f"Unexpected evaluation split labels: {unexpected}"
    )


print("\n====================================")
print("CONTROLLED UK-DALE SAMPLE CHECK")
print("====================================")

print("\nReference windows:", len(uk_ref))

print("\nEvaluation split counts:")
print(
    uk_eval["split_norm"]
    .value_counts()
    .reindex(["train", "validation", "test"])
)

print(
    "\nUnique reference series:",
    uk_ref[SERIES_COL].nunique()
)

print(
    "Unique evaluation series:",
    uk_eval[SERIES_COL].nunique()
)


# ------------------------------------------------------------
# Preserve row identifiers
# ------------------------------------------------------------

if (
    "source_row_id" in uk_ref.columns
    and "source_row_id" in uk_eval.columns
):

    ref_ids = set(
        uk_ref["source_row_id"]
        .dropna()
        .astype(str)
    )

    eval_ids = set(
        uk_eval["source_row_id"]
        .dropna()
        .astype(str)
    )

    identical_ids = ref_ids.intersection(eval_ids)

    print(
        "\nIdentical source_row_id reuse:",
        len(identical_ids)
    )

    assert len(identical_ids) == 0, \
        "Unexpected identical source_row_id reuse detected."

else:

    print(
        "\nsource_row_id is not available in both files; "
        "row-ID uniqueness cannot be checked here."
    )


# ------------------------------------------------------------
# Add permanent audit IDs
# ------------------------------------------------------------

uk_ref = uk_ref.reset_index(drop=True)
uk_eval = uk_eval.reset_index(drop=True)

uk_ref["audit_reference_id"] = np.arange(
    len(uk_ref),
    dtype=np.int64
)

uk_eval["audit_evaluation_id"] = np.arange(
    len(uk_eval),
    dtype=np.int64
)


# ------------------------------------------------------------
# Efficient interval-overlap helper
#
# Windows are minute-level and window_end represents the final
# included timestamp. Therefore two windows overlap if:
#
#     A.start <= B.end
# and
#     B.start <= A.end
#
# This function checks whether each query interval overlaps
# at least one interval in a preceding/reference pool from
# the SAME series.
# ------------------------------------------------------------

def find_overlapping_query_rows(
    query_df,
    preceding_df,
    series_col,
    query_id_col
):

    conflicting_ids = []
    details = []

    preceding_groups = {
        key: g.sort_values("window_start").copy()
        for key, g in preceding_df.groupby(
            series_col,
            sort=False
        )
    }

    for series, qg in query_df.groupby(
        series_col,
        sort=False
    ):

        if series not in preceding_groups:
            continue

        pg = preceding_groups[series]

        p_start = (
            pg["window_start"]
            .astype("int64")
            .to_numpy()
        )

        p_end = (
            pg["window_end"]
            .astype("int64")
            .to_numpy()
        )

        order = np.argsort(p_start)

        p_start = p_start[order]
        p_end = p_end[order]

        # Prefix maximum end timestamp
        prefix_max_end = np.maximum.accumulate(
            p_end
        )

        for _, row in qg.iterrows():

            q_start = row["window_start"].value
            q_end = row["window_end"].value

            # Find all preceding intervals whose start
            # occurs no later than this query's end.
            pos = np.searchsorted(
                p_start,
                q_end,
                side="right"
            ) - 1

            overlaps = False

            if pos >= 0:
                overlaps = (
                    prefix_max_end[pos]
                    >= q_start
                )

            if overlaps:

                qid = int(
                    row[query_id_col]
                )

                conflicting_ids.append(qid)

                details.append({
                    "audit_evaluation_id": qid,
                    "series_key": series,
                    "split": row["split_norm"],
                    "window_start": row["window_start"],
                    "window_end": row["window_end"],
                })

    return (
        sorted(set(conflicting_ids)),
        pd.DataFrame(details)
    )


# ============================================================
# STEP A — TRAIN-EVALUATION vs REFERENCE
# ============================================================

uk_train_eval = uk_eval[
    uk_eval["split_norm"] == "train"
].copy()

uk_val_eval = uk_eval[
    uk_eval["split_norm"] == "validation"
].copy()

uk_test_eval = uk_eval[
    uk_eval["split_norm"] == "test"
].copy()


train_conflict_ids, train_conflicts = (
    find_overlapping_query_rows(
        query_df=uk_train_eval,
        preceding_df=uk_ref,
        series_col=SERIES_COL,
        query_id_col="audit_evaluation_id"
    )
)


# ============================================================
# STEP B — VALIDATION vs preceding reference + retained train
#
# First remove conflicting training-evaluation rows.
# ============================================================

uk_train_retained = uk_train_eval[
    ~uk_train_eval[
        "audit_evaluation_id"
    ].isin(train_conflict_ids)
].copy()

preceding_for_validation = pd.concat(
    [
        uk_ref,
        uk_train_retained
    ],
    ignore_index=True,
    sort=False
)

val_conflict_ids, val_conflicts = (
    find_overlapping_query_rows(
        query_df=uk_val_eval,
        preceding_df=preceding_for_validation,
        series_col=SERIES_COL,
        query_id_col="audit_evaluation_id"
    )
)


# ============================================================
# STEP C — TEST vs all preceding retained partitions
# ============================================================

uk_val_retained = uk_val_eval[
    ~uk_val_eval[
        "audit_evaluation_id"
    ].isin(val_conflict_ids)
].copy()

preceding_for_test = pd.concat(
    [
        uk_ref,
        uk_train_retained,
        uk_val_retained
    ],
    ignore_index=True,
    sort=False
)

test_conflict_ids, test_conflicts = (
    find_overlapping_query_rows(
        query_df=uk_test_eval,
        preceding_df=preceding_for_test,
        series_col=SERIES_COL,
        query_id_col="audit_evaluation_id"
    )
)


# ------------------------------------------------------------
# Consolidate unique removals
# ------------------------------------------------------------

all_conflict_ids = sorted(
    set(
        train_conflict_ids
        + val_conflict_ids
        + test_conflict_ids
    )
)

uk_controlled_conflicts = pd.concat(
    [
        train_conflicts.assign(
            conflict_type="reference_vs_train_evaluation"
        ),
        val_conflicts.assign(
            conflict_type="preceding_vs_validation"
        ),
        test_conflicts.assign(
            conflict_type="preceding_vs_test"
        ),
    ],
    ignore_index=True,
    sort=False
)


# ------------------------------------------------------------
# Corrected evaluation preview
# ------------------------------------------------------------

uk_eval_purged_preview = uk_eval[
    ~uk_eval[
        "audit_evaluation_id"
    ].isin(all_conflict_ids)
].copy()

corrected_counts = (
    uk_eval_purged_preview[
        "split_norm"
    ]
    .value_counts()
    .reindex([
        "train",
        "validation",
        "test"
    ])
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# Print final results
# ------------------------------------------------------------

print("\n====================================")
print("CONTROLLED UK-DALE OVERLAP RESULT")
print("====================================")

print(
    "\nTrain-evaluation windows overlapping reference:",
    len(train_conflict_ids)
)

print(
    "Validation windows overlapping preceding retained data:",
    len(val_conflict_ids)
)

print(
    "Test windows overlapping preceding retained data:",
    len(test_conflict_ids)
)

print(
    "\nTotal unique evaluation rows to remove:",
    len(all_conflict_ids)
)

print(
    "Affected evaluation percentage:",
    f"{100 * len(all_conflict_ids) / len(uk_eval):.6f}%"
)

print("\nOriginal evaluation counts:")
print(
    uk_eval["split_norm"]
    .value_counts()
    .reindex([
        "train",
        "validation",
        "test"
    ])
)

print("\nCorrected evaluation counts:")
print(corrected_counts)

print(
    "\nCorrected total evaluation rows:",
    len(uk_eval_purged_preview)
)


# ------------------------------------------------------------
# Show validation/test conflicts explicitly
# ------------------------------------------------------------

print("\nValidation conflicts:")

if len(val_conflicts) == 0:
    print("None")
else:
    print(
        val_conflicts[
            [
                "audit_evaluation_id",
                "series_key",
                "window_start",
                "window_end"
            ]
        ].to_string(index=False)
    )


print("\nTest conflicts:")

if len(test_conflicts) == 0:
    print("None")
else:
    print(
        test_conflicts[
            [
                "audit_evaluation_id",
                "series_key",
                "window_start",
                "window_end"
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# Compare against the prior Work audit
# ------------------------------------------------------------

print("\nPrior Work-audit expectation:")
print(
    "1299 train + 2 validation + 2 test = 1303"
)

print("\nObserved exact result:")
print(
    f"{len(train_conflict_ids)} train + "
    f"{len(val_conflict_ids)} validation + "
    f"{len(test_conflict_ids)} test = "
    f"{len(all_conflict_ids)}"
)


if (
    len(train_conflict_ids) == 1299
    and len(val_conflict_ids) == 2
    and len(test_conflict_ids) == 2
    and len(all_conflict_ids) == 1303
):

    print(
        "\nPASS: Prior controlled UK-DALE audit "
        "confirmed exactly."
    )

else:

    print(
        "\nNOTE: Full saved-sample result differs from "
        "the prior audit. Use the exact values above."
    )


# ------------------------------------------------------------
# Save audit artifacts only
# ------------------------------------------------------------

UK_CONTROLLED_AUDIT_OUT = (
    CORRECTION
    / "ukdale_controlled_overlap_audit_summary.csv"
)

UK_CONTROLLED_CONFLICTS_OUT = (
    CORRECTION
    / "ukdale_controlled_conflicting_evaluation_rows.csv"
)


summary = pd.DataFrame([{
    "dataset": "UK-DALE",
    "reference_windows": len(uk_ref),
    "evaluation_train_original": len(uk_train_eval),
    "evaluation_validation_original": len(uk_val_eval),
    "evaluation_test_original": len(uk_test_eval),
    "train_eval_removed": len(train_conflict_ids),
    "validation_removed": len(val_conflict_ids),
    "test_removed": len(test_conflict_ids),
    "total_removed": len(all_conflict_ids),
    "corrected_train_eval": int(
        corrected_counts["train"]
    ),
    "corrected_validation": int(
        corrected_counts["validation"]
    ),
    "corrected_test": int(
        corrected_counts["test"]
    ),
    "corrected_total_evaluation": len(
        uk_eval_purged_preview
    ),
    "affected_percent": (
        100
        * len(all_conflict_ids)
        / len(uk_eval)
    )
}])


summary.to_csv(
    UK_CONTROLLED_AUDIT_OUT,
    index=False
)

uk_controlled_conflicts.to_csv(
    UK_CONTROLLED_CONFLICTS_OUT,
    index=False
)


print("\nAudit summary saved to:")
print(UK_CONTROLLED_AUDIT_OUT)

print("\nConflicting evaluation rows saved to:")
print(UK_CONTROLLED_CONFLICTS_OUT)

print(
    "\nIMPORTANT: "
    "No reference/evaluation sample was modified."
)

In [ ]:
# CELL 12 — Exact Controlled AMPds2 Reference/Evaluation Raw-Time Overlap Audit

import pandas as pd
import numpy as np

AMP_REF_FILE = (
    CROSS_MODEL_EXP
    / "tables"
    / "ampds2_lof_reference_sample.csv"
)

AMP_EVAL_FILE = (
    CROSS_MODEL_EXP
    / "tables"
    / "ampds2_cross_model_evaluation_sample.csv"
)

print("Reference file:", AMP_REF_FILE)
print("Reference exists:", AMP_REF_FILE.exists())

print("\nEvaluation file:", AMP_EVAL_FILE)
print("Evaluation exists:", AMP_EVAL_FILE.exists())


# ------------------------------------------------------------
# Inspect schemas
# ------------------------------------------------------------

amp_ref_cols = pd.read_csv(
    AMP_REF_FILE,
    nrows=0
).columns.tolist()

amp_eval_cols = pd.read_csv(
    AMP_EVAL_FILE,
    nrows=0
).columns.tolist()

print("\nReference columns:")
print(amp_ref_cols)

print("\nEvaluation columns:")
print(amp_eval_cols)


# ------------------------------------------------------------
# Required timestamp columns
# ------------------------------------------------------------

for col in ["window_start", "window_end"]:

    assert col in amp_ref_cols, \
        f"Reference sample missing required column: {col}"

    assert col in amp_eval_cols, \
        f"Evaluation sample missing required column: {col}"


# ------------------------------------------------------------
# Determine the evaluation split column
# ------------------------------------------------------------

if "evaluation_split" in amp_eval_cols:
    AMP_SPLIT_COL = "evaluation_split"

elif "split" in amp_eval_cols:
    AMP_SPLIT_COL = "split"

else:
    raise RuntimeError(
        "No split/evaluation_split column found in AMPds2 evaluation sample."
    )

print("\nEvaluation split column used:", AMP_SPLIT_COL)


# ------------------------------------------------------------
# Determine common series/channel identifier
# ------------------------------------------------------------

if (
    "series_key" in amp_ref_cols
    and "series_key" in amp_eval_cols
):
    AMP_SERIES_COL = "series_key"

elif (
    "channel_code" in amp_ref_cols
    and "channel_code" in amp_eval_cols
):
    AMP_SERIES_COL = "channel_code"

else:
    raise RuntimeError(
        "Could not identify a common AMPds2 channel identifier. "
        "STOP and send me the printed column lists."
    )

print("Series/channel column used:", AMP_SERIES_COL)


# ------------------------------------------------------------
# Load complete saved samples
# ------------------------------------------------------------

amp_ref = pd.read_csv(
    AMP_REF_FILE
)

amp_eval = pd.read_csv(
    AMP_EVAL_FILE
)


# ------------------------------------------------------------
# Parse timestamps
# ------------------------------------------------------------

for df_name, df in [
    ("reference", amp_ref),
    ("evaluation", amp_eval)
]:

    df["window_start"] = pd.to_datetime(
        df["window_start"],
        utc=True,
        errors="coerce"
    )

    df["window_end"] = pd.to_datetime(
        df["window_end"],
        utc=True,
        errors="coerce"
    )

    assert df["window_start"].notna().all(), \
        f"{df_name}: invalid window_start found."

    assert df["window_end"].notna().all(), \
        f"{df_name}: invalid window_end found."


# ------------------------------------------------------------
# Normalize evaluation split names
# ------------------------------------------------------------

amp_eval["split_norm"] = (
    amp_eval[AMP_SPLIT_COL]
    .replace({
        "val": "validation"
    })
)

unexpected_amp_controlled = set(
    amp_eval["split_norm"]
    .dropna()
    .unique()
) - {
    "train",
    "validation",
    "test"
}

if unexpected_amp_controlled:

    raise RuntimeError(
        "Unexpected evaluation split labels: "
        f"{unexpected_amp_controlled}"
    )


print("\n====================================")
print("CONTROLLED AMPds2 SAMPLE CHECK")
print("====================================")

print("\nReference windows:", len(amp_ref))

print("\nEvaluation split counts:")
print(
    amp_eval["split_norm"]
    .value_counts()
    .reindex([
        "train",
        "validation",
        "test"
    ])
)

print(
    "\nUnique reference channels:",
    amp_ref[AMP_SERIES_COL].nunique()
)

print(
    "Unique evaluation channels:",
    amp_eval[AMP_SERIES_COL].nunique()
)


# ------------------------------------------------------------
# Verify that exact saved source rows were not reused
# ------------------------------------------------------------

if (
    "source_row_id" in amp_ref.columns
    and "source_row_id" in amp_eval.columns
):

    amp_ref_ids = set(
        amp_ref["source_row_id"]
        .dropna()
        .astype(str)
    )

    amp_eval_ids = set(
        amp_eval["source_row_id"]
        .dropna()
        .astype(str)
    )

    identical_amp_ids = (
        amp_ref_ids
        .intersection(amp_eval_ids)
    )

    print(
        "\nIdentical source_row_id reuse:",
        len(identical_amp_ids)
    )

    assert len(identical_amp_ids) == 0, \
        "Unexpected identical AMPds2 source_row_id reuse detected."

else:

    print(
        "\nsource_row_id is not available in both files; "
        "exact row-ID reuse cannot be checked."
    )


# ------------------------------------------------------------
# Add stable audit IDs
# ------------------------------------------------------------

amp_ref = (
    amp_ref
    .reset_index(drop=True)
)

amp_eval = (
    amp_eval
    .reset_index(drop=True)
)

amp_ref["audit_reference_id"] = np.arange(
    len(amp_ref),
    dtype=np.int64
)

amp_eval["audit_evaluation_id"] = np.arange(
    len(amp_eval),
    dtype=np.int64
)


# ------------------------------------------------------------
# Interval-overlap helper
#
# Windows overlap when:
#
# query_start <= preceding_end
# AND
# preceding_start <= query_end
#
# Checks are performed only within the same channel.
# ------------------------------------------------------------

def find_amp_overlapping_query_rows(
    query_df,
    preceding_df,
    series_col,
    query_id_col
):

    conflicting_ids = []
    details = []

    preceding_groups = {
        key: g.sort_values(
            "window_start"
        ).copy()

        for key, g in preceding_df.groupby(
            series_col,
            sort=False
        )
    }

    for series, qg in query_df.groupby(
        series_col,
        sort=False
    ):

        if series not in preceding_groups:
            continue

        pg = preceding_groups[series]

        p_start = (
            pg["window_start"]
            .astype("int64")
            .to_numpy()
        )

        p_end = (
            pg["window_end"]
            .astype("int64")
            .to_numpy()
        )

        order = np.argsort(
            p_start
        )

        p_start = p_start[order]
        p_end = p_end[order]

        prefix_max_end = (
            np.maximum.accumulate(
                p_end
            )
        )

        for _, row in qg.iterrows():

            q_start = (
                row["window_start"].value
            )

            q_end = (
                row["window_end"].value
            )

            pos = np.searchsorted(
                p_start,
                q_end,
                side="right"
            ) - 1

            overlaps = False

            if pos >= 0:

                overlaps = (
                    prefix_max_end[pos]
                    >= q_start
                )

            if overlaps:

                qid = int(
                    row[query_id_col]
                )

                conflicting_ids.append(
                    qid
                )

                details.append({
                    "audit_evaluation_id":
                        qid,

                    "series_key":
                        series,

                    "split":
                        row["split_norm"],

                    "window_start":
                        row["window_start"],

                    "window_end":
                        row["window_end"],
                })


    return (
        sorted(
            set(conflicting_ids)
        ),
        pd.DataFrame(details)
    )


# ============================================================
# STEP A — TRAIN-EVALUATION vs REFERENCE
# ============================================================

amp_train_eval = amp_eval[
    amp_eval["split_norm"] == "train"
].copy()

amp_val_eval = amp_eval[
    amp_eval["split_norm"] == "validation"
].copy()

amp_test_eval = amp_eval[
    amp_eval["split_norm"] == "test"
].copy()


amp_train_conflict_ids, amp_train_conflicts = (
    find_amp_overlapping_query_rows(
        query_df=amp_train_eval,
        preceding_df=amp_ref,
        series_col=AMP_SERIES_COL,
        query_id_col="audit_evaluation_id"
    )
)


# ============================================================
# STEP B — VALIDATION vs REFERENCE + RETAINED TRAIN-EVALUATION
# ============================================================

amp_train_retained = amp_train_eval[
    ~amp_train_eval[
        "audit_evaluation_id"
    ].isin(
        amp_train_conflict_ids
    )
].copy()


amp_preceding_validation = pd.concat(
    [
        amp_ref,
        amp_train_retained
    ],
    ignore_index=True,
    sort=False
)


amp_val_conflict_ids, amp_val_conflicts = (
    find_amp_overlapping_query_rows(
        query_df=amp_val_eval,
        preceding_df=amp_preceding_validation,
        series_col=AMP_SERIES_COL,
        query_id_col="audit_evaluation_id"
    )
)


# ============================================================
# STEP C — TEST vs ALL RETAINED PRECEDING DATA
# ============================================================

amp_val_retained = amp_val_eval[
    ~amp_val_eval[
        "audit_evaluation_id"
    ].isin(
        amp_val_conflict_ids
    )
].copy()


amp_preceding_test = pd.concat(
    [
        amp_ref,
        amp_train_retained,
        amp_val_retained
    ],
    ignore_index=True,
    sort=False
)


amp_test_conflict_ids, amp_test_conflicts = (
    find_amp_overlapping_query_rows(
        query_df=amp_test_eval,
        preceding_df=amp_preceding_test,
        series_col=AMP_SERIES_COL,
        query_id_col="audit_evaluation_id"
    )
)


# ------------------------------------------------------------
# Consolidate unique conflicts
# ------------------------------------------------------------

amp_all_conflict_ids = sorted(
    set(
        amp_train_conflict_ids
        + amp_val_conflict_ids
        + amp_test_conflict_ids
    )
)


amp_controlled_conflicts = pd.concat(
    [
        amp_train_conflicts.assign(
            conflict_type=
            "reference_vs_train_evaluation"
        ),

        amp_val_conflicts.assign(
            conflict_type=
            "preceding_vs_validation"
        ),

        amp_test_conflicts.assign(
            conflict_type=
            "preceding_vs_test"
        ),
    ],
    ignore_index=True,
    sort=False
)


# ------------------------------------------------------------
# Corrected evaluation preview
# ------------------------------------------------------------

amp_eval_purged_preview = amp_eval[
    ~amp_eval[
        "audit_evaluation_id"
    ].isin(
        amp_all_conflict_ids
    )
].copy()


amp_corrected_eval_counts = (
    amp_eval_purged_preview[
        "split_norm"
    ]
    .value_counts()
    .reindex([
        "train",
        "validation",
        "test"
    ])
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print("\n====================================")
print("CONTROLLED AMPds2 OVERLAP RESULT")
print("====================================")

print(
    "\nTrain-evaluation windows overlapping reference:",
    len(amp_train_conflict_ids)
)

print(
    "Validation windows overlapping preceding retained data:",
    len(amp_val_conflict_ids)
)

print(
    "Test windows overlapping preceding retained data:",
    len(amp_test_conflict_ids)
)

print(
    "\nTotal unique evaluation rows to remove:",
    len(amp_all_conflict_ids)
)

print(
    "Affected evaluation percentage:",
    f"{100 * len(amp_all_conflict_ids) / len(amp_eval):.6f}%"
)


print("\nOriginal evaluation counts:")
print(
    amp_eval["split_norm"]
    .value_counts()
    .reindex([
        "train",
        "validation",
        "test"
    ])
)


print("\nCorrected evaluation counts:")
print(
    amp_corrected_eval_counts
)

print(
    "\nCorrected total evaluation rows:",
    len(amp_eval_purged_preview)
)


# ------------------------------------------------------------
# Show validation and test conflicts
# ------------------------------------------------------------

print("\nValidation conflicts:")

if len(amp_val_conflicts) == 0:

    print("None")

else:

    print(
        amp_val_conflicts[
            [
                "audit_evaluation_id",
                "series_key",
                "window_start",
                "window_end"
            ]
        ].to_string(index=False)
    )


print("\nTest conflicts:")

if len(amp_test_conflicts) == 0:

    print("None")

else:

    print(
        amp_test_conflicts[
            [
                "audit_evaluation_id",
                "series_key",
                "window_start",
                "window_end"
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# Compare against prior Work audit
# ------------------------------------------------------------

print("\nPrior Work-audit expectation:")
print(
    "373 train + 1 validation + 0 test = 374"
)

print("\nObserved exact result:")
print(
    f"{len(amp_train_conflict_ids)} train + "
    f"{len(amp_val_conflict_ids)} validation + "
    f"{len(amp_test_conflict_ids)} test = "
    f"{len(amp_all_conflict_ids)}"
)


if (
    len(amp_train_conflict_ids) == 373
    and len(amp_val_conflict_ids) == 1
    and len(amp_test_conflict_ids) == 0
    and len(amp_all_conflict_ids) == 374
):

    print(
        "\nPASS: Prior controlled AMPds2 audit "
        "confirmed exactly."
    )

else:

    print(
        "\nNOTE: Full saved-sample result differs "
        "from the prior audit. "
        "Use the exact values above."
    )


# ------------------------------------------------------------
# Save audit artifacts only
# ------------------------------------------------------------

AMP_CONTROLLED_AUDIT_OUT = (
    CORRECTION
    / "ampds2_controlled_overlap_audit_summary.csv"
)

AMP_CONTROLLED_CONFLICTS_OUT = (
    CORRECTION
    / "ampds2_controlled_conflicting_evaluation_rows.csv"
)


amp_controlled_summary = pd.DataFrame([{
    "dataset": "AMPds2",

    "reference_windows":
        len(amp_ref),

    "evaluation_train_original":
        len(amp_train_eval),

    "evaluation_validation_original":
        len(amp_val_eval),

    "evaluation_test_original":
        len(amp_test_eval),

    "train_eval_removed":
        len(amp_train_conflict_ids),

    "validation_removed":
        len(amp_val_conflict_ids),

    "test_removed":
        len(amp_test_conflict_ids),

    "total_removed":
        len(amp_all_conflict_ids),

    "corrected_train_eval":
        int(
            amp_corrected_eval_counts[
                "train"
            ]
        ),

    "corrected_validation":
        int(
            amp_corrected_eval_counts[
                "validation"
            ]
        ),

    "corrected_test":
        int(
            amp_corrected_eval_counts[
                "test"
            ]
        ),

    "corrected_total_evaluation":
        len(amp_eval_purged_preview),

    "affected_percent":
        (
            100
            * len(amp_all_conflict_ids)
            / len(amp_eval)
        )
}])


amp_controlled_summary.to_csv(
    AMP_CONTROLLED_AUDIT_OUT,
    index=False
)

amp_controlled_conflicts.to_csv(
    AMP_CONTROLLED_CONFLICTS_OUT,
    index=False
)


print("\nAudit summary saved to:")
print(
    AMP_CONTROLLED_AUDIT_OUT
)

print("\nConflicting evaluation rows saved to:")
print(
    AMP_CONTROLLED_CONFLICTS_OUT
)

print(
    "\nIMPORTANT: "
    "No reference/evaluation sample was modified."
)

In [ ]:
# CELL 13 — Correction-Phase Artifact Inventory and Threshold Audit

import pandas as pd
from pathlib import Path
import os

print("==========================================")
print("PROBLEM 2 — CORRECTION-PHASE INVENTORY")
print("==========================================")


# ============================================================
# A. FINAL VERIFIED AUDIT COUNTS
# ============================================================

verified_summary = pd.DataFrame([
    {
        "branch": "NAB 60/30",
        "original_total": 103526,
        "validation_removed": 57,
        "test_removed": 57,
        "total_removed": 114,
        "corrected_total": 103412,
    },
    {
        "branch": "NAB 60/15",
        "original_total": 207033,
        "validation_removed": 170,
        "test_removed": 170,
        "total_removed": 340,
        "corrected_total": 206693,
    },
    {
        "branch": "UK-DALE full",
        "original_total": 363940,
        "validation_removed": 16,
        "test_removed": 15,
        "total_removed": 31,
        "corrected_total": 363909,
    },
    {
        "branch": "AMPds2 full",
        "original_total": 735819,
        "validation_removed": 21,
        "test_removed": 21,
        "total_removed": 42,
        "corrected_total": 735777,
    },
    {
        "branch": "Controlled UK-DALE evaluation",
        "original_total": 19694,
        "validation_removed": 2,
        "test_removed": 2,
        "total_removed": 1303,
        "corrected_total": 18391,
    },
    {
        "branch": "Controlled AMPds2 evaluation",
        "original_total": 26250,
        "validation_removed": 1,
        "test_removed": 0,
        "total_removed": 374,
        "corrected_total": 25876,
    },
])

print("\nVerified audit summary:")
print(verified_summary.to_string(index=False))


# ============================================================
# B. LIST NAB 60/30 ARTIFACTS
# ============================================================

NAB_6030_RESULT_DIR = (
    BASE
    / "results"
    / "EXP_NAB_60min_stride30"
)

NAB_6015_RESULT_DIR = (
    BASE
    / "results"
    / "EXP_NAB_60min_stride15"
)

print("\n==========================================")
print("NAB 60/30 RESULT ARTIFACTS")
print("==========================================")

if NAB_6030_RESULT_DIR.exists():
    for p in sorted(NAB_6030_RESULT_DIR.iterdir()):
        print(p.name, " | ", p.stat().st_size, "bytes")
else:
    print("Directory not found:", NAB_6030_RESULT_DIR)


print("\n==========================================")
print("NAB 60/15 RESULT ARTIFACTS")
print("==========================================")

if NAB_6015_RESULT_DIR.exists():
    for p in sorted(NAB_6015_RESULT_DIR.iterdir()):
        print(p.name, " | ", p.stat().st_size, "bytes")
else:
    print("Directory not found:", NAB_6015_RESULT_DIR)


# ============================================================
# C. SEARCH FOR NAB MODEL / SCORE / VALIDATION ARTIFACTS
# ============================================================

print("\n==========================================")
print("NAB MODEL / VALIDATION / SCORE ARTIFACTS")
print("==========================================")

nab_keywords = [
    "model",
    "validation",
    "val_",
    "score",
    "threshold",
    "scaler",
    "encoder",
    "isolation",
    "lof",
]

nab_candidates = []

for p in BASE.rglob("*"):
    if not p.is_file():
        continue

    name = p.name.lower()
    path_lower = str(p).lower()

    # Restrict search to NAB-related paths
    if "nab" not in path_lower:
        continue

    if any(k in name for k in nab_keywords):
        nab_candidates.append(p)

for p in sorted(nab_candidates)[:150]:
    print(p)

print("\nNAB candidate artifact count:", len(nab_candidates))


# ============================================================
# D. INSPECT NAB METRIC / PREDICTION SCHEMAS
# ============================================================

nab_files_to_inspect = [
    BASE / "results" / "EXP_NAB_60min_stride30"
         / "metrics_nab_60min_stride30_all_models.csv",

    BASE / "results" / "EXP_NAB_60min_stride30"
         / "test_predictions_nab_60min_stride30.csv",

    BASE / "results" / "EXP_NAB_60min_stride15"
         / "metrics_nab_60min_stride15.csv",

    BASE / "results" / "EXP_NAB_60min_stride15"
         / "metrics_nab_60min_stride15_all_models.csv",

    BASE / "results" / "EXP_NAB_60min_stride15"
         / "test_predictions_nab_60min_stride15.csv",

    BASE / "results" / "EXP_NAB_60min_stride15"
         / "test_predictions_nab_60min_stride15_supervised_models.csv",
]

print("\n==========================================")
print("NAB CSV SCHEMAS")
print("==========================================")

for path in nab_files_to_inspect:

    print("\nFILE:", path)

    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        sample = pd.read_csv(path, nrows=5)

        print("Rows previewed:", len(sample))
        print("Columns:")
        print(sample.columns.tolist())

        print("\nFirst rows:")
        print(sample.head().to_string(index=False))

    except Exception as e:
        print("Could not read:", repr(e))


# ============================================================
# E. UK-DALE FULL-PIPELINE THRESHOLD AUDIT
# ============================================================

print("\n==========================================")
print("UK-DALE SAVED THRESHOLD / SCORE AUDIT")
print("==========================================")

uk_threshold_cols = [
    c for c in [
        "split",
        "series_key",
        "window_start",
        "window_end",
        "anomaly_score",
        "if_decision_function",
        "if_threshold_percentile",
        "if_threshold",
        "predicted_anomaly",
    ]
    if c in uk_header
]

uk_threshold_df = pd.read_csv(
    UKDALE_SCORED_FILE,
    usecols=uk_threshold_cols
)

print("\nColumns loaded:")
print(uk_threshold_df.columns.tolist())

if "if_threshold_percentile" in uk_threshold_df.columns:
    print("\nUnique threshold percentiles:")
    print(
        uk_threshold_df[
            "if_threshold_percentile"
        ].dropna().unique()
    )

if "if_threshold" in uk_threshold_df.columns:
    vals = (
        uk_threshold_df["if_threshold"]
        .dropna()
        .unique()
    )

    print("\nNumber of unique saved IF thresholds:", len(vals))
    print("First up to 20 values:")
    print(vals[:20])

if "anomaly_score" in uk_threshold_df.columns:

    print("\nUK-DALE anomaly-score counts by split:")
    print(
        uk_threshold_df
        .groupby("split")["anomaly_score"]
        .agg(["count", "min", "median", "max"])
    )


# ============================================================
# F. AMPds2 FULL-PIPELINE THRESHOLD AUDIT
# ============================================================

print("\n==========================================")
print("AMPds2 SAVED THRESHOLD / SCORE AUDIT")
print("==========================================")

amp_threshold_cols = [
    c for c in [
        "split",
        "series_key",
        "window_start",
        "window_end",
        "anomaly_score",
        "if_decision_function",
        "if_threshold_percentile",
        "if_threshold",
        "predicted_anomaly",
    ]
    if c in amp_header
]

amp_threshold_df = pd.read_csv(
    AMPDS2_SCORED_FILE,
    usecols=amp_threshold_cols
)

print("\nColumns loaded:")
print(amp_threshold_df.columns.tolist())

if "if_threshold_percentile" in amp_threshold_df.columns:
    print("\nUnique threshold percentiles:")
    print(
        amp_threshold_df[
            "if_threshold_percentile"
        ].dropna().unique()
    )

if "if_threshold" in amp_threshold_df.columns:
    vals = (
        amp_threshold_df["if_threshold"]
        .dropna()
        .unique()
    )

    print("\nNumber of unique saved IF thresholds:", len(vals))
    print("First up to 20 values:")
    print(vals[:20])

if "anomaly_score" in amp_threshold_df.columns:

    print("\nAMPds2 anomaly-score counts by split:")
    print(
        amp_threshold_df
        .groupby("split")["anomaly_score"]
        .agg(["count", "min", "median", "max"])
    )


# ============================================================
# G. CONTROLLED CROSS-MODEL TABLE INVENTORY
# ============================================================

print("\n==========================================")
print("CONTROLLED CROSS-MODEL TABLE INVENTORY")
print("==========================================")

CROSS_TABLE_DIR = CROSS_MODEL_EXP / "tables"

for p in sorted(CROSS_TABLE_DIR.iterdir()):

    if p.is_file():
        print(
            p.name,
            " | ",
            p.stat().st_size,
            "bytes"
        )


# ============================================================
# H. CONTROLLED SAVED MODEL INVENTORY
# ============================================================

print("\n==========================================")
print("CONTROLLED SAVED MODEL INVENTORY")
print("==========================================")

CROSS_MODEL_DIR = CROSS_MODEL_EXP / "models"

if CROSS_MODEL_DIR.exists():

    for p in sorted(
        CROSS_MODEL_DIR.rglob("*")
    ):
        if p.is_file():
            print(
                p.relative_to(CROSS_MODEL_DIR),
                " | ",
                p.stat().st_size,
                "bytes"
            )

else:
    print("No model directory found:", CROSS_MODEL_DIR)


# ============================================================
# I. SEARCH FOR AUTOENCODER TRAINING HISTORY / EARLY STOPPING
# ============================================================

print("\n==========================================")
print("AUTOENCODER HISTORY / EARLY-STOPPING ARTIFACTS")
print("==========================================")

ae_keywords = [
    "history",
    "early",
    "checkpoint",
    "autoencoder",
    "ae_",
    "keras",
    "h5",
    "weights",
]

ae_candidates = []

for p in CROSS_MODEL_EXP.rglob("*"):

    if not p.is_file():
        continue

    if any(
        k in p.name.lower()
        for k in ae_keywords
    ):
        ae_candidates.append(p)

for p in sorted(ae_candidates):
    print(p)

print(
    "\nAutoencoder-related candidate count:",
    len(ae_candidates)
)


# ============================================================
# J. SAVE INVENTORY CHECKPOINT
# ============================================================

INVENTORY_TXT = (
    CORRECTION
    / "correction_phase_artifact_inventory.txt"
)

with open(
    INVENTORY_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "Problem 2 Correction-Phase Artifact Inventory\n"
    )

    f.write(
        "============================================\n\n"
    )

    f.write(
        "Verified boundary corrections:\n"
    )

    f.write(
        verified_summary.to_string(index=False)
    )

    f.write(
        "\n\nOriginal experiment files were not modified "
        "during the audit phase.\n"
    )


print("\n==========================================")
print("INVENTORY COMPLETE")
print("==========================================")

print("\nSaved checkpoint:")
print(INVENTORY_TXT)

print(
    "\nIMPORTANT: "
    "No model, score, threshold, prediction, or dataset "
    "was modified by CELL 13."
)

In [ ]:
# CELL 14 — Controlled Autoencoder Dependency and Saved-Score Audit

import pandas as pd
import numpy as np
from pathlib import Path
import json
import re

print("===================================================")
print("CONTROLLED AUTOENCODER DEPENDENCY / SCORE AUDIT")
print("===================================================")


# ============================================================
# A. INSPECT FINAL CONTROLLED SCORED TABLES
# ============================================================

CONTROLLED_SCORED_FILES = {
    "UK-DALE": (
        CROSS_MODEL_EXP
        / "tables"
        / "ukdale_balanced_if_lof_ae_scored_evaluation.csv"
    ),

    "AMPds2": (
        CROSS_MODEL_EXP
        / "tables"
        / "ampds2_balanced_if_lof_ae_scored_evaluation.csv"
    ),
}


for dataset, path in CONTROLLED_SCORED_FILES.items():

    print("\n" + "=" * 55)
    print(dataset, "FINAL CONTROLLED SCORED TABLE")
    print("=" * 55)

    print("File:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    header = pd.read_csv(
        path,
        nrows=0
    ).columns.tolist()

    print("\nColumns:")
    print(header)

    relevant_cols = [
        c for c in header

        if any(
            token in c.lower()

            for token in [
                "split",
                "score",
                "threshold",
                "pred",
                "flag",
                "source_row",
                "window_start",
                "window_end",
            ]
        )
    ]

    print("\nRelevant score / threshold columns:")
    print(relevant_cols)

    sample = pd.read_csv(
        path,
        usecols=relevant_cols
    )

    print("\nRows:", len(sample))

    # --------------------------------------------------------
    # Split counts
    # --------------------------------------------------------

    split_candidates = [
        c for c in relevant_cols
        if "split" in c.lower()
    ]

    for split_col in split_candidates:

        print(
            f"\nCounts for {split_col}:"
        )

        print(
            sample[split_col]
            .value_counts(dropna=False)
        )


    # --------------------------------------------------------
    # Unique thresholds
    # --------------------------------------------------------

    threshold_cols = [
        c for c in relevant_cols
        if "threshold" in c.lower()
    ]

    for col in threshold_cols:

        vals = (
            sample[col]
            .dropna()
            .unique()
        )

        print(
            f"\n{col}: "
            f"{len(vals)} unique non-null value(s)"
        )

        print(vals[:20])


    # --------------------------------------------------------
    # Score ranges
    # --------------------------------------------------------

    score_cols = [
        c for c in relevant_cols

        if "score" in c.lower()
        and pd.api.types.is_numeric_dtype(
            sample[c]
        )
    ]

    if score_cols:

        print("\nScore summaries:")

        for col in score_cols:

            print(
                f"{col}: "
                f"min={sample[col].min():.8f}, "
                f"median={sample[col].median():.8f}, "
                f"max={sample[col].max():.8f}"
            )


# ============================================================
# B. INSPECT AUTOENCODER TRAINING HISTORIES
# ============================================================

AE_HISTORY_FILES = {
    "UK-DALE": (
        CROSS_MODEL_EXP
        / "tables"
        / "ukdale_autoencoder_training_history.csv"
    ),

    "AMPds2": (
        CROSS_MODEL_EXP
        / "tables"
        / "ampds2_autoencoder_training_history.csv"
    ),
}


for dataset, path in AE_HISTORY_FILES.items():

    print("\n" + "=" * 55)
    print(dataset, "AUTOENCODER TRAINING HISTORY")
    print("=" * 55)

    print("File:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    hist = pd.read_csv(path)

    print("\nColumns:")
    print(hist.columns.tolist())

    print("\nEpoch rows:", len(hist))

    print("\nFirst 5 rows:")
    print(
        hist.head().to_string(index=False)
    )

    print("\nLast 5 rows:")
    print(
        hist.tail().to_string(index=False)
    )

    if "val_loss" in hist.columns:

        best_idx = (
            hist["val_loss"]
            .astype(float)
            .idxmin()
        )

        print(
            "\nBest val_loss row:"
        )

        print(
            hist.loc[best_idx]
            .to_string()
        )


# ============================================================
# C. LOCATE THESIS NOTEBOOKS
# ============================================================

print("\n" + "=" * 55)
print("THESIS NOTEBOOK SEARCH")
print("=" * 55)

NOTEBOOK_SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/Colab Notebooks"),
    BASE / "notebooks",
    BASE,
]

notebook_paths = []

seen = set()

for root in NOTEBOOK_SEARCH_ROOTS:

    if not root.exists():
        continue

    try:
        for p in root.rglob("*.ipynb"):

            # Only inspect notebooks plausibly related to thesis work.
            name_lower = p.name.lower()

            if (
                "thesis" in name_lower
                or "master" in name_lower
                or "anomaly" in name_lower
            ):

                resolved = str(p)

                if resolved not in seen:
                    notebook_paths.append(p)
                    seen.add(resolved)

    except Exception as e:
        print(
            "Search warning for",
            root,
            ":",
            repr(e)
        )


print("\nCandidate notebooks:")

for p in notebook_paths:

    try:
        size_mb = (
            p.stat().st_size
            / (1024 ** 2)
        )
    except:
        size_mb = np.nan

    print(
        f"{p} | {size_mb:.2f} MB"
    )


# ============================================================
# D. SEARCH NOTEBOOK CELLS FOR AE TRAINING LOGIC
# ============================================================

AE_SEARCH_TERMS = [
    "EarlyStopping",
    "validation_split",
    "validation_data",
    "ukdale_dense_autoencoder",
    "ampds2_dense_autoencoder",
    "ukdale_autoencoder_training_history",
    "ampds2_autoencoder_training_history",
    "balanced_if_lof_ae",
    "EXP_UKDALE_AMPDS2_Cross_Model_Comparison",
]


print("\n" + "=" * 55)
print("AUTOENCODER TRAINING-CODE MATCHES")
print("=" * 55)


notebook_matches = []


for nb_path in notebook_paths:

    print("\nScanning:", nb_path)

    try:

        with open(
            nb_path,
            "r",
            encoding="utf-8"
        ) as f:

            nb = json.load(f)

    except Exception as e:

        print(
            "Could not parse notebook:",
            repr(e)
        )

        continue


    cells = nb.get(
        "cells",
        []
    )


    for cell_num, cell in enumerate(
        cells,
        start=1
    ):

        source = cell.get(
            "source",
            []
        )

        if isinstance(source, list):
            text = "".join(source)
        else:
            text = str(source)


        found_terms = [
            term
            for term in AE_SEARCH_TERMS
            if term.lower() in text.lower()
        ]


        if not found_terms:
            continue


        notebook_matches.append({
            "notebook": str(nb_path),
            "cell_number": cell_num,
            "terms": ", ".join(found_terms),
        })


        print(
            "\n---------------------------------------"
        )

        print(
            "Notebook:",
            nb_path.name
        )

        print(
            "Cell:",
            cell_num
        )

        print(
            "Matched:",
            found_terms
        )


        # Print limited context around first match only
        lowered = text.lower()

        first_positions = [
            lowered.find(term.lower())
            for term in found_terms
            if lowered.find(term.lower()) >= 0
        ]

        if first_positions:

            pos = min(first_positions)

            start = max(
                0,
                pos - 1000
            )

            end = min(
                len(text),
                pos + 2500
            )

            snippet = text[
                start:end
            ]

            print("\nSnippet:")
            print(snippet)


# ============================================================
# E. SEARCH TEXT / LOG FILES FOR SAME TRAINING SETTINGS
# ============================================================

print("\n" + "=" * 55)
print("TEXT / LOG AUTOENCODER MATCHES")
print("=" * 55)

TEXT_ROOTS = [
    BASE / "experiment_logs",
    BASE / "documentation",
    CROSS_MODEL_EXP,
]

text_extensions = {
    ".md",
    ".txt",
    ".py",
    ".json",
}

text_match_count = 0


for root in TEXT_ROOTS:

    if not root.exists():
        continue


    for p in root.rglob("*"):

        if (
            not p.is_file()
            or p.suffix.lower()
            not in text_extensions
        ):
            continue


        try:

            text = p.read_text(
                encoding="utf-8",
                errors="ignore"
            )

        except Exception:

            continue


        found_terms = [
            term
            for term in AE_SEARCH_TERMS
            if term.lower() in text.lower()
        ]


        if not found_terms:
            continue


        text_match_count += 1

        print(
            "\n---------------------------------------"
        )

        print("File:", p)

        print(
            "Matched:",
            found_terms
        )


        lowered = text.lower()

        first_positions = [
            lowered.find(term.lower())
            for term in found_terms
            if lowered.find(term.lower()) >= 0
        ]

        if first_positions:

            pos = min(first_positions)

            start = max(
                0,
                pos - 800
            )

            end = min(
                len(text),
                pos + 2200
            )

            print("\nSnippet:")
            print(
                text[start:end]
            )


print(
    "\nText/log files with relevant matches:",
    text_match_count
)


# ============================================================
# F. SAVE MATCH INDEX
# ============================================================

AE_DEPENDENCY_INDEX = (
    CORRECTION
    / "autoencoder_dependency_search_index.csv"
)

pd.DataFrame(
    notebook_matches
).to_csv(
    AE_DEPENDENCY_INDEX,
    index=False
)

print("\nSearch index saved to:")
print(AE_DEPENDENCY_INDEX)


print("\n===================================================")
print("CELL 14 COMPLETE")
print("===================================================")

print(
    "\nIMPORTANT: "
    "No model, score, threshold, sample, or source file "
    "was modified."
)

In [ ]:
# CELL 15 — Verify Controlled Autoencoder Internal Validation Source

import json
from pathlib import Path

THESIS_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)

print("Notebook:", THESIS_NOTEBOOK)
print("Exists:", THESIS_NOTEBOOK.exists())

with open(
    THESIS_NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:
    thesis_nb = json.load(f)

cells = thesis_nb["cells"]


# ------------------------------------------------------------
# Notebook cell numbers found by CELL 14
# ------------------------------------------------------------

TARGET_NOTEBOOK_CELLS = {
    "UK-DALE controlled Autoencoder": 551,
    "AMPds2 controlled Autoencoder": 553,
}


search_terms = [
    "INTERNAL_VALIDATION_RATIO",
    "reference_df",
    "train_test_split",
    "X_reference",
    "X_reference_all",
    "X_train",
    "X_val",
    "validation_data",
    ".fit(",
]


for label, notebook_cell_number in TARGET_NOTEBOOK_CELLS.items():

    print("\n" + "=" * 100)
    print(label)
    print("=" * 100)

    # CELL 14 numbered notebook cells starting from 1,
    # while Python lists are zero-indexed.
    cell = cells[notebook_cell_number - 1]

    source = cell.get("source", [])

    if isinstance(source, list):
        text = "".join(source)
    else:
        text = str(source)

    print(
        f"\nNotebook cell {notebook_cell_number}"
    )

    print(
        "Total characters:",
        len(text)
    )

    print("\nRelevant code blocks:")

    lines = text.splitlines()

    relevant_line_numbers = []

    for i, line in enumerate(lines):

        if any(
            term.lower() in line.lower()
            for term in search_terms
        ):
            relevant_line_numbers.append(i)

    # Merge nearby matches into readable contexts
    intervals = []

    for idx in relevant_line_numbers:

        start = max(0, idx - 8)
        end = min(len(lines), idx + 15)

        if (
            intervals
            and start <= intervals[-1][1] + 2
        ):
            intervals[-1] = (
                intervals[-1][0],
                max(intervals[-1][1], end)
            )
        else:
            intervals.append((start, end))


    for block_num, (start, end) in enumerate(
        intervals,
        start=1
    ):

        print(
            f"\n--- Relevant block {block_num} "
            f"(lines {start + 1}–{end}) ---"
        )

        for line_no in range(start, end):

            print(
                f"{line_no + 1:04d}: "
                f"{lines[line_no]}"
            )


# ------------------------------------------------------------
# Also search explicitly for whether external evaluation rows
# were ever supplied to model.fit()
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("EXTERNAL VALIDATION DEPENDENCY CHECK")
print("=" * 100)

for label, notebook_cell_number in TARGET_NOTEBOOK_CELLS.items():

    cell = cells[notebook_cell_number - 1]

    source = cell.get("source", [])

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

    lower = text.lower()

    print(f"\n{label}:")

    checks = {
        "evaluation_df appears":
            "evaluation_df" in lower,

        "internal validation ratio appears":
            "internal_validation_ratio" in lower,

        "validation_data appears":
            "validation_data" in lower,

        "train_test_split appears":
            "train_test_split" in lower,
    }

    for key, value in checks.items():
        print(f"{key}: {value}")


print(
    "\nIMPORTANT: "
    "This cell only reads notebook source code. "
    "No models or artifacts were modified."
)

In [ ]:
# CELL 16 — NAB 60/30 and 60/15 Original-Code Reconstruction Audit

import json
import pandas as pd
import numpy as np
from pathlib import Path

THESIS_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)

print("Notebook:", THESIS_NOTEBOOK)
print("Exists:", THESIS_NOTEBOOK.exists())

with open(
    THESIS_NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:
    thesis_nb = json.load(f)

cells = thesis_nb.get("cells", [])


# ============================================================
# A. SEARCH FOR THE EXACT NAB 60/30 AND 60/15 BRANCHES
# ============================================================

SEARCH_TERMS = [
    "EXP_NAB_60min_stride30",
    "EXP_NAB_60min_stride15",
    "metrics_nab_60min_stride30_all_models",
    "metrics_nab_60min_stride15_all_models",
    "test_predictions_nab_60min_stride30",
    "test_predictions_nab_60min_stride15",
    "train_unsupervised_and_get_scores",
    "evaluate_scores_local",
    "X_train_scaled_6030",
    "X_val_scaled_6030",
    "X_test_scaled_6030",
    "X_train_scaled_6015",
    "X_val_scaled_6015",
    "X_test_scaled_6015",
]


matches = []

for cell_number, cell in enumerate(
    cells,
    start=1
):

    source = cell.get("source", [])

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

    found = [
        term
        for term in SEARCH_TERMS
        if term.lower() in text.lower()
    ]

    if found:

        matches.append({
            "cell_number": cell_number,
            "matched_terms": found,
            "text": text,
        })


print("\n==========================================")
print("MATCHING NOTEBOOK CELLS")
print("==========================================")

for item in matches:

    print(
        f"\nCell {item['cell_number']}: "
        f"{item['matched_terms']}"
    )

print(
    "\nTotal matching cells:",
    len(matches)
)


# ============================================================
# B. TERMS WE NEED TO INSPECT INSIDE THOSE CELLS
# ============================================================

INSPECTION_TERMS = [
    # Split construction
    "groupby",
    "0.60",
    "0.80",
    "train_end",
    "val_end",
    "split",

    # Feature preprocessing
    "StandardScaler",
    "fit_transform",
    "transform",
    "scaler",

    # Models
    "IsolationForest",
    "LocalOutlierFactor",
    "Autoencoder",
    "RandomForestClassifier",
    "XGBClassifier",
    "LogisticRegression",
    "Sequential",
    "MLP",

    # Training
    ".fit(",
    "EarlyStopping",
    "validation_data",
    "validation_split",

    # Scores
    "decision_function",
    "predict_proba",
    "predict(",
    "reconstruction",

    # Threshold selection
    "selected_percentile",
    "selected_threshold",
    "percentile",
    "evaluate_scores_local",
    "validation",

    # Saving
    "to_csv",
    "joblib.dump",
    ".save(",
]


# ============================================================
# C. PRINT MERGED RELEVANT CODE BLOCKS
# ============================================================

for item in matches:

    cell_number = item["cell_number"]
    text = item["text"]

    lines = text.splitlines()

    relevant_indices = []

    for i, line in enumerate(lines):

        if any(
            term.lower() in line.lower()
            for term in INSPECTION_TERMS
        ):
            relevant_indices.append(i)


    # Merge nearby matches into larger readable blocks
    intervals = []

    for idx in relevant_indices:

        start = max(
            0,
            idx - 6
        )

        end = min(
            len(lines),
            idx + 12
        )

        if (
            intervals
            and start <= intervals[-1][1] + 3
        ):

            intervals[-1] = (
                intervals[-1][0],
                max(
                    intervals[-1][1],
                    end
                )
            )

        else:

            intervals.append(
                (start, end)
            )


    print("\n\n")
    print("=" * 110)
    print(
        f"NOTEBOOK CELL {cell_number}"
    )
    print("=" * 110)


    if not intervals:

        print(
            "No relevant implementation blocks found."
        )

        continue


    for block_num, (start, end) in enumerate(
        intervals,
        start=1
    ):

        print(
            f"\n--- Relevant block {block_num} "
            f"(lines {start + 1}–{end}) ---"
        )

        for line_no in range(
            start,
            end
        ):

            print(
                f"{line_no + 1:04d}: "
                f"{lines[line_no]}"
            )


# ============================================================
# D. AUTOMATIC DEPENDENCY MATRIX
# ============================================================

print("\n\n")
print("=" * 110)
print("AUTOMATIC NAB DEPENDENCY MATRIX")
print("=" * 110)


dependency_rows = []


for item in matches:

    text_lower = item["text"].lower()

    dependency_rows.append({
        "cell_number":
            item["cell_number"],

        "mentions_6030":
            (
                "6030" in text_lower
                or "60min_stride30" in text_lower
            ),

        "mentions_6015":
            (
                "6015" in text_lower
                or "60min_stride15" in text_lower
            ),

        "uses_fit":
            ".fit(" in text_lower,

        "uses_validation_data":
            "validation_data" in text_lower,

        "uses_validation_split":
            "validation_split" in text_lower,

        "uses_early_stopping":
            "earlystopping" in text_lower,

        "uses_if":
            "isolationforest" in text_lower,

        "uses_lof":
            "localoutlierfactor" in text_lower,

        "uses_autoencoder":
            "autoencoder" in text_lower,

        "uses_rf":
            "randomforestclassifier" in text_lower,

        "uses_xgb":
            "xgbclassifier" in text_lower,

        "uses_logreg":
            "logisticregression" in text_lower,

        "uses_mlp":
            (
                "mlp" in text_lower
                or "sequential" in text_lower
            ),

        "uses_percentile_threshold":
            "percentile" in text_lower,

        "saves_csv":
            "to_csv" in text_lower,

        "saves_joblib":
            "joblib.dump" in text_lower,

        "saves_keras_model":
            ".save(" in text_lower,
    })


dependency_df = pd.DataFrame(
    dependency_rows
)

print(
    dependency_df.to_string(
        index=False
    )
)


# ============================================================
# E. SEARCH ALL NOTEBOOK CELLS FOR MODEL SAVING RELATED
#    SPECIFICALLY TO THE NAB 60/30 OR 60/15 OBJECTS
# ============================================================

print("\n")
print("=" * 110)
print("SEARCH FOR SAVED NAB 60/30 / 60/15 MODEL OBJECTS")
print("=" * 110)


MODEL_OBJECT_TERMS = [
    "iso_6030",
    "lof_6030",
    "ae_6030",
    "rf_6030",
    "xgb_6030",
    "lr_6030",
    "mlp_6030",

    "iso_6015",
    "lof_6015",
    "ae_6015",
    "rf_6015",
    "xgb_6015",
    "lr_6015",
    "mlp_6015",
]

SAVE_TERMS = [
    "joblib.dump",
    ".save(",
    "pickle.dump",
]


save_matches = []


for cell_number, cell in enumerate(
    cells,
    start=1
):

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

    lower = text.lower()

    model_terms_found = [
        term
        for term in MODEL_OBJECT_TERMS
        if term.lower() in lower
    ]

    save_terms_found = [
        term
        for term in SAVE_TERMS
        if term.lower() in lower
    ]

    if (
        model_terms_found
        and save_terms_found
    ):

        save_matches.append({
            "cell_number":
                cell_number,

            "model_terms":
                ", ".join(
                    model_terms_found
                ),

            "save_terms":
                ", ".join(
                    save_terms_found
                ),
        })


if save_matches:

    print(
        pd.DataFrame(
            save_matches
        ).to_string(
            index=False
        )
    )

else:

    print(
        "No notebook cell was found that both "
        "references the 60/30 or 60/15 model objects "
        "and saves a model."
    )


# ============================================================
# F. SEARCH FOR VALIDATION SCORE FILE EXPORTS
# ============================================================

print("\n")
print("=" * 110)
print("SEARCH FOR NAB VALIDATION-SCORE EXPORTS")
print("=" * 110)


VALIDATION_EXPORT_TERMS = [
    "val_scores_iso_6030",
    "val_scores_lof_6030",
    "val_scores_ae_6030",
    "val_scores_rf_6030",
    "val_scores_xgb_6030",
    "val_scores_lr_6030",
    "val_scores_mlp_6030",

    "val_scores_iso_6015",
    "val_scores_lof_6015",
    "val_scores_ae_6015",
    "val_scores_rf_6015",
    "val_scores_xgb_6015",
    "val_scores_lr_6015",
    "val_scores_mlp_6015",
]


validation_export_matches = []


for cell_number, cell in enumerate(
    cells,
    start=1
):

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

    lower = text.lower()

    score_terms_found = [
        term
        for term in VALIDATION_EXPORT_TERMS
        if term.lower() in lower
    ]

    if not score_terms_found:
        continue


    saves_data = (
        "to_csv" in lower
        or "np.save" in lower
        or "pickle.dump" in lower
        or "joblib.dump" in lower
    )


    validation_export_matches.append({
        "cell_number":
            cell_number,

        "score_terms":
            ", ".join(
                score_terms_found
            ),

        "contains_export_command":
            saves_data,
    })


if validation_export_matches:

    print(
        pd.DataFrame(
            validation_export_matches
        ).to_string(
            index=False
        )
    )

else:

    print(
        "No validation-score variable references found."
    )


# ============================================================
# G. SAVE SOURCE-AUDIT INDEX
# ============================================================

SOURCE_AUDIT_INDEX = (
    CORRECTION
    / "nab_60_30_60_15_source_audit_index.csv"
)

dependency_df.to_csv(
    SOURCE_AUDIT_INDEX,
    index=False
)


print("\nSource-audit index saved to:")
print(SOURCE_AUDIT_INDEX)


print("\n==========================================")
print("CELL 16 COMPLETE")
print("==========================================")

print(
    "\nIMPORTANT: "
    "This cell only inspected notebook source code. "
    "No NAB model, score, result, or dataset was modified."
)

In [ ]:
# CELL 17 — Extract Exact NAB 60/15 Split, Preprocessing, Model, and Threshold Code

import json
from pathlib import Path

THESIS_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)

print("Notebook:", THESIS_NOTEBOOK)
print("Exists:", THESIS_NOTEBOOK.exists())

with open(
    THESIS_NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:
    thesis_nb = json.load(f)

cells = thesis_nb["cells"]


# ------------------------------------------------------------
# The original 60/15 experiment is around notebook cells
# 458–478. Print this whole neighborhood so that we preserve
# the exact implementation instead of reconstructing it from
# memory.
# ------------------------------------------------------------

TARGET_RANGE = range(458, 479)

important_terms = [
    "chronological_split_by_series",
    "train_df",
    "val_df",
    "test_df",
    "feature_cols",
    "StandardScaler",
    "fit_transform",
    "IsolationForest",
    "LocalOutlierFactor",
    "Autoencoder",
    "RandomForestClassifier",
    "XGBClassifier",
    "LogisticRegression",
    "MLP",
    "Sequential",
    "EarlyStopping",
    "validation_data",
    "validation_split",
    "select_threshold",
    "evaluate_model",
    "evaluate_scores",
    "percentile",
    "test_scores",
    "val_scores",
    "class_weight",
    "scale_pos_weight",
]


print("\n" + "=" * 110)
print("NAB 60/15 ORIGINAL IMPLEMENTATION")
print("=" * 110)


for notebook_cell_number in TARGET_RANGE:

    cell = cells[
        notebook_cell_number - 1
    ]

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

    if not any(
        term.lower() in text.lower()
        for term in important_terms
    ):
        continue

    print("\n\n")
    print("=" * 110)
    print(
        f"NOTEBOOK CELL {notebook_cell_number}"
    )
    print("=" * 110)

    lines = text.splitlines()

    # Print full cell when it is reasonably sized.
    # Otherwise print merged relevant contexts.
    if len(lines) <= 350:

        for i, line in enumerate(
            lines,
            start=1
        ):
            print(
                f"{i:04d}: {line}"
            )

    else:

        relevant_indices = []

        for i, line in enumerate(lines):

            if any(
                term.lower() in line.lower()
                for term in important_terms
            ):
                relevant_indices.append(i)

        intervals = []

        for idx in relevant_indices:

            start = max(
                0,
                idx - 8
            )

            end = min(
                len(lines),
                idx + 18
            )

            if (
                intervals
                and start <= intervals[-1][1] + 3
            ):

                intervals[-1] = (
                    intervals[-1][0],
                    max(
                        intervals[-1][1],
                        end
                    )
                )

            else:

                intervals.append(
                    (start, end)
                )

        for block_number, (
            start,
            end
        ) in enumerate(
            intervals,
            start=1
        ):

            print(
                f"\n--- Relevant block "
                f"{block_number} "
                f"(lines {start + 1}–{end}) ---"
            )

            for line_no in range(
                start,
                end
            ):

                print(
                    f"{line_no + 1:04d}: "
                    f"{lines[line_no]}"
                )


# ------------------------------------------------------------
# Locate the actual definition of chronological_split_by_series
# ------------------------------------------------------------

print("\n\n")
print("=" * 110)
print("chronological_split_by_series DEFINITION")
print("=" * 110)

split_definition_found = False

for notebook_cell_number, cell in enumerate(
    cells,
    start=1
):

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

    if (
        "def chronological_split_by_series"
        not in text
    ):
        continue

    split_definition_found = True

    print(
        f"\nFound in notebook cell "
        f"{notebook_cell_number}"
    )

    lines = text.splitlines()

    start_idx = next(
        i
        for i, line in enumerate(lines)
        if "def chronological_split_by_series"
        in line
    )

    end_idx = min(
        len(lines),
        start_idx + 80
    )

    for i in range(
        start_idx,
        end_idx
    ):

        print(
            f"{i + 1:04d}: "
            f"{lines[i]}"
        )


if not split_definition_found:

    print(
        "Definition was not found."
    )


# ------------------------------------------------------------
# Search specifically for model persistence or validation-score
# persistence within the original 60/15 neighborhood.
# ------------------------------------------------------------

print("\n\n")
print("=" * 110)
print("60/15 PERSISTENCE CHECK")
print("=" * 110)

persistence_terms = [
    "joblib.dump",
    "pickle.dump",
    "np.save",
    ".save(",
    "to_csv",
]

for notebook_cell_number in TARGET_RANGE:

    cell = cells[
        notebook_cell_number - 1
    ]

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

    lower = text.lower()

    found = [
        term
        for term in persistence_terms
        if term.lower() in lower
    ]

    if not found:
        continue

    print(
        f"\nCell {notebook_cell_number}: "
        f"{found}"
    )

    # Print only persistence-related lines
    for i, line in enumerate(
        text.splitlines(),
        start=1
    ):

        if any(
            term.lower() in line.lower()
            for term in persistence_terms
        ):

            print(
                f"{i:04d}: {line}"
            )


print("\n" + "=" * 110)
print("CELL 17 COMPLETE")
print("=" * 110)

print(
    "\nIMPORTANT: "
    "This cell only reads the original notebook source. "
    "No models, scores, windows, or metrics were modified."
)

In [ ]:
# CELL 18 — Create and Verify Purged NAB 60/30 and 60/15 Correction Partitions

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 100)
print("CREATE PURGED NAB CORRECTION PARTITIONS")
print("=" * 100)


# ============================================================
# Output directories
# ============================================================

NAB_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
)

NAB_6030_CORRECTION_DIR = (
    NAB_CORRECTION_DIR
    / "60_30"
)

NAB_6015_CORRECTION_DIR = (
    NAB_CORRECTION_DIR
    / "60_15"
)

for folder in [
    NAB_CORRECTION_DIR,
    NAB_6030_CORRECTION_DIR,
    NAB_6015_CORRECTION_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# Exact original chronological split function
# Reproduced from Thesis.ipynb Cell 464
# ============================================================

def chronological_split_by_series_exact(
    df,
    train_ratio=0.60,
    val_ratio=0.20
):
    train_parts = []
    val_parts = []
    test_parts = []

    for series_key, g in df.groupby("series"):

        g = (
            g.sort_values("window_start")
            .reset_index(drop=True)
        )

        n = len(g)

        if n < 5:
            continue

        train_end = int(
            n * train_ratio
        )

        val_end = int(
            n * (
                train_ratio
                + val_ratio
            )
        )

        train_parts.append(
            g.iloc[:train_end].copy()
        )

        val_parts.append(
            g.iloc[
                train_end:val_end
            ].copy()
        )

        test_parts.append(
            g.iloc[val_end:].copy()
        )

    train_df = pd.concat(
        train_parts,
        ignore_index=True
    )

    val_df = pd.concat(
        val_parts,
        ignore_index=True
    )

    test_df = pd.concat(
        test_parts,
        ignore_index=True
    )

    return (
        train_df,
        val_df,
        test_df
    )


# ============================================================
# Boundary purge function
#
# NAB windows use time intervals of the form:
# [window_start, window_end)
#
# Therefore:
# later_start < preceding_max_end
# means raw temporal overlap.
# ============================================================

def purge_nab_split_boundaries(
    train_df,
    val_df,
    test_df
):
    purged_val_parts = []
    purged_test_parts = []

    val_removed_parts = []
    test_removed_parts = []

    audit_rows = []


    all_series = sorted(
        set(train_df["series"])
        | set(val_df["series"])
        | set(test_df["series"])
    )


    for series in all_series:

        train = (
            train_df[
                train_df["series"] == series
            ]
            .sort_values("window_start")
            .copy()
        )

        val = (
            val_df[
                val_df["series"] == series
            ]
            .sort_values("window_start")
            .copy()
        )

        test = (
            test_df[
                test_df["series"] == series
            ]
            .sort_values("window_start")
            .copy()
        )


        if (
            len(train) == 0
            or len(val) == 0
            or len(test) == 0
        ):
            raise RuntimeError(
                f"Missing split for series: {series}"
            )


        # --------------------------------------------
        # TRAIN -> VALIDATION
        # --------------------------------------------

        train_max_end = (
            train["window_end"].max()
        )

        val_removed = val[
            val["window_start"]
            < train_max_end
        ].copy()

        val_retained = val[
            val["window_start"]
            >= train_max_end
        ].copy()


        if len(val_retained) == 0:
            raise RuntimeError(
                f"No validation rows remain for {series}"
            )


        # --------------------------------------------
        # VALIDATION -> TEST
        # --------------------------------------------

        retained_val_max_end = (
            val_retained[
                "window_end"
            ].max()
        )

        test_removed = test[
            test["window_start"]
            < retained_val_max_end
        ].copy()

        test_retained = test[
            test["window_start"]
            >= retained_val_max_end
        ].copy()


        if len(test_retained) == 0:
            raise RuntimeError(
                f"No test rows remain for {series}"
            )


        purged_val_parts.append(
            val_retained
        )

        purged_test_parts.append(
            test_retained
        )


        if len(val_removed) > 0:

            val_removed = (
                val_removed.copy()
            )

            val_removed[
                "removed_at_boundary"
            ] = "train_validation"

            val_removed_parts.append(
                val_removed
            )


        if len(test_removed) > 0:

            test_removed = (
                test_removed.copy()
            )

            test_removed[
                "removed_at_boundary"
            ] = "validation_test"

            test_removed_parts.append(
                test_removed
            )


        audit_rows.append({
            "series":
                series,

            "train_windows":
                len(train),

            "validation_original":
                len(val),

            "validation_removed":
                len(val_removed),

            "validation_retained":
                len(val_retained),

            "test_original":
                len(test),

            "test_removed":
                len(test_removed),

            "test_retained":
                len(test_retained),

            "train_max_end":
                train_max_end,

            "first_retained_validation_start":
                val_retained[
                    "window_start"
                ].min(),

            "retained_validation_max_end":
                retained_val_max_end,

            "first_retained_test_start":
                test_retained[
                    "window_start"
                ].min(),
        })


    purged_val = pd.concat(
        purged_val_parts,
        ignore_index=True
    )

    purged_test = pd.concat(
        purged_test_parts,
        ignore_index=True
    )


    if val_removed_parts:

        removed_val = pd.concat(
            val_removed_parts,
            ignore_index=True
        )

    else:

        removed_val = pd.DataFrame()


    if test_removed_parts:

        removed_test = pd.concat(
            test_removed_parts,
            ignore_index=True
        )

    else:

        removed_test = pd.DataFrame()


    audit = pd.DataFrame(
        audit_rows
    )


    return (
        train_df.copy(),
        purged_val,
        purged_test,
        removed_val,
        removed_test,
        audit
    )


# ============================================================
# Process one NAB configuration
# ============================================================

def create_corrected_nab_partitions(
    source_file,
    config_name,
    output_dir,
    expected_original,
    expected_corrected
):

    print("\n")
    print("=" * 100)
    print(
        f"NAB {config_name}"
    )
    print("=" * 100)

    print(
        "Source:",
        source_file
    )

    windows = pd.read_csv(
        source_file,
        low_memory=False
    )


    # --------------------------------------------------------
    # Parse timestamps
    # --------------------------------------------------------

    windows["window_start"] = pd.to_datetime(
        windows["window_start"],
        utc=True,
        errors="raise"
    )

    windows["window_end"] = pd.to_datetime(
        windows["window_end"],
        utc=True,
        errors="raise"
    )


    print(
        "\nLoaded windows:",
        len(windows)
    )

    print(
        "Unique series:",
        windows["series"].nunique()
    )

    print(
        "Anomaly windows:",
        int(windows["label"].sum())
    )


    # --------------------------------------------------------
    # Recreate original split
    # --------------------------------------------------------

    train, val, test = (
        chronological_split_by_series_exact(
            windows
        )
    )


    original_counts = {
        "train":
            len(train),

        "validation":
            len(val),

        "test":
            len(test),

        "total":
            (
                len(train)
                + len(val)
                + len(test)
            ),
    }


    print(
        "\nReconstructed historical counts:"
    )

    print(
        original_counts
    )


    assert (
        original_counts
        == expected_original
    ), (
        f"{config_name}: original split "
        "counts do not match."
    )


    # --------------------------------------------------------
    # Purge boundaries
    # --------------------------------------------------------

    (
        train_corrected,
        val_corrected,
        test_corrected,
        removed_val,
        removed_test,
        audit
    ) = purge_nab_split_boundaries(
        train,
        val,
        test
    )


    corrected_counts = {
        "train":
            len(train_corrected),

        "validation":
            len(val_corrected),

        "test":
            len(test_corrected),

        "total":
            (
                len(train_corrected)
                + len(val_corrected)
                + len(test_corrected)
            ),
    }


    print(
        "\nCorrected counts:"
    )

    print(
        corrected_counts
    )


    assert (
        corrected_counts
        == expected_corrected
    ), (
        f"{config_name}: corrected split "
        "counts do not match the verified audit."
    )


    # --------------------------------------------------------
    # Confirm training is byte-for-byte row-equivalent
    # in the relevant source columns
    # --------------------------------------------------------

    assert len(
        train_corrected
    ) == len(train)

    check_cols = [
        "series",
        "window_start",
        "window_end",
        "label",
    ]

    pd.testing.assert_frame_equal(
        train_corrected[
            check_cols
        ].reset_index(drop=True),

        train[
            check_cols
        ].reset_index(drop=True),

        check_dtype=True
    )

    print(
        "\nPASS: training partition unchanged."
    )


    # --------------------------------------------------------
    # Strict post-purge boundary verification
    # --------------------------------------------------------

    violations = []

    for series in sorted(
        train_corrected["series"].unique()
    ):

        tr = train_corrected[
            train_corrected[
                "series"
            ] == series
        ]

        va = val_corrected[
            val_corrected[
                "series"
            ] == series
        ]

        te = test_corrected[
            test_corrected[
                "series"
            ] == series
        ]


        train_max_end = (
            tr["window_end"].max()
        )

        val_min_start = (
            va["window_start"].min()
        )

        val_max_end = (
            va["window_end"].max()
        )

        test_min_start = (
            te["window_start"].min()
        )


        if (
            val_min_start
            < train_max_end
        ):
            violations.append({
                "series":
                    series,

                "boundary":
                    "train_validation",

                "previous_max_end":
                    train_max_end,

                "next_min_start":
                    val_min_start,
            })


        if (
            test_min_start
            < val_max_end
        ):
            violations.append({
                "series":
                    series,

                "boundary":
                    "validation_test",

                "previous_max_end":
                    val_max_end,

                "next_min_start":
                    test_min_start,
            })


    violations_df = pd.DataFrame(
        violations
    )


    print(
        "\nPost-purge raw-overlap violations:",
        len(violations_df)
    )

    assert len(
        violations_df
    ) == 0, (
        f"{config_name}: boundary leakage "
        "still present after purge."
    )


    # --------------------------------------------------------
    # Label summaries
    # --------------------------------------------------------

    label_summary = pd.DataFrame([
        {
            "split":
                "train",

            "windows":
                len(train_corrected),

            "normal_windows":
                int(
                    (
                        train_corrected[
                            "label"
                        ] == 0
                    ).sum()
                ),

            "anomaly_windows":
                int(
                    train_corrected[
                        "label"
                    ].sum()
                ),

            "anomaly_ratio":
                float(
                    train_corrected[
                        "label"
                    ].mean()
                ),
        },

        {
            "split":
                "validation",

            "windows":
                len(val_corrected),

            "normal_windows":
                int(
                    (
                        val_corrected[
                            "label"
                        ] == 0
                    ).sum()
                ),

            "anomaly_windows":
                int(
                    val_corrected[
                        "label"
                    ].sum()
                ),

            "anomaly_ratio":
                float(
                    val_corrected[
                        "label"
                    ].mean()
                ),
        },

        {
            "split":
                "test",

            "windows":
                len(test_corrected),

            "normal_windows":
                int(
                    (
                        test_corrected[
                            "label"
                        ] == 0
                    ).sum()
                ),

            "anomaly_windows":
                int(
                    test_corrected[
                        "label"
                    ].sum()
                ),

            "anomaly_ratio":
                float(
                    test_corrected[
                        "label"
                    ].mean()
                ),
        },
    ])


    print(
        "\nCorrected label distribution:"
    )

    print(
        label_summary.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Add split labels to saved correction copies
    # --------------------------------------------------------

    train_save = (
        train_corrected.copy()
    )

    val_save = (
        val_corrected.copy()
    )

    test_save = (
        test_corrected.copy()
    )

    train_save[
        "split"
    ] = "train"

    val_save[
        "split"
    ] = "validation"

    test_save[
        "split"
    ] = "test"


    corrected_all = pd.concat(
        [
            train_save,
            val_save,
            test_save,
        ],
        ignore_index=True
    )


    # --------------------------------------------------------
    # Save corrected copies
    # --------------------------------------------------------

    train_path = (
        output_dir
        / "train_corrected.csv"
    )

    val_path = (
        output_dir
        / "validation_corrected.csv"
    )

    test_path = (
        output_dir
        / "test_corrected.csv"
    )

    all_path = (
        output_dir
        / "all_corrected_partitions.csv"
    )

    removed_path = (
        output_dir
        / "removed_boundary_windows.csv"
    )

    audit_path = (
        output_dir
        / "boundary_purge_audit.csv"
    )

    label_path = (
        output_dir
        / "corrected_split_label_summary.csv"
    )


    train_save.to_csv(
        train_path,
        index=False
    )

    val_save.to_csv(
        val_path,
        index=False
    )

    test_save.to_csv(
        test_path,
        index=False
    )

    corrected_all.to_csv(
        all_path,
        index=False
    )


    removed_all = pd.concat(
        [
            removed_val,
            removed_test,
        ],
        ignore_index=True,
        sort=False
    )

    removed_all.to_csv(
        removed_path,
        index=False
    )

    audit.to_csv(
        audit_path,
        index=False
    )

    label_summary.to_csv(
        label_path,
        index=False
    )


    print(
        "\nSaved corrected partition copies:"
    )

    print(train_path)
    print(val_path)
    print(test_path)
    print(all_path)

    print(
        "\nSaved removed windows:"
    )

    print(
        removed_path
    )

    print(
        "\nSaved purge audit:"
    )

    print(
        audit_path
    )


    print(
        "\nIMPORTANT: original window file "
        "was NOT modified."
    )


    return {
        "train":
            train_corrected,

        "validation":
            val_corrected,

        "test":
            test_corrected,

        "removed":
            removed_all,

        "audit":
            audit,

        "label_summary":
            label_summary,

        "corrected_all":
            corrected_all,
    }


# ============================================================
# NAB 60/30
# ============================================================

nab6030_corrected = (
    create_corrected_nab_partitions(

        source_file=(
            BASE
            / "processed_data"
            / "PH4"
            / "EXP_NAB_60min_stride30"
            / "nab_windows_60min_stride30.csv"
        ),

        config_name="60/30",

        output_dir=(
            NAB_6030_CORRECTION_DIR
        ),

        expected_original={
            "train":
                62096,

            "validation":
                20705,

            "test":
                20725,

            "total":
                103526,
        },

        expected_corrected={
            "train":
                62096,

            "validation":
                20648,

            "test":
                20668,

            "total":
                103412,
        },
    )
)


# ============================================================
# NAB 60/15
# ============================================================

nab6015_corrected = (
    create_corrected_nab_partitions(

        source_file=(
            BASE
            / "processed_data"
            / "PH4"
            / "EXP_NAB_60min_stride15"
            / "nab_windows_60min_stride15.csv"
        ),

        config_name="60/15",

        output_dir=(
            NAB_6015_CORRECTION_DIR
        ),

        expected_original={
            "train":
                124204,

            "validation":
                41405,

            "test":
                41424,

            "total":
                207033,
        },

        expected_corrected={
            "train":
                124204,

            "validation":
                41235,

            "test":
                41254,

            "total":
                206693,
        },
    )
)


# ============================================================
# Final correction checkpoint
# ============================================================

print("\n")
print("=" * 100)
print("CELL 18 FINAL CHECKPOINT")
print("=" * 100)

checkpoint = pd.DataFrame([
    {
        "configuration":
            "60/30",

        "train":
            len(
                nab6030_corrected[
                    "train"
                ]
            ),

        "validation":
            len(
                nab6030_corrected[
                    "validation"
                ]
            ),

        "test":
            len(
                nab6030_corrected[
                    "test"
                ]
            ),

        "removed":
            len(
                nab6030_corrected[
                    "removed"
                ]
            ),
    },

    {
        "configuration":
            "60/15",

        "train":
            len(
                nab6015_corrected[
                    "train"
                ]
            ),

        "validation":
            len(
                nab6015_corrected[
                    "validation"
                ]
            ),

        "test":
            len(
                nab6015_corrected[
                    "test"
                ]
            ),

        "removed":
            len(
                nab6015_corrected[
                    "removed"
                ]
            ),
    },
])


print(
    checkpoint.to_string(
        index=False
    )
)


CHECKPOINT_PATH = (
    NAB_CORRECTION_DIR
    / "nab_corrected_partition_checkpoint.csv"
)

checkpoint.to_csv(
    CHECKPOINT_PATH,
    index=False
)

print(
    "\nCheckpoint saved to:"
)

print(
    CHECKPOINT_PATH
)

print(
    "\nCELL 18 COMPLETE — "
    "corrected NAB partition copies created; "
    "no model training performed."
)

In [ ]:
# CELL 19 — NAB Reconstruction Environment and Historical-Reproduction Setup

import sys
import platform
import json
import numpy as np
import pandas as pd

import sklearn
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import tensorflow as tf


print("=" * 105)
print("NAB MODEL-RECONSTRUCTION SETUP")
print("=" * 105)


# ============================================================
# A. Runtime / library environment
# ============================================================

environment_info = {
    "python":
        sys.version.replace("\n", " "),

    "platform":
        platform.platform(),

    "numpy":
        np.__version__,

    "pandas":
        pd.__version__,

    "scikit_learn":
        sklearn.__version__,

    "xgboost":
        xgb.__version__,

    "tensorflow":
        tf.__version__,
}


print("\nRuntime environment:")

for key, value in environment_info.items():
    print(f"{key}: {value}")


# ============================================================
# B. Reproducibility seeds
# ============================================================

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)

tf.keras.utils.set_random_seed(
    RANDOM_STATE
)

try:
    tf.config.experimental.enable_op_determinism()

    print(
        "\nTensorFlow deterministic operations enabled."
    )

except Exception as exc:

    print(
        "\nTensorFlow deterministic-operation request "
        f"returned: {exc}"
    )


# ============================================================
# C. Exact original NAB split function
# ============================================================

def reconstruct_original_nab_split(
    df,
    train_ratio=0.60,
    val_ratio=0.20
):
    train_parts = []
    val_parts = []
    test_parts = []

    for series_key, g in df.groupby(
        "series"
    ):

        g = (
            g.sort_values(
                "window_start"
            )
            .reset_index(drop=True)
        )

        n = len(g)

        if n < 5:
            continue

        train_end = int(
            n * train_ratio
        )

        val_end = int(
            n * (
                train_ratio
                + val_ratio
            )
        )

        train_parts.append(
            g.iloc[
                :train_end
            ].copy()
        )

        val_parts.append(
            g.iloc[
                train_end:val_end
            ].copy()
        )

        test_parts.append(
            g.iloc[
                val_end:
            ].copy()
        )

    return (
        pd.concat(
            train_parts,
            ignore_index=True
        ),

        pd.concat(
            val_parts,
            ignore_index=True
        ),

        pd.concat(
            test_parts,
            ignore_index=True
        ),
    )


# ============================================================
# D. Configuration paths
# ============================================================

NAB_RECONSTRUCTION_CONFIGS = {

    "60/30": {
        "source":
            BASE
            / "processed_data"
            / "PH4"
            / "EXP_NAB_60min_stride30"
            / "nab_windows_60min_stride30.csv",

        "corrected_dir":
            NAB_6030_CORRECTION_DIR,

        "historical_metrics":
            BASE
            / "results"
            / "EXP_NAB_60min_stride30"
            / "metrics_nab_60min_stride30_all_models.csv",

        "expected_original":
            (62096, 20705, 20725),

        "expected_corrected":
            (62096, 20648, 20668),
    },

    "60/15": {
        "source":
            BASE
            / "processed_data"
            / "PH4"
            / "EXP_NAB_60min_stride15"
            / "nab_windows_60min_stride15.csv",

        "corrected_dir":
            NAB_6015_CORRECTION_DIR,

        "historical_metrics":
            BASE
            / "results"
            / "EXP_NAB_60min_stride15"
            / "metrics_nab_60min_stride15_all_models.csv",

        "expected_original":
            (124204, 41405, 41424),

        "expected_corrected":
            (124204, 41235, 41254),
    },
}


# ============================================================
# E. Build exact original + corrected matrices
# ============================================================

nab_reconstruction = {}

setup_rows = []


for config_name, cfg in (
    NAB_RECONSTRUCTION_CONFIGS.items()
):

    print("\n")
    print("=" * 105)
    print(f"CONFIGURATION {config_name}")
    print("=" * 105)


    # --------------------------------------------------------
    # Load original complete window table
    # --------------------------------------------------------

    original_windows = pd.read_csv(
        cfg["source"],
        low_memory=False
    )

    original_windows[
        "window_start"
    ] = pd.to_datetime(
        original_windows[
            "window_start"
        ],
        utc=True,
        errors="raise"
    )

    original_windows[
        "window_end"
    ] = pd.to_datetime(
        original_windows[
            "window_end"
        ],
        utc=True,
        errors="raise"
    )


    (
        original_train,
        original_val,
        original_test
    ) = reconstruct_original_nab_split(
        original_windows
    )


    assert (
        len(original_train),
        len(original_val),
        len(original_test),
    ) == cfg["expected_original"]


    # --------------------------------------------------------
    # Load corrected partition copies
    # --------------------------------------------------------

    corrected_train = pd.read_csv(
        cfg["corrected_dir"]
        / "train_corrected.csv",
        low_memory=False
    )

    corrected_val = pd.read_csv(
        cfg["corrected_dir"]
        / "validation_corrected.csv",
        low_memory=False
    )

    corrected_test = pd.read_csv(
        cfg["corrected_dir"]
        / "test_corrected.csv",
        low_memory=False
    )


    for df in [
        corrected_train,
        corrected_val,
        corrected_test,
    ]:

        df["window_start"] = pd.to_datetime(
            df["window_start"],
            utc=True,
            errors="raise"
        )

        df["window_end"] = pd.to_datetime(
            df["window_end"],
            utc=True,
            errors="raise"
        )


    assert (
        len(corrected_train),
        len(corrected_val),
        len(corrected_test),
    ) == cfg["expected_corrected"]


    # --------------------------------------------------------
    # Recover exact feature schema using original code rule
    # --------------------------------------------------------

    meta_cols = [
        "series",
        "domain",
        "window_start",
        "window_end",
        "label",
    ]

    feature_cols = [
        c
        for c in original_windows.columns
        if c not in meta_cols
    ]

    feature_cols = [
        c
        for c in feature_cols

        if (
            pd.api.types.is_numeric_dtype(
                original_windows[c]
            )
            and c != "label"
        )
    ]


    print(
        "\nRecovered feature count:",
        len(feature_cols)
    )

    print(
        "Recovered feature order:"
    )

    print(
        feature_cols
    )


    assert len(feature_cols) == 23, (
        f"{config_name}: expected 23 features, "
        f"found {len(feature_cols)}."
    )


    # --------------------------------------------------------
    # Exact training-row identity check
    # --------------------------------------------------------

    identity_cols = [
        "series",
        "window_start",
        "window_end",
        "label",
    ] + feature_cols


    pd.testing.assert_frame_equal(

        original_train[
            identity_cols
        ].reset_index(drop=True),

        corrected_train[
            identity_cols
        ].reset_index(drop=True),

        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=0,
    )


    print(
        "\nPASS: corrected training partition is "
        "identical to historical training partition."
    )


    # --------------------------------------------------------
    # Prepare original matrices exactly as original notebook
    # --------------------------------------------------------

    def feature_matrix(df):

        return (
            df[
                feature_cols
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .fillna(0)
        )


    X_train_original = (
        feature_matrix(
            original_train
        )
    )

    X_val_original = (
        feature_matrix(
            original_val
        )
    )

    X_test_original = (
        feature_matrix(
            original_test
        )
    )


    y_train_original = (
        original_train[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_val_original = (
        original_val[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_test_original = (
        original_test[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )


    # --------------------------------------------------------
    # Prepare corrected matrices
    # --------------------------------------------------------

    X_train_corrected = (
        feature_matrix(
            corrected_train
        )
    )

    X_val_corrected = (
        feature_matrix(
            corrected_val
        )
    )

    X_test_corrected = (
        feature_matrix(
            corrected_test
        )
    )


    y_train_corrected = (
        corrected_train[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_val_corrected = (
        corrected_val[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_test_corrected = (
        corrected_test[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )


    # --------------------------------------------------------
    # Fit historical training-only StandardScaler
    #
    # Since training is unchanged, this same scaler is valid
    # for both historical and corrected evaluations.
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = (
        scaler.fit_transform(
            X_train_original
        )
    )

    X_val_original_scaled = (
        scaler.transform(
            X_val_original
        )
    )

    X_test_original_scaled = (
        scaler.transform(
            X_test_original
        )
    )

    X_val_corrected_scaled = (
        scaler.transform(
            X_val_corrected
        )
    )

    X_test_corrected_scaled = (
        scaler.transform(
            X_test_corrected
        )
    )


    # Corrected train must give exactly same transformed matrix
    X_train_corrected_scaled = (
        scaler.transform(
            X_train_corrected
        )
    )


    np.testing.assert_allclose(
        X_train_scaled,
        X_train_corrected_scaled,
        rtol=0,
        atol=0
    )


    print(
        "PASS: training-only StandardScaler is "
        "unchanged by boundary correction."
    )


    # --------------------------------------------------------
    # Historical metrics for reproduction target
    # --------------------------------------------------------

    historical_metrics = pd.read_csv(
        cfg["historical_metrics"]
    )


    print(
        "\nHistorical metrics target:"
    )

    print(
        historical_metrics[
            [
                "model",
                "selected_percentile",
                "selected_threshold",
                "test_f1",
                "test_fpr",
                "test_roc_auc",
                "test_pr_auc",
            ]
        ].to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Store everything for next cells
    # --------------------------------------------------------

    nab_reconstruction[
        config_name
    ] = {
        "feature_cols":
            feature_cols,

        "scaler":
            scaler,

        "original": {
            "train_df":
                original_train,

            "val_df":
                original_val,

            "test_df":
                original_test,

            "X_train":
                X_train_scaled,

            "X_val":
                X_val_original_scaled,

            "X_test":
                X_test_original_scaled,

            "y_train":
                y_train_original,

            "y_val":
                y_val_original,

            "y_test":
                y_test_original,
        },

        "corrected": {
            "train_df":
                corrected_train,

            "val_df":
                corrected_val,

            "test_df":
                corrected_test,

            "X_train":
                X_train_corrected_scaled,

            "X_val":
                X_val_corrected_scaled,

            "X_test":
                X_test_corrected_scaled,

            "y_train":
                y_train_corrected,

            "y_val":
                y_val_corrected,

            "y_test":
                y_test_corrected,
        },

        "historical_metrics":
            historical_metrics,
    }


    # --------------------------------------------------------
    # Setup manifest row
    # --------------------------------------------------------

    setup_rows.append({
        "configuration":
            config_name,

        "feature_count":
            len(feature_cols),

        "original_train":
            len(original_train),

        "original_validation":
            len(original_val),

        "original_test":
            len(original_test),

        "corrected_train":
            len(corrected_train),

        "corrected_validation":
            len(corrected_val),

        "corrected_test":
            len(corrected_test),

        "original_val_anomalies":
            int(
                y_val_original.sum()
            ),

        "corrected_val_anomalies":
            int(
                y_val_corrected.sum()
            ),

        "original_test_anomalies":
            int(
                y_test_original.sum()
            ),

        "corrected_test_anomalies":
            int(
                y_test_corrected.sum()
            ),
    })


# ============================================================
# F. Cross-configuration feature consistency
# ============================================================

features_6030 = (
    nab_reconstruction[
        "60/30"
    ][
        "feature_cols"
    ]
)

features_6015 = (
    nab_reconstruction[
        "60/15"
    ][
        "feature_cols"
    ]
)


assert (
    features_6030
    == features_6015
), (
    "60/30 and 60/15 feature order differs."
)


print("\n")
print("=" * 105)
print("CROSS-CONFIGURATION CHECK")
print("=" * 105)

print(
    "\nPASS: 60/30 and 60/15 use the same "
    "23 features in the same order."
)


# ============================================================
# G. Save reconstruction setup manifests
# ============================================================

setup_manifest = pd.DataFrame(
    setup_rows
)

print(
    "\nReconstruction setup summary:"
)

print(
    setup_manifest.to_string(
        index=False
    )
)


SETUP_MANIFEST_PATH = (
    NAB_CORRECTION_DIR
    / "nab_model_reconstruction_setup.csv"
)

setup_manifest.to_csv(
    SETUP_MANIFEST_PATH,
    index=False
)


ENVIRONMENT_PATH = (
    NAB_CORRECTION_DIR
    / "nab_reconstruction_environment.json"
)

with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        environment_info,
        f,
        indent=2
    )


print(
    "\nSetup manifest saved to:"
)

print(
    SETUP_MANIFEST_PATH
)

print(
    "\nEnvironment manifest saved to:"
)

print(
    ENVIRONMENT_PATH
)


print("\n")
print("=" * 105)
print("CELL 19 COMPLETE")
print("=" * 105)

print(
    "\nNo model has been trained yet. "
    "Historical and corrected matrices are now "
    "prepared from the same unchanged training data."
)

In [ ]:
# CELL 19 — NAB Reconstruction Environment and Numerically Robust Historical-Reproduction Setup

import sys
import platform
import json
import numpy as np
import pandas as pd

import sklearn
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import tensorflow as tf


print("=" * 105)
print("NAB MODEL-RECONSTRUCTION SETUP")
print("=" * 105)


# ============================================================
# A. Runtime / library environment
# ============================================================

environment_info = {
    "python":
        sys.version.replace("\n", " "),

    "platform":
        platform.platform(),

    "numpy":
        np.__version__,

    "pandas":
        pd.__version__,

    "scikit_learn":
        sklearn.__version__,

    "xgboost":
        xgb.__version__,

    "tensorflow":
        tf.__version__,
}


print("\nRuntime environment:")

for key, value in environment_info.items():
    print(f"{key}: {value}")


# ============================================================
# B. Reproducibility seeds
# ============================================================

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)

tf.keras.utils.set_random_seed(
    RANDOM_STATE
)

try:
    tf.config.experimental.enable_op_determinism()

    print(
        "\nTensorFlow deterministic operations enabled."
    )

except Exception as exc:

    print(
        "\nTensorFlow deterministic-operation request "
        f"returned: {exc}"
    )


# ============================================================
# C. Exact original NAB split function
# ============================================================

def reconstruct_original_nab_split(
    df,
    train_ratio=0.60,
    val_ratio=0.20
):

    train_parts = []
    val_parts = []
    test_parts = []

    for series_key, g in df.groupby(
        "series"
    ):

        g = (
            g.sort_values(
                "window_start"
            )
            .reset_index(drop=True)
        )

        n = len(g)

        if n < 5:
            continue

        train_end = int(
            n * train_ratio
        )

        val_end = int(
            n * (
                train_ratio
                + val_ratio
            )
        )

        train_parts.append(
            g.iloc[
                :train_end
            ].copy()
        )

        val_parts.append(
            g.iloc[
                train_end:val_end
            ].copy()
        )

        test_parts.append(
            g.iloc[
                val_end:
            ].copy()
        )

    return (
        pd.concat(
            train_parts,
            ignore_index=True
        ),

        pd.concat(
            val_parts,
            ignore_index=True
        ),

        pd.concat(
            test_parts,
            ignore_index=True
        ),
    )


# ============================================================
# D. Configuration paths
# ============================================================

NAB_RECONSTRUCTION_CONFIGS = {

    "60/30": {
        "source":
            BASE
            / "processed_data"
            / "PH4"
            / "EXP_NAB_60min_stride30"
            / "nab_windows_60min_stride30.csv",

        "corrected_dir":
            NAB_6030_CORRECTION_DIR,

        "historical_metrics":
            BASE
            / "results"
            / "EXP_NAB_60min_stride30"
            / "metrics_nab_60min_stride30_all_models.csv",

        "expected_original":
            (62096, 20705, 20725),

        "expected_corrected":
            (62096, 20648, 20668),
    },

    "60/15": {
        "source":
            BASE
            / "processed_data"
            / "PH4"
            / "EXP_NAB_60min_stride15"
            / "nab_windows_60min_stride15.csv",

        "corrected_dir":
            NAB_6015_CORRECTION_DIR,

        "historical_metrics":
            BASE
            / "results"
            / "EXP_NAB_60min_stride15"
            / "metrics_nab_60min_stride15_all_models.csv",

        "expected_original":
            (124204, 41405, 41424),

        "expected_corrected":
            (124204, 41235, 41254),
    },
}


# ============================================================
# E. Numerical tolerance
#
# Corrected partitions were written to CSV in CELL 18 and are
# read back here. Decimal-to-binary round-tripping can produce
# differences around machine precision (e.g. ~1e-15).
#
# Structural identifiers must still match exactly.
# ============================================================

NUMERIC_RTOL = 1e-12
NUMERIC_ATOL = 1e-12


# ============================================================
# F. Build exact original + corrected matrices
# ============================================================

nab_reconstruction = {}

setup_rows = []

numeric_identity_rows = []


for config_name, cfg in (
    NAB_RECONSTRUCTION_CONFIGS.items()
):

    print("\n")
    print("=" * 105)
    print(f"CONFIGURATION {config_name}")
    print("=" * 105)


    # --------------------------------------------------------
    # Load original complete window table
    # --------------------------------------------------------

    original_windows = pd.read_csv(
        cfg["source"],
        low_memory=False
    )

    original_windows[
        "window_start"
    ] = pd.to_datetime(
        original_windows[
            "window_start"
        ],
        utc=True,
        errors="raise"
    )

    original_windows[
        "window_end"
    ] = pd.to_datetime(
        original_windows[
            "window_end"
        ],
        utc=True,
        errors="raise"
    )


    (
        original_train,
        original_val,
        original_test
    ) = reconstruct_original_nab_split(
        original_windows
    )


    assert (
        len(original_train),
        len(original_val),
        len(original_test),
    ) == cfg["expected_original"]


    # --------------------------------------------------------
    # Load corrected partition copies
    # --------------------------------------------------------

    corrected_train = pd.read_csv(
        cfg["corrected_dir"]
        / "train_corrected.csv",
        low_memory=False
    )

    corrected_val = pd.read_csv(
        cfg["corrected_dir"]
        / "validation_corrected.csv",
        low_memory=False
    )

    corrected_test = pd.read_csv(
        cfg["corrected_dir"]
        / "test_corrected.csv",
        low_memory=False
    )


    for df in [
        corrected_train,
        corrected_val,
        corrected_test,
    ]:

        df["window_start"] = pd.to_datetime(
            df["window_start"],
            utc=True,
            errors="raise"
        )

        df["window_end"] = pd.to_datetime(
            df["window_end"],
            utc=True,
            errors="raise"
        )


    assert (
        len(corrected_train),
        len(corrected_val),
        len(corrected_test),
    ) == cfg["expected_corrected"]


    # --------------------------------------------------------
    # Recover exact historical feature schema
    # --------------------------------------------------------

    meta_cols = [
        "series",
        "domain",
        "window_start",
        "window_end",
        "label",
    ]

    feature_cols = [
        c
        for c in original_windows.columns
        if c not in meta_cols
    ]

    feature_cols = [
        c
        for c in feature_cols
        if (
            pd.api.types.is_numeric_dtype(
                original_windows[c]
            )
            and c != "label"
        )
    ]


    print(
        "\nRecovered feature count:",
        len(feature_cols)
    )

    print(
        "Recovered feature order:"
    )

    print(
        feature_cols
    )


    assert len(feature_cols) == 23, (
        f"{config_name}: expected 23 features, "
        f"found {len(feature_cols)}."
    )


    # ========================================================
    # G. Structural training-row identity
    # Must be exact.
    # ========================================================

    structural_cols = [
        "series",
        "domain",
        "window_start",
        "window_end",
        "label",
    ]


    pd.testing.assert_frame_equal(

        original_train[
            structural_cols
        ].reset_index(drop=True),

        corrected_train[
            structural_cols
        ].reset_index(drop=True),

        check_dtype=False,
        check_exact=True
    )


    print(
        "\nPASS: training series, timestamps, domains, "
        "labels, and row order are exactly unchanged."
    )


    # ========================================================
    # H. Numerical training-feature identity
    # Compare with machine-precision tolerance because one
    # copy underwent CSV serialization.
    # ========================================================

    original_feature_np = (
        original_train[
            feature_cols
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    corrected_feature_np = (
        corrected_train[
            feature_cols
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    finite_pair_mask = (
        np.isfinite(
            original_feature_np
        )
        &
        np.isfinite(
            corrected_feature_np
        )
    )


    abs_diff = np.zeros_like(
        original_feature_np,
        dtype=np.float64
    )

    abs_diff[
        finite_pair_mask
    ] = np.abs(
        original_feature_np[
            finite_pair_mask
        ]
        -
        corrected_feature_np[
            finite_pair_mask
        ]
    )


    max_abs_diff = float(
        np.max(
            abs_diff
        )
    )


    exact_numeric_cells = (
        original_feature_np
        == corrected_feature_np
    )

    exact_numeric_cells |= (
        np.isnan(
            original_feature_np
        )
        &
        np.isnan(
            corrected_feature_np
        )
    )


    nonexact_numeric_cells = int(
        np.size(
            exact_numeric_cells
        )
        -
        np.sum(
            exact_numeric_cells
        )
    )


    total_numeric_cells = int(
        exact_numeric_cells.size
    )


    nonexact_percent = (
        100
        * nonexact_numeric_cells
        / total_numeric_cells
    )


    print(
        "\nNumeric CSV round-trip diagnostic:"
    )

    print(
        "Total numeric feature cells:",
        total_numeric_cells
    )

    print(
        "Cells not bit-for-bit identical:",
        nonexact_numeric_cells
    )

    print(
        "Nonexact cell percentage:",
        f"{nonexact_percent:.8f}%"
    )

    print(
        "Maximum absolute feature difference:",
        f"{max_abs_diff:.18e}"
    )


    np.testing.assert_allclose(
        original_feature_np,
        corrected_feature_np,
        rtol=NUMERIC_RTOL,
        atol=NUMERIC_ATOL,
        equal_nan=True
    )


    print(
        "PASS: all training feature values are "
        "numerically identical within 1e-12 tolerance."
    )


    numeric_identity_rows.append({
        "configuration":
            config_name,

        "numeric_cells":
            total_numeric_cells,

        "nonexact_binary_cells":
            nonexact_numeric_cells,

        "nonexact_binary_percent":
            nonexact_percent,

        "max_absolute_difference":
            max_abs_diff,

        "rtol":
            NUMERIC_RTOL,

        "atol":
            NUMERIC_ATOL,
    })


    # ========================================================
    # I. Prepare feature matrices exactly as original notebook
    # ========================================================

    def feature_matrix(df):

        return (
            df[
                feature_cols
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .fillna(0)
        )


    X_train_original = (
        feature_matrix(
            original_train
        )
    )

    X_val_original = (
        feature_matrix(
            original_val
        )
    )

    X_test_original = (
        feature_matrix(
            original_test
        )
    )


    y_train_original = (
        original_train[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_val_original = (
        original_val[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_test_original = (
        original_test[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )


    # --------------------------------------------------------
    # Corrected matrices
    # --------------------------------------------------------

    X_train_corrected = (
        feature_matrix(
            corrected_train
        )
    )

    X_val_corrected = (
        feature_matrix(
            corrected_val
        )
    )

    X_test_corrected = (
        feature_matrix(
            corrected_test
        )
    )


    y_train_corrected = (
        corrected_train[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_val_corrected = (
        corrected_val[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    y_test_corrected = (
        corrected_test[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )


    # ========================================================
    # J. Historical training-only StandardScaler
    # ========================================================

    scaler = StandardScaler()

    X_train_scaled = (
        scaler.fit_transform(
            X_train_original
        )
    )

    X_val_original_scaled = (
        scaler.transform(
            X_val_original
        )
    )

    X_test_original_scaled = (
        scaler.transform(
            X_test_original
        )
    )

    X_val_corrected_scaled = (
        scaler.transform(
            X_val_corrected
        )
    )

    X_test_corrected_scaled = (
        scaler.transform(
            X_test_corrected
        )
    )

    X_train_corrected_scaled = (
        scaler.transform(
            X_train_corrected
        )
    )


    scaled_max_abs_diff = float(
        np.max(
            np.abs(
                X_train_scaled
                -
                X_train_corrected_scaled
            )
        )
    )


    print(
        "\nMaximum absolute difference after "
        "training-only scaling:",
        f"{scaled_max_abs_diff:.18e}"
    )


    np.testing.assert_allclose(
        X_train_scaled,
        X_train_corrected_scaled,
        rtol=NUMERIC_RTOL,
        atol=NUMERIC_ATOL
    )


    print(
        "PASS: training-only StandardScaler output "
        "is numerically unchanged."
    )


    # ========================================================
    # K. Historical metrics for reproduction target
    # ========================================================

    historical_metrics = pd.read_csv(
        cfg["historical_metrics"]
    )


    print(
        "\nHistorical metrics target:"
    )

    print(
        historical_metrics[
            [
                "model",
                "selected_percentile",
                "selected_threshold",
                "test_f1",
                "test_fpr",
                "test_roc_auc",
                "test_pr_auc",
            ]
        ].to_string(
            index=False
        )
    )


    # ========================================================
    # L. Store matrices for following cells
    # ========================================================

    nab_reconstruction[
        config_name
    ] = {
        "feature_cols":
            feature_cols,

        "scaler":
            scaler,

        "original": {
            "train_df":
                original_train,

            "val_df":
                original_val,

            "test_df":
                original_test,

            "X_train":
                X_train_scaled,

            "X_val":
                X_val_original_scaled,

            "X_test":
                X_test_original_scaled,

            "y_train":
                y_train_original,

            "y_val":
                y_val_original,

            "y_test":
                y_test_original,
        },

        "corrected": {
            "train_df":
                corrected_train,

            "val_df":
                corrected_val,

            "test_df":
                corrected_test,

            "X_train":
                X_train_corrected_scaled,

            "X_val":
                X_val_corrected_scaled,

            "X_test":
                X_test_corrected_scaled,

            "y_train":
                y_train_corrected,

            "y_val":
                y_val_corrected,

            "y_test":
                y_test_corrected,
        },

        "historical_metrics":
            historical_metrics,
    }


    # ========================================================
    # M. Setup manifest row
    # ========================================================

    setup_rows.append({
        "configuration":
            config_name,

        "feature_count":
            len(feature_cols),

        "original_train":
            len(original_train),

        "original_validation":
            len(original_val),

        "original_test":
            len(original_test),

        "corrected_train":
            len(corrected_train),

        "corrected_validation":
            len(corrected_val),

        "corrected_test":
            len(corrected_test),

        "original_val_anomalies":
            int(
                y_val_original.sum()
            ),

        "corrected_val_anomalies":
            int(
                y_val_corrected.sum()
            ),

        "original_test_anomalies":
            int(
                y_test_original.sum()
            ),

        "corrected_test_anomalies":
            int(
                y_test_corrected.sum()
            ),
    })


# ============================================================
# N. Cross-configuration feature consistency
# ============================================================

features_6030 = (
    nab_reconstruction[
        "60/30"
    ][
        "feature_cols"
    ]
)

features_6015 = (
    nab_reconstruction[
        "60/15"
    ][
        "feature_cols"
    ]
)


assert (
    features_6030
    == features_6015
), (
    "60/30 and 60/15 feature order differs."
)


print("\n")
print("=" * 105)
print("CROSS-CONFIGURATION CHECK")
print("=" * 105)

print(
    "\nPASS: 60/30 and 60/15 use the same "
    "23 features in the same order."
)


# ============================================================
# O. Save reconstruction manifests
# ============================================================

setup_manifest = pd.DataFrame(
    setup_rows
)

numeric_identity_manifest = pd.DataFrame(
    numeric_identity_rows
)


print(
    "\nReconstruction setup summary:"
)

print(
    setup_manifest.to_string(
        index=False
    )
)


print(
    "\nNumerical identity summary:"
)

print(
    numeric_identity_manifest.to_string(
        index=False
    )
)


SETUP_MANIFEST_PATH = (
    NAB_CORRECTION_DIR
    / "nab_model_reconstruction_setup.csv"
)

setup_manifest.to_csv(
    SETUP_MANIFEST_PATH,
    index=False
)


NUMERIC_IDENTITY_PATH = (
    NAB_CORRECTION_DIR
    / "nab_training_numeric_identity_audit.csv"
)

numeric_identity_manifest.to_csv(
    NUMERIC_IDENTITY_PATH,
    index=False
)


ENVIRONMENT_PATH = (
    NAB_CORRECTION_DIR
    / "nab_reconstruction_environment.json"
)

with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        environment_info,
        f,
        indent=2
    )


print(
    "\nSetup manifest saved to:"
)

print(
    SETUP_MANIFEST_PATH
)

print(
    "\nNumerical identity audit saved to:"
)

print(
    NUMERIC_IDENTITY_PATH
)

print(
    "\nEnvironment manifest saved to:"
)

print(
    ENVIRONMENT_PATH
)


print("\n")
print("=" * 105)
print("CELL 19 COMPLETE")
print("=" * 105)

print(
    "\nNo model has been trained yet. "
    "Historical and corrected matrices are now "
    "prepared from the same unchanged training population."
)

In [ ]:
# CELL 20 — NAB Raw-Feature Round-Trip Precision and Independent-Scaler Audit

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("=" * 110)
print("NAB RAW-FEATURE ROUND-TRIP PRECISION AUDIT")
print("=" * 110)

precision_rows = []
scaler_rows = []


for config_name in ["60/30", "60/15"]:

    print("\n")
    print("=" * 110)
    print(f"CONFIGURATION {config_name}")
    print("=" * 110)

    bundle = nab_reconstruction[config_name]

    feature_cols = bundle["feature_cols"]

    original_train = (
        bundle["original"]["train_df"]
        .reset_index(drop=True)
    )

    corrected_train = (
        bundle["corrected"]["train_df"]
        .reset_index(drop=True)
    )


    # ========================================================
    # A. Raw numerical matrices
    # ========================================================

    A = (
        original_train[feature_cols]
        .to_numpy(dtype=np.float64)
    )

    B = (
        corrected_train[feature_cols]
        .to_numpy(dtype=np.float64)
    )

    assert A.shape == B.shape


    # ========================================================
    # B. Feature-by-feature precision diagnostics
    # ========================================================

    feature_diagnostics = []

    for j, feature in enumerate(feature_cols):

        a = A[:, j]
        b = B[:, j]

        finite = (
            np.isfinite(a)
            & np.isfinite(b)
        )

        a_f = a[finite]
        b_f = b[finite]

        abs_diff = np.abs(
            a_f - b_f
        )

        # Relative difference with a safe denominator.
        denominator = np.maximum(
            np.maximum(
                np.abs(a_f),
                np.abs(b_f)
            ),
            1.0
        )

        rel_diff = (
            abs_diff
            / denominator
        )

        nonexact = (
            a_f != b_f
        )

        max_abs = (
            float(abs_diff.max())
            if len(abs_diff)
            else 0.0
        )

        max_rel = (
            float(rel_diff.max())
            if len(rel_diff)
            else 0.0
        )

        nonexact_count = int(
            nonexact.sum()
        )

        # Locate the row with the largest absolute difference
        if len(abs_diff):

            local_idx = int(
                np.argmax(abs_diff)
            )

            finite_indices = np.flatnonzero(
                finite
            )

            row_idx = int(
                finite_indices[
                    local_idx
                ]
            )

            original_value = float(
                a[row_idx]
            )

            corrected_value = float(
                b[row_idx]
            )

            series = original_train.loc[
                row_idx,
                "series"
            ]

            window_start = original_train.loc[
                row_idx,
                "window_start"
            ]

        else:

            row_idx = None
            original_value = np.nan
            corrected_value = np.nan
            series = None
            window_start = None


        feature_diagnostics.append({
            "configuration":
                config_name,

            "feature":
                feature,

            "finite_cells":
                int(finite.sum()),

            "nonexact_cells":
                nonexact_count,

            "max_absolute_difference":
                max_abs,

            "max_relative_difference":
                max_rel,

            "row_of_max_abs_difference":
                row_idx,

            "series_of_max_abs_difference":
                series,

            "window_start_of_max_abs_difference":
                window_start,

            "original_value":
                original_value,

            "corrected_value":
                corrected_value,
        })


    diag_df = pd.DataFrame(
        feature_diagnostics
    )

    precision_rows.extend(
        feature_diagnostics
    )


    print(
        "\nFeatures sorted by maximum absolute difference:"
    )

    print(
        diag_df[
            [
                "feature",
                "nonexact_cells",
                "max_absolute_difference",
                "max_relative_difference",
                "original_value",
                "corrected_value",
            ]
        ]
        .sort_values(
            "max_absolute_difference",
            ascending=False
        )
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.16e}"
        )
    )


    global_max_relative = float(
        diag_df[
            "max_relative_difference"
        ].max()
    )

    print(
        "\nMaximum relative difference across "
        "all 23 features:"
    )

    print(
        f"{global_max_relative:.18e}"
    )


    # Strong relative-precision requirement
    assert (
        global_max_relative
        <= 1e-12
    ), (
        f"{config_name}: raw CSV round-trip "
        "relative difference exceeded 1e-12."
    )

    print(
        "PASS: all raw feature discrepancies are "
        "within 1e-12 relative precision."
    )


    # ========================================================
    # C. Independently fit a scaler to EACH copy
    #
    # This is stronger than using the same scaler for both:
    # it verifies that CSV round-trip precision does not
    # materially alter learned training preprocessing.
    # ========================================================

    def clean_features(df):

        return (
            df[feature_cols]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .fillna(0)
            .to_numpy(
                dtype=np.float64
            )
        )


    X_original = clean_features(
        original_train
    )

    X_corrected = clean_features(
        corrected_train
    )


    scaler_original = (
        StandardScaler()
        .fit(X_original)
    )

    scaler_corrected = (
        StandardScaler()
        .fit(X_corrected)
    )


    mean_abs_diff = np.abs(
        scaler_original.mean_
        - scaler_corrected.mean_
    )

    scale_abs_diff = np.abs(
        scaler_original.scale_
        - scaler_corrected.scale_
    )


    mean_rel_diff = (
        mean_abs_diff
        / np.maximum(
            np.maximum(
                np.abs(
                    scaler_original.mean_
                ),
                np.abs(
                    scaler_corrected.mean_
                )
            ),
            1.0
        )
    )

    scale_rel_diff = (
        scale_abs_diff
        / np.maximum(
            np.maximum(
                np.abs(
                    scaler_original.scale_
                ),
                np.abs(
                    scaler_corrected.scale_
                )
            ),
            1.0
        )
    )


    max_mean_abs = float(
        mean_abs_diff.max()
    )

    max_mean_rel = float(
        mean_rel_diff.max()
    )

    max_scale_abs = float(
        scale_abs_diff.max()
    )

    max_scale_rel = float(
        scale_rel_diff.max()
    )


    # Transform each copy using the scaler fitted
    # independently on that copy.
    Z_original = (
        scaler_original
        .transform(X_original)
    )

    Z_corrected = (
        scaler_corrected
        .transform(X_corrected)
    )


    transformed_abs_diff = np.abs(
        Z_original
        - Z_corrected
    )

    transformed_max_abs = float(
        transformed_abs_diff.max()
    )


    print(
        "\nIndependent StandardScaler comparison:"
    )

    print(
        "Maximum scaler mean absolute difference:",
        f"{max_mean_abs:.18e}"
    )

    print(
        "Maximum scaler mean relative difference:",
        f"{max_mean_rel:.18e}"
    )

    print(
        "Maximum scaler scale absolute difference:",
        f"{max_scale_abs:.18e}"
    )

    print(
        "Maximum scaler scale relative difference:",
        f"{max_scale_rel:.18e}"
    )

    print(
        "Maximum transformed training-matrix difference:",
        f"{transformed_max_abs:.18e}"
    )


    assert max_mean_rel <= 1e-12

    assert max_scale_rel <= 1e-12

    np.testing.assert_allclose(
        Z_original,
        Z_corrected,
        rtol=1e-12,
        atol=1e-12
    )


    print(
        "PASS: independently fitted training "
        "StandardScalers are numerically equivalent."
    )


    scaler_rows.append({
        "configuration":
            config_name,

        "max_scaler_mean_abs_difference":
            max_mean_abs,

        "max_scaler_mean_relative_difference":
            max_mean_rel,

        "max_scaler_scale_abs_difference":
            max_scale_abs,

        "max_scaler_scale_relative_difference":
            max_scale_rel,

        "max_independently_scaled_matrix_difference":
            transformed_max_abs,
    })


# ============================================================
# D. Save diagnostic artifacts
# ============================================================

precision_df = pd.DataFrame(
    precision_rows
)

scaler_audit_df = pd.DataFrame(
    scaler_rows
)


PRECISION_OUT = (
    NAB_CORRECTION_DIR
    / "nab_raw_feature_roundtrip_precision_audit.csv"
)

SCALER_OUT = (
    NAB_CORRECTION_DIR
    / "nab_independent_scaler_equivalence_audit.csv"
)


precision_df.to_csv(
    PRECISION_OUT,
    index=False
)

scaler_audit_df.to_csv(
    SCALER_OUT,
    index=False
)


print("\n")
print("=" * 110)
print("FINAL PRECISION CHECKPOINT")
print("=" * 110)

print(
    scaler_audit_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.16e}"
    )
)


print(
    "\nRaw precision audit saved to:"
)

print(
    PRECISION_OUT
)

print(
    "\nScaler equivalence audit saved to:"
)

print(
    SCALER_OUT
)


print("\n")
print("=" * 110)
print("CELL 20 COMPLETE")
print("=" * 110)

print(
    "\nNo model was trained. "
    "This cell only quantified CSV round-trip precision "
    "and independently verified preprocessing equivalence."
)

In [ ]:
# CELL 21 — NAB Five Non-Neural Model Historical Reproduction Gate

import time
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import (
    IsolationForest,
    RandomForestClassifier,
)

from sklearn.neighbors import (
    LocalOutlierFactor,
)

from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

from xgboost import (
    XGBClassifier,
)


print("=" * 110)
print("NAB FIVE NON-NEURAL MODEL HISTORICAL REPRODUCTION GATE")
print("=" * 110)


# ============================================================
# A. Exact historical threshold grid
# ============================================================

PERCENTILE_GRID = [
    80,
    85,
    90,
    92,
    94,
    95,
    96,
    97,
    98,
    99,
    99.2,
    99.5,
    99.7,
    99.9,
]


# ============================================================
# B. Exact historical metric functions
# ============================================================

def compute_metrics_reproduction(
    y_true,
    y_pred,
    scores=None
):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    roc_auc = np.nan
    pr_auc = np.nan

    if (
        scores is not None
        and len(
            np.unique(y_true)
        ) > 1
    ):

        try:
            roc_auc = roc_auc_score(
                y_true,
                scores,
            )
        except Exception:
            pass

        try:
            pr_auc = average_precision_score(
                y_true,
                scores,
            )
        except Exception:
            pass


    return {
        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "accuracy":
            accuracy,

        "fpr":
            fpr,

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),

        "roc_auc":
            roc_auc,

        "pr_auc":
            pr_auc,
    }


def select_threshold_by_val_f1_reproduction(
    y_val,
    val_scores
):

    best = None

    for percentile in PERCENTILE_GRID:

        threshold = np.percentile(
            val_scores,
            percentile,
        )

        val_pred = (
            val_scores
            >= threshold
        ).astype(int)

        metrics = (
            compute_metrics_reproduction(
                y_val,
                val_pred,
                val_scores,
            )
        )

        candidate = {
            "percentile":
                percentile,

            "threshold":
                float(threshold),

            **metrics,
        }

        # Exact historical tie behavior:
        # first percentile wins unless later F1 is larger.
        if (
            best is None
            or candidate["f1"]
            > best["f1"]
        ):
            best = candidate

    return best


def evaluate_reproduction(
    model_name,
    y_val,
    y_test,
    val_scores,
    test_scores
):

    best = (
        select_threshold_by_val_f1_reproduction(
            y_val,
            val_scores,
        )
    )

    test_pred = (
        test_scores
        >= best["threshold"]
    ).astype(int)

    test_metrics = (
        compute_metrics_reproduction(
            y_test,
            test_pred,
            test_scores,
        )
    )


    return {
        "model":
            model_name,

        "selected_percentile":
            best["percentile"],

        "selected_threshold":
            best["threshold"],

        "val_precision":
            best["precision"],

        "val_recall":
            best["recall"],

        "val_f1":
            best["f1"],

        "test_precision":
            test_metrics["precision"],

        "test_recall":
            test_metrics["recall"],

        "test_f1":
            test_metrics["f1"],

        "test_accuracy":
            test_metrics["accuracy"],

        "test_fpr":
            test_metrics["fpr"],

        "test_roc_auc":
            test_metrics["roc_auc"],

        "test_pr_auc":
            test_metrics["pr_auc"],

        "tn":
            test_metrics["tn"],

        "fp":
            test_metrics["fp"],

        "fn":
            test_metrics["fn"],

        "tp":
            test_metrics["tp"],
    }


# ============================================================
# C. Model constructor
# ============================================================

def build_non_neural_models(
    y_train
):

    negative_count = int(
        (
            y_train == 0
        ).sum()
    )

    positive_count = int(
        (
            y_train == 1
        ).sum()
    )

    scale_pos_weight = (
        negative_count
        / max(
            positive_count,
            1,
        )
    )


    models = {
        "Isolation Forest":
            IsolationForest(
                n_estimators=500,
                max_samples="auto",
                contamination="auto",
                random_state=42,
                n_jobs=-1,
            ),

        "LOF":
            LocalOutlierFactor(
                n_neighbors=35,
                novelty=True,
                contamination="auto",
            ),

        "Random Forest":
            RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            ),

        "XGBoost":
            XGBClassifier(
                n_estimators=500,
                max_depth=5,
                learning_rate=0.03,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
            ),

        "Logistic Regression":
            LogisticRegression(
                class_weight="balanced",
                solver="liblinear",
                max_iter=5000,
                random_state=42,
            ),
    }

    return (
        models,
        scale_pos_weight,
    )


# ============================================================
# D. Scoring helper
# ============================================================

def get_scores(
    model_name,
    model,
    X
):

    if model_name == "Isolation Forest":

        return (
            -model.decision_function(X)
        )

    if model_name == "LOF":

        return (
            -model.decision_function(X)
        )

    if model_name in [
        "Random Forest",
        "XGBoost",
        "Logistic Regression",
    ]:

        return (
            model.predict_proba(X)[:, 1]
        )

    raise RuntimeError(
        f"Unknown model: {model_name}"
    )


# ============================================================
# E. Reproduce both configurations
# ============================================================

reproduction_rows = []
comparison_rows = []

nab_reconstructed_non_neural_models = {}


for config_name in [
    "60/30",
    "60/15",
]:

    print("\n")
    print("=" * 110)
    print(
        f"HISTORICAL REPRODUCTION — {config_name}"
    )
    print("=" * 110)


    bundle = (
        nab_reconstruction[
            config_name
        ]
    )

    original = (
        bundle["original"]
    )

    historical = (
        bundle[
            "historical_metrics"
        ]
        .copy()
    )


    X_train = (
        original["X_train"]
    )

    X_val = (
        original["X_val"]
    )

    X_test = (
        original["X_test"]
    )

    y_train = (
        original["y_train"]
    )

    y_val = (
        original["y_val"]
    )

    y_test = (
        original["y_test"]
    )


    models_dict, scale_pos_weight = (
        build_non_neural_models(
            y_train
        )
    )


    print(
        "\nscale_pos_weight:",
        f"{scale_pos_weight:.12f}"
    )


    config_models = {}


    for model_name, model in (
        models_dict.items()
    ):

        print(
            f"\nTraining {model_name}..."
        )

        start = time.perf_counter()

        model.fit(
            X_train,
            y_train
            if model_name in [
                "Random Forest",
                "XGBoost",
                "Logistic Regression",
            ]
            else None
        ) if model_name in [
            "Random Forest",
            "XGBoost",
            "Logistic Regression",
        ] else model.fit(
            X_train
        )

        training_seconds = (
            time.perf_counter()
            - start
        )


        val_scores = get_scores(
            model_name,
            model,
            X_val,
        )

        test_scores = get_scores(
            model_name,
            model,
            X_test,
        )


        result = (
            evaluate_reproduction(
                model_name,
                y_val,
                y_test,
                val_scores,
                test_scores,
            )
        )


        result[
            "configuration"
        ] = config_name

        result[
            "training_seconds_reconstruction"
        ] = training_seconds


        reproduction_rows.append(
            result
        )

        config_models[
            model_name
        ] = {
            "model":
                model,

            "val_scores_original":
                val_scores,

            "test_scores_original":
                test_scores,
        }


        # ----------------------------------------------------
        # Compare to frozen historical row
        # ----------------------------------------------------

        hist_row = historical[
            historical["model"]
            == model_name
        ]

        assert len(hist_row) == 1, (
            f"{config_name} / {model_name}: "
            "historical row missing or duplicated."
        )

        hist_row = (
            hist_row.iloc[0]
        )


        compare = {
            "configuration":
                config_name,

            "model":
                model_name,

            "historical_percentile":
                float(
                    hist_row[
                        "selected_percentile"
                    ]
                ),

            "reproduced_percentile":
                float(
                    result[
                        "selected_percentile"
                    ]
                ),

            "historical_threshold":
                float(
                    hist_row[
                        "selected_threshold"
                    ]
                ),

            "reproduced_threshold":
                float(
                    result[
                        "selected_threshold"
                    ]
                ),

            "historical_test_f1":
                float(
                    hist_row[
                        "test_f1"
                    ]
                ),

            "reproduced_test_f1":
                float(
                    result[
                        "test_f1"
                    ]
                ),

            "historical_test_fpr":
                float(
                    hist_row[
                        "test_fpr"
                    ]
                ),

            "reproduced_test_fpr":
                float(
                    result[
                        "test_fpr"
                    ]
                ),

            "historical_roc_auc":
                float(
                    hist_row[
                        "test_roc_auc"
                    ]
                ),

            "reproduced_roc_auc":
                float(
                    result[
                        "test_roc_auc"
                    ]
                ),

            "historical_pr_auc":
                float(
                    hist_row[
                        "test_pr_auc"
                    ]
                ),

            "reproduced_pr_auc":
                float(
                    result[
                        "test_pr_auc"
                    ]
                ),

            "historical_tn":
                int(
                    hist_row["tn"]
                ),

            "reproduced_tn":
                int(
                    result["tn"]
                ),

            "historical_fp":
                int(
                    hist_row["fp"]
                ),

            "reproduced_fp":
                int(
                    result["fp"]
                ),

            "historical_fn":
                int(
                    hist_row["fn"]
                ),

            "reproduced_fn":
                int(
                    result["fn"]
                ),

            "historical_tp":
                int(
                    hist_row["tp"]
                ),

            "reproduced_tp":
                int(
                    result["tp"]
                ),
        }


        compare[
            "threshold_abs_delta"
        ] = abs(
            compare[
                "reproduced_threshold"
            ]
            -
            compare[
                "historical_threshold"
            ]
        )

        compare[
            "test_f1_abs_delta"
        ] = abs(
            compare[
                "reproduced_test_f1"
            ]
            -
            compare[
                "historical_test_f1"
            ]
        )

        compare[
            "test_fpr_abs_delta"
        ] = abs(
            compare[
                "reproduced_test_fpr"
            ]
            -
            compare[
                "historical_test_fpr"
            ]
        )

        compare[
            "roc_auc_abs_delta"
        ] = abs(
            compare[
                "reproduced_roc_auc"
            ]
            -
            compare[
                "historical_roc_auc"
            ]
        )

        compare[
            "pr_auc_abs_delta"
        ] = abs(
            compare[
                "reproduced_pr_auc"
            ]
            -
            compare[
                "historical_pr_auc"
            ]
        )


        confusion_exact = (
            compare[
                "historical_tn"
            ]
            == compare[
                "reproduced_tn"
            ]
            and
            compare[
                "historical_fp"
            ]
            == compare[
                "reproduced_fp"
            ]
            and
            compare[
                "historical_fn"
            ]
            == compare[
                "reproduced_fn"
            ]
            and
            compare[
                "historical_tp"
            ]
            == compare[
                "reproduced_tp"
            ]
        )


        percentile_exact = (
            compare[
                "historical_percentile"
            ]
            == compare[
                "reproduced_percentile"
            ]
        )


        # ----------------------------------------------------
        # Reproduction status
        # ----------------------------------------------------

        if (
            confusion_exact
            and percentile_exact
            and compare[
                "threshold_abs_delta"
            ] <= 1e-10
            and compare[
                "test_f1_abs_delta"
            ] <= 1e-10
            and compare[
                "test_fpr_abs_delta"
            ] <= 1e-10
            and compare[
                "roc_auc_abs_delta"
            ] <= 1e-10
            and compare[
                "pr_auc_abs_delta"
            ] <= 1e-10
        ):

            status = "PASS_EXACT"

        elif (
            percentile_exact
            and compare[
                "threshold_abs_delta"
            ] <= 1e-6
            and compare[
                "test_f1_abs_delta"
            ] <= 1e-6
            and compare[
                "test_fpr_abs_delta"
            ] <= 1e-6
            and compare[
                "roc_auc_abs_delta"
            ] <= 1e-6
            and compare[
                "pr_auc_abs_delta"
            ] <= 1e-6
        ):

            status = "PASS_NUMERIC"

        else:

            status = "REVIEW"


        compare[
            "confusion_matrix_exact"
        ] = confusion_exact

        compare[
            "selected_percentile_exact"
        ] = percentile_exact

        compare[
            "reproduction_status"
        ] = status


        comparison_rows.append(
            compare
        )


        print(
            f"{model_name}: "
            f"status={status} | "
            f"percentile="
            f"{result['selected_percentile']} | "
            f"threshold="
            f"{result['selected_threshold']:.9f} | "
            f"F1="
            f"{result['test_f1']:.6f} | "
            f"FPR="
            f"{result['test_fpr']:.6f}"
        )


    # --------------------------------------------------------
    # Save regenerated fitted models only in correction folder
    # --------------------------------------------------------

    model_dir = (
        NAB_6030_CORRECTION_DIR
        if config_name == "60/30"
        else NAB_6015_CORRECTION_DIR
    ) / "reconstructed_models"

    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    safe_names = {
        "Isolation Forest":
            "isolation_forest.joblib",

        "LOF":
            "lof.joblib",

        "Random Forest":
            "random_forest.joblib",

        "XGBoost":
            "xgboost.joblib",

        "Logistic Regression":
            "logistic_regression.joblib",
    }


    for model_name, info in (
        config_models.items()
    ):

        joblib.dump(
            info["model"],
            model_dir
            / safe_names[
                model_name
            ],
        )


    joblib.dump(
        bundle["scaler"],
        model_dir
        / "standard_scaler.joblib",
    )


    nab_reconstructed_non_neural_models[
        config_name
    ] = config_models


    print(
        "\nReconstructed models saved under:"
    )

    print(
        model_dir
    )


# ============================================================
# F. Consolidated reproduction audit
# ============================================================

reproduction_df = pd.DataFrame(
    reproduction_rows
)

comparison_df = pd.DataFrame(
    comparison_rows
)


print("\n")
print("=" * 110)
print("REPRODUCTION COMPARISON")
print("=" * 110)


display_cols = [
    "configuration",
    "model",
    "historical_percentile",
    "reproduced_percentile",
    "threshold_abs_delta",
    "test_f1_abs_delta",
    "test_fpr_abs_delta",
    "roc_auc_abs_delta",
    "pr_auc_abs_delta",
    "confusion_matrix_exact",
    "reproduction_status",
]


print(
    comparison_df[
        display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.12e}"
    )
)


# ============================================================
# G. Summary counts
# ============================================================

print(
    "\nReproduction status counts:"
)

print(
    comparison_df[
        "reproduction_status"
    ].value_counts()
)


review_rows = (
    comparison_df[
        comparison_df[
            "reproduction_status"
        ] == "REVIEW"
    ]
)


if len(review_rows) == 0:

    print(
        "\nPASS: all five non-neural models "
        "reproduce the historical frozen results "
        "within the defined numerical gate."
    )

else:

    print(
        "\nATTENTION: one or more models require review "
        "before corrected metrics are generated."
    )

    print(
        review_rows[
            display_cols
        ].to_string(
            index=False
        )
    )


# ============================================================
# H. Save audit artifacts
# ============================================================

REPRODUCTION_RESULTS_PATH = (
    NAB_CORRECTION_DIR
    / "nab_non_neural_historical_reproduction_metrics.csv"
)

REPRODUCTION_COMPARISON_PATH = (
    NAB_CORRECTION_DIR
    / "nab_non_neural_historical_reproduction_gate.csv"
)


reproduction_df.to_csv(
    REPRODUCTION_RESULTS_PATH,
    index=False
)

comparison_df.to_csv(
    REPRODUCTION_COMPARISON_PATH,
    index=False
)


print(
    "\nReproduced historical metrics saved to:"
)

print(
    REPRODUCTION_RESULTS_PATH
)

print(
    "\nReproduction gate saved to:"
)

print(
    REPRODUCTION_COMPARISON_PATH
)


print("\n")
print("=" * 110)
print("CELL 21 COMPLETE")
print("=" * 110)

print(
    "\nOnly reconstructed copies were created inside "
    "the correction experiment. "
    "No frozen historical artifact was overwritten."
)

In [ ]:
# CELL 22 — NAB Frozen-Test-Score Reproduction and Model-Drift Audit

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

print("=" * 115)
print("NAB FROZEN TEST-SCORE REPRODUCTION / MODEL-DRIFT AUDIT")
print("=" * 115)


# ============================================================
# A. Frozen historical prediction files
# ============================================================

FROZEN_PREDICTION_FILES = {

    "60/30_unsupervised": (
        BASE
        / "results"
        / "EXP_NAB_60min_stride30"
        / "test_predictions_nab_60min_stride30.csv"
    ),

    "60/15_unsupervised": (
        BASE
        / "results"
        / "EXP_NAB_60min_stride15"
        / "test_predictions_nab_60min_stride15.csv"
    ),

    "60/15_supervised": (
        BASE
        / "results"
        / "EXP_NAB_60min_stride15"
        / "test_predictions_nab_60min_stride15_supervised_models.csv"
    ),
}


for name, path in FROZEN_PREDICTION_FILES.items():

    print(
        f"\n{name}:"
    )

    print(
        "  exists:",
        path.exists()
    )

    print(
        "  path:",
        path
    )


# ============================================================
# B. Helpers
# ============================================================

IDENTITY_COLUMNS = [
    "series",
    "domain",
    "window_start",
    "window_end",
    "label",
]


def normalize_identity_timestamps(
    df
):

    out = df.copy()

    out["window_start"] = pd.to_datetime(
        out["window_start"],
        utc=True,
        errors="raise"
    )

    out["window_end"] = pd.to_datetime(
        out["window_end"],
        utc=True,
        errors="raise"
    )

    return out


def compare_score_vectors(
    configuration,
    model,
    frozen_scores,
    reconstructed_scores
):

    frozen_scores = np.asarray(
        frozen_scores,
        dtype=np.float64
    )

    reconstructed_scores = np.asarray(
        reconstructed_scores,
        dtype=np.float64
    )


    assert len(
        frozen_scores
    ) == len(
        reconstructed_scores
    )


    finite = (
        np.isfinite(
            frozen_scores
        )
        &
        np.isfinite(
            reconstructed_scores
        )
    )


    f = frozen_scores[
        finite
    ]

    r = reconstructed_scores[
        finite
    ]


    abs_diff = np.abs(
        f - r
    )


    max_abs = float(
        np.max(
            abs_diff
        )
    )

    mean_abs = float(
        np.mean(
            abs_diff
        )
    )

    median_abs = float(
        np.median(
            abs_diff
        )
    )


    denom = np.maximum(
        np.maximum(
            np.abs(f),
            np.abs(r)
        ),
        1e-15
    )

    relative_diff = (
        abs_diff
        / denom
    )


    max_relative = float(
        np.max(
            relative_diff
        )
    )


    if np.std(f) > 0 and np.std(r) > 0:

        pearson = float(
            pearsonr(
                f,
                r
            ).statistic
        )

        spearman = float(
            spearmanr(
                f,
                r
            ).statistic
        )

    else:

        pearson = np.nan
        spearman = np.nan


    exact_count = int(
        np.sum(
            f == r
        )
    )


    close_1e10 = int(
        np.sum(
            np.isclose(
                f,
                r,
                rtol=1e-10,
                atol=1e-10
            )
        )
    )


    close_1e6 = int(
        np.sum(
            np.isclose(
                f,
                r,
                rtol=1e-6,
                atol=1e-6
            )
        )
    )


    return {
        "configuration":
            configuration,

        "model":
            model,

        "rows_compared":
            len(f),

        "exact_score_rows":
            exact_count,

        "close_rows_1e10":
            close_1e10,

        "close_rows_1e6":
            close_1e6,

        "max_absolute_score_difference":
            max_abs,

        "mean_absolute_score_difference":
            mean_abs,

        "median_absolute_score_difference":
            median_abs,

        "max_relative_score_difference":
            max_relative,

        "pearson_score_correlation":
            pearson,

        "spearman_rank_correlation":
            spearman,
    }


# ============================================================
# C. Load frozen files
# ============================================================

frozen_6030_unsup = normalize_identity_timestamps(
    pd.read_csv(
        FROZEN_PREDICTION_FILES[
            "60/30_unsupervised"
        ],
        low_memory=False
    )
)

frozen_6015_unsup = normalize_identity_timestamps(
    pd.read_csv(
        FROZEN_PREDICTION_FILES[
            "60/15_unsupervised"
        ],
        low_memory=False
    )
)

frozen_6015_sup = normalize_identity_timestamps(
    pd.read_csv(
        FROZEN_PREDICTION_FILES[
            "60/15_supervised"
        ],
        low_memory=False
    )
)


print(
    "\nFrozen prediction shapes:"
)

print(
    "60/30 unsupervised:",
    frozen_6030_unsup.shape
)

print(
    "60/15 unsupervised:",
    frozen_6015_unsup.shape
)

print(
    "60/15 supervised:",
    frozen_6015_sup.shape
)


# ============================================================
# D. Confirm frozen files represent the exact historical
#    test populations
# ============================================================

historical_test_6030 = (
    nab_reconstruction[
        "60/30"
    ][
        "original"
    ][
        "test_df"
    ]
    .reset_index(drop=True)
)

historical_test_6015 = (
    nab_reconstruction[
        "60/15"
    ][
        "original"
    ][
        "test_df"
    ]
    .reset_index(drop=True)
)


for label, frozen, expected in [

    (
        "60/30 unsupervised",
        frozen_6030_unsup,
        historical_test_6030
    ),

    (
        "60/15 unsupervised",
        frozen_6015_unsup,
        historical_test_6015
    ),

    (
        "60/15 supervised",
        frozen_6015_sup,
        historical_test_6015
    ),
]:

    print(
        "\nChecking row identity:",
        label
    )


    assert len(
        frozen
    ) == len(
        expected
    )


    pd.testing.assert_frame_equal(

        frozen[
            IDENTITY_COLUMNS
        ].reset_index(drop=True),

        expected[
            IDENTITY_COLUMNS
        ].reset_index(drop=True),

        check_dtype=False,
        check_exact=True
    )


    print(
        "PASS: frozen prediction rows exactly match "
        "the historical test population."
    )


# ============================================================
# E. Compare frozen vs reconstructed scores
# ============================================================

score_audit_rows = []


# ------------------------------------------------------------
# 60/30 — Isolation Forest and LOF
# ------------------------------------------------------------

recon_6030 = (
    nab_reconstructed_non_neural_models[
        "60/30"
    ]
)


score_audit_rows.append(
    compare_score_vectors(

        configuration="60/30",

        model="Isolation Forest",

        frozen_scores=(
            frozen_6030_unsup[
                "iso_score"
            ].to_numpy()
        ),

        reconstructed_scores=(
            recon_6030[
                "Isolation Forest"
            ][
                "test_scores_original"
            ]
        ),
    )
)


score_audit_rows.append(
    compare_score_vectors(

        configuration="60/30",

        model="LOF",

        frozen_scores=(
            frozen_6030_unsup[
                "lof_score"
            ].to_numpy()
        ),

        reconstructed_scores=(
            recon_6030[
                "LOF"
            ][
                "test_scores_original"
            ]
        ),
    )
)


# ------------------------------------------------------------
# 60/15 — Isolation Forest and LOF
# ------------------------------------------------------------

recon_6015 = (
    nab_reconstructed_non_neural_models[
        "60/15"
    ]
)


score_audit_rows.append(
    compare_score_vectors(

        configuration="60/15",

        model="Isolation Forest",

        frozen_scores=(
            frozen_6015_unsup[
                "iso_score"
            ].to_numpy()
        ),

        reconstructed_scores=(
            recon_6015[
                "Isolation Forest"
            ][
                "test_scores_original"
            ]
        ),
    )
)


score_audit_rows.append(
    compare_score_vectors(

        configuration="60/15",

        model="LOF",

        frozen_scores=(
            frozen_6015_unsup[
                "lof_score"
            ].to_numpy()
        ),

        reconstructed_scores=(
            recon_6015[
                "LOF"
            ][
                "test_scores_original"
            ]
        ),
    )
)


# ------------------------------------------------------------
# 60/15 — RF, XGBoost, Logistic Regression
# ------------------------------------------------------------

score_audit_rows.append(
    compare_score_vectors(

        configuration="60/15",

        model="Random Forest",

        frozen_scores=(
            frozen_6015_sup[
                "rf_score"
            ].to_numpy()
        ),

        reconstructed_scores=(
            recon_6015[
                "Random Forest"
            ][
                "test_scores_original"
            ]
        ),
    )
)


score_audit_rows.append(
    compare_score_vectors(

        configuration="60/15",

        model="XGBoost",

        frozen_scores=(
            frozen_6015_sup[
                "xgb_score"
            ].to_numpy()
        ),

        reconstructed_scores=(
            recon_6015[
                "XGBoost"
            ][
                "test_scores_original"
            ]
        ),
    )
)


score_audit_rows.append(
    compare_score_vectors(

        configuration="60/15",

        model="Logistic Regression",

        frozen_scores=(
            frozen_6015_sup[
                "logreg_score"
            ].to_numpy()
        ),

        reconstructed_scores=(
            recon_6015[
                "Logistic Regression"
            ][
                "test_scores_original"
            ]
        ),
    )
)


score_audit_df = pd.DataFrame(
    score_audit_rows
)


# ============================================================
# F. Display score-level reproduction results
# ============================================================

print("\n")
print("=" * 115)
print("FROZEN VS RECONSTRUCTED TEST-SCORE COMPARISON")
print("=" * 115)


display_cols = [
    "configuration",
    "model",
    "rows_compared",
    "max_absolute_score_difference",
    "mean_absolute_score_difference",
    "median_absolute_score_difference",
    "pearson_score_correlation",
    "spearman_rank_correlation",
    "exact_score_rows",
    "close_rows_1e10",
    "close_rows_1e6",
]


print(
    score_audit_df[
        display_cols
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.12e}"
    )
)


# ============================================================
# G. Historical frozen-prediction metric verification
#
# This confirms the frozen prediction files themselves still
# reproduce the frozen metric table for models available in
# those files.
# ============================================================

print("\n")
print("=" * 115)
print("FROZEN PREDICTION FILE SELF-CONSISTENCY")
print("=" * 115)


self_consistency_rows = []


def frozen_test_metric_check(
    configuration,
    model_name,
    frozen_df,
    score_col,
    historical_metrics_df
):

    hist = historical_metrics_df[
        historical_metrics_df[
            "model"
        ] == model_name
    ].iloc[0]


    threshold = float(
        hist[
            "selected_threshold"
        ]
    )

    y = (
        frozen_df[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    scores = (
        frozen_df[
            score_col
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    pred = (
        scores
        >= threshold
    ).astype(int)


    metrics = (
        compute_metrics_reproduction(
            y,
            pred,
            scores
        )
    )


    row = {
        "configuration":
            configuration,

        "model":
            model_name,

        "historical_threshold":
            threshold,

        "frozen_score_test_f1":
            metrics["f1"],

        "historical_test_f1":
            float(
                hist[
                    "test_f1"
                ]
            ),

        "frozen_score_test_fpr":
            metrics["fpr"],

        "historical_test_fpr":
            float(
                hist[
                    "test_fpr"
                ]
            ),

        "frozen_score_roc_auc":
            metrics["roc_auc"],

        "historical_roc_auc":
            float(
                hist[
                    "test_roc_auc"
                ]
            ),

        "frozen_score_pr_auc":
            metrics["pr_auc"],

        "historical_pr_auc":
            float(
                hist[
                    "test_pr_auc"
                ]
            ),

        "frozen_tn":
            metrics["tn"],

        "historical_tn":
            int(
                hist[
                    "tn"
                ]
            ),

        "frozen_fp":
            metrics["fp"],

        "historical_fp":
            int(
                hist[
                    "fp"
                ]
            ),

        "frozen_fn":
            metrics["fn"],

        "historical_fn":
            int(
                hist[
                    "fn"
                ]
            ),

        "frozen_tp":
            metrics["tp"],

        "historical_tp":
            int(
                hist[
                    "tp"
                ]
            ),
    }


    row[
        "confusion_exact"
    ] = (
        row["frozen_tn"]
        == row["historical_tn"]

        and row["frozen_fp"]
        == row["historical_fp"]

        and row["frozen_fn"]
        == row["historical_fn"]

        and row["frozen_tp"]
        == row["historical_tp"]
    )


    return row


hist_6030 = (
    nab_reconstruction[
        "60/30"
    ][
        "historical_metrics"
    ]
)

hist_6015 = (
    nab_reconstruction[
        "60/15"
    ][
        "historical_metrics"
    ]
)


for model_name, score_col in [
    (
        "Isolation Forest",
        "iso_score"
    ),
    (
        "LOF",
        "lof_score"
    ),
]:

    self_consistency_rows.append(
        frozen_test_metric_check(
            "60/30",
            model_name,
            frozen_6030_unsup,
            score_col,
            hist_6030,
        )
    )

    self_consistency_rows.append(
        frozen_test_metric_check(
            "60/15",
            model_name,
            frozen_6015_unsup,
            score_col,
            hist_6015,
        )
    )


for model_name, score_col in [
    (
        "Random Forest",
        "rf_score"
    ),
    (
        "XGBoost",
        "xgb_score"
    ),
    (
        "Logistic Regression",
        "logreg_score"
    ),
]:

    self_consistency_rows.append(
        frozen_test_metric_check(
            "60/15",
            model_name,
            frozen_6015_sup,
            score_col,
            hist_6015,
        )
    )


self_consistency_df = pd.DataFrame(
    self_consistency_rows
)


print(
    self_consistency_df[
        [
            "configuration",
            "model",
            "confusion_exact",
            "frozen_score_test_f1",
            "historical_test_f1",
            "frozen_score_test_fpr",
            "historical_test_fpr",
            "frozen_score_roc_auc",
            "historical_roc_auc",
            "frozen_score_pr_auc",
            "historical_pr_auc",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.12e}"
    )
)


# ============================================================
# H. Save audit outputs
# ============================================================

SCORE_AUDIT_PATH = (
    NAB_CORRECTION_DIR
    / "nab_frozen_vs_reconstructed_test_score_audit.csv"
)

SELF_CONSISTENCY_PATH = (
    NAB_CORRECTION_DIR
    / "nab_frozen_prediction_self_consistency_audit.csv"
)


score_audit_df.to_csv(
    SCORE_AUDIT_PATH,
    index=False
)

self_consistency_df.to_csv(
    SELF_CONSISTENCY_PATH,
    index=False
)


print(
    "\nScore-level audit saved to:"
)

print(
    SCORE_AUDIT_PATH
)


print(
    "\nFrozen prediction self-consistency audit saved to:"
)

print(
    SELF_CONSISTENCY_PATH
)


print("\n")
print("=" * 115)
print("CELL 22 COMPLETE")
print("=" * 115)

print(
    "\nNo model was trained or modified in this cell."
)

In [ ]:
# CELL 23 — NAB Historical Artifact Provenance and Prediction-Threshold Consistency Audit

import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


print("=" * 115)
print("NAB HISTORICAL ARTIFACT PROVENANCE / CONSISTENCY AUDIT")
print("=" * 115)


# ============================================================
# A. Historical artifact paths
# ============================================================

ARTIFACTS = {

    "60/30 metrics": (
        BASE
        / "results"
        / "EXP_NAB_60min_stride30"
        / "metrics_nab_60min_stride30_all_models.csv"
    ),

    "60/30 unsupervised predictions": (
        BASE
        / "results"
        / "EXP_NAB_60min_stride30"
        / "test_predictions_nab_60min_stride30.csv"
    ),

    "60/15 metrics": (
        BASE
        / "results"
        / "EXP_NAB_60min_stride15"
        / "metrics_nab_60min_stride15_all_models.csv"
    ),

    "60/15 unsupervised predictions": (
        BASE
        / "results"
        / "EXP_NAB_60min_stride15"
        / "test_predictions_nab_60min_stride15.csv"
    ),

    "60/15 supervised predictions": (
        BASE
        / "results"
        / "EXP_NAB_60min_stride15"
        / "test_predictions_nab_60min_stride15_supervised_models.csv"
    ),
}


print("\nHistorical artifact timestamps:")

timestamp_rows = []

for name, path in ARTIFACTS.items():

    exists = path.exists()

    if exists:

        stat = path.stat()

        modified = datetime.fromtimestamp(
            stat.st_mtime
        )

        timestamp_rows.append({
            "artifact":
                name,

            "path":
                str(path),

            "size_bytes":
                stat.st_size,

            "modified_local":
                modified.isoformat(
                    timespec="seconds"
                ),
        })

        print(
            f"{name}: "
            f"{modified.isoformat(timespec='seconds')} | "
            f"{stat.st_size:,} bytes"
        )

    else:

        print(
            f"{name}: MISSING"
        )


timestamp_df = pd.DataFrame(
    timestamp_rows
)


# ============================================================
# B. Prediction-column consistency with stored score columns
# ============================================================

metrics_6030 = pd.read_csv(
    ARTIFACTS[
        "60/30 metrics"
    ]
)

metrics_6015 = pd.read_csv(
    ARTIFACTS[
        "60/15 metrics"
    ]
)

pred_6030 = pd.read_csv(
    ARTIFACTS[
        "60/30 unsupervised predictions"
    ],
    low_memory=False
)

pred_6015_unsup = pd.read_csv(
    ARTIFACTS[
        "60/15 unsupervised predictions"
    ],
    low_memory=False
)

pred_6015_sup = pd.read_csv(
    ARTIFACTS[
        "60/15 supervised predictions"
    ],
    low_memory=False
)


CONSISTENCY_SPECS = [

    {
        "configuration":
            "60/30",

        "model":
            "Isolation Forest",

        "df":
            pred_6030,

        "score_col":
            "iso_score",

        "stored_pred_col":
            "iso_pred",

        "metrics":
            metrics_6030,
    },

    {
        "configuration":
            "60/30",

        "model":
            "LOF",

        "df":
            pred_6030,

        "score_col":
            "lof_score",

        "stored_pred_col":
            "lof_pred",

        "metrics":
            metrics_6030,
    },

    {
        "configuration":
            "60/30",

        "model":
            "Autoencoder",

        "df":
            pred_6030,

        "score_col":
            "ae_score",

        "stored_pred_col":
            "ae_pred",

        "metrics":
            metrics_6030,
    },

    {
        "configuration":
            "60/15",

        "model":
            "Isolation Forest",

        "df":
            pred_6015_unsup,

        "score_col":
            "iso_score",

        "stored_pred_col":
            "iso_pred",

        "metrics":
            metrics_6015,
    },

    {
        "configuration":
            "60/15",

        "model":
            "LOF",

        "df":
            pred_6015_unsup,

        "score_col":
            "lof_score",

        "stored_pred_col":
            "lof_pred",

        "metrics":
            metrics_6015,
    },

    {
        "configuration":
            "60/15",

        "model":
            "Autoencoder",

        "df":
            pred_6015_unsup,

        "score_col":
            "ae_score",

        "stored_pred_col":
            "ae_pred",

        "metrics":
            metrics_6015,
    },

    {
        "configuration":
            "60/15",

        "model":
            "Random Forest",

        "df":
            pred_6015_sup,

        "score_col":
            "rf_score",

        "stored_pred_col":
            "rf_pred",

        "metrics":
            metrics_6015,
    },

    {
        "configuration":
            "60/15",

        "model":
            "XGBoost",

        "df":
            pred_6015_sup,

        "score_col":
            "xgb_score",

        "stored_pred_col":
            "xgb_pred",

        "metrics":
            metrics_6015,
    },

    {
        "configuration":
            "60/15",

        "model":
            "Logistic Regression",

        "df":
            pred_6015_sup,

        "score_col":
            "logreg_score",

        "stored_pred_col":
            "logreg_pred",

        "metrics":
            metrics_6015,
    },

    {
        "configuration":
            "60/15",

        "model":
            "MLP",

        "df":
            pred_6015_sup,

        "score_col":
            "mlp_score",

        "stored_pred_col":
            "mlp_pred",

        "metrics":
            metrics_6015,
    },
]


consistency_rows = []


for spec in CONSISTENCY_SPECS:

    df = spec["df"]

    hist_row = spec[
        "metrics"
    ][
        spec["metrics"]["model"]
        == spec["model"]
    ]

    if len(hist_row) != 1:

        raise RuntimeError(
            f"Historical metric row problem: "
            f"{spec['configuration']} / {spec['model']}"
        )

    hist_row = hist_row.iloc[0]

    threshold = float(
        hist_row[
            "selected_threshold"
        ]
    )

    scores = (
        df[
            spec["score_col"]
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    stored_pred = (
        df[
            spec["stored_pred_col"]
        ]
        .to_numpy(
            dtype=int
        )
    )

    threshold_pred = (
        scores
        >= threshold
    ).astype(int)

    y_true = (
        df["label"]
        .to_numpy(
            dtype=int
        )
    )


    threshold_metrics = (
        compute_metrics_reproduction(
            y_true,
            threshold_pred,
            scores
        )
    )

    stored_pred_metrics = (
        compute_metrics_reproduction(
            y_true,
            stored_pred,
            scores
        )
    )


    prediction_agreement = int(
        np.sum(
            threshold_pred
            == stored_pred
        )
    )

    prediction_disagreement = int(
        len(df)
        - prediction_agreement
    )


    hist_confusion = (
        int(hist_row["tn"]),
        int(hist_row["fp"]),
        int(hist_row["fn"]),
        int(hist_row["tp"]),
    )

    threshold_confusion = (
        threshold_metrics["tn"],
        threshold_metrics["fp"],
        threshold_metrics["fn"],
        threshold_metrics["tp"],
    )

    stored_confusion = (
        stored_pred_metrics["tn"],
        stored_pred_metrics["fp"],
        stored_pred_metrics["fn"],
        stored_pred_metrics["tp"],
    )


    consistency_rows.append({

        "configuration":
            spec["configuration"],

        "model":
            spec["model"],

        "historical_threshold":
            threshold,

        "rows":
            len(df),

        "stored_vs_threshold_pred_disagreements":
            prediction_disagreement,

        "stored_pred_positive_count":
            int(
                stored_pred.sum()
            ),

        "threshold_pred_positive_count":
            int(
                threshold_pred.sum()
            ),

        "historical_tn":
            hist_confusion[0],

        "historical_fp":
            hist_confusion[1],

        "historical_fn":
            hist_confusion[2],

        "historical_tp":
            hist_confusion[3],

        "stored_tn":
            stored_confusion[0],

        "stored_fp":
            stored_confusion[1],

        "stored_fn":
            stored_confusion[2],

        "stored_tp":
            stored_confusion[3],

        "threshold_tn":
            threshold_confusion[0],

        "threshold_fp":
            threshold_confusion[1],

        "threshold_fn":
            threshold_confusion[2],

        "threshold_tp":
            threshold_confusion[3],

        "stored_confusion_matches_historical":
            (
                stored_confusion
                == hist_confusion
            ),

        "threshold_confusion_matches_historical":
            (
                threshold_confusion
                == hist_confusion
            ),

        "stored_test_f1":
            stored_pred_metrics["f1"],

        "threshold_test_f1":
            threshold_metrics["f1"],

        "historical_test_f1":
            float(
                hist_row["test_f1"]
            ),
    })


consistency_df = pd.DataFrame(
    consistency_rows
)


print("\n")
print("=" * 115)
print("STORED PREDICTION / HISTORICAL THRESHOLD CONSISTENCY")
print("=" * 115)


print(
    consistency_df[
        [
            "configuration",
            "model",
            "stored_vs_threshold_pred_disagreements",
            "stored_pred_positive_count",
            "threshold_pred_positive_count",
            "stored_confusion_matches_historical",
            "threshold_confusion_matches_historical",
            "stored_test_f1",
            "threshold_test_f1",
            "historical_test_f1",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.12f}"
    )
)


# ============================================================
# C. Inspect notebook execution metadata for critical cells
# ============================================================

THESIS_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)

with open(
    THESIS_NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:

    notebook = json.load(f)


# Cells involved in creation / later rewriting
CRITICAL_NOTEBOOK_CELLS = [
    466,  # 60/15 IF
    467,  # 60/15 LOF
    468,  # 60/15 AE
    469,  # 60/15 unsup save
    472,  # 60/15 RF
    473,  # 60/15 XGB
    474,  # 60/15 LR
    475,  # 60/15 MLP
    476,  # 60/15 all metrics save
    477,  # 60/15 supervised prediction save
    482,  # 60/30 seven-model training / metrics
    501,  # later 60/30 unsupervised re-run / overwrite
]


execution_rows = []


for cell_number in CRITICAL_NOTEBOOK_CELLS:

    cell = notebook[
        "cells"
    ][
        cell_number - 1
    ]

    execution_count = cell.get(
        "execution_count"
    )

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )


    first_nonempty = ""

    for line in text.splitlines():

        if line.strip():

            first_nonempty = (
                line.strip()
            )

            break


    outputs = cell.get(
        "outputs",
        []
    )


    output_text_fragments = []

    for output in outputs:

        if "text" in output:

            text_field = output[
                "text"
            ]

            if isinstance(
                text_field,
                list
            ):

                output_text_fragments.append(
                    "".join(
                        text_field
                    )
                )

            else:

                output_text_fragments.append(
                    str(
                        text_field
                    )
                )


    output_preview = (
        "\n".join(
            output_text_fragments
        )[:1200]
    )


    execution_rows.append({

        "cell_number":
            cell_number,

        "execution_count":
            execution_count,

        "first_source_line":
            first_nonempty,

        "output_preview":
            output_preview,
    })


execution_df = pd.DataFrame(
    execution_rows
)


print("\n")
print("=" * 115)
print("NOTEBOOK EXECUTION METADATA")
print("=" * 115)


print(
    execution_df[
        [
            "cell_number",
            "execution_count",
            "first_source_line",
        ]
    ].to_string(
        index=False
    )
)


print(
    "\nCritical output previews:"
)


for _, row in execution_df.iterrows():

    if row[
        "output_preview"
    ]:

        print(
            "\n-----------------------------------------"
        )

        print(
            "Notebook cell:",
            row[
                "cell_number"
            ]
        )

        print(
            "Execution count:",
            row[
                "execution_count"
            ]
        )

        print(
            row[
                "output_preview"
            ]
        )


# ============================================================
# D. Explicit source-path overwrite search
# ============================================================

TARGET_FILENAMES = [
    "test_predictions_nab_60min_stride30.csv",
    "test_predictions_nab_60min_stride15.csv",
    "test_predictions_nab_60min_stride15_supervised_models.csv",
    "metrics_nab_60min_stride30_all_models.csv",
    "metrics_nab_60min_stride15_all_models.csv",
]


write_rows = []


for cell_number, cell in enumerate(
    notebook["cells"],
    start=1
):

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )

    for filename in TARGET_FILENAMES:

        if filename not in text:
            continue

        write_rows.append({

            "cell_number":
                cell_number,

            "execution_count":
                cell.get(
                    "execution_count"
                ),

            "filename":
                filename,

            "contains_to_csv":
                "to_csv" in text,

            "contains_read_csv":
                "read_csv" in text,

            "source_preview":
                text[:1000],
        })


write_df = pd.DataFrame(
    write_rows
)


print("\n")
print("=" * 115)
print("HISTORICAL ARTIFACT WRITE / READ LOCATIONS")
print("=" * 115)


if len(write_df):

    print(
        write_df[
            [
                "cell_number",
                "execution_count",
                "filename",
                "contains_to_csv",
                "contains_read_csv",
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "No matching notebook references found."
    )


# ============================================================
# E. Save provenance evidence
# ============================================================

TIMESTAMP_PATH = (
    NAB_CORRECTION_DIR
    / "nab_historical_artifact_timestamps.csv"
)

CONSISTENCY_PATH = (
    NAB_CORRECTION_DIR
    / "nab_historical_prediction_threshold_consistency.csv"
)

EXECUTION_PATH = (
    NAB_CORRECTION_DIR
    / "nab_historical_notebook_execution_metadata.csv"
)

WRITE_PATH = (
    NAB_CORRECTION_DIR
    / "nab_historical_artifact_write_locations.csv"
)


timestamp_df.to_csv(
    TIMESTAMP_PATH,
    index=False
)

consistency_df.to_csv(
    CONSISTENCY_PATH,
    index=False
)

execution_df.to_csv(
    EXECUTION_PATH,
    index=False
)

write_df.to_csv(
    WRITE_PATH,
    index=False
)


print("\n")
print("=" * 115)
print("CELL 23 COMPLETE")
print("=" * 115)

print(
    "\nSaved:"
)

print(
    TIMESTAMP_PATH
)

print(
    CONSISTENCY_PATH
)

print(
    EXECUTION_PATH
)

print(
    WRITE_PATH
)

print(
    "\nNo historical artifact, model, or metric "
    "was modified."
)

In [ ]:
# CELL 24 — NAB Five Non-Neural Models Same-Runtime Unpurged vs Purged Correction

import numpy as np
import pandas as pd


print("=" * 115)
print("NAB NON-NEURAL SAME-RUNTIME UNPURGED VS PURGED CORRECTION")
print("=" * 115)


# ============================================================
# A. Models included
# ============================================================

NON_NEURAL_MODELS = [
    "Isolation Forest",
    "LOF",
    "Random Forest",
    "XGBoost",
    "Logistic Regression",
]


# ============================================================
# B. Helper to obtain anomaly/confidence scores
# ============================================================

def score_non_neural_model(
    model_name,
    model,
    X
):

    if model_name in [
        "Isolation Forest",
        "LOF",
    ]:

        return (
            -model.decision_function(X)
        )

    if model_name in [
        "Random Forest",
        "XGBoost",
        "Logistic Regression",
    ]:

        return (
            model.predict_proba(X)[:, 1]
        )

    raise ValueError(
        f"Unsupported model: {model_name}"
    )


# ============================================================
# C. Exact historical/current threshold-selection protocol
# ============================================================

def evaluate_scores_same_runtime(
    model_name,
    y_val,
    y_test,
    val_scores,
    test_scores
):

    best = (
        select_threshold_by_val_f1_reproduction(
            y_val,
            val_scores
        )
    )

    test_pred = (
        test_scores
        >= best["threshold"]
    ).astype(int)

    test_metrics = (
        compute_metrics_reproduction(
            y_test,
            test_pred,
            test_scores
        )
    )


    return {
        "model":
            model_name,

        "selected_percentile":
            best["percentile"],

        "selected_threshold":
            float(
                best["threshold"]
            ),

        "val_precision":
            best["precision"],

        "val_recall":
            best["recall"],

        "val_f1":
            best["f1"],

        "test_precision":
            test_metrics["precision"],

        "test_recall":
            test_metrics["recall"],

        "test_f1":
            test_metrics["f1"],

        "test_accuracy":
            test_metrics["accuracy"],

        "test_fpr":
            test_metrics["fpr"],

        "test_roc_auc":
            test_metrics["roc_auc"],

        "test_pr_auc":
            test_metrics["pr_auc"],

        "tn":
            test_metrics["tn"],

        "fp":
            test_metrics["fp"],

        "fn":
            test_metrics["fn"],

        "tp":
            test_metrics["tp"],
    }, test_pred


# ============================================================
# D. Run same fitted models on:
#
#    1. original unpurged validation/test
#    2. corrected purged validation/test
#
# Training model is identical within each comparison.
# ============================================================

all_metric_rows = []
delta_rows = []

corrected_prediction_tables = {}


for config_name in [
    "60/30",
    "60/15",
]:

    print("\n")
    print("=" * 115)
    print(
        f"CONFIGURATION {config_name}"
    )
    print("=" * 115)


    bundle = (
        nab_reconstruction[
            config_name
        ]
    )

    original = (
        bundle["original"]
    )

    corrected = (
        bundle["corrected"]
    )

    model_bundle = (
        nab_reconstructed_non_neural_models[
            config_name
        ]
    )


    print(
        "\nOriginal evaluation counts:"
    )

    print(
        "Validation:",
        len(
            original["y_val"]
        ),
        "| Test:",
        len(
            original["y_test"]
        )
    )


    print(
        "Corrected evaluation counts:"
    )

    print(
        "Validation:",
        len(
            corrected["y_val"]
        ),
        "| Test:",
        len(
            corrected["y_test"]
        )
    )


    # --------------------------------------------------------
    # Corrected test prediction base table
    # --------------------------------------------------------

    corrected_test_table = (
        corrected[
            "test_df"
        ][
            [
                "series",
                "domain",
                "window_start",
                "window_end",
                "label",
            ]
        ]
        .reset_index(drop=True)
        .copy()
    )


    # --------------------------------------------------------
    # Each reconstructed fitted model
    # --------------------------------------------------------

    for model_name in NON_NEURAL_MODELS:

        print(
            f"\nScoring {model_name}..."
        )


        model = (
            model_bundle[
                model_name
            ][
                "model"
            ]
        )


        # ====================================================
        # SAME-RUNTIME UNPURGED
        # ====================================================

        original_val_scores = (
            score_non_neural_model(
                model_name,
                model,
                original["X_val"]
            )
        )

        original_test_scores = (
            score_non_neural_model(
                model_name,
                model,
                original["X_test"]
            )
        )


        (
            original_result,
            original_test_pred
        ) = evaluate_scores_same_runtime(
            model_name,
            original["y_val"],
            original["y_test"],
            original_val_scores,
            original_test_scores,
        )


        original_result[
            "configuration"
        ] = config_name

        original_result[
            "evaluation_version"
        ] = "same_runtime_unpurged"


        all_metric_rows.append(
            original_result
        )


        # ====================================================
        # PURGED CORRECTED EVALUATION
        # ====================================================

        corrected_val_scores = (
            score_non_neural_model(
                model_name,
                model,
                corrected["X_val"]
            )
        )

        corrected_test_scores = (
            score_non_neural_model(
                model_name,
                model,
                corrected["X_test"]
            )
        )


        (
            corrected_result,
            corrected_test_pred
        ) = evaluate_scores_same_runtime(
            model_name,
            corrected["y_val"],
            corrected["y_test"],
            corrected_val_scores,
            corrected_test_scores,
        )


        corrected_result[
            "configuration"
        ] = config_name

        corrected_result[
            "evaluation_version"
        ] = "purged_corrected"


        all_metric_rows.append(
            corrected_result
        )


        # ====================================================
        # Purge-only delta
        #
        # Because the fitted model is identical, these deltas
        # isolate the effect of removing boundary-overlap rows.
        # ====================================================

        delta_row = {
            "configuration":
                config_name,

            "model":
                model_name,

            "unpurged_selected_percentile":
                original_result[
                    "selected_percentile"
                ],

            "corrected_selected_percentile":
                corrected_result[
                    "selected_percentile"
                ],

            "selected_percentile_changed":
                (
                    original_result[
                        "selected_percentile"
                    ]
                    !=
                    corrected_result[
                        "selected_percentile"
                    ]
                ),

            "unpurged_threshold":
                original_result[
                    "selected_threshold"
                ],

            "corrected_threshold":
                corrected_result[
                    "selected_threshold"
                ],

            "threshold_delta":
                (
                    corrected_result[
                        "selected_threshold"
                    ]
                    -
                    original_result[
                        "selected_threshold"
                    ]
                ),

            "unpurged_test_precision":
                original_result[
                    "test_precision"
                ],

            "corrected_test_precision":
                corrected_result[
                    "test_precision"
                ],

            "precision_delta":
                (
                    corrected_result[
                        "test_precision"
                    ]
                    -
                    original_result[
                        "test_precision"
                    ]
                ),

            "unpurged_test_recall":
                original_result[
                    "test_recall"
                ],

            "corrected_test_recall":
                corrected_result[
                    "test_recall"
                ],

            "recall_delta":
                (
                    corrected_result[
                        "test_recall"
                    ]
                    -
                    original_result[
                        "test_recall"
                    ]
                ),

            "unpurged_test_f1":
                original_result[
                    "test_f1"
                ],

            "corrected_test_f1":
                corrected_result[
                    "test_f1"
                ],

            "f1_delta":
                (
                    corrected_result[
                        "test_f1"
                    ]
                    -
                    original_result[
                        "test_f1"
                    ]
                ),

            "unpurged_test_fpr":
                original_result[
                    "test_fpr"
                ],

            "corrected_test_fpr":
                corrected_result[
                    "test_fpr"
                ],

            "fpr_delta":
                (
                    corrected_result[
                        "test_fpr"
                    ]
                    -
                    original_result[
                        "test_fpr"
                    ]
                ),

            "unpurged_test_roc_auc":
                original_result[
                    "test_roc_auc"
                ],

            "corrected_test_roc_auc":
                corrected_result[
                    "test_roc_auc"
                ],

            "roc_auc_delta":
                (
                    corrected_result[
                        "test_roc_auc"
                    ]
                    -
                    original_result[
                        "test_roc_auc"
                    ]
                ),

            "unpurged_test_pr_auc":
                original_result[
                    "test_pr_auc"
                ],

            "corrected_test_pr_auc":
                corrected_result[
                    "test_pr_auc"
                ],

            "pr_auc_delta":
                (
                    corrected_result[
                        "test_pr_auc"
                    ]
                    -
                    original_result[
                        "test_pr_auc"
                    ]
                ),

            "unpurged_tn":
                original_result["tn"],

            "corrected_tn":
                corrected_result["tn"],

            "unpurged_fp":
                original_result["fp"],

            "corrected_fp":
                corrected_result["fp"],

            "unpurged_fn":
                original_result["fn"],

            "corrected_fn":
                corrected_result["fn"],

            "unpurged_tp":
                original_result["tp"],

            "corrected_tp":
                corrected_result["tp"],
        }


        delta_rows.append(
            delta_row
        )


        # ====================================================
        # Add corrected test scores/predictions to table
        # ====================================================

        safe_prefix = {
            "Isolation Forest":
                "if",

            "LOF":
                "lof",

            "Random Forest":
                "rf",

            "XGBoost":
                "xgb",

            "Logistic Regression":
                "logreg",
        }[
            model_name
        ]


        corrected_test_table[
            f"{safe_prefix}_score"
        ] = corrected_test_scores

        corrected_test_table[
            f"{safe_prefix}_pred"
        ] = corrected_test_pred

        corrected_test_table[
            f"{safe_prefix}_threshold"
        ] = corrected_result[
            "selected_threshold"
        ]

        corrected_test_table[
            f"{safe_prefix}_selected_percentile"
        ] = corrected_result[
            "selected_percentile"
        ]


        print(
            f"{model_name}: "
            f"percentile "
            f"{original_result['selected_percentile']} "
            f"→ "
            f"{corrected_result['selected_percentile']} | "
            f"F1 "
            f"{original_result['test_f1']:.6f} "
            f"→ "
            f"{corrected_result['test_f1']:.6f} | "
            f"FPR "
            f"{original_result['test_fpr']:.6f} "
            f"→ "
            f"{corrected_result['test_fpr']:.6f}"
        )


    corrected_prediction_tables[
        config_name
    ] = corrected_test_table


# ============================================================
# E. Consolidate metrics and purge-only deltas
# ============================================================

same_runtime_metrics_df = pd.DataFrame(
    all_metric_rows
)

purge_delta_df = pd.DataFrame(
    delta_rows
)


print("\n")
print("=" * 115)
print("PURGE-ONLY EFFECT SUMMARY")
print("=" * 115)


summary_columns = [
    "configuration",
    "model",
    "unpurged_selected_percentile",
    "corrected_selected_percentile",
    "threshold_delta",
    "unpurged_test_f1",
    "corrected_test_f1",
    "f1_delta",
    "unpurged_test_fpr",
    "corrected_test_fpr",
    "fpr_delta",
    "unpurged_test_roc_auc",
    "corrected_test_roc_auc",
    "roc_auc_delta",
    "unpurged_test_pr_auc",
    "corrected_test_pr_auc",
    "pr_auc_delta",
]


print(
    purge_delta_df[
        summary_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# F. Confirm corrected confusion-matrix totals
# ============================================================

print("\n")
print("=" * 115)
print("CORRECTED TEST CONFUSION-MATRIX TOTAL CHECK")
print("=" * 115)


for _, row in purge_delta_df.iterrows():

    corrected_total = (
        row["corrected_tn"]
        + row["corrected_fp"]
        + row["corrected_fn"]
        + row["corrected_tp"]
    )

    expected_total = (
        20668
        if row["configuration"] == "60/30"
        else 41254
    )

    assert (
        corrected_total
        == expected_total
    )

    print(
        f"{row['configuration']} | "
        f"{row['model']}: "
        f"{int(corrected_total):,} "
        f"PASS"
    )


# ============================================================
# G. Save corrected artifacts
# ============================================================

SAME_RUNTIME_METRICS_PATH = (
    NAB_CORRECTION_DIR
    / "nab_non_neural_same_runtime_unpurged_and_corrected_metrics.csv"
)

PURGE_DELTA_PATH = (
    NAB_CORRECTION_DIR
    / "nab_non_neural_purge_only_effect.csv"
)


same_runtime_metrics_df.to_csv(
    SAME_RUNTIME_METRICS_PATH,
    index=False
)

purge_delta_df.to_csv(
    PURGE_DELTA_PATH,
    index=False
)


for config_name, prediction_df in (
    corrected_prediction_tables.items()
):

    output_dir = (
        NAB_6030_CORRECTION_DIR
        if config_name == "60/30"
        else NAB_6015_CORRECTION_DIR
    )

    prediction_path = (
        output_dir
        / "test_predictions_non_neural_corrected.csv"
    )

    prediction_df.to_csv(
        prediction_path,
        index=False
    )

    print(
        f"\nCorrected {config_name} "
        f"prediction table saved to:"
    )

    print(
        prediction_path
    )


print(
    "\nSame-runtime metric table saved to:"
)

print(
    SAME_RUNTIME_METRICS_PATH
)

print(
    "\nPurge-only effect table saved to:"
)

print(
    PURGE_DELTA_PATH
)


# ============================================================
# H. Save methodological interpretation note
# ============================================================

PROVENANCE_NOTE_PATH = (
    NAB_CORRECTION_DIR
    / "README_NAB_CORRECTION_PROVENANCE.md"
)


provenance_note = """# NAB Split-Boundary Correction Provenance

The original NAB 60/30 and 60/15 experiments used overlapping
60-minute windows and chronological 60/20/20 splitting by series.
A later audit identified boundary-adjacent validation and test
windows that shared raw temporal observations with the preceding
partition.

The final correction preserves all training windows and all
training-only preprocessing. Only conflicting validation and test
windows are removed.

Historical artifact inspection also identified provenance
limitations in some archived prediction files. In particular, the
60/30 test-prediction file was overwritten by a later notebook
experiment after the original metric table had been generated.
The archived 60/15 XGBoost score and binary-prediction columns were
also not mutually consistent with the archived validation-selected
threshold.

For this reason, final corrected NAB results are reconstructed under
one documented software environment using the original model
configurations, feature representation, training populations,
validation percentile grid, and test-evaluation protocol.

For non-neural models, the same reconstructed fitted model is scored
on both the original unpurged and corrected purged evaluation
partitions. Their difference therefore isolates the effect of the
split-boundary purge under the current reconstruction environment.

Historical files are preserved unchanged as archival artifacts.
"""


with open(
    PROVENANCE_NOTE_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        provenance_note
    )


print(
    "\nCorrection provenance note saved to:"
)

print(
    PROVENANCE_NOTE_PATH
)


print("\n")
print("=" * 115)
print("CELL 24 COMPLETE")
print("=" * 115)

print(
    "\nNo historical artifact was overwritten. "
    "No additional model fitting was performed."
)

In [ ]:
# CELL 25 — NAB 60/30 Autoencoder and MLP Boundary-Correction Rerun

import random
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import (
    layers,
    models,
    callbacks,
)

from sklearn.utils.class_weight import (
    compute_class_weight,
)


print("=" * 115)
print("NAB 60/30 NEURAL BOUNDARY-CORRECTION RERUN")
print("=" * 115)


# ============================================================
# A. Fixed reproducibility protocol
# ============================================================

RANDOM_STATE = 42


def reset_neural_state():

    tf.keras.backend.clear_session()

    random.seed(
        RANDOM_STATE
    )

    np.random.seed(
        RANDOM_STATE
    )

    tf.keras.utils.set_random_seed(
        RANDOM_STATE
    )

    try:
        tf.config.experimental.enable_op_determinism()

    except Exception:
        pass


# ============================================================
# B. Exact historical architectures
# ============================================================

def build_nab_autoencoder(
    input_dim
):

    model = models.Sequential([
        layers.Input(
            shape=(input_dim,)
        ),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.Dense(
            32,
            activation="relu"
        ),

        layers.Dense(
            16,
            activation="relu"
        ),

        layers.Dense(
            32,
            activation="relu"
        ),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.Dense(
            input_dim,
            activation="linear"
        ),
    ])

    # Exact original specification
    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


def build_nab_mlp(
    input_dim
):

    model = models.Sequential([
        layers.Input(
            shape=(input_dim,)
        ),

        layers.Dense(
            128,
            activation="relu"
        ),

        layers.BatchNormalization(),

        layers.Dropout(
            0.25
        ),

        layers.Dense(
            64,
            activation="relu"
        ),

        layers.BatchNormalization(),

        layers.Dropout(
            0.25
        ),

        layers.Dense(
            32,
            activation="relu"
        ),

        layers.Dense(
            1,
            activation="sigmoid"
        ),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),

        loss="binary_crossentropy",

        metrics=[
            tf.keras.metrics.Precision(
                name="precision"
            ),

            tf.keras.metrics.Recall(
                name="recall"
            ),
        ]
    )

    return model


# ============================================================
# C. Evaluation helper
# Uses the exact NAB percentile grid and >= threshold rule
# already reconstructed in CELL 21.
# ============================================================

def evaluate_neural_scores(
    model_name,
    y_val,
    y_test,
    val_scores,
    test_scores
):

    best = (
        select_threshold_by_val_f1_reproduction(
            y_val,
            val_scores
        )
    )

    test_pred = (
        test_scores
        >= best["threshold"]
    ).astype(int)

    metrics = (
        compute_metrics_reproduction(
            y_test,
            test_pred,
            test_scores
        )
    )


    result = {
        "model":
            model_name,

        "selected_percentile":
            best["percentile"],

        "selected_threshold":
            float(
                best["threshold"]
            ),

        "val_precision":
            best["precision"],

        "val_recall":
            best["recall"],

        "val_f1":
            best["f1"],

        "test_precision":
            metrics["precision"],

        "test_recall":
            metrics["recall"],

        "test_f1":
            metrics["f1"],

        "test_accuracy":
            metrics["accuracy"],

        "test_fpr":
            metrics["fpr"],

        "test_roc_auc":
            metrics["roc_auc"],

        "test_pr_auc":
            metrics["pr_auc"],

        "tn":
            metrics["tn"],

        "fp":
            metrics["fp"],

        "fn":
            metrics["fn"],

        "tp":
            metrics["tp"],
    }

    return (
        result,
        test_pred
    )


# ============================================================
# D. Load reconstructed 60/30 matrices
# ============================================================

bundle = (
    nab_reconstruction[
        "60/30"
    ]
)

original = (
    bundle["original"]
)

corrected = (
    bundle["corrected"]
)


X_train = (
    original["X_train"]
)

y_train = (
    original["y_train"]
)


assert len(
    X_train
) == 62096

assert len(
    corrected["X_train"]
) == 62096


np.testing.assert_allclose(
    X_train,
    corrected["X_train"],
    rtol=1e-12,
    atol=1e-12
)


print(
    "\nTraining rows:",
    len(X_train)
)

print(
    "Original validation rows:",
    len(original["X_val"])
)

print(
    "Corrected validation rows:",
    len(corrected["X_val"])
)

print(
    "Original test rows:",
    len(original["X_test"])
)

print(
    "Corrected test rows:",
    len(corrected["X_test"])
)


input_dim = (
    X_train.shape[1]
)

assert input_dim == 23


# ============================================================
# E. MLP class weights — training population unchanged
# ============================================================

classes = np.array(
    [0, 1]
)

class_weights_array = (
    compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
)

class_weight_dict = {
    0:
        float(
            class_weights_array[0]
        ),

    1:
        float(
            class_weights_array[1]
        ),
}


print(
    "\nMLP class weights:"
)

print(
    class_weight_dict
)


# ============================================================
# F. Experiment versions
# ============================================================

VERSIONS = {

    "same_runtime_unpurged": {
        "X_val":
            original["X_val"],

        "y_val":
            original["y_val"],

        "X_test":
            original["X_test"],

        "y_test":
            original["y_test"],
    },

    "purged_corrected": {
        "X_val":
            corrected["X_val"],

        "y_val":
            corrected["y_val"],

        "X_test":
            corrected["X_test"],

        "y_test":
            corrected["y_test"],
    },
}


NEURAL_MODEL_DIR = (
    NAB_6030_CORRECTION_DIR
    / "reconstructed_models"
    / "neural"
)

NEURAL_HISTORY_DIR = (
    NAB_6030_CORRECTION_DIR
    / "neural_histories"
)

NEURAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

NEURAL_HISTORY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


metric_rows = []

prediction_tables = {}

history_summary_rows = []


# ============================================================
# G. Train same-runtime unpurged and purged versions
# ============================================================

for version_name, data in (
    VERSIONS.items()
):

    print("\n")
    print("=" * 115)
    print(
        version_name.upper()
    )
    print("=" * 115)


    X_val = data[
        "X_val"
    ]

    y_val = data[
        "y_val"
    ]

    X_test = data[
        "X_test"
    ]

    y_test = data[
        "y_test"
    ]


    # ========================================================
    # G1. AUTOENCODER
    # ========================================================

    print(
        "\nTraining Autoencoder..."
    )

    reset_neural_state()

    autoencoder = (
        build_nab_autoencoder(
            input_dim
        )
    )


    ae_early_stop = (
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    )


    ae_start = (
        time.perf_counter()
    )

    ae_history = (
        autoencoder.fit(
            X_train,
            X_train,

            validation_data=(
                X_val,
                X_val
            ),

            epochs=100,

            batch_size=256,

            callbacks=[
                ae_early_stop
            ],

            verbose=2
        )
    )

    ae_seconds = (
        time.perf_counter()
        - ae_start
    )


    ae_history_df = pd.DataFrame(
        ae_history.history
    )

    ae_history_df[
        "epoch"
    ] = (
        np.arange(
            1,
            len(ae_history_df) + 1
        )
    )


    ae_best_epoch = int(
        ae_history_df[
            "val_loss"
        ].idxmin()
        + 1
    )

    ae_best_val_loss = float(
        ae_history_df[
            "val_loss"
        ].min()
    )


    val_recon = (
        autoencoder.predict(
            X_val,
            batch_size=512,
            verbose=0
        )
    )

    test_recon = (
        autoencoder.predict(
            X_test,
            batch_size=512,
            verbose=0
        )
    )


    ae_val_scores = np.mean(
        (
            X_val
            - val_recon
        ) ** 2,
        axis=1
    )

    ae_test_scores = np.mean(
        (
            X_test
            - test_recon
        ) ** 2,
        axis=1
    )


    (
        ae_result,
        ae_test_pred
    ) = evaluate_neural_scores(
        "Autoencoder",
        y_val,
        y_test,
        ae_val_scores,
        ae_test_scores
    )


    ae_result[
        "configuration"
    ] = "60/30"

    ae_result[
        "evaluation_version"
    ] = version_name

    ae_result[
        "epochs_completed"
    ] = len(
        ae_history_df
    )

    ae_result[
        "best_epoch"
    ] = ae_best_epoch

    ae_result[
        "best_val_loss"
    ] = ae_best_val_loss

    ae_result[
        "training_seconds"
    ] = ae_seconds


    metric_rows.append(
        ae_result
    )


    history_summary_rows.append({
        "configuration":
            "60/30",

        "model":
            "Autoencoder",

        "evaluation_version":
            version_name,

        "epochs_completed":
            len(ae_history_df),

        "best_epoch":
            ae_best_epoch,

        "best_val_loss":
            ae_best_val_loss,

        "training_seconds":
            ae_seconds,
    })


    ae_history_path = (
        NEURAL_HISTORY_DIR
        / (
            f"autoencoder_"
            f"{version_name}_history.csv"
        )
    )

    ae_history_df.to_csv(
        ae_history_path,
        index=False
    )


    ae_model_path = (
        NEURAL_MODEL_DIR
        / (
            f"autoencoder_"
            f"{version_name}.keras"
        )
    )

    autoencoder.save(
        ae_model_path
    )


    print(
        "Autoencoder:"
    )

    print(
        f"  epochs completed: "
        f"{len(ae_history_df)}"
    )

    print(
        f"  best epoch: "
        f"{ae_best_epoch}"
    )

    print(
        f"  selected percentile: "
        f"{ae_result['selected_percentile']}"
    )

    print(
        f"  threshold: "
        f"{ae_result['selected_threshold']:.9f}"
    )

    print(
        f"  test F1: "
        f"{ae_result['test_f1']:.6f}"
    )

    print(
        f"  test FPR: "
        f"{ae_result['test_fpr']:.6f}"
    )


    # ========================================================
    # G2. MLP
    # ========================================================

    print(
        "\nTraining MLP..."
    )

    reset_neural_state()

    mlp = (
        build_nab_mlp(
            input_dim
        )
    )


    mlp_early_stop = (
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    )


    mlp_start = (
        time.perf_counter()
    )

    mlp_history = (
        mlp.fit(
            X_train,
            y_train,

            validation_data=(
                X_val,
                y_val
            ),

            epochs=100,

            batch_size=512,

            class_weight=(
                class_weight_dict
            ),

            callbacks=[
                mlp_early_stop
            ],

            verbose=2
        )
    )

    mlp_seconds = (
        time.perf_counter()
        - mlp_start
    )


    mlp_history_df = pd.DataFrame(
        mlp_history.history
    )

    mlp_history_df[
        "epoch"
    ] = (
        np.arange(
            1,
            len(mlp_history_df) + 1
        )
    )


    mlp_best_epoch = int(
        mlp_history_df[
            "val_loss"
        ].idxmin()
        + 1
    )

    mlp_best_val_loss = float(
        mlp_history_df[
            "val_loss"
        ].min()
    )


    mlp_val_scores = (
        mlp.predict(
            X_val,
            batch_size=512,
            verbose=0
        )
        .ravel()
    )

    mlp_test_scores = (
        mlp.predict(
            X_test,
            batch_size=512,
            verbose=0
        )
        .ravel()
    )


    (
        mlp_result,
        mlp_test_pred
    ) = evaluate_neural_scores(
        "MLP",
        y_val,
        y_test,
        mlp_val_scores,
        mlp_test_scores
    )


    mlp_result[
        "configuration"
    ] = "60/30"

    mlp_result[
        "evaluation_version"
    ] = version_name

    mlp_result[
        "epochs_completed"
    ] = len(
        mlp_history_df
    )

    mlp_result[
        "best_epoch"
    ] = mlp_best_epoch

    mlp_result[
        "best_val_loss"
    ] = mlp_best_val_loss

    mlp_result[
        "training_seconds"
    ] = mlp_seconds


    metric_rows.append(
        mlp_result
    )


    history_summary_rows.append({
        "configuration":
            "60/30",

        "model":
            "MLP",

        "evaluation_version":
            version_name,

        "epochs_completed":
            len(mlp_history_df),

        "best_epoch":
            mlp_best_epoch,

        "best_val_loss":
            mlp_best_val_loss,

        "training_seconds":
            mlp_seconds,
    })


    mlp_history_path = (
        NEURAL_HISTORY_DIR
        / (
            f"mlp_"
            f"{version_name}_history.csv"
        )
    )

    mlp_history_df.to_csv(
        mlp_history_path,
        index=False
    )


    mlp_model_path = (
        NEURAL_MODEL_DIR
        / (
            f"mlp_"
            f"{version_name}.keras"
        )
    )

    mlp.save(
        mlp_model_path
    )


    print(
        "MLP:"
    )

    print(
        f"  epochs completed: "
        f"{len(mlp_history_df)}"
    )

    print(
        f"  best epoch: "
        f"{mlp_best_epoch}"
    )

    print(
        f"  selected percentile: "
        f"{mlp_result['selected_percentile']}"
    )

    print(
        f"  threshold: "
        f"{mlp_result['selected_threshold']:.9f}"
    )

    print(
        f"  test F1: "
        f"{mlp_result['test_f1']:.6f}"
    )

    print(
        f"  test FPR: "
        f"{mlp_result['test_fpr']:.6f}"
    )


    # ========================================================
    # G3. Save version-specific test predictions
    # ========================================================

    test_df = (
        data[
            "X_test"
        ]
    )

    source_test_df = (
        original["test_df"]
        if version_name
        == "same_runtime_unpurged"
        else corrected["test_df"]
    )


    pred_table = (
        source_test_df[
            [
                "series",
                "domain",
                "window_start",
                "window_end",
                "label",
            ]
        ]
        .reset_index(drop=True)
        .copy()
    )


    pred_table[
        "ae_score"
    ] = ae_test_scores

    pred_table[
        "ae_pred"
    ] = ae_test_pred

    pred_table[
        "ae_threshold"
    ] = ae_result[
        "selected_threshold"
    ]

    pred_table[
        "ae_selected_percentile"
    ] = ae_result[
        "selected_percentile"
    ]


    pred_table[
        "mlp_score"
    ] = mlp_test_scores

    pred_table[
        "mlp_pred"
    ] = mlp_test_pred

    pred_table[
        "mlp_threshold"
    ] = mlp_result[
        "selected_threshold"
    ]

    pred_table[
        "mlp_selected_percentile"
    ] = mlp_result[
        "selected_percentile"
    ]


    prediction_tables[
        version_name
    ] = pred_table


# ============================================================
# H. Metrics and boundary-correction effect
# ============================================================

neural_metrics_df = pd.DataFrame(
    metric_rows
)

history_summary_df = pd.DataFrame(
    history_summary_rows
)


delta_rows = []


for model_name in [
    "Autoencoder",
    "MLP",
]:

    unpurged = (
        neural_metrics_df[
            (
                neural_metrics_df[
                    "model"
                ] == model_name
            )
            &
            (
                neural_metrics_df[
                    "evaluation_version"
                ]
                ==
                "same_runtime_unpurged"
            )
        ]
        .iloc[0]
    )

    corrected_row = (
        neural_metrics_df[
            (
                neural_metrics_df[
                    "model"
                ] == model_name
            )
            &
            (
                neural_metrics_df[
                    "evaluation_version"
                ]
                ==
                "purged_corrected"
            )
        ]
        .iloc[0]
    )


    delta_rows.append({
        "configuration":
            "60/30",

        "model":
            model_name,

        "unpurged_percentile":
            unpurged[
                "selected_percentile"
            ],

        "corrected_percentile":
            corrected_row[
                "selected_percentile"
            ],

        "unpurged_threshold":
            unpurged[
                "selected_threshold"
            ],

        "corrected_threshold":
            corrected_row[
                "selected_threshold"
            ],

        "threshold_delta":
            (
                corrected_row[
                    "selected_threshold"
                ]
                -
                unpurged[
                    "selected_threshold"
                ]
            ),

        "unpurged_f1":
            unpurged[
                "test_f1"
            ],

        "corrected_f1":
            corrected_row[
                "test_f1"
            ],

        "f1_delta":
            (
                corrected_row[
                    "test_f1"
                ]
                -
                unpurged[
                    "test_f1"
                ]
            ),

        "unpurged_fpr":
            unpurged[
                "test_fpr"
            ],

        "corrected_fpr":
            corrected_row[
                "test_fpr"
            ],

        "fpr_delta":
            (
                corrected_row[
                    "test_fpr"
                ]
                -
                unpurged[
                    "test_fpr"
                ]
            ),

        "unpurged_roc_auc":
            unpurged[
                "test_roc_auc"
            ],

        "corrected_roc_auc":
            corrected_row[
                "test_roc_auc"
            ],

        "roc_auc_delta":
            (
                corrected_row[
                    "test_roc_auc"
                ]
                -
                unpurged[
                    "test_roc_auc"
                ]
            ),

        "unpurged_pr_auc":
            unpurged[
                "test_pr_auc"
            ],

        "corrected_pr_auc":
            corrected_row[
                "test_pr_auc"
            ],

        "pr_auc_delta":
            (
                corrected_row[
                    "test_pr_auc"
                ]
                -
                unpurged[
                    "test_pr_auc"
                ]
            ),

        "unpurged_epochs":
            int(
                unpurged[
                    "epochs_completed"
                ]
            ),

        "corrected_epochs":
            int(
                corrected_row[
                    "epochs_completed"
                ]
            ),

        "unpurged_best_epoch":
            int(
                unpurged[
                    "best_epoch"
                ]
            ),

        "corrected_best_epoch":
            int(
                corrected_row[
                    "best_epoch"
                ]
            ),
    })


neural_delta_df = pd.DataFrame(
    delta_rows
)


print("\n")
print("=" * 115)
print("60/30 NEURAL BOUNDARY-CORRECTION EFFECT")
print("=" * 115)


print(
    neural_delta_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# I. Compare current unpurged metrics with frozen historical
#    metrics — descriptive provenance only
# ============================================================

historical_metrics = (
    bundle[
        "historical_metrics"
    ]
)


historical_comparison_rows = []


for model_name in [
    "Autoencoder",
    "MLP",
]:

    historical = (
        historical_metrics[
            historical_metrics[
                "model"
            ] == model_name
        ]
        .iloc[0]
    )

    current = (
        neural_metrics_df[
            (
                neural_metrics_df[
                    "model"
                ] == model_name
            )
            &
            (
                neural_metrics_df[
                    "evaluation_version"
                ]
                ==
                "same_runtime_unpurged"
            )
        ]
        .iloc[0]
    )


    historical_comparison_rows.append({
        "model":
            model_name,

        "historical_percentile":
            historical[
                "selected_percentile"
            ],

        "current_percentile":
            current[
                "selected_percentile"
            ],

        "historical_threshold":
            historical[
                "selected_threshold"
            ],

        "current_threshold":
            current[
                "selected_threshold"
            ],

        "historical_f1":
            historical[
                "test_f1"
            ],

        "current_f1":
            current[
                "test_f1"
            ],

        "historical_fpr":
            historical[
                "test_fpr"
            ],

        "current_fpr":
            current[
                "test_fpr"
            ],

        "historical_roc_auc":
            historical[
                "test_roc_auc"
            ],

        "current_roc_auc":
            current[
                "test_roc_auc"
            ],

        "historical_pr_auc":
            historical[
                "test_pr_auc"
            ],

        "current_pr_auc":
            current[
                "test_pr_auc"
            ],
    })


historical_comparison_df = pd.DataFrame(
    historical_comparison_rows
)


print("\n")
print("=" * 115)
print("FROZEN HISTORICAL VS CURRENT SAME-RUNTIME UNPURGED")
print("=" * 115)


print(
    historical_comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# J. Corrected confusion total check
# ============================================================

corrected_metrics_only = (
    neural_metrics_df[
        neural_metrics_df[
            "evaluation_version"
        ]
        ==
        "purged_corrected"
    ]
)


for _, row in (
    corrected_metrics_only.iterrows()
):

    total = (
        row["tn"]
        + row["fp"]
        + row["fn"]
        + row["tp"]
    )

    assert int(
        total
    ) == 20668


print(
    "\nPASS: corrected Autoencoder and MLP "
    "confusion matrices each total 20,668 test windows."
)


# ============================================================
# K. Save results
# ============================================================

NEURAL_METRICS_PATH = (
    NAB_6030_CORRECTION_DIR
    / "neural_same_runtime_unpurged_and_corrected_metrics.csv"
)

NEURAL_DELTA_PATH = (
    NAB_6030_CORRECTION_DIR
    / "neural_boundary_correction_effect.csv"
)

NEURAL_HISTORY_SUMMARY_PATH = (
    NAB_6030_CORRECTION_DIR
    / "neural_training_history_summary.csv"
)

NEURAL_HISTORICAL_COMPARISON_PATH = (
    NAB_6030_CORRECTION_DIR
    / "neural_historical_vs_current_unpurged.csv"
)


neural_metrics_df.to_csv(
    NEURAL_METRICS_PATH,
    index=False
)

neural_delta_df.to_csv(
    NEURAL_DELTA_PATH,
    index=False
)

history_summary_df.to_csv(
    NEURAL_HISTORY_SUMMARY_PATH,
    index=False
)

historical_comparison_df.to_csv(
    NEURAL_HISTORICAL_COMPARISON_PATH,
    index=False
)


for version_name, pred_table in (
    prediction_tables.items()
):

    pred_path = (
        NAB_6030_CORRECTION_DIR
        / (
            f"test_predictions_neural_"
            f"{version_name}.csv"
        )
    )

    pred_table.to_csv(
        pred_path,
        index=False
    )


print(
    "\nMetrics saved to:"
)

print(
    NEURAL_METRICS_PATH
)

print(
    "\nBoundary-correction effect saved to:"
)

print(
    NEURAL_DELTA_PATH
)

print(
    "\nTraining summary saved to:"
)

print(
    NEURAL_HISTORY_SUMMARY_PATH
)

print(
    "\nHistorical comparison saved to:"
)

print(
    NEURAL_HISTORICAL_COMPARISON_PATH
)


print("\n")
print("=" * 115)
print("CELL 25 COMPLETE")
print("=" * 115)

print(
    "\nOriginal NAB artifacts were not modified. "
    "Four new deterministic neural fits were created "
    "inside the correction experiment only."
)

In [ ]:
# CELL 26 — NAB 60/15 Autoencoder and MLP Boundary-Correction Rerun

import time
import numpy as np
import pandas as pd

from sklearn.utils.class_weight import (
    compute_class_weight,
)

from tensorflow.keras import (
    callbacks,
)


print("=" * 115)
print("NAB 60/15 NEURAL BOUNDARY-CORRECTION RERUN")
print("=" * 115)


# ============================================================
# A. Load reconstructed 60/15 matrices
# ============================================================

bundle_6015 = (
    nab_reconstruction[
        "60/15"
    ]
)

original_6015 = (
    bundle_6015[
        "original"
    ]
)

corrected_6015 = (
    bundle_6015[
        "corrected"
    ]
)


X_train_6015 = (
    original_6015[
        "X_train"
    ]
)

y_train_6015 = (
    original_6015[
        "y_train"
    ]
)


# Training population must remain unchanged
assert len(
    X_train_6015
) == 124204

assert len(
    corrected_6015[
        "X_train"
    ]
) == 124204


np.testing.assert_allclose(
    X_train_6015,
    corrected_6015[
        "X_train"
    ],
    rtol=1e-12,
    atol=1e-12
)


print(
    "\nTraining rows:",
    len(X_train_6015)
)

print(
    "Original validation rows:",
    len(
        original_6015[
            "X_val"
        ]
    )
)

print(
    "Corrected validation rows:",
    len(
        corrected_6015[
            "X_val"
        ]
    )
)

print(
    "Original test rows:",
    len(
        original_6015[
            "X_test"
        ]
    )
)

print(
    "Corrected test rows:",
    len(
        corrected_6015[
            "X_test"
        ]
    )
)


input_dim_6015 = (
    X_train_6015.shape[1]
)

assert (
    input_dim_6015 == 23
)


# ============================================================
# B. Exact MLP class weights from unchanged training labels
# ============================================================

classes_6015 = np.array(
    [0, 1]
)

class_weights_array_6015 = (
    compute_class_weight(
        class_weight="balanced",
        classes=classes_6015,
        y=y_train_6015
    )
)

class_weight_dict_6015 = {
    0:
        float(
            class_weights_array_6015[
                0
            ]
        ),

    1:
        float(
            class_weights_array_6015[
                1
            ]
        ),
}


print(
    "\nMLP class weights:"
)

print(
    class_weight_dict_6015
)


# ============================================================
# C. Evaluation versions
# ============================================================

VERSIONS_6015 = {

    "same_runtime_unpurged": {
        "X_val":
            original_6015[
                "X_val"
            ],

        "y_val":
            original_6015[
                "y_val"
            ],

        "X_test":
            original_6015[
                "X_test"
            ],

        "y_test":
            original_6015[
                "y_test"
            ],

        "test_df":
            original_6015[
                "test_df"
            ],
    },

    "purged_corrected": {
        "X_val":
            corrected_6015[
                "X_val"
            ],

        "y_val":
            corrected_6015[
                "y_val"
            ],

        "X_test":
            corrected_6015[
                "X_test"
            ],

        "y_test":
            corrected_6015[
                "y_test"
            ],

        "test_df":
            corrected_6015[
                "test_df"
            ],
    },
}


# ============================================================
# D. Output folders
# ============================================================

NEURAL_MODEL_DIR_6015 = (
    NAB_6015_CORRECTION_DIR
    / "reconstructed_models"
    / "neural"
)

NEURAL_HISTORY_DIR_6015 = (
    NAB_6015_CORRECTION_DIR
    / "neural_histories"
)


NEURAL_MODEL_DIR_6015.mkdir(
    parents=True,
    exist_ok=True
)

NEURAL_HISTORY_DIR_6015.mkdir(
    parents=True,
    exist_ok=True
)


metric_rows_6015 = []

history_summary_rows_6015 = []

prediction_tables_6015 = {}


# ============================================================
# E. Run unpurged and corrected versions
# ============================================================

for version_name, data in (
    VERSIONS_6015.items()
):

    print("\n")
    print("=" * 115)
    print(
        version_name.upper()
    )
    print("=" * 115)


    X_val = data[
        "X_val"
    ]

    y_val = data[
        "y_val"
    ]

    X_test = data[
        "X_test"
    ]

    y_test = data[
        "y_test"
    ]


    # ========================================================
    # E1. AUTOENCODER
    # ========================================================

    print(
        "\nTraining Autoencoder..."
    )


    reset_neural_state()


    autoencoder_6015 = (
        build_nab_autoencoder(
            input_dim_6015
        )
    )


    ae_early_stop_6015 = (
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    )


    ae_start = (
        time.perf_counter()
    )


    ae_history_6015 = (
        autoencoder_6015.fit(

            X_train_6015,
            X_train_6015,

            validation_data=(
                X_val,
                X_val
            ),

            epochs=100,

            batch_size=256,

            callbacks=[
                ae_early_stop_6015
            ],

            verbose=2
        )
    )


    ae_seconds = (
        time.perf_counter()
        - ae_start
    )


    ae_history_df_6015 = pd.DataFrame(
        ae_history_6015.history
    )


    ae_history_df_6015[
        "epoch"
    ] = np.arange(
        1,
        len(
            ae_history_df_6015
        ) + 1
    )


    ae_best_epoch = int(
        ae_history_df_6015[
            "val_loss"
        ].idxmin()
        + 1
    )


    ae_best_val_loss = float(
        ae_history_df_6015[
            "val_loss"
        ].min()
    )


    val_recon = (
        autoencoder_6015.predict(
            X_val,
            batch_size=512,
            verbose=0
        )
    )


    test_recon = (
        autoencoder_6015.predict(
            X_test,
            batch_size=512,
            verbose=0
        )
    )


    ae_val_scores = np.mean(
        (
            X_val
            - val_recon
        ) ** 2,
        axis=1
    )


    ae_test_scores = np.mean(
        (
            X_test
            - test_recon
        ) ** 2,
        axis=1
    )


    (
        ae_result,
        ae_test_pred
    ) = evaluate_neural_scores(

        "Autoencoder",

        y_val,
        y_test,

        ae_val_scores,
        ae_test_scores
    )


    ae_result[
        "configuration"
    ] = "60/15"

    ae_result[
        "evaluation_version"
    ] = version_name

    ae_result[
        "epochs_completed"
    ] = len(
        ae_history_df_6015
    )

    ae_result[
        "best_epoch"
    ] = ae_best_epoch

    ae_result[
        "best_val_loss"
    ] = ae_best_val_loss

    ae_result[
        "training_seconds"
    ] = ae_seconds


    metric_rows_6015.append(
        ae_result
    )


    history_summary_rows_6015.append({

        "configuration":
            "60/15",

        "model":
            "Autoencoder",

        "evaluation_version":
            version_name,

        "epochs_completed":
            len(
                ae_history_df_6015
            ),

        "best_epoch":
            ae_best_epoch,

        "best_val_loss":
            ae_best_val_loss,

        "training_seconds":
            ae_seconds,
    })


    ae_history_path = (
        NEURAL_HISTORY_DIR_6015
        / (
            f"autoencoder_"
            f"{version_name}_history.csv"
        )
    )


    ae_history_df_6015.to_csv(
        ae_history_path,
        index=False
    )


    ae_model_path = (
        NEURAL_MODEL_DIR_6015
        / (
            f"autoencoder_"
            f"{version_name}.keras"
        )
    )


    autoencoder_6015.save(
        ae_model_path
    )


    print(
        "Autoencoder:"
    )

    print(
        f"  epochs completed: "
        f"{len(ae_history_df_6015)}"
    )

    print(
        f"  best epoch: "
        f"{ae_best_epoch}"
    )

    print(
        f"  selected percentile: "
        f"{ae_result['selected_percentile']}"
    )

    print(
        f"  threshold: "
        f"{ae_result['selected_threshold']:.9f}"
    )

    print(
        f"  test F1: "
        f"{ae_result['test_f1']:.6f}"
    )

    print(
        f"  test FPR: "
        f"{ae_result['test_fpr']:.6f}"
    )


    # ========================================================
    # E2. MLP
    # ========================================================

    print(
        "\nTraining MLP..."
    )


    reset_neural_state()


    mlp_6015 = (
        build_nab_mlp(
            input_dim_6015
        )
    )


    mlp_early_stop_6015 = (
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    )


    mlp_start = (
        time.perf_counter()
    )


    mlp_history_6015 = (
        mlp_6015.fit(

            X_train_6015,
            y_train_6015,

            validation_data=(
                X_val,
                y_val
            ),

            epochs=100,

            batch_size=512,

            class_weight=(
                class_weight_dict_6015
            ),

            callbacks=[
                mlp_early_stop_6015
            ],

            verbose=2
        )
    )


    mlp_seconds = (
        time.perf_counter()
        - mlp_start
    )


    mlp_history_df_6015 = pd.DataFrame(
        mlp_history_6015.history
    )


    mlp_history_df_6015[
        "epoch"
    ] = np.arange(
        1,
        len(
            mlp_history_df_6015
        ) + 1
    )


    mlp_best_epoch = int(
        mlp_history_df_6015[
            "val_loss"
        ].idxmin()
        + 1
    )


    mlp_best_val_loss = float(
        mlp_history_df_6015[
            "val_loss"
        ].min()
    )


    mlp_val_scores = (
        mlp_6015.predict(
            X_val,
            batch_size=512,
            verbose=0
        )
        .ravel()
    )


    mlp_test_scores = (
        mlp_6015.predict(
            X_test,
            batch_size=512,
            verbose=0
        )
        .ravel()
    )


    (
        mlp_result,
        mlp_test_pred
    ) = evaluate_neural_scores(

        "MLP",

        y_val,
        y_test,

        mlp_val_scores,
        mlp_test_scores
    )


    mlp_result[
        "configuration"
    ] = "60/15"

    mlp_result[
        "evaluation_version"
    ] = version_name

    mlp_result[
        "epochs_completed"
    ] = len(
        mlp_history_df_6015
    )

    mlp_result[
        "best_epoch"
    ] = mlp_best_epoch

    mlp_result[
        "best_val_loss"
    ] = mlp_best_val_loss

    mlp_result[
        "training_seconds"
    ] = mlp_seconds


    metric_rows_6015.append(
        mlp_result
    )


    history_summary_rows_6015.append({

        "configuration":
            "60/15",

        "model":
            "MLP",

        "evaluation_version":
            version_name,

        "epochs_completed":
            len(
                mlp_history_df_6015
            ),

        "best_epoch":
            mlp_best_epoch,

        "best_val_loss":
            mlp_best_val_loss,

        "training_seconds":
            mlp_seconds,
    })


    mlp_history_path = (
        NEURAL_HISTORY_DIR_6015
        / (
            f"mlp_"
            f"{version_name}_history.csv"
        )
    )


    mlp_history_df_6015.to_csv(
        mlp_history_path,
        index=False
    )


    mlp_model_path = (
        NEURAL_MODEL_DIR_6015
        / (
            f"mlp_"
            f"{version_name}.keras"
        )
    )


    mlp_6015.save(
        mlp_model_path
    )


    print(
        "MLP:"
    )

    print(
        f"  epochs completed: "
        f"{len(mlp_history_df_6015)}"
    )

    print(
        f"  best epoch: "
        f"{mlp_best_epoch}"
    )

    print(
        f"  selected percentile: "
        f"{mlp_result['selected_percentile']}"
    )

    print(
        f"  threshold: "
        f"{mlp_result['selected_threshold']:.9f}"
    )

    print(
        f"  test F1: "
        f"{mlp_result['test_f1']:.6f}"
    )

    print(
        f"  test FPR: "
        f"{mlp_result['test_fpr']:.6f}"
    )


    # ========================================================
    # E3. Save test scores and predictions
    # ========================================================

    source_test_df = (
        data[
            "test_df"
        ]
    )


    pred_table = (
        source_test_df[
            [
                "series",
                "domain",
                "window_start",
                "window_end",
                "label",
            ]
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )


    pred_table[
        "ae_score"
    ] = ae_test_scores

    pred_table[
        "ae_pred"
    ] = ae_test_pred

    pred_table[
        "ae_threshold"
    ] = ae_result[
        "selected_threshold"
    ]

    pred_table[
        "ae_selected_percentile"
    ] = ae_result[
        "selected_percentile"
    ]


    pred_table[
        "mlp_score"
    ] = mlp_test_scores

    pred_table[
        "mlp_pred"
    ] = mlp_test_pred

    pred_table[
        "mlp_threshold"
    ] = mlp_result[
        "selected_threshold"
    ]

    pred_table[
        "mlp_selected_percentile"
    ] = mlp_result[
        "selected_percentile"
    ]


    prediction_tables_6015[
        version_name
    ] = pred_table


# ============================================================
# F. Consolidate metrics
# ============================================================

neural_metrics_6015_df = pd.DataFrame(
    metric_rows_6015
)

history_summary_6015_df = pd.DataFrame(
    history_summary_rows_6015
)


# ============================================================
# G. Boundary-purge effect
# ============================================================

delta_rows_6015 = []


for model_name in [
    "Autoencoder",
    "MLP",
]:

    unpurged = (
        neural_metrics_6015_df[
            (
                neural_metrics_6015_df[
                    "model"
                ] == model_name
            )
            &
            (
                neural_metrics_6015_df[
                    "evaluation_version"
                ]
                ==
                "same_runtime_unpurged"
            )
        ]
        .iloc[0]
    )


    corrected_row = (
        neural_metrics_6015_df[
            (
                neural_metrics_6015_df[
                    "model"
                ] == model_name
            )
            &
            (
                neural_metrics_6015_df[
                    "evaluation_version"
                ]
                ==
                "purged_corrected"
            )
        ]
        .iloc[0]
    )


    delta_rows_6015.append({

        "configuration":
            "60/15",

        "model":
            model_name,

        "unpurged_percentile":
            unpurged[
                "selected_percentile"
            ],

        "corrected_percentile":
            corrected_row[
                "selected_percentile"
            ],

        "unpurged_threshold":
            unpurged[
                "selected_threshold"
            ],

        "corrected_threshold":
            corrected_row[
                "selected_threshold"
            ],

        "threshold_delta":
            (
                corrected_row[
                    "selected_threshold"
                ]
                -
                unpurged[
                    "selected_threshold"
                ]
            ),

        "unpurged_f1":
            unpurged[
                "test_f1"
            ],

        "corrected_f1":
            corrected_row[
                "test_f1"
            ],

        "f1_delta":
            (
                corrected_row[
                    "test_f1"
                ]
                -
                unpurged[
                    "test_f1"
                ]
            ),

        "unpurged_fpr":
            unpurged[
                "test_fpr"
            ],

        "corrected_fpr":
            corrected_row[
                "test_fpr"
            ],

        "fpr_delta":
            (
                corrected_row[
                    "test_fpr"
                ]
                -
                unpurged[
                    "test_fpr"
                ]
            ),

        "unpurged_roc_auc":
            unpurged[
                "test_roc_auc"
            ],

        "corrected_roc_auc":
            corrected_row[
                "test_roc_auc"
            ],

        "roc_auc_delta":
            (
                corrected_row[
                    "test_roc_auc"
                ]
                -
                unpurged[
                    "test_roc_auc"
                ]
            ),

        "unpurged_pr_auc":
            unpurged[
                "test_pr_auc"
            ],

        "corrected_pr_auc":
            corrected_row[
                "test_pr_auc"
            ],

        "pr_auc_delta":
            (
                corrected_row[
                    "test_pr_auc"
                ]
                -
                unpurged[
                    "test_pr_auc"
                ]
            ),

        "unpurged_epochs":
            int(
                unpurged[
                    "epochs_completed"
                ]
            ),

        "corrected_epochs":
            int(
                corrected_row[
                    "epochs_completed"
                ]
            ),

        "unpurged_best_epoch":
            int(
                unpurged[
                    "best_epoch"
                ]
            ),

        "corrected_best_epoch":
            int(
                corrected_row[
                    "best_epoch"
                ]
            ),
    })


neural_delta_6015_df = pd.DataFrame(
    delta_rows_6015
)


print("\n")
print("=" * 115)
print("60/15 NEURAL BOUNDARY-CORRECTION EFFECT")
print("=" * 115)


print(
    neural_delta_6015_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# H. Frozen historical vs current same-runtime unpurged
# ============================================================

historical_metrics_6015 = (
    bundle_6015[
        "historical_metrics"
    ]
)


historical_comparison_rows_6015 = []


for model_name in [
    "Autoencoder",
    "MLP",
]:

    historical = (
        historical_metrics_6015[
            historical_metrics_6015[
                "model"
            ]
            ==
            model_name
        ]
        .iloc[0]
    )


    current = (
        neural_metrics_6015_df[
            (
                neural_metrics_6015_df[
                    "model"
                ]
                ==
                model_name
            )
            &
            (
                neural_metrics_6015_df[
                    "evaluation_version"
                ]
                ==
                "same_runtime_unpurged"
            )
        ]
        .iloc[0]
    )


    historical_comparison_rows_6015.append({

        "model":
            model_name,

        "historical_percentile":
            historical[
                "selected_percentile"
            ],

        "current_percentile":
            current[
                "selected_percentile"
            ],

        "historical_threshold":
            historical[
                "selected_threshold"
            ],

        "current_threshold":
            current[
                "selected_threshold"
            ],

        "historical_f1":
            historical[
                "test_f1"
            ],

        "current_f1":
            current[
                "test_f1"
            ],

        "historical_fpr":
            historical[
                "test_fpr"
            ],

        "current_fpr":
            current[
                "test_fpr"
            ],

        "historical_roc_auc":
            historical[
                "test_roc_auc"
            ],

        "current_roc_auc":
            current[
                "test_roc_auc"
            ],

        "historical_pr_auc":
            historical[
                "test_pr_auc"
            ],

        "current_pr_auc":
            current[
                "test_pr_auc"
            ],
    })


historical_comparison_6015_df = pd.DataFrame(
    historical_comparison_rows_6015
)


print("\n")
print("=" * 115)
print("FROZEN HISTORICAL VS CURRENT SAME-RUNTIME UNPURGED")
print("=" * 115)


print(
    historical_comparison_6015_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# I. Corrected confusion-matrix total
# ============================================================

corrected_metrics_6015_only = (
    neural_metrics_6015_df[
        neural_metrics_6015_df[
            "evaluation_version"
        ]
        ==
        "purged_corrected"
    ]
)


for _, row in (
    corrected_metrics_6015_only.iterrows()
):

    total = (
        row["tn"]
        + row["fp"]
        + row["fn"]
        + row["tp"]
    )

    assert (
        int(total)
        == 41254
    )


print(
    "\nPASS: corrected Autoencoder and MLP "
    "confusion matrices each total 41,254 test windows."
)


# ============================================================
# J. Save outputs
# ============================================================

NEURAL_METRICS_6015_PATH = (
    NAB_6015_CORRECTION_DIR
    / "neural_same_runtime_unpurged_and_corrected_metrics.csv"
)

NEURAL_DELTA_6015_PATH = (
    NAB_6015_CORRECTION_DIR
    / "neural_boundary_correction_effect.csv"
)

NEURAL_HISTORY_SUMMARY_6015_PATH = (
    NAB_6015_CORRECTION_DIR
    / "neural_training_history_summary.csv"
)

NEURAL_HISTORICAL_COMPARISON_6015_PATH = (
    NAB_6015_CORRECTION_DIR
    / "neural_historical_vs_current_unpurged.csv"
)


neural_metrics_6015_df.to_csv(
    NEURAL_METRICS_6015_PATH,
    index=False
)

neural_delta_6015_df.to_csv(
    NEURAL_DELTA_6015_PATH,
    index=False
)

history_summary_6015_df.to_csv(
    NEURAL_HISTORY_SUMMARY_6015_PATH,
    index=False
)

historical_comparison_6015_df.to_csv(
    NEURAL_HISTORICAL_COMPARISON_6015_PATH,
    index=False
)


for version_name, pred_table in (
    prediction_tables_6015.items()
):

    pred_path = (
        NAB_6015_CORRECTION_DIR
        / (
            f"test_predictions_neural_"
            f"{version_name}.csv"
        )
    )

    pred_table.to_csv(
        pred_path,
        index=False
    )


print(
    "\nMetrics saved to:"
)

print(
    NEURAL_METRICS_6015_PATH
)

print(
    "\nBoundary-correction effect saved to:"
)

print(
    NEURAL_DELTA_6015_PATH
)

print(
    "\nTraining summary saved to:"
)

print(
    NEURAL_HISTORY_SUMMARY_6015_PATH
)

print(
    "\nHistorical comparison saved to:"
)

print(
    NEURAL_HISTORICAL_COMPARISON_6015_PATH
)


print("\n")
print("=" * 115)
print("CELL 26 COMPLETE")
print("=" * 115)

print(
    "\nOriginal NAB artifacts were not modified. "
    "Four deterministic 60/15 neural fits were saved "
    "inside the correction experiment only."
)

In [ ]:
# CELL 27 — Consolidate Final Leakage-Corrected NAB Seven-Model Results

import numpy as np
import pandas as pd
from pathlib import Path


print("=" * 120)
print("FINAL LEAKAGE-CORRECTED NAB SEVEN-MODEL CONSOLIDATION")
print("=" * 120)


# ============================================================
# A. Paths to correction outputs
# ============================================================

NON_NEURAL_METRICS_PATH = (
    NAB_CORRECTION_DIR
    / "nab_non_neural_same_runtime_unpurged_and_corrected_metrics.csv"
)

NEURAL_6030_PATH = (
    NAB_6030_CORRECTION_DIR
    / "neural_same_runtime_unpurged_and_corrected_metrics.csv"
)

NEURAL_6015_PATH = (
    NAB_6015_CORRECTION_DIR
    / "neural_same_runtime_unpurged_and_corrected_metrics.csv"
)


print("\nInput artifacts:")

for path in [
    NON_NEURAL_METRICS_PATH,
    NEURAL_6030_PATH,
    NEURAL_6015_PATH,
]:

    print(
        path,
        "| exists:",
        path.exists()
    )

    assert path.exists()


# ============================================================
# B. Load all current-environment metrics
# ============================================================

non_neural_df = pd.read_csv(
    NON_NEURAL_METRICS_PATH
)

neural_6030_df = pd.read_csv(
    NEURAL_6030_PATH
)

neural_6015_df = pd.read_csv(
    NEURAL_6015_PATH
)


print(
    "\nLoaded rows:"
)

print(
    "Non-neural:",
    len(non_neural_df)
)

print(
    "Neural 60/30:",
    len(neural_6030_df)
)

print(
    "Neural 60/15:",
    len(neural_6015_df)
)


# ============================================================
# C. Keep corrected rows only
# ============================================================

non_neural_corrected = (
    non_neural_df[
        non_neural_df[
            "evaluation_version"
        ] == "purged_corrected"
    ]
    .copy()
)


neural_corrected = pd.concat(
    [
        neural_6030_df[
            neural_6030_df[
                "evaluation_version"
            ] == "purged_corrected"
        ],

        neural_6015_df[
            neural_6015_df[
                "evaluation_version"
            ] == "purged_corrected"
        ],
    ],
    ignore_index=True,
    sort=False
)


print(
    "\nCorrected rows:"
)

print(
    "Non-neural:",
    len(non_neural_corrected)
)

print(
    "Neural:",
    len(neural_corrected)
)


assert len(
    non_neural_corrected
) == 10

assert len(
    neural_corrected
) == 4


# ============================================================
# D. Common final metric schema
# ============================================================

FINAL_COLUMNS = [
    "configuration",
    "model",

    "selected_percentile",
    "selected_threshold",

    "val_precision",
    "val_recall",
    "val_f1",

    "test_precision",
    "test_recall",
    "test_f1",
    "test_accuracy",
    "test_fpr",
    "test_roc_auc",
    "test_pr_auc",

    "tn",
    "fp",
    "fn",
    "tp",
]


for col in FINAL_COLUMNS:

    assert col in non_neural_corrected.columns, (
        f"Missing non-neural column: {col}"
    )

    assert col in neural_corrected.columns, (
        f"Missing neural column: {col}"
    )


final_corrected_df = pd.concat(
    [
        non_neural_corrected[
            FINAL_COLUMNS
        ],

        neural_corrected[
            FINAL_COLUMNS
        ],
    ],
    ignore_index=True
)


# ============================================================
# E. Normalize model order
# ============================================================

MODEL_ORDER = [
    "Isolation Forest",
    "LOF",
    "Autoencoder",
    "Random Forest",
    "XGBoost",
    "Logistic Regression",
    "MLP",
]


CONFIG_ORDER = [
    "60/30",
    "60/15",
]


final_corrected_df[
    "model"
] = pd.Categorical(
    final_corrected_df[
        "model"
    ],
    categories=MODEL_ORDER,
    ordered=True
)


final_corrected_df[
    "configuration"
] = pd.Categorical(
    final_corrected_df[
        "configuration"
    ],
    categories=CONFIG_ORDER,
    ordered=True
)


final_corrected_df = (
    final_corrected_df
    .sort_values(
        [
            "configuration",
            "model",
        ]
    )
    .reset_index(drop=True)
)


assert len(
    final_corrected_df
) == 14


# ============================================================
# F. Structural checks
# ============================================================

counts_per_config = (
    final_corrected_df
    .groupby(
        "configuration",
        observed=False
    )
    .size()
)


print(
    "\nModels per configuration:"
)

print(
    counts_per_config
)


assert (
    counts_per_config.loc[
        "60/30"
    ] == 7
)

assert (
    counts_per_config.loc[
        "60/15"
    ] == 7
)


for config in CONFIG_ORDER:

    models_found = (
        final_corrected_df[
            final_corrected_df[
                "configuration"
            ] == config
        ][
            "model"
        ]
        .astype(str)
        .tolist()
    )

    assert (
        models_found
        == MODEL_ORDER
    ), (
        f"{config}: unexpected model list."
    )


print(
    "\nPASS: exactly seven models are present "
    "for both configurations."
)


# ============================================================
# G. Corrected split counts and confusion-total checks
# ============================================================

EXPECTED_TEST_COUNTS = {
    "60/30":
        20668,

    "60/15":
        41254,
}


for _, row in (
    final_corrected_df.iterrows()
):

    total = int(
        row["tn"]
        + row["fp"]
        + row["fn"]
        + row["tp"]
    )

    config = str(
        row[
            "configuration"
        ]
    )

    assert (
        total
        == EXPECTED_TEST_COUNTS[
            config
        ]
    )


print(
    "PASS: all 14 confusion matrices "
    "match their corrected test-set sizes."
)


# ============================================================
# H. Print final corrected metrics
# ============================================================

print("\n")
print("=" * 120)
print("FINAL CORRECTED NAB METRICS")
print("=" * 120)


DISPLAY_COLUMNS = [
    "configuration",
    "model",

    "selected_percentile",
    "selected_threshold",

    "test_precision",
    "test_recall",
    "test_f1",
    "test_fpr",
    "test_roc_auc",
    "test_pr_auc",

    "tn",
    "fp",
    "fn",
    "tp",
]


print(
    final_corrected_df[
        DISPLAY_COLUMNS
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# I. Rank models separately within each configuration
# ============================================================

ranking_rows = []


for config in CONFIG_ORDER:

    config_df = (
        final_corrected_df[
            final_corrected_df[
                "configuration"
            ] == config
        ]
        .copy()
    )


    # Primary ranking: test F1 descending
    # Secondary descriptive tie breakers:
    # recall, PR-AUC
    config_df = (
        config_df
        .sort_values(
            [
                "test_f1",
                "test_recall",
                "test_pr_auc",
            ],
            ascending=[
                False,
                False,
                False,
            ]
        )
        .reset_index(drop=True)
    )


    config_df[
        "f1_rank"
    ] = (
        np.arange(
            1,
            len(config_df) + 1
        )
    )


    ranking_rows.append(
        config_df[
            [
                "configuration",
                "model",
                "f1_rank",
                "test_precision",
                "test_recall",
                "test_f1",
                "test_fpr",
                "test_roc_auc",
                "test_pr_auc",
            ]
        ]
    )


ranking_df = pd.concat(
    ranking_rows,
    ignore_index=True
)


print("\n")
print("=" * 120)
print("CORRECTED MODEL RANKING BY TEST F1")
print("=" * 120)


print(
    ranking_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# J. Direct 60/15 minus 60/30 comparison
# ============================================================

metrics_for_comparison = [
    "selected_threshold",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_fpr",
    "test_roc_auc",
    "test_pr_auc",
]


wide = (
    final_corrected_df
    .pivot(
        index="model",
        columns="configuration",
        values=metrics_for_comparison
    )
)


comparison_rows = []


for model_name in MODEL_ORDER:

    row = {
        "model":
            model_name
    }


    for metric in metrics_for_comparison:

        value_6030 = float(
            wide.loc[
                model_name,
                (
                    metric,
                    "60/30"
                )
            ]
        )

        value_6015 = float(
            wide.loc[
                model_name,
                (
                    metric,
                    "60/15"
                )
            ]
        )


        row[
            f"{metric}_60_30"
        ] = value_6030

        row[
            f"{metric}_60_15"
        ] = value_6015

        row[
            f"{metric}_delta_60_15_minus_60_30"
        ] = (
            value_6015
            - value_6030
        )


    comparison_rows.append(
        row
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


print("\n")
print("=" * 120)
print("CORRECTED 60/15 VS 60/30 COMPARISON")
print("=" * 120)


comparison_display = comparison_df[
    [
        "model",

        "test_f1_60_30",
        "test_f1_60_15",
        "test_f1_delta_60_15_minus_60_30",

        "test_fpr_60_30",
        "test_fpr_60_15",
        "test_fpr_delta_60_15_minus_60_30",

        "test_roc_auc_60_30",
        "test_roc_auc_60_15",
        "test_roc_auc_delta_60_15_minus_60_30",

        "test_pr_auc_60_30",
        "test_pr_auc_60_15",
        "test_pr_auc_delta_60_15_minus_60_30",
    ]
]


print(
    comparison_display.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# K. Best model per configuration
# ============================================================

best_rows = []


for config in CONFIG_ORDER:

    config_ranked = (
        ranking_df[
            ranking_df[
                "configuration"
            ] == config
        ]
        .sort_values(
            "f1_rank"
        )
    )


    best = (
        config_ranked.iloc[0]
    )


    best_rows.append({
        "configuration":
            config,

        "best_model_by_test_f1":
            str(
                best["model"]
            ),

        "test_precision":
            float(
                best[
                    "test_precision"
                ]
            ),

        "test_recall":
            float(
                best[
                    "test_recall"
                ]
            ),

        "test_f1":
            float(
                best[
                    "test_f1"
                ]
            ),

        "test_fpr":
            float(
                best[
                    "test_fpr"
                ]
            ),

        "test_roc_auc":
            float(
                best[
                    "test_roc_auc"
                ]
            ),

        "test_pr_auc":
            float(
                best[
                    "test_pr_auc"
                ]
            ),
    })


best_models_df = pd.DataFrame(
    best_rows
)


print("\n")
print("=" * 120)
print("BEST CORRECTED MODEL PER CONFIGURATION")
print("=" * 120)


print(
    best_models_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# L. Corrected split-population table
# ============================================================

split_population_rows = []


for config_name, data in [
    (
        "60/30",
        nab6030_corrected
    ),
    (
        "60/15",
        nab6015_corrected
    ),
]:

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        df = (
            data[
                split_name
            ]
        )

        total = len(df)

        anomalies = int(
            df[
                "label"
            ].sum()
        )

        normals = (
            total
            - anomalies
        )

        split_population_rows.append({
            "configuration":
                config_name,

            "split":
                split_name,

            "total_windows":
                total,

            "normal_windows":
                normals,

            "anomaly_windows":
                anomalies,

            "anomaly_ratio":
                (
                    anomalies
                    / total
                ),
        })


split_population_df = pd.DataFrame(
    split_population_rows
)


print("\n")
print("=" * 120)
print("FINAL CORRECTED NAB SPLIT POPULATIONS")
print("=" * 120)


print(
    split_population_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# M. Save official corrected NAB artifacts
# ============================================================

FINAL_METRICS_PATH = (
    NAB_CORRECTION_DIR
    / "FINAL_nab_corrected_7model_metrics.csv"
)

FINAL_RANKING_PATH = (
    NAB_CORRECTION_DIR
    / "FINAL_nab_corrected_model_ranking.csv"
)

FINAL_COMPARISON_PATH = (
    NAB_CORRECTION_DIR
    / "FINAL_nab_corrected_60_15_vs_60_30.csv"
)

FINAL_BEST_PATH = (
    NAB_CORRECTION_DIR
    / "FINAL_nab_corrected_best_models.csv"
)

FINAL_SPLITS_PATH = (
    NAB_CORRECTION_DIR
    / "FINAL_nab_corrected_split_populations.csv"
)


final_corrected_df.to_csv(
    FINAL_METRICS_PATH,
    index=False
)

ranking_df.to_csv(
    FINAL_RANKING_PATH,
    index=False
)

comparison_df.to_csv(
    FINAL_COMPARISON_PATH,
    index=False
)

best_models_df.to_csv(
    FINAL_BEST_PATH,
    index=False
)

split_population_df.to_csv(
    FINAL_SPLITS_PATH,
    index=False
)


# ============================================================
# N. Human-readable summary
# ============================================================

SUMMARY_PATH = (
    NAB_CORRECTION_DIR
    / "FINAL_nab_corrected_summary.txt"
)


best_6030 = (
    best_models_df[
        best_models_df[
            "configuration"
        ] == "60/30"
    ]
    .iloc[0]
)

best_6015 = (
    best_models_df[
        best_models_df[
            "configuration"
        ] == "60/15"
    ]
    .iloc[0]
)


summary_text = f"""
FINAL LEAKAGE-CORRECTED NAB RESULTS

Corrected split sizes
---------------------
60/30:
Train = 62,096
Validation = 20,648
Test = 20,668
Total = 103,412

60/15:
Train = 124,204
Validation = 41,235
Test = 41,254
Total = 206,693

Boundary removals
-----------------
60/30: 114 windows removed
60/15: 340 windows removed

Best corrected model by test F1
-------------------------------
60/30:
{best_6030['best_model_by_test_f1']}
F1 = {best_6030['test_f1']:.6f}
Precision = {best_6030['test_precision']:.6f}
Recall = {best_6030['test_recall']:.6f}
FPR = {best_6030['test_fpr']:.6f}
ROC-AUC = {best_6030['test_roc_auc']:.6f}
PR-AUC = {best_6030['test_pr_auc']:.6f}

60/15:
{best_6015['best_model_by_test_f1']}
F1 = {best_6015['test_f1']:.6f}
Precision = {best_6015['test_precision']:.6f}
Recall = {best_6015['test_recall']:.6f}
FPR = {best_6015['test_fpr']:.6f}
ROC-AUC = {best_6015['test_roc_auc']:.6f}
PR-AUC = {best_6015['test_pr_auc']:.6f}

Interpretation guardrail
------------------------
These metrics are from a single internally consistent reconstruction
environment after timestamp-level split-boundary purging. Historical
prediction artifacts were preserved unchanged and are not mixed with
the corrected result set.
"""


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        summary_text.strip()
    )


print("\n")
print("=" * 120)
print("SAVED FINAL NAB ARTIFACTS")
print("=" * 120)


for path in [
    FINAL_METRICS_PATH,
    FINAL_RANKING_PATH,
    FINAL_COMPARISON_PATH,
    FINAL_BEST_PATH,
    FINAL_SPLITS_PATH,
    SUMMARY_PATH,
]:

    print(
        path
    )


print("\n")
print("=" * 120)
print("CELL 27 COMPLETE")
print("=" * 120)

print(
    "\nNAB model correction and consolidation are complete. "
    "No model training occurred in this cell and no historical "
    "artifact was overwritten."
)

In [ ]:
# CELL 28 — UK-DALE Final Selected-Pipeline Threshold and Summary Logic Audit

import json
from pathlib import Path


print("=" * 115)
print("UK-DALE FINAL SELECTED-PIPELINE SOURCE AUDIT")
print("=" * 115)


# ============================================================
# A. Notebook
# ============================================================

THESIS_NOTEBOOK = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)

print(
    "\nNotebook:",
    THESIS_NOTEBOOK
)

print(
    "Exists:",
    THESIS_NOTEBOOK.exists()
)

assert THESIS_NOTEBOOK.exists()


with open(
    THESIS_NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:

    nb = json.load(f)


# ============================================================
# B. Search terms for the final UK-DALE selected pipeline
# ============================================================

SEARCH_TERMS = [
    "UKDALE_Selected_Pipeline",
    "ukdale_clean_main_if_scored_no_injection.csv",
    "if_threshold_percentile",
    "predicted_anomaly",
    "anomaly_score",
    "95",
    "percentile",
    "series_key",
    "washing_machine",
    "dishwasher",
    "kettle",
    "microwave",
    "fridge",
    "freezer",
    "category",
    "flagged",
]


matching_cells = []


for cell_number, cell in enumerate(
    nb["cells"],
    start=1
):

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(source, list)
        else str(source)
    )


    matched = [
        term
        for term in SEARCH_TERMS
        if term.lower()
        in text.lower()
    ]


    # Require a strong UK-DALE anchor rather than
    # printing every generic percentile/category cell.
    strong_anchor = (
        "UKDALE_Selected_Pipeline"
        in text

        or
        "ukdale_clean_main_if_scored_no_injection.csv"
        in text

        or
        (
            "UK-DALE"
            in text
            and
            "anomaly_score"
            in text
        )
    )


    if matched and strong_anchor:

        matching_cells.append({
            "cell_number":
                cell_number,

            "execution_count":
                cell.get(
                    "execution_count"
                ),

            "matched_terms":
                matched,

            "text":
                text,
        })


print(
    "\nMatching cells:",
    len(matching_cells)
)


for item in matching_cells:

    print(
        f"\nCell {item['cell_number']}: "
        f"{item['matched_terms']}"
    )


# ============================================================
# C. Print relevant implementation blocks
# ============================================================

IMPORTANT_LINES = [
    "percentile",
    "threshold",
    "anomaly_score",
    "predicted_anomaly",
    "decision_function",
    "split",
    "series_key",
    "category",
    "appliance",
    "flagged",
    "groupby",
    "mean(",
    "sum(",
    "to_csv",
]


for item in matching_cells:

    text = item[
        "text"
    ]

    lines = (
        text.splitlines()
    )

    relevant_indices = []


    for i, line in enumerate(
        lines
    ):

        if any(
            term.lower()
            in line.lower()

            for term
            in IMPORTANT_LINES
        ):

            relevant_indices.append(
                i
            )


    if not relevant_indices:
        continue


    # Merge overlapping context intervals
    intervals = []


    for idx in relevant_indices:

        start = max(
            0,
            idx - 8
        )

        end = min(
            len(lines),
            idx + 14
        )


        if (
            intervals
            and start
            <= intervals[-1][1] + 2
        ):

            intervals[-1] = (
                intervals[-1][0],
                max(
                    intervals[-1][1],
                    end
                )
            )

        else:

            intervals.append(
                (
                    start,
                    end
                )
            )


    print("\n")
    print("=" * 115)
    print(
        f"NOTEBOOK CELL "
        f"{item['cell_number']}"
    )
    print("=" * 115)


    for block_number, (
        start,
        end
    ) in enumerate(
        intervals,
        start=1
    ):

        print(
            f"\n--- Relevant block "
            f"{block_number} "
            f"(lines {start + 1}–{end}) ---"
        )


        for line_no in range(
            start,
            end
        ):

            print(
                f"{line_no + 1:04d}: "
                f"{lines[line_no]}"
            )


# ============================================================
# D. Inspect final scored UK-DALE file
# ============================================================

UK_SCORED_PATH = (
    BASE
    / "processed_data"
    / "UKDALE_Selected_Pipeline"
    / "ukdale_clean_main_if_scored_no_injection.csv"
)


print("\n")
print("=" * 115)
print("FINAL SCORED UK-DALE FILE")
print("=" * 115)

print(
    "Path:",
    UK_SCORED_PATH
)

print(
    "Exists:",
    UK_SCORED_PATH.exists()
)

assert UK_SCORED_PATH.exists()


uk_scored = pd.read_csv(
    UK_SCORED_PATH,
    low_memory=False
)


print(
    "\nShape:",
    uk_scored.shape
)

print(
    "\nColumns:"
)

print(
    uk_scored.columns.tolist()
)


print(
    "\nSplit counts:"
)

print(
    uk_scored[
        "split"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nStored threshold percentiles:"
)

print(
    uk_scored[
        "if_threshold_percentile"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nStored threshold values:"
)

print(
    uk_scored[
        "if_threshold"
    ].value_counts(
        dropna=False
    ).head(10)
)


print(
    "\nStored predicted-anomaly counts by split:"
)

print(
    pd.crosstab(
        uk_scored[
            "split"
        ],
        uk_scored[
            "predicted_anomaly"
        ]
    )
)


print(
    "\nUnique series:",
    uk_scored[
        "series_key"
    ].nunique()
)


print(
    "\nSeries keys:"
)

for key in sorted(
    uk_scored[
        "series_key"
    ].dropna().unique()
):

    print(
        key
    )


# ============================================================
# E. Test which threshold operator reproduces stored flags
# ============================================================

stored_threshold = float(
    uk_scored[
        "if_threshold"
    ].dropna().iloc[0]
)


scores = (
    uk_scored[
        "anomaly_score"
    ]
    .to_numpy(
        dtype=float
    )
)

stored_pred = (
    uk_scored[
        "predicted_anomaly"
    ]
    .to_numpy(
        dtype=int
    )
)


pred_ge = (
    scores
    >= stored_threshold
).astype(int)

pred_gt = (
    scores
    > stored_threshold
).astype(int)


ge_disagreements = int(
    np.sum(
        pred_ge
        != stored_pred
    )
)

gt_disagreements = int(
    np.sum(
        pred_gt
        != stored_pred
    )
)


print("\n")
print("=" * 115)
print("THRESHOLD OPERATOR REPRODUCTION")
print("=" * 115)


print(
    "Stored threshold:",
    f"{stored_threshold:.18f}"
)

print(
    "Disagreements using >= :",
    ge_disagreements
)

print(
    "Disagreements using >  :",
    gt_disagreements
)


if ge_disagreements == 0:

    print(
        "\nPASS: stored predictions use "
        "anomaly_score >= threshold."
    )

elif gt_disagreements == 0:

    print(
        "\nPASS: stored predictions use "
        "anomaly_score > threshold."
    )

else:

    print(
        "\nATTENTION: neither simple threshold operator "
        "fully reproduces the stored predictions."
    )


# ============================================================
# F. Save audit index
# ============================================================

audit_index = pd.DataFrame([
    {
        "scored_file":
            str(
                UK_SCORED_PATH
            ),

        "rows":
            len(
                uk_scored
            ),

        "series_count":
            uk_scored[
                "series_key"
            ].nunique(),

        "stored_threshold_percentile":
            float(
                uk_scored[
                    "if_threshold_percentile"
                ].dropna().iloc[0]
            ),

        "stored_threshold":
            stored_threshold,

        "ge_disagreements":
            ge_disagreements,

        "gt_disagreements":
            gt_disagreements,

        "matching_notebook_cells":
            ",".join(
                str(
                    x[
                        "cell_number"
                    ]
                )
                for x
                in matching_cells
            ),
    }
])


UK_AUDIT_INDEX_PATH = (
    CORRECTION
    / "ukdale_final_selected_pipeline_source_audit.csv"
)


audit_index.to_csv(
    UK_AUDIT_INDEX_PATH,
    index=False
)


print(
    "\nAudit index saved to:"
)

print(
    UK_AUDIT_INDEX_PATH
)


print("\n")
print("=" * 115)
print("CELL 28 COMPLETE")
print("=" * 115)

print(
    "\nThis cell only inspected the final UK-DALE "
    "implementation and stored scores. "
    "No result or dataset was modified."
)

In [ ]:
# CELL 29 — UK-DALE Final Selected-Pipeline Split-Boundary Correction

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 120)
print("UK-DALE FINAL SELECTED-PIPELINE SPLIT-BOUNDARY CORRECTION")
print("=" * 120)


# ============================================================
# A. Paths
# ============================================================

UK_SCORED_PATH = (
    BASE
    / "processed_data"
    / "UKDALE_Selected_Pipeline"
    / "ukdale_clean_main_if_scored_no_injection.csv"
)

UK_CORRECTION_DIR = (
    CORRECTION
    / "UKDALE"
)

UK_CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "\nSource scored file:",
    UK_SCORED_PATH
)

print(
    "Exists:",
    UK_SCORED_PATH.exists()
)

assert UK_SCORED_PATH.exists()


# ============================================================
# B. Load frozen scored data
# ============================================================

df = pd.read_csv(
    UK_SCORED_PATH,
    low_memory=False
)


df["window_start"] = pd.to_datetime(
    df["window_start"],
    utc=True,
    errors="raise"
)

df["window_end"] = pd.to_datetime(
    df["window_end"],
    utc=True,
    errors="raise"
)


print(
    "\nLoaded rows:",
    len(df)
)

print(
    "Unique series:",
    df["series_key"].nunique()
)

print(
    "\nHistorical split counts:"
)

print(
    df["split"]
    .value_counts()
    .reindex(
        [
            "train",
            "val",
            "test",
        ]
    )
)


assert len(df) == 363940

assert int(
    (df["split"] == "train").sum()
) == 218358

assert int(
    (df["split"] == "val").sum()
) == 72787

assert int(
    (df["split"] == "test").sum()
) == 72795

assert (
    df["series_key"].nunique()
    == 16
)


# ============================================================
# C. Preserve original source-row identity
# ============================================================

df = df.reset_index(
    drop=True
)

df[
    "original_source_row_id"
] = np.arange(
    len(df),
    dtype=np.int64
)


# ============================================================
# D. Reconstruct timestamp-level boundary purge
#
# UK-DALE window_end is treated as an inclusive endpoint.
#
# Therefore:
# validation starts must be strictly later than the maximum
# training end time in the same series;
#
# test starts must be strictly later than the maximum RETAINED
# validation end time in the same series.
# ============================================================

corrected_parts = []

removed_rows = []

purge_audit_rows = []


for series_key, g in df.groupby(
    "series_key",
    sort=True
):

    g = (
        g.sort_values(
            [
                "window_start",
                "window_end",
                "original_source_row_id",
            ]
        )
        .copy()
    )


    train = (
        g[
            g["split"] == "train"
        ]
        .copy()
    )

    validation = (
        g[
            g["split"] == "val"
        ]
        .copy()
    )

    test = (
        g[
            g["split"] == "test"
        ]
        .copy()
    )


    assert len(train) > 0
    assert len(validation) > 0
    assert len(test) > 0


    # --------------------------------------------------------
    # Train -> validation boundary
    # --------------------------------------------------------

    max_train_end = (
        train["window_end"].max()
    )


    validation_conflict_mask = (
        validation["window_start"]
        <= max_train_end
    )


    validation_removed = (
        validation[
            validation_conflict_mask
        ]
        .copy()
    )


    validation_retained = (
        validation[
            ~validation_conflict_mask
        ]
        .copy()
    )


    if len(validation_removed):

        validation_removed[
            "purge_reason"
        ] = (
            "train_validation_raw_time_overlap"
        )

        removed_rows.append(
            validation_removed
        )


    # --------------------------------------------------------
    # Retained validation -> test boundary
    # --------------------------------------------------------

    assert len(
        validation_retained
    ) > 0


    max_retained_validation_end = (
        validation_retained[
            "window_end"
        ].max()
    )


    test_conflict_mask = (
        test["window_start"]
        <= max_retained_validation_end
    )


    test_removed = (
        test[
            test_conflict_mask
        ]
        .copy()
    )


    test_retained = (
        test[
            ~test_conflict_mask
        ]
        .copy()
    )


    if len(test_removed):

        test_removed[
            "purge_reason"
        ] = (
            "validation_test_raw_time_overlap"
        )

        removed_rows.append(
            test_removed
        )


    # --------------------------------------------------------
    # Audit row
    # --------------------------------------------------------

    purge_audit_rows.append({

        "series_key":
            series_key,

        "original_train":
            len(train),

        "original_validation":
            len(validation),

        "original_test":
            len(test),

        "validation_removed":
            len(
                validation_removed
            ),

        "test_removed":
            len(
                test_removed
            ),

        "corrected_train":
            len(train),

        "corrected_validation":
            len(
                validation_retained
            ),

        "corrected_test":
            len(
                test_retained
            ),

        "max_train_end":
            max_train_end,

        "first_retained_validation_start":
            validation_retained[
                "window_start"
            ].min(),

        "max_retained_validation_end":
            max_retained_validation_end,

        "first_retained_test_start":
            test_retained[
                "window_start"
            ].min(),
    })


    corrected_parts.extend(
        [
            train,
            validation_retained,
            test_retained,
        ]
    )


corrected_df = pd.concat(
    corrected_parts,
    ignore_index=True
)


removed_df = pd.concat(
    removed_rows,
    ignore_index=True
)


purge_audit_df = pd.DataFrame(
    purge_audit_rows
)


# ============================================================
# E. Verify exact corrected population
# ============================================================

corrected_counts = (
    corrected_df[
        "split"
    ]
    .value_counts()
    .reindex(
        [
            "train",
            "val",
            "test",
        ]
    )
)


print("\n")
print("=" * 120)
print("CORRECTED POPULATION")
print("=" * 120)


print(
    corrected_counts
)


print(
    "\nRemoved rows by boundary:"
)

print(
    removed_df[
        "purge_reason"
    ].value_counts()
)


assert int(
    corrected_counts[
        "train"
    ]
) == 218358

assert int(
    corrected_counts[
        "val"
    ]
) == 72771

assert int(
    corrected_counts[
        "test"
    ]
) == 72780

assert len(
    corrected_df
) == 363909

assert len(
    removed_df
) == 31

assert int(
    (
        removed_df[
            "purge_reason"
        ]
        ==
        "train_validation_raw_time_overlap"
    ).sum()
) == 16

assert int(
    (
        removed_df[
            "purge_reason"
        ]
        ==
        "validation_test_raw_time_overlap"
    ).sum()
) == 15


# Training rows must be exactly unchanged
original_train_ids = (
    df.loc[
        df["split"] == "train",
        "original_source_row_id"
    ]
    .to_numpy()
)

corrected_train_ids = (
    corrected_df.loc[
        corrected_df[
            "split"
        ] == "train",
        "original_source_row_id"
    ]
    .to_numpy()
)


assert np.array_equal(
    np.sort(
        original_train_ids
    ),
    np.sort(
        corrected_train_ids
    )
)


print(
    "\nPASS: training population is exactly unchanged."
)


# ============================================================
# F. Verify zero post-purge boundary overlap
# ============================================================

violations = []


for series_key, g in corrected_df.groupby(
    "series_key",
    sort=True
):

    train = (
        g[
            g["split"] == "train"
        ]
    )

    validation = (
        g[
            g["split"] == "val"
        ]
    )

    test = (
        g[
            g["split"] == "test"
        ]
    )


    max_train_end = (
        train[
            "window_end"
        ].max()
    )

    min_validation_start = (
        validation[
            "window_start"
        ].min()
    )


    max_validation_end = (
        validation[
            "window_end"
        ].max()
    )

    min_test_start = (
        test[
            "window_start"
        ].min()
    )


    if (
        min_validation_start
        <= max_train_end
    ):

        violations.append(
            (
                series_key,
                "train_validation"
            )
        )


    if (
        min_test_start
        <= max_validation_end
    ):

        violations.append(
            (
                series_key,
                "validation_test"
            )
        )


print(
    "Post-purge boundary violations:",
    len(violations)
)

assert len(
    violations
) == 0


print(
    "PASS: no raw temporal observations cross "
    "the corrected UK-DALE split boundaries."
)


# ============================================================
# G. Historical threshold and original summaries
# ============================================================

historical_threshold_percentile = int(
    df[
        "if_threshold_percentile"
    ].dropna().iloc[0]
)

historical_threshold = float(
    df[
        "if_threshold"
    ].dropna().iloc[0]
)


assert (
    historical_threshold_percentile
    == 95
)


def make_split_summary(
    data,
    flag_column,
    threshold_value
):

    rows = []


    for split_name in [
        "train",
        "val",
        "test",
    ]:

        split_df = (
            data[
                data["split"]
                == split_name
            ]
        )

        total = len(
            split_df
        )

        flagged = int(
            split_df[
                flag_column
            ].sum()
        )


        rows.append({

            "split":
                split_name,

            "windows":
                total,

            "flagged_windows":
                flagged,

            "flagged_ratio":
                flagged
                / total,

            "flagged_ratio_percent":
                100
                * flagged
                / total,

            "mean_anomaly_score":
                split_df[
                    "anomaly_score"
                ].mean(),

            "median_anomaly_score":
                split_df[
                    "anomaly_score"
                ].median(),

            "score_q95":
                split_df[
                    "anomaly_score"
                ].quantile(
                    0.95
                ),

            "score_q99":
                split_df[
                    "anomaly_score"
                ].quantile(
                    0.99
                ),

            "maximum_anomaly_score":
                split_df[
                    "anomaly_score"
                ].max(),

            "validation_threshold":
                threshold_value,

            "threshold_percentile":
                95,
        })


    return pd.DataFrame(
        rows
    )


historical_summary = (
    make_split_summary(
        df,
        "predicted_anomaly",
        historical_threshold
    )
)


# ============================================================
# H. Recalculate corrected 95th-percentile threshold
# ============================================================

corrected_validation_scores = (
    corrected_df.loc[
        corrected_df[
            "split"
        ] == "val",
        "anomaly_score"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


corrected_threshold = float(
    np.percentile(
        corrected_validation_scores,
        95
    )
)


print("\n")
print("=" * 120)
print("THRESHOLD RECALCULATION")
print("=" * 120)


print(
    "Historical threshold:",
    f"{historical_threshold:.18f}"
)

print(
    "Corrected threshold:",
    f"{corrected_threshold:.18f}"
)

print(
    "Threshold delta:",
    f"{corrected_threshold - historical_threshold:.18e}"
)


# ============================================================
# I. Resolve > versus >= empirically
#
# The historical file had no equality cases.
# We test the corrected threshold too.
# ============================================================

corrected_scores = (
    corrected_df[
        "anomaly_score"
    ].to_numpy(
        dtype=np.float64
    )
)


flags_gt = (
    corrected_scores
    > corrected_threshold
).astype(
    np.int8
)


flags_ge = (
    corrected_scores
    >= corrected_threshold
).astype(
    np.int8
)


operator_disagreements = int(
    np.sum(
        flags_gt
        != flags_ge
    )
)


print(
    "\nCorrected > versus >= disagreements:",
    operator_disagreements
)


if operator_disagreements != 0:

    raise RuntimeError(
        "Corrected UK-DALE scores contain values exactly "
        "equal to the threshold. The original final IF source "
        "operator must be recovered before proceeding."
    )


# Downstream selected-pipeline code consistently used >
# and it is numerically identical to >= here.
corrected_df[
    "predicted_anomaly_corrected"
] = flags_gt


corrected_df[
    "if_threshold_percentile_corrected"
] = 95


corrected_df[
    "if_threshold_corrected"
] = corrected_threshold


print(
    "PASS: > and >= produce identical corrected flags."
)


# ============================================================
# J. Corrected overall summary
# ============================================================

corrected_summary = (
    make_split_summary(
        corrected_df,
        "predicted_anomaly_corrected",
        corrected_threshold
    )
)


summary_comparison = (
    historical_summary.merge(
        corrected_summary,
        on="split",
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


summary_comparison[
    "windows_delta"
] = (
    summary_comparison[
        "windows_corrected"
    ]
    -
    summary_comparison[
        "windows_historical"
    ]
)


summary_comparison[
    "flagged_windows_delta"
] = (
    summary_comparison[
        "flagged_windows_corrected"
    ]
    -
    summary_comparison[
        "flagged_windows_historical"
    ]
)


summary_comparison[
    "flagged_ratio_percent_delta"
] = (
    summary_comparison[
        "flagged_ratio_percent_corrected"
    ]
    -
    summary_comparison[
        "flagged_ratio_percent_historical"
    ]
)


print("\n")
print("=" * 120)
print("UK-DALE HISTORICAL VS CORRECTED SPLIT SUMMARY")
print("=" * 120)


print(
    summary_comparison[
        [
            "split",

            "windows_historical",
            "windows_corrected",
            "windows_delta",

            "flagged_windows_historical",
            "flagged_windows_corrected",
            "flagged_windows_delta",

            "flagged_ratio_percent_historical",
            "flagged_ratio_percent_corrected",
            "flagged_ratio_percent_delta",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# ============================================================
# K. Historical and corrected TEST category summaries
# ============================================================

def make_category_summary(
    data,
    flag_column
):

    test_df = (
        data[
            data["split"]
            == "test"
        ]
        .copy()
    )


    category_summary = (
        test_df
        .groupby(
            "primary_category",
            dropna=False
        )
        .agg(
            test_windows=(
                flag_column,
                "size"
            ),

            flagged_windows=(
                flag_column,
                "sum"
            ),

            mean_anomaly_score=(
                "anomaly_score",
                "mean"
            ),

            median_anomaly_score=(
                "anomaly_score",
                "median"
            ),

            maximum_anomaly_score=(
                "anomaly_score",
                "max"
            ),
        )
        .reset_index()
    )


    category_summary[
        "flagged_ratio"
    ] = (
        category_summary[
            "flagged_windows"
        ]
        /
        category_summary[
            "test_windows"
        ]
    )


    category_summary[
        "flagged_ratio_percent"
    ] = (
        100
        *
        category_summary[
            "flagged_ratio"
        ]
    )


    return (
        category_summary
        .sort_values(
            "flagged_ratio",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


historical_category = (
    make_category_summary(
        df,
        "predicted_anomaly"
    )
)


corrected_category = (
    make_category_summary(
        corrected_df,
        "predicted_anomaly_corrected"
    )
)


category_comparison = (
    historical_category[
        [
            "primary_category",
            "test_windows",
            "flagged_windows",
            "flagged_ratio_percent",
        ]
    ]
    .merge(
        corrected_category[
            [
                "primary_category",
                "test_windows",
                "flagged_windows",
                "flagged_ratio_percent",
            ]
        ],
        on="primary_category",
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


category_comparison[
    "test_windows_delta"
] = (
    category_comparison[
        "test_windows_corrected"
    ]
    -
    category_comparison[
        "test_windows_historical"
    ]
)


category_comparison[
    "flagged_windows_delta"
] = (
    category_comparison[
        "flagged_windows_corrected"
    ]
    -
    category_comparison[
        "flagged_windows_historical"
    ]
)


category_comparison[
    "flagged_ratio_percent_delta"
] = (
    category_comparison[
        "flagged_ratio_percent_corrected"
    ]
    -
    category_comparison[
        "flagged_ratio_percent_historical"
    ]
)


category_comparison = (
    category_comparison
    .sort_values(
        "flagged_ratio_percent_corrected",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\n")
print("=" * 120)
print("UK-DALE TEST CATEGORY SUMMARY — HISTORICAL VS CORRECTED")
print("=" * 120)


print(
    category_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# ============================================================
# L. Save corrected artifacts
# ============================================================

CORRECTED_SCORED_PATH = (
    UK_CORRECTION_DIR
    / "ukdale_if_scored_boundary_corrected.csv"
)

REMOVED_PATH = (
    UK_CORRECTION_DIR
    / "ukdale_removed_boundary_windows.csv"
)

PURGE_AUDIT_PATH = (
    UK_CORRECTION_DIR
    / "ukdale_boundary_purge_audit.csv"
)

CORRECTED_SUMMARY_PATH = (
    UK_CORRECTION_DIR
    / "ukdale_if_corrected_split_summary.csv"
)

SUMMARY_COMPARISON_PATH = (
    UK_CORRECTION_DIR
    / "ukdale_if_historical_vs_corrected_split_summary.csv"
)

CORRECTED_CATEGORY_PATH = (
    UK_CORRECTION_DIR
    / "ukdale_if_corrected_test_category_summary.csv"
)

CATEGORY_COMPARISON_PATH = (
    UK_CORRECTION_DIR
    / "ukdale_if_historical_vs_corrected_test_category_summary.csv"
)


corrected_df.to_csv(
    CORRECTED_SCORED_PATH,
    index=False
)

removed_df.to_csv(
    REMOVED_PATH,
    index=False
)

purge_audit_df.to_csv(
    PURGE_AUDIT_PATH,
    index=False
)

corrected_summary.to_csv(
    CORRECTED_SUMMARY_PATH,
    index=False
)

summary_comparison.to_csv(
    SUMMARY_COMPARISON_PATH,
    index=False
)

corrected_category.to_csv(
    CORRECTED_CATEGORY_PATH,
    index=False
)

category_comparison.to_csv(
    CATEGORY_COMPARISON_PATH,
    index=False
)


# ============================================================
# M. Final checkpoint
# ============================================================

print("\n")
print("=" * 120)
print("UK-DALE CORRECTION CHECKPOINT")
print("=" * 120)


print(
    f"Historical total: {len(df):,}"
)

print(
    f"Corrected total:  {len(corrected_df):,}"
)

print(
    f"Removed:          {len(removed_df):,}"
)

print(
    f"Historical threshold: "
    f"{historical_threshold:.12f}"
)

print(
    f"Corrected threshold:  "
    f"{corrected_threshold:.12f}"
)


corrected_test = (
    corrected_summary[
        corrected_summary[
            "split"
        ] == "test"
    ]
    .iloc[0]
)


print(
    f"Corrected test flagged windows: "
    f"{int(corrected_test['flagged_windows']):,}"
)

print(
    f"Corrected test flagged ratio: "
    f"{corrected_test['flagged_ratio_percent']:.6f}%"
)


print(
    "\nSaved corrected artifacts under:"
)

print(
    UK_CORRECTION_DIR
)


print("\n")
print("=" * 120)
print("CELL 29 COMPLETE")
print("=" * 120)

print(
    "\nNo UK-DALE model was retrained. "
    "The original scored dataset was not modified."
)

In [ ]:
# CELL 30 — AMPds2 Final Selected-Pipeline Split-Boundary Correction

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 120)
print("AMPds2 FINAL SELECTED-PIPELINE SPLIT-BOUNDARY CORRECTION")
print("=" * 120)


# ============================================================
# A. Paths
# ============================================================

AMP_SCORED_PATH = (
    BASE
    / "processed_data"
    / "AMPDS2_Selected_Pipeline"
    / "ampds2_p_if_scored_no_injection.csv"
)

AMP_CORRECTION_DIR = (
    CORRECTION
    / "AMPDS2"
)

AMP_CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "\nSource scored file:",
    AMP_SCORED_PATH
)

print(
    "Exists:",
    AMP_SCORED_PATH.exists()
)

assert AMP_SCORED_PATH.exists()


# ============================================================
# B. Load frozen scored data
# ============================================================

df = pd.read_csv(
    AMP_SCORED_PATH,
    low_memory=False
)


df["window_start"] = pd.to_datetime(
    df["window_start"],
    utc=True,
    errors="raise"
)

df["window_end"] = pd.to_datetime(
    df["window_end"],
    utc=True,
    errors="raise"
)


df = df.reset_index(
    drop=True
)

df[
    "original_source_row_id"
] = np.arange(
    len(df),
    dtype=np.int64
)


print(
    "\nLoaded rows:",
    len(df)
)

print(
    "Unique series:",
    df[
        "series_key"
    ].nunique()
)

print(
    "\nHistorical split counts:"
)

print(
    df[
        "split"
    ]
    .value_counts()
    .reindex(
        [
            "train",
            "val",
            "test",
        ]
    )
)


assert len(df) == 735819

assert int(
    (
        df["split"]
        == "train"
    ).sum()
) == 441483

assert int(
    (
        df["split"]
        == "val"
    ).sum()
) == 147168

assert int(
    (
        df["split"]
        == "test"
    ).sum()
) == 147168

assert (
    df["series_key"].nunique()
    == 21
)


# ============================================================
# C. Timestamp-level boundary purge
#
# AMPds2 uses the same inclusive endpoint interpretation:
#
# validation_start <= max_train_end
#     -> purge
#
# test_start <= max_retained_validation_end
#     -> purge
# ============================================================

corrected_parts = []

removed_parts = []

audit_rows = []


for series_key, g in df.groupby(
    "series_key",
    sort=True
):

    g = (
        g.sort_values(
            [
                "window_start",
                "window_end",
                "original_source_row_id",
            ]
        )
        .copy()
    )


    train = (
        g[
            g["split"]
            == "train"
        ]
        .copy()
    )

    validation = (
        g[
            g["split"]
            == "val"
        ]
        .copy()
    )

    test = (
        g[
            g["split"]
            == "test"
        ]
        .copy()
    )


    assert len(train) > 0
    assert len(validation) > 0
    assert len(test) > 0


    # --------------------------------------------------------
    # Train -> validation
    # --------------------------------------------------------

    max_train_end = (
        train[
            "window_end"
        ].max()
    )


    val_conflict_mask = (
        validation[
            "window_start"
        ]
        <= max_train_end
    )


    val_removed = (
        validation[
            val_conflict_mask
        ]
        .copy()
    )


    val_retained = (
        validation[
            ~val_conflict_mask
        ]
        .copy()
    )


    if len(
        val_removed
    ):

        val_removed[
            "purge_reason"
        ] = (
            "train_validation_raw_time_overlap"
        )

        removed_parts.append(
            val_removed
        )


    # --------------------------------------------------------
    # Retained validation -> test
    # --------------------------------------------------------

    assert len(
        val_retained
    ) > 0


    max_retained_val_end = (
        val_retained[
            "window_end"
        ].max()
    )


    test_conflict_mask = (
        test[
            "window_start"
        ]
        <= max_retained_val_end
    )


    test_removed = (
        test[
            test_conflict_mask
        ]
        .copy()
    )


    test_retained = (
        test[
            ~test_conflict_mask
        ]
        .copy()
    )


    if len(
        test_removed
    ):

        test_removed[
            "purge_reason"
        ] = (
            "validation_test_raw_time_overlap"
        )

        removed_parts.append(
            test_removed
        )


    audit_rows.append({

        "series_key":
            series_key,

        "original_train":
            len(train),

        "original_validation":
            len(validation),

        "original_test":
            len(test),

        "validation_removed":
            len(
                val_removed
            ),

        "test_removed":
            len(
                test_removed
            ),

        "corrected_train":
            len(train),

        "corrected_validation":
            len(
                val_retained
            ),

        "corrected_test":
            len(
                test_retained
            ),

        "max_train_end":
            max_train_end,

        "first_retained_validation_start":
            val_retained[
                "window_start"
            ].min(),

        "max_retained_validation_end":
            max_retained_val_end,

        "first_retained_test_start":
            test_retained[
                "window_start"
            ].min(),
    })


    corrected_parts.extend(
        [
            train,
            val_retained,
            test_retained,
        ]
    )


corrected_df = pd.concat(
    corrected_parts,
    ignore_index=True
)


removed_df = pd.concat(
    removed_parts,
    ignore_index=True
)


purge_audit_df = pd.DataFrame(
    audit_rows
)


# ============================================================
# D. Exact corrected-population checks
# ============================================================

corrected_counts = (
    corrected_df[
        "split"
    ]
    .value_counts()
    .reindex(
        [
            "train",
            "val",
            "test",
        ]
    )
)


print("\n")
print("=" * 120)
print("CORRECTED POPULATION")
print("=" * 120)


print(
    corrected_counts
)


print(
    "\nRemoved rows by boundary:"
)

print(
    removed_df[
        "purge_reason"
    ].value_counts()
)


assert int(
    corrected_counts[
        "train"
    ]
) == 441483

assert int(
    corrected_counts[
        "val"
    ]
) == 147147

assert int(
    corrected_counts[
        "test"
    ]
) == 147147

assert len(
    corrected_df
) == 735777

assert len(
    removed_df
) == 42

assert int(
    (
        removed_df[
            "purge_reason"
        ]
        ==
        "train_validation_raw_time_overlap"
    ).sum()
) == 21

assert int(
    (
        removed_df[
            "purge_reason"
        ]
        ==
        "validation_test_raw_time_overlap"
    ).sum()
) == 21


# Training population must be identical
original_train_ids = (
    df.loc[
        df[
            "split"
        ] == "train",
        "original_source_row_id"
    ]
    .to_numpy()
)

corrected_train_ids = (
    corrected_df.loc[
        corrected_df[
            "split"
        ] == "train",
        "original_source_row_id"
    ]
    .to_numpy()
)


assert np.array_equal(
    np.sort(
        original_train_ids
    ),
    np.sort(
        corrected_train_ids
    )
)


print(
    "\nPASS: AMPds2 training population "
    "is exactly unchanged."
)


# ============================================================
# E. Zero-overlap post-purge verification
# ============================================================

violations = []


for series_key, g in corrected_df.groupby(
    "series_key",
    sort=True
):

    train = (
        g[
            g["split"]
            == "train"
        ]
    )

    validation = (
        g[
            g["split"]
            == "val"
        ]
    )

    test = (
        g[
            g["split"]
            == "test"
        ]
    )


    if (
        validation[
            "window_start"
        ].min()
        <=
        train[
            "window_end"
        ].max()
    ):

        violations.append(
            (
                series_key,
                "train_validation"
            )
        )


    if (
        test[
            "window_start"
        ].min()
        <=
        validation[
            "window_end"
        ].max()
    ):

        violations.append(
            (
                series_key,
                "validation_test"
            )
        )


print(
    "Post-purge boundary violations:",
    len(
        violations
    )
)

assert len(
    violations
) == 0


print(
    "PASS: no raw temporal observations "
    "cross AMPds2 split boundaries."
)


# ============================================================
# F. Historical threshold
# ============================================================

historical_threshold_percentile = int(
    df[
        "if_threshold_percentile"
    ]
    .dropna()
    .iloc[0]
)


historical_threshold = float(
    df[
        "if_threshold"
    ]
    .dropna()
    .iloc[0]
)


assert (
    historical_threshold_percentile
    == 95
)


# ============================================================
# G. Recalculate threshold from corrected validation
# ============================================================

corrected_validation_scores = (
    corrected_df.loc[
        corrected_df[
            "split"
        ] == "val",
        "anomaly_score"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


corrected_threshold = float(
    np.percentile(
        corrected_validation_scores,
        95
    )
)


print("\n")
print("=" * 120)
print("THRESHOLD RECALCULATION")
print("=" * 120)


print(
    "Historical threshold:",
    f"{historical_threshold:.18f}"
)

print(
    "Corrected threshold:",
    f"{corrected_threshold:.18f}"
)

print(
    "Threshold delta:",
    f"{corrected_threshold - historical_threshold:.18e}"
)


# ============================================================
# H. Check > versus >=
# ============================================================

scores = (
    corrected_df[
        "anomaly_score"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


flags_gt = (
    scores
    > corrected_threshold
).astype(
    np.int8
)


flags_ge = (
    scores
    >= corrected_threshold
).astype(
    np.int8
)


operator_disagreements = int(
    np.sum(
        flags_gt
        != flags_ge
    )
)


print(
    "\nCorrected > versus >= disagreements:",
    operator_disagreements
)


if operator_disagreements != 0:

    raise RuntimeError(
        "AMPds2 contains score values exactly equal "
        "to the corrected threshold. Stop before "
        "choosing an operator."
    )


corrected_df[
    "predicted_anomaly_corrected"
] = flags_gt


corrected_df[
    "if_threshold_percentile_corrected"
] = 95


corrected_df[
    "if_threshold_corrected"
] = corrected_threshold


print(
    "PASS: > and >= produce identical "
    "corrected AMPds2 flags."
)


# ============================================================
# I. Overall-summary helper
# ============================================================

def make_amp_split_summary(
    data,
    flag_column,
    threshold
):

    rows = []


    for split_name in [
        "train",
        "val",
        "test",
    ]:

        split_df = (
            data[
                data[
                    "split"
                ]
                == split_name
            ]
        )


        total = len(
            split_df
        )

        flagged = int(
            split_df[
                flag_column
            ].sum()
        )


        rows.append({

            "split":
                split_name,

            "windows":
                total,

            "flagged_windows":
                flagged,

            "flagged_ratio":
                flagged
                / total,

            "flagged_ratio_percent":
                100
                * flagged
                / total,

            "mean_anomaly_score":
                split_df[
                    "anomaly_score"
                ].mean(),

            "median_anomaly_score":
                split_df[
                    "anomaly_score"
                ].median(),

            "score_q95":
                split_df[
                    "anomaly_score"
                ].quantile(
                    0.95
                ),

            "score_q99":
                split_df[
                    "anomaly_score"
                ].quantile(
                    0.99
                ),

            "maximum_anomaly_score":
                split_df[
                    "anomaly_score"
                ].max(),

            "validation_threshold":
                threshold,

            "threshold_percentile":
                95,
        })


    return pd.DataFrame(
        rows
    )


historical_summary = (
    make_amp_split_summary(
        df,
        "predicted_anomaly",
        historical_threshold
    )
)


corrected_summary = (
    make_amp_split_summary(
        corrected_df,
        "predicted_anomaly_corrected",
        corrected_threshold
    )
)


summary_comparison = (
    historical_summary.merge(
        corrected_summary,
        on="split",
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


summary_comparison[
    "windows_delta"
] = (
    summary_comparison[
        "windows_corrected"
    ]
    -
    summary_comparison[
        "windows_historical"
    ]
)


summary_comparison[
    "flagged_windows_delta"
] = (
    summary_comparison[
        "flagged_windows_corrected"
    ]
    -
    summary_comparison[
        "flagged_windows_historical"
    ]
)


summary_comparison[
    "flagged_ratio_percent_delta"
] = (
    summary_comparison[
        "flagged_ratio_percent_corrected"
    ]
    -
    summary_comparison[
        "flagged_ratio_percent_historical"
    ]
)


print("\n")
print("=" * 120)
print("AMPds2 HISTORICAL VS CORRECTED SPLIT SUMMARY")
print("=" * 120)


print(
    summary_comparison[
        [
            "split",

            "windows_historical",
            "windows_corrected",
            "windows_delta",

            "flagged_windows_historical",
            "flagged_windows_corrected",
            "flagged_windows_delta",

            "flagged_ratio_percent_historical",
            "flagged_ratio_percent_corrected",
            "flagged_ratio_percent_delta",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# ============================================================
# J. Channel/category test-summary helper
# ============================================================

def make_amp_channel_summary(
    data,
    flag_column
):

    test_df = (
        data[
            data[
                "split"
            ] == "test"
        ]
        .copy()
    )


    summary = (
        test_df
        .groupby(
            [
                "channel_code",
                "channel_name",
                "primary_category",
            ],
            dropna=False
        )
        .agg(
            test_windows=(
                flag_column,
                "size"
            ),

            flagged_windows=(
                flag_column,
                "sum"
            ),

            mean_anomaly_score=(
                "anomaly_score",
                "mean"
            ),

            median_anomaly_score=(
                "anomaly_score",
                "median"
            ),

            maximum_anomaly_score=(
                "anomaly_score",
                "max"
            ),
        )
        .reset_index()
    )


    summary[
        "flagged_ratio"
    ] = (
        summary[
            "flagged_windows"
        ]
        /
        summary[
            "test_windows"
        ]
    )


    summary[
        "flagged_ratio_percent"
    ] = (
        100
        *
        summary[
            "flagged_ratio"
        ]
    )


    return (
        summary
        .sort_values(
            "flagged_ratio",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


historical_channel_summary = (
    make_amp_channel_summary(
        df,
        "predicted_anomaly"
    )
)


corrected_channel_summary = (
    make_amp_channel_summary(
        corrected_df,
        "predicted_anomaly_corrected"
    )
)


channel_comparison = (
    historical_channel_summary[
        [
            "channel_code",
            "channel_name",
            "primary_category",
            "test_windows",
            "flagged_windows",
            "flagged_ratio_percent",
        ]
    ]
    .merge(
        corrected_channel_summary[
            [
                "channel_code",
                "channel_name",
                "primary_category",
                "test_windows",
                "flagged_windows",
                "flagged_ratio_percent",
            ]
        ],
        on=[
            "channel_code",
            "channel_name",
            "primary_category",
        ],
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


channel_comparison[
    "test_windows_delta"
] = (
    channel_comparison[
        "test_windows_corrected"
    ]
    -
    channel_comparison[
        "test_windows_historical"
    ]
)


channel_comparison[
    "flagged_windows_delta"
] = (
    channel_comparison[
        "flagged_windows_corrected"
    ]
    -
    channel_comparison[
        "flagged_windows_historical"
    ]
)


channel_comparison[
    "flagged_ratio_percent_delta"
] = (
    channel_comparison[
        "flagged_ratio_percent_corrected"
    ]
    -
    channel_comparison[
        "flagged_ratio_percent_historical"
    ]
)


channel_comparison = (
    channel_comparison
    .sort_values(
        "flagged_ratio_percent_corrected",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\n")
print("=" * 120)
print("AMPds2 TEST CHANNEL SUMMARY — TOP 10 CORRECTED")
print("=" * 120)


print(
    channel_comparison
    .head(10)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# ============================================================
# K. Save corrected artifacts
# ============================================================

CORRECTED_SCORED_PATH = (
    AMP_CORRECTION_DIR
    / "ampds2_if_scored_boundary_corrected.csv"
)

REMOVED_PATH = (
    AMP_CORRECTION_DIR
    / "ampds2_removed_boundary_windows.csv"
)

PURGE_AUDIT_PATH = (
    AMP_CORRECTION_DIR
    / "ampds2_boundary_purge_audit.csv"
)

CORRECTED_SUMMARY_PATH = (
    AMP_CORRECTION_DIR
    / "ampds2_if_corrected_split_summary.csv"
)

SUMMARY_COMPARISON_PATH = (
    AMP_CORRECTION_DIR
    / "ampds2_if_historical_vs_corrected_split_summary.csv"
)

CORRECTED_CHANNEL_PATH = (
    AMP_CORRECTION_DIR
    / "ampds2_if_corrected_test_channel_summary.csv"
)

CHANNEL_COMPARISON_PATH = (
    AMP_CORRECTION_DIR
    / "ampds2_if_historical_vs_corrected_test_channel_summary.csv"
)


corrected_df.to_csv(
    CORRECTED_SCORED_PATH,
    index=False
)

removed_df.to_csv(
    REMOVED_PATH,
    index=False
)

purge_audit_df.to_csv(
    PURGE_AUDIT_PATH,
    index=False
)

corrected_summary.to_csv(
    CORRECTED_SUMMARY_PATH,
    index=False
)

summary_comparison.to_csv(
    SUMMARY_COMPARISON_PATH,
    index=False
)

corrected_channel_summary.to_csv(
    CORRECTED_CHANNEL_PATH,
    index=False
)

channel_comparison.to_csv(
    CHANNEL_COMPARISON_PATH,
    index=False
)


# ============================================================
# L. Final checkpoint
# ============================================================

corrected_test = (
    corrected_summary[
        corrected_summary[
            "split"
        ]
        == "test"
    ]
    .iloc[0]
)


print("\n")
print("=" * 120)
print("AMPds2 CORRECTION CHECKPOINT")
print("=" * 120)


print(
    f"Historical total: {len(df):,}"
)

print(
    f"Corrected total:  {len(corrected_df):,}"
)

print(
    f"Removed:          {len(removed_df):,}"
)


print(
    f"\nHistorical threshold: "
    f"{historical_threshold:.12f}"
)

print(
    f"Corrected threshold:  "
    f"{corrected_threshold:.12f}"
)


print(
    f"\nCorrected test flagged windows: "
    f"{int(corrected_test['flagged_windows']):,}"
)

print(
    f"Corrected test flagged ratio: "
    f"{corrected_test['flagged_ratio_percent']:.6f}%"
)


print(
    "\nSaved corrected artifacts under:"
)

print(
    AMP_CORRECTION_DIR
)


print("\n")
print("=" * 120)
print("CELL 30 COMPLETE")
print("=" * 120)

print(
    "\nNo AMPds2 model was retrained. "
    "The original scored dataset was not modified."
)

In [ ]:
# CELL 31 — Controlled Residential Final-Artifact Identifier and Correction-Input Audit

from pathlib import Path
import pandas as pd
import numpy as np


print("=" * 120)
print("CONTROLLED UK-DALE / AMPds2 CORRECTION-INPUT AUDIT")
print("=" * 120)


# ============================================================
# A. Paths
# ============================================================

CONTROLLED_TABLE_DIR = (
    BASE
    / "experiments"
    / "EXP_UKDALE_AMPDS2_Cross_Model_Comparison"
    / "tables"
)


CONTROLLED_INPUTS = {

    "UK-DALE": {

        "reference":
            CONTROLLED_TABLE_DIR
            / "ukdale_lof_reference_sample.csv",

        "evaluation":
            CONTROLLED_TABLE_DIR
            / "ukdale_cross_model_evaluation_sample.csv",

        "final_scored":
            CONTROLLED_TABLE_DIR
            / "ukdale_balanced_if_lof_ae_scored_evaluation.csv",

        "conflicts":
            CORRECTION
            / "ukdale_controlled_conflicting_evaluation_rows.csv",
    },

    "AMPds2": {

        "reference":
            CONTROLLED_TABLE_DIR
            / "ampds2_lof_reference_sample.csv",

        "evaluation":
            CONTROLLED_TABLE_DIR
            / "ampds2_cross_model_evaluation_sample.csv",

        "final_scored":
            CONTROLLED_TABLE_DIR
            / "ampds2_balanced_if_lof_ae_scored_evaluation.csv",

        "conflicts":
            CORRECTION
            / "ampds2_controlled_conflicting_evaluation_rows.csv",
    },
}


# ============================================================
# B. Inspect each dataset
# ============================================================

audit_rows = []


for dataset_name, paths in (
    CONTROLLED_INPUTS.items()
):

    print("\n")
    print("=" * 120)
    print(dataset_name)
    print("=" * 120)


    for label, path in (
        paths.items()
    ):

        print(
            f"\n{label}:"
        )

        print(
            "  exists:",
            path.exists()
        )

        print(
            "  path:",
            path
        )

        assert path.exists(), (
            f"Missing {dataset_name} {label}: {path}"
        )


    reference_df = pd.read_csv(
        paths["reference"],
        low_memory=False
    )

    evaluation_df = pd.read_csv(
        paths["evaluation"],
        low_memory=False
    )

    final_df = pd.read_csv(
        paths["final_scored"],
        low_memory=False
    )

    conflict_df = pd.read_csv(
        paths["conflicts"],
        low_memory=False
    )


    print(
        "\nShapes:"
    )

    print(
        "  reference:",
        reference_df.shape
    )

    print(
        "  evaluation:",
        evaluation_df.shape
    )

    print(
        "  final scored:",
        final_df.shape
    )

    print(
        "  conflict rows:",
        conflict_df.shape
    )


    print(
        "\nEvaluation columns:"
    )

    print(
        evaluation_df.columns.tolist()
    )


    print(
        "\nFinal scored columns:"
    )

    print(
        final_df.columns.tolist()
    )


    print(
        "\nConflict-file columns:"
    )

    print(
        conflict_df.columns.tolist()
    )


    # --------------------------------------------------------
    # Split counts
    # --------------------------------------------------------

    print(
        "\nOriginal evaluation split counts:"
    )

    print(
        evaluation_df[
            "evaluation_split"
        ]
        .value_counts()
        .reindex(
            [
                "train",
                "val",
                "test",
            ]
        )
    )


    print(
        "\nFinal scored split counts:"
    )

    print(
        final_df[
            "evaluation_split"
        ]
        .value_counts()
        .reindex(
            [
                "train",
                "val",
                "test",
            ]
        )
    )


    # --------------------------------------------------------
    # Verify final scored table contains exactly the same
    # evaluation rows.
    # --------------------------------------------------------

    assert len(
        evaluation_df
    ) == len(
        final_df
    )


    candidate_identity_columns = [
        "source_row_id",
        "series_key",
        "evaluation_split",
        "window_start",
        "window_end",
    ]


    common_identity_columns = [
        col
        for col
        in candidate_identity_columns
        if (
            col in evaluation_df.columns
            and
            col in final_df.columns
        )
    ]


    print(
        "\nCommon identity columns:"
    )

    print(
        common_identity_columns
    )


    assert len(
        common_identity_columns
    ) >= 4


    left = (
        evaluation_df[
            common_identity_columns
        ]
        .copy()
        .reset_index(drop=True)
    )

    right = (
        final_df[
            common_identity_columns
        ]
        .copy()
        .reset_index(drop=True)
    )


    for timestamp_col in [
        "window_start",
        "window_end",
    ]:

        if (
            timestamp_col
            in common_identity_columns
        ):

            left[
                timestamp_col
            ] = pd.to_datetime(
                left[
                    timestamp_col
                ],
                utc=True,
                errors="raise"
            )

            right[
                timestamp_col
            ] = pd.to_datetime(
                right[
                    timestamp_col
                ],
                utc=True,
                errors="raise"
            )


    pd.testing.assert_frame_equal(
        left,
        right,
        check_dtype=False,
        check_exact=True
    )


    print(
        "PASS: final scored table preserves "
        "the original evaluation-row order and identity."
    )


    # --------------------------------------------------------
    # Critical score columns
    # --------------------------------------------------------

    required_score_columns = [
        "if_balanced_anomaly_score",
        "lof_anomaly_score",
        "ae_anomaly_score",
    ]


    missing_scores = [
        col
        for col
        in required_score_columns
        if col
        not in final_df.columns
    ]


    print(
        "\nMissing required score columns:",
        missing_scores
    )

    assert not missing_scores


    # --------------------------------------------------------
    # Historical threshold columns
    # --------------------------------------------------------

    threshold_columns = [
        col
        for col
        in final_df.columns
        if "threshold"
        in col.lower()
    ]


    print(
        "\nThreshold-related columns:"
    )

    print(
        threshold_columns
    )


    for col in threshold_columns:

        unique_values = (
            final_df[
                col
            ]
            .dropna()
            .unique()
        )

        if len(
            unique_values
        ) <= 5:

            print(
                f"  {col}:",
                unique_values
            )


    # --------------------------------------------------------
    # Conflict preview
    # --------------------------------------------------------

    print(
        "\nConflict rows preview:"
    )

    preview_columns = [
        col
        for col
        in [
            "audit_evaluation_id",
            "source_row_id",
            "series_key",
            "evaluation_split",
            "window_start",
            "window_end",
            "purge_reason",
        ]
        if col
        in conflict_df.columns
    ]


    print(
        conflict_df[
            preview_columns
        ]
        .head(10)
        .to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Expected audit counts from previous exact audit
    # --------------------------------------------------------

    if (
        dataset_name
        == "UK-DALE"
    ):

        expected_reference = 12000
        expected_evaluation = 19694
        expected_conflicts = 1303

        expected_split_counts = {
            "train":
                4000,

            "val":
                7847,

            "test":
                7847,
        }

    else:

        expected_reference = 15750
        expected_evaluation = 26250
        expected_conflicts = 374

        expected_split_counts = {
            "train":
                5250,

            "val":
                10500,

            "test":
                10500,
        }


    assert len(
        reference_df
    ) == expected_reference

    assert len(
        evaluation_df
    ) == expected_evaluation

    assert len(
        conflict_df
    ) == expected_conflicts


    for split_name, expected_count in (
        expected_split_counts.items()
    ):

        actual_count = int(
            (
                evaluation_df[
                    "evaluation_split"
                ]
                == split_name
            ).sum()
        )

        assert (
            actual_count
            == expected_count
        )


    audit_rows.append({

        "dataset":
            dataset_name,

        "reference_rows":
            len(
                reference_df
            ),

        "evaluation_rows":
            len(
                evaluation_df
            ),

        "conflicting_evaluation_rows":
            len(
                conflict_df
            ),

        "common_identity_columns":
            "|".join(
                common_identity_columns
            ),

        "required_scores_present":
            True,

        "final_scored_row_identity_pass":
            True,
    })


# ============================================================
# C. Consolidated checkpoint
# ============================================================

controlled_input_audit_df = pd.DataFrame(
    audit_rows
)


print("\n")
print("=" * 120)
print("CONTROLLED CORRECTION INPUT CHECKPOINT")
print("=" * 120)


print(
    controlled_input_audit_df.to_string(
        index=False
    )
)


OUTPUT_PATH = (
    CORRECTION
    / "controlled_residential_correction_input_audit.csv"
)


controlled_input_audit_df.to_csv(
    OUTPUT_PATH,
    index=False
)


print(
    "\nSaved audit to:"
)

print(
    OUTPUT_PATH
)


print("\n")
print("=" * 120)
print("CELL 31 COMPLETE")
print("=" * 120)

print(
    "\nNo model, sample, score, threshold, or historical "
    "artifact was modified."
)

In [ ]:
# CELL 32 — Controlled UK-DALE and AMPds2 Three-Model Boundary Correction

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


print("=" * 125)
print("CONTROLLED UK-DALE / AMPds2 THREE-MODEL SPLIT-BOUNDARY CORRECTION")
print("=" * 125)


# ============================================================
# A. Configuration
# ============================================================

CONTROLLED_CORRECTION_DIR = (
    CORRECTION
    / "CONTROLLED_RESIDENTIAL"
)

CONTROLLED_CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CONTROLLED_TABLE_DIR = (
    BASE
    / "experiments"
    / "EXP_UKDALE_AMPDS2_Cross_Model_Comparison"
    / "tables"
)


DATASETS = {

    "UK-DALE": {

        "slug":
            "ukdale",

        "final_scored":
            CONTROLLED_TABLE_DIR
            / "ukdale_balanced_if_lof_ae_scored_evaluation.csv",

        "conflicts":
            CORRECTION
            / "ukdale_controlled_conflicting_evaluation_rows.csv",

        "expected_original":
            {
                "train": 4000,
                "val": 7847,
                "test": 7847,
            },

        "expected_removed":
            {
                "train": 1299,
                "val": 2,
                "test": 2,
            },

        "expected_corrected":
            {
                "train": 2701,
                "val": 7845,
                "test": 7845,
            },

        "category_columns":
            [
                "primary_category",
            ],
    },

    "AMPds2": {

        "slug":
            "ampds2",

        "final_scored":
            CONTROLLED_TABLE_DIR
            / "ampds2_balanced_if_lof_ae_scored_evaluation.csv",

        "conflicts":
            CORRECTION
            / "ampds2_controlled_conflicting_evaluation_rows.csv",

        "expected_original":
            {
                "train": 5250,
                "val": 10500,
                "test": 10500,
            },

        "expected_removed":
            {
                "train": 373,
                "val": 1,
                "test": 0,
            },

        "expected_corrected":
            {
                "train": 4877,
                "val": 10499,
                "test": 10500,
            },

        "category_columns":
            [
                "channel_code",
                "channel_name",
                "primary_category",
            ],
    },
}


MODELS = {

    "Isolation Forest": {
        "score":
            "if_balanced_anomaly_score",

        "old_flag":
            "if_balanced_predicted_anomaly",

        "old_threshold":
            "if_balanced_threshold",

        "new_flag":
            "if_balanced_predicted_anomaly_corrected",

        "new_threshold":
            "if_balanced_threshold_corrected",
    },

    "Local Outlier Factor": {
        "score":
            "lof_anomaly_score",

        "old_flag":
            "lof_predicted_anomaly",

        "old_threshold":
            "lof_threshold",

        "new_flag":
            "lof_predicted_anomaly_corrected",

        "new_threshold":
            "lof_threshold_corrected",
    },

    "Dense Autoencoder": {
        "score":
            "ae_anomaly_score",

        "old_flag":
            "ae_predicted_anomaly",

        "old_threshold":
            "ae_threshold",

        "new_flag":
            "ae_predicted_anomaly_corrected",

        "new_threshold":
            "ae_threshold_corrected",
    },
}


# ============================================================
# B. Helpers
# ============================================================

def prepare_timestamps(
    df
):

    out = df.copy()

    out[
        "window_start"
    ] = pd.to_datetime(
        out[
            "window_start"
        ],
        utc=True,
        errors="raise"
    )

    out[
        "window_end"
    ] = pd.to_datetime(
        out[
            "window_end"
        ],
        utc=True,
        errors="raise"
    )

    return out


def split_counts(
    df
):

    return (
        df[
            "evaluation_split"
        ]
        .value_counts()
        .reindex(
            [
                "train",
                "val",
                "test",
            ],
            fill_value=0
        )
    )


def model_split_summary(
    dataset_name,
    df,
    flag_columns,
    score_columns
):

    rows = []

    for model_name in MODELS:

        flag_col = flag_columns[
            model_name
        ]

        score_col = score_columns[
            model_name
        ]

        for split_name in [
            "train",
            "val",
            "test",
        ]:

            part = (
                df[
                    df[
                        "evaluation_split"
                    ]
                    ==
                    split_name
                ]
            )

            total = len(
                part
            )

            flagged = int(
                part[
                    flag_col
                ].sum()
            )

            rows.append({

                "dataset":
                    dataset_name,

                "model":
                    model_name,

                "split":
                    split_name,

                "windows":
                    total,

                "flagged_windows":
                    flagged,

                "flagged_ratio":
                    flagged
                    / total,

                "flagged_ratio_percent":
                    100
                    * flagged
                    / total,

                "median_score":
                    part[
                        score_col
                    ].median(),

                "score_q95":
                    part[
                        score_col
                    ].quantile(
                        0.95
                    ),

                "score_q99":
                    part[
                        score_col
                    ].quantile(
                        0.99
                    ),
            })

    return pd.DataFrame(
        rows
    )


def pairwise_test_summary(
    dataset_name,
    df,
    flag_columns,
    score_columns
):

    test_df = (
        df[
            df[
                "evaluation_split"
            ]
            ==
            "test"
        ]
        .copy()
    )

    pairs = [
        (
            "Isolation Forest",
            "Local Outlier Factor",
        ),
        (
            "Isolation Forest",
            "Dense Autoencoder",
        ),
        (
            "Local Outlier Factor",
            "Dense Autoencoder",
        ),
    ]

    rows = []

    for model_a, model_b in pairs:

        flags_a = (
            test_df[
                flag_columns[
                    model_a
                ]
            ]
            .astype(int)
            .to_numpy()
        )

        flags_b = (
            test_df[
                flag_columns[
                    model_b
                ]
            ]
            .astype(int)
            .to_numpy()
        )

        scores_a = (
            test_df[
                score_columns[
                    model_a
                ]
            ]
            .to_numpy()
        )

        scores_b = (
            test_df[
                score_columns[
                    model_b
                ]
            ]
            .to_numpy()
        )

        both = int(
            (
                (flags_a == 1)
                &
                (flags_b == 1)
            ).sum()
        )

        only_a = int(
            (
                (flags_a == 1)
                &
                (flags_b == 0)
            ).sum()
        )

        only_b = int(
            (
                (flags_a == 0)
                &
                (flags_b == 1)
            ).sum()
        )

        neither = int(
            (
                (flags_a == 0)
                &
                (flags_b == 0)
            ).sum()
        )

        union = (
            both
            + only_a
            + only_b
        )

        rank_result = (
            spearmanr(
                scores_a,
                scores_b
            )
        )

        rows.append({

            "dataset":
                dataset_name,

            "model_a":
                model_a,

            "model_b":
                model_b,

            "test_windows":
                len(
                    test_df
                ),

            "model_a_flagged":
                int(
                    flags_a.sum()
                ),

            "model_b_flagged":
                int(
                    flags_b.sum()
                ),

            "both_flagged":
                both,

            "model_a_only":
                only_a,

            "model_b_only":
                only_b,

            "neither_flagged":
                neither,

            "flag_jaccard_index":
                (
                    both / union
                    if union
                    else np.nan
                ),

            "score_spearman_correlation":
                rank_result.statistic,

            "score_spearman_p_value":
                rank_result.pvalue,
        })

    return pd.DataFrame(
        rows
    )


def consensus_test_summary(
    dataset_name,
    df,
    flag_columns
):

    test_df = (
        df[
            df[
                "evaluation_split"
            ]
            ==
            "test"
        ]
        .copy()
    )

    flag_count = (
        test_df[
            flag_columns[
                "Isolation Forest"
            ]
        ].astype(int)
        +
        test_df[
            flag_columns[
                "Local Outlier Factor"
            ]
        ].astype(int)
        +
        test_df[
            flag_columns[
                "Dense Autoencoder"
            ]
        ].astype(int)
    )

    total = len(
        test_df
    )

    all_three = int(
        (
            flag_count == 3
        ).sum()
    )

    at_least_two = int(
        (
            flag_count >= 2
        ).sum()
    )

    any_model = int(
        (
            flag_count >= 1
        ).sum()
    )

    return pd.DataFrame([
        {
            "dataset":
                dataset_name,

            "test_windows":
                total,

            "all_three_models_flagged":
                all_three,

            "all_three_ratio_percent":
                100
                * all_three
                / total,

            "at_least_two_models_flagged":
                at_least_two,

            "at_least_two_ratio_percent":
                100
                * at_least_two
                / total,

            "any_model_flagged":
                any_model,

            "any_model_ratio_percent":
                100
                * any_model
                / total,
        }
    ])


def category_test_summary(
    dataset_name,
    df,
    category_columns,
    flag_columns
):

    test_df = (
        df[
            df[
                "evaluation_split"
            ]
            ==
            "test"
        ]
        .copy()
    )

    rows = []

    grouper = (
        category_columns[0]
        if len(
            category_columns
        ) == 1
        else category_columns
    )

    for group_key, group in (
        test_df.groupby(
            grouper,
            dropna=False
        )
    ):

        if not isinstance(
            group_key,
            tuple
        ):

            group_key = (
                group_key,
            )

        row = {
            col:
                value
            for col, value
            in zip(
                category_columns,
                group_key
            )
        }

        row[
            "test_windows"
        ] = len(
            group
        )

        flag_count = np.zeros(
            len(group),
            dtype=int
        )

        short_names = {
            "Isolation Forest":
                "if",

            "Local Outlier Factor":
                "lof",

            "Dense Autoencoder":
                "ae",
        }

        for model_name in MODELS:

            flag_col = (
                flag_columns[
                    model_name
                ]
            )

            flags = (
                group[
                    flag_col
                ]
                .astype(int)
                .to_numpy()
            )

            flagged = int(
                flags.sum()
            )

            short = (
                short_names[
                    model_name
                ]
            )

            row[
                f"{short}_flagged_windows"
            ] = flagged

            row[
                f"{short}_flagged_ratio_percent"
            ] = (
                100
                * flagged
                / len(group)
            )

            flag_count += (
                flags
            )

        row[
            "all_three_models_flagged"
        ] = int(
            (
                flag_count == 3
            ).sum()
        )

        row[
            "at_least_two_models_flagged"
        ] = int(
            (
                flag_count >= 2
            ).sum()
        )

        row[
            "at_least_two_models_ratio_percent"
        ] = (
            100
            * row[
                "at_least_two_models_flagged"
            ]
            / len(group)
        )

        rows.append(
            row
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "at_least_two_models_ratio_percent",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# C. Correct each dataset
# ============================================================

all_threshold_rows = []

all_split_historical = []
all_split_corrected = []

all_pairwise_historical = []
all_pairwise_corrected = []

all_consensus_historical = []
all_consensus_corrected = []

all_category_historical = {}
all_category_corrected = {}

corrected_scored_tables = {}


for dataset_name, spec in (
    DATASETS.items()
):

    print("\n")
    print("=" * 125)
    print(
        dataset_name
    )
    print("=" * 125)


    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    final_df = prepare_timestamps(
        pd.read_csv(
            spec[
                "final_scored"
            ],
            low_memory=False
        )
    )

    conflict_df = prepare_timestamps(
        pd.read_csv(
            spec[
                "conflicts"
            ],
            low_memory=False
        )
    )


    # Conflict file calls the evaluation split simply "split".
    conflict_df = (
        conflict_df.rename(
            columns={
                "split":
                    "evaluation_split"
            }
        )
    )


    # --------------------------------------------------------
    # Verify historical counts
    # --------------------------------------------------------

    original_counts = (
        split_counts(
            final_df
        )
    )


    print(
        "\nOriginal split counts:"
    )

    print(
        original_counts
    )


    for split_name, expected in (
        spec[
            "expected_original"
        ].items()
    ):

        assert int(
            original_counts[
                split_name
            ]
        ) == expected


    # --------------------------------------------------------
    # Match conflict rows by exact temporal identity
    # --------------------------------------------------------

    match_columns = [
        "series_key",
        "evaluation_split",
        "window_start",
        "window_end",
    ]


    assert not (
        conflict_df[
            match_columns
        ].duplicated()
        .any()
    )


    assert not (
        final_df[
            match_columns
        ].duplicated()
        .any()
    )


    final_identity_index = (
        pd.MultiIndex.from_frame(
            final_df[
                match_columns
            ]
        )
    )

    conflict_identity_index = (
        pd.MultiIndex.from_frame(
            conflict_df[
                match_columns
            ]
        )
    )


    removal_mask = (
        final_identity_index.isin(
            conflict_identity_index
        )
    )


    matched_removed_count = int(
        removal_mask.sum()
    )


    print(
        "\nConflict rows requested:",
        len(
            conflict_df
        )
    )

    print(
        "Exact conflict rows matched:",
        matched_removed_count
    )


    assert (
        matched_removed_count
        == len(
            conflict_df
        )
    )


    # Every conflict must exist once in final table.
    matched_identity_index = (
        final_identity_index[
            removal_mask
        ]
    )

    assert (
        len(
            matched_identity_index
        )
        ==
        len(
            conflict_identity_index
        )
    )


    removed_df = (
        final_df.loc[
            removal_mask
        ]
        .copy()
    )


    corrected_df = (
        final_df.loc[
            ~removal_mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Verify conflict counts by split
    # --------------------------------------------------------

    removed_counts = (
        removed_df[
            "evaluation_split"
        ]
        .value_counts()
        .reindex(
            [
                "train",
                "val",
                "test",
            ],
            fill_value=0
        )
    )


    corrected_counts = (
        split_counts(
            corrected_df
        )
    )


    print(
        "\nRemoved by split:"
    )

    print(
        removed_counts
    )


    print(
        "\nCorrected split counts:"
    )

    print(
        corrected_counts
    )


    for split_name, expected in (
        spec[
            "expected_removed"
        ].items()
    ):

        assert int(
            removed_counts[
                split_name
            ]
        ) == expected


    for split_name, expected in (
        spec[
            "expected_corrected"
        ].items()
    ):

        assert int(
            corrected_counts[
                split_name
            ]
        ) == expected


    # --------------------------------------------------------
    # Threshold recalculation from retained validation rows
    # --------------------------------------------------------

    print(
        "\nThreshold recalculation:"
    )


    historical_flag_columns = {}

    corrected_flag_columns = {}

    score_columns = {}


    for model_name, model_spec in (
        MODELS.items()
    ):

        score_col = (
            model_spec[
                "score"
            ]
        )

        old_flag_col = (
            model_spec[
                "old_flag"
            ]
        )

        old_threshold_col = (
            model_spec[
                "old_threshold"
            ]
        )

        new_flag_col = (
            model_spec[
                "new_flag"
            ]
        )

        new_threshold_col = (
            model_spec[
                "new_threshold"
            ]
        )


        score_columns[
            model_name
        ] = score_col

        historical_flag_columns[
            model_name
        ] = old_flag_col

        corrected_flag_columns[
            model_name
        ] = new_flag_col


        # Historical stored threshold
        historical_threshold_values = (
            final_df[
                old_threshold_col
            ]
            .dropna()
            .unique()
        )

        assert len(
            historical_threshold_values
        ) == 1


        stored_historical_threshold = float(
            historical_threshold_values[
                0
            ]
        )


        # Recompute original threshold as provenance check
        original_val_scores = (
            final_df.loc[
                final_df[
                    "evaluation_split"
                ]
                ==
                "val",
                score_col
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        recomputed_original_threshold = float(
            np.percentile(
                original_val_scores,
                95
            )
        )


        # Corrected threshold
        corrected_val_scores = (
            corrected_df.loc[
                corrected_df[
                    "evaluation_split"
                ]
                ==
                "val",
                score_col
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        corrected_threshold = float(
            np.percentile(
                corrected_val_scores,
                95
            )
        )


        # Controlled experiment source explicitly used >
        corrected_flags = (
            corrected_df[
                score_col
            ].to_numpy(
                dtype=np.float64
            )
            >
            corrected_threshold
        ).astype(
            np.int8
        )


        corrected_df[
            new_flag_col
        ] = corrected_flags


        corrected_df[
            new_threshold_col
        ] = corrected_threshold


        # Test >= only as a diagnostic.
        corrected_flags_ge = (
            corrected_df[
                score_col
            ].to_numpy(
                dtype=np.float64
            )
            >=
            corrected_threshold
        ).astype(
            np.int8
        )


        operator_difference = int(
            np.sum(
                corrected_flags
                !=
                corrected_flags_ge
            )
        )


        print(
            f"{model_name}:"
        )

        print(
            f"  stored historical threshold    = "
            f"{stored_historical_threshold:.12f}"
        )

        print(
            f"  recomputed original threshold  = "
            f"{recomputed_original_threshold:.12f}"
        )

        print(
            f"  corrected threshold            = "
            f"{corrected_threshold:.12f}"
        )

        print(
            f"  corrected - historical         = "
            f"{corrected_threshold - stored_historical_threshold:.12e}"
        )

        print(
            f"  > vs >= corrected differences  = "
            f"{operator_difference}"
        )


        all_threshold_rows.append({

            "dataset":
                dataset_name,

            "model":
                model_name,

            "stored_historical_threshold":
                stored_historical_threshold,

            "recomputed_original_threshold":
                recomputed_original_threshold,

            "historical_threshold_recompute_delta":
                (
                    recomputed_original_threshold
                    -
                    stored_historical_threshold
                ),

            "corrected_threshold":
                corrected_threshold,

            "corrected_minus_historical_threshold":
                (
                    corrected_threshold
                    -
                    stored_historical_threshold
                ),

            "corrected_gt_vs_ge_differences":
                operator_difference,

            "original_validation_rows":
                int(
                    original_counts[
                        "val"
                    ]
                ),

            "corrected_validation_rows":
                int(
                    corrected_counts[
                        "val"
                    ]
                ),
        })


    # --------------------------------------------------------
    # Split summaries
    # --------------------------------------------------------

    historical_split_summary = (
        model_split_summary(
            dataset_name,
            final_df,
            historical_flag_columns,
            score_columns
        )
    )


    corrected_split_summary = (
        model_split_summary(
            dataset_name,
            corrected_df,
            corrected_flag_columns,
            score_columns
        )
    )


    all_split_historical.append(
        historical_split_summary
    )

    all_split_corrected.append(
        corrected_split_summary
    )


    # --------------------------------------------------------
    # Pairwise TEST summaries
    # --------------------------------------------------------

    historical_pairwise = (
        pairwise_test_summary(
            dataset_name,
            final_df,
            historical_flag_columns,
            score_columns
        )
    )


    corrected_pairwise = (
        pairwise_test_summary(
            dataset_name,
            corrected_df,
            corrected_flag_columns,
            score_columns
        )
    )


    all_pairwise_historical.append(
        historical_pairwise
    )

    all_pairwise_corrected.append(
        corrected_pairwise
    )


    # --------------------------------------------------------
    # Consensus TEST summaries
    # --------------------------------------------------------

    historical_consensus = (
        consensus_test_summary(
            dataset_name,
            final_df,
            historical_flag_columns
        )
    )


    corrected_consensus = (
        consensus_test_summary(
            dataset_name,
            corrected_df,
            corrected_flag_columns
        )
    )


    all_consensus_historical.append(
        historical_consensus
    )

    all_consensus_corrected.append(
        corrected_consensus
    )


    # --------------------------------------------------------
    # Category / channel summaries
    # --------------------------------------------------------

    historical_category = (
        category_test_summary(
            dataset_name,
            final_df,
            spec[
                "category_columns"
            ],
            historical_flag_columns
        )
    )


    corrected_category = (
        category_test_summary(
            dataset_name,
            corrected_df,
            spec[
                "category_columns"
            ],
            corrected_flag_columns
        )
    )


    all_category_historical[
        dataset_name
    ] = historical_category

    all_category_corrected[
        dataset_name
    ] = corrected_category


    corrected_scored_tables[
        dataset_name
    ] = corrected_df


    # --------------------------------------------------------
    # Save dataset-specific corrected and removed rows
    # --------------------------------------------------------

    dataset_dir = (
        CONTROLLED_CORRECTION_DIR
        / spec[
            "slug"
        ]
    )

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    corrected_df.to_csv(
        dataset_dir
        / (
            f"{spec['slug']}_"
            f"controlled_three_model_scored_corrected.csv"
        ),
        index=False
    )


    removed_df.to_csv(
        dataset_dir
        / (
            f"{spec['slug']}_"
            f"controlled_removed_evaluation_rows.csv"
        ),
        index=False
    )


# ============================================================
# D. Consolidate
# ============================================================

threshold_df = pd.DataFrame(
    all_threshold_rows
)


historical_split_df = pd.concat(
    all_split_historical,
    ignore_index=True
)

corrected_split_df = pd.concat(
    all_split_corrected,
    ignore_index=True
)


historical_pairwise_df = pd.concat(
    all_pairwise_historical,
    ignore_index=True
)

corrected_pairwise_df = pd.concat(
    all_pairwise_corrected,
    ignore_index=True
)


historical_consensus_df = pd.concat(
    all_consensus_historical,
    ignore_index=True
)

corrected_consensus_df = pd.concat(
    all_consensus_corrected,
    ignore_index=True
)


# ============================================================
# E. Historical vs corrected split-level comparison
# ============================================================

split_comparison_df = (
    historical_split_df[
        [
            "dataset",
            "model",
            "split",
            "windows",
            "flagged_windows",
            "flagged_ratio_percent",
        ]
    ]
    .merge(
        corrected_split_df[
            [
                "dataset",
                "model",
                "split",
                "windows",
                "flagged_windows",
                "flagged_ratio_percent",
            ]
        ],
        on=[
            "dataset",
            "model",
            "split",
        ],
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


split_comparison_df[
    "windows_delta"
] = (
    split_comparison_df[
        "windows_corrected"
    ]
    -
    split_comparison_df[
        "windows_historical"
    ]
)


split_comparison_df[
    "flagged_windows_delta"
] = (
    split_comparison_df[
        "flagged_windows_corrected"
    ]
    -
    split_comparison_df[
        "flagged_windows_historical"
    ]
)


split_comparison_df[
    "flagged_ratio_percent_delta"
] = (
    split_comparison_df[
        "flagged_ratio_percent_corrected"
    ]
    -
    split_comparison_df[
        "flagged_ratio_percent_historical"
    ]
)


print("\n")
print("=" * 125)
print("CONTROLLED MODEL THRESHOLD CORRECTION")
print("=" * 125)


print(
    threshold_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.12f}"
    )
)


print("\n")
print("=" * 125)
print("CONTROLLED TEST FLAGGED-RATIO SUMMARY — HISTORICAL VS CORRECTED")
print("=" * 125)


print(
    split_comparison_df[
        split_comparison_df[
            "split"
        ]
        ==
        "test"
    ][
        [
            "dataset",
            "model",

            "windows_historical",
            "windows_corrected",

            "flagged_windows_historical",
            "flagged_windows_corrected",

            "flagged_ratio_percent_historical",
            "flagged_ratio_percent_corrected",
            "flagged_ratio_percent_delta",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# F. Pairwise historical vs corrected
# ============================================================

pairwise_comparison_df = (
    historical_pairwise_df.merge(
        corrected_pairwise_df,
        on=[
            "dataset",
            "model_a",
            "model_b",
        ],
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


print("\n")
print("=" * 125)
print("CONTROLLED TEST PAIRWISE AGREEMENT — HISTORICAL VS CORRECTED")
print("=" * 125)


print(
    pairwise_comparison_df[
        [
            "dataset",
            "model_a",
            "model_b",

            "test_windows_historical",
            "test_windows_corrected",

            "flag_jaccard_index_historical",
            "flag_jaccard_index_corrected",

            "score_spearman_correlation_historical",
            "score_spearman_correlation_corrected",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# G. Consensus historical vs corrected
# ============================================================

consensus_comparison_df = (
    historical_consensus_df.merge(
        corrected_consensus_df,
        on="dataset",
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


print("\n")
print("=" * 125)
print("CONTROLLED TEST CONSENSUS — HISTORICAL VS CORRECTED")
print("=" * 125)


print(
    consensus_comparison_df[
        [
            "dataset",

            "test_windows_historical",
            "test_windows_corrected",

            "all_three_models_flagged_historical",
            "all_three_models_flagged_corrected",

            "all_three_ratio_percent_historical",
            "all_three_ratio_percent_corrected",

            "at_least_two_models_flagged_historical",
            "at_least_two_models_flagged_corrected",

            "at_least_two_ratio_percent_historical",
            "at_least_two_ratio_percent_corrected",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# H. Corrected category/channel summaries
# ============================================================

print("\n")
print("=" * 125)
print("UK-DALE CORRECTED CONTROLLED TEST CATEGORY SUMMARY")
print("=" * 125)


print(
    all_category_corrected[
        "UK-DALE"
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n")
print("=" * 125)
print("AMPds2 CORRECTED CONTROLLED TEST CHANNEL SUMMARY — TOP 10")
print("=" * 125)


print(
    all_category_corrected[
        "AMPds2"
    ]
    .head(10)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# I. Save consolidated corrected outputs
# ============================================================

threshold_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_corrected_thresholds.csv",
    index=False
)


corrected_split_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_corrected_split_summary.csv",
    index=False
)


split_comparison_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_historical_vs_corrected_split_summary.csv",
    index=False
)


corrected_pairwise_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_corrected_test_pairwise_agreement.csv",
    index=False
)


pairwise_comparison_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_historical_vs_corrected_test_pairwise_agreement.csv",
    index=False
)


corrected_consensus_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_corrected_test_consensus_summary.csv",
    index=False
)


consensus_comparison_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_historical_vs_corrected_test_consensus_summary.csv",
    index=False
)


all_category_corrected[
    "UK-DALE"
].to_csv(
    CONTROLLED_CORRECTION_DIR
    / "ukdale_controlled_corrected_test_category_summary.csv",
    index=False
)


all_category_corrected[
    "AMPds2"
].to_csv(
    CONTROLLED_CORRECTION_DIR
    / "ampds2_controlled_corrected_test_channel_summary.csv",
    index=False
)


print(
    "\nSaved corrected controlled-comparison outputs under:"
)

print(
    CONTROLLED_CORRECTION_DIR
)


print("\n")
print("=" * 125)
print("CELL 32 COMPLETE")
print("=" * 125)

print(
    "\nThe reference samples and fitted IF/LOF/AE models "
    "were unchanged. No model was retrained and no historical "
    "artifact was overwritten."
)

In [ ]:
# CELL 32A — Diagnose Unmatched Controlled Residential Conflict Rows

import numpy as np
import pandas as pd


print("=" * 120)
print("CONTROLLED CONFLICT-ROW MATCH DIAGNOSTIC")
print("=" * 120)


for dataset_name, spec in DATASETS.items():

    print("\n")
    print("=" * 120)
    print(dataset_name)
    print("=" * 120)


    final_df = pd.read_csv(
        spec["final_scored"],
        low_memory=False
    )

    conflict_df = pd.read_csv(
        spec["conflicts"],
        low_memory=False
    )


    # --------------------------------------------------------
    # Normalize timestamps only
    # --------------------------------------------------------

    for frame in [
        final_df,
        conflict_df,
    ]:

        frame["window_start"] = pd.to_datetime(
            frame["window_start"],
            utc=True,
            errors="raise"
        )

        frame["window_end"] = pd.to_datetime(
            frame["window_end"],
            utc=True,
            errors="raise"
        )


    print("\nConflict-file split values:")

    print(
        conflict_df["split"]
        .value_counts(
            dropna=False
        )
    )


    print("\nFinal scored evaluation_split values:")

    print(
        final_df["evaluation_split"]
        .value_counts(
            dropna=False
        )
    )


    # --------------------------------------------------------
    # Exact identity with split included
    # --------------------------------------------------------

    final_exact = set(
        zip(
            final_df["series_key"],
            final_df["evaluation_split"],
            final_df["window_start"],
            final_df["window_end"],
        )
    )


    conflict_exact = list(
        zip(
            conflict_df["series_key"],
            conflict_df["split"],
            conflict_df["window_start"],
            conflict_df["window_end"],
        )
    )


    exact_match_mask = np.array(
        [
            key in final_exact
            for key in conflict_exact
        ],
        dtype=bool
    )


    unmatched = (
        conflict_df.loc[
            ~exact_match_mask
        ]
        .copy()
    )


    print(
        "\nExact matches:",
        int(
            exact_match_mask.sum()
        ),
        "/",
        len(
            conflict_df
        )
    )


    print(
        "Unmatched rows:",
        len(
            unmatched
        )
    )


    if len(
        unmatched
    ) == 0:

        print(
            "PASS: no unmatched rows."
        )

        continue


    print(
        "\nUnmatched conflict rows:"
    )

    print(
        unmatched.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Search same temporal row IGNORING split name
    # --------------------------------------------------------

    print(
        "\nCandidate rows when split label is ignored:"
    )


    diagnostic_rows = []


    for _, conflict_row in unmatched.iterrows():

        candidate_mask = (
            (
                final_df["series_key"]
                ==
                conflict_row["series_key"]
            )
            &
            (
                final_df["window_start"]
                ==
                conflict_row["window_start"]
            )
            &
            (
                final_df["window_end"]
                ==
                conflict_row["window_end"]
            )
        )


        candidates = (
            final_df.loc[
                candidate_mask,
                [
                    "source_row_id",
                    "series_key",
                    "evaluation_split",
                    "window_start",
                    "window_end",
                ]
            ]
            .copy()
        )


        diagnostic_rows.append({

            "audit_evaluation_id":
                conflict_row.get(
                    "audit_evaluation_id",
                    np.nan
                ),

            "conflict_series_key":
                conflict_row["series_key"],

            "conflict_split":
                conflict_row["split"],

            "window_start":
                conflict_row["window_start"],

            "window_end":
                conflict_row["window_end"],

            "candidate_count_ignoring_split":
                len(
                    candidates
                ),

            "candidate_final_splits":
                "|".join(
                    candidates[
                        "evaluation_split"
                    ]
                    .astype(str)
                    .tolist()
                ),

            "candidate_source_row_ids":
                "|".join(
                    candidates[
                        "source_row_id"
                    ]
                    .astype(str)
                    .tolist()
                ),
        })


    diagnostic_df = pd.DataFrame(
        diagnostic_rows
    )


    print(
        diagnostic_df.to_string(
            index=False
        )
    )


    output_path = (
        CORRECTION
        /
        (
            dataset_name
            .lower()
            .replace("-", "")
            .replace(" ", "_")
            +
            "_controlled_unmatched_conflict_diagnostic.csv"
        )
    )


    diagnostic_df.to_csv(
        output_path,
        index=False
    )


    print(
        "\nSaved diagnostic:",
        output_path
    )


print("\n")
print("=" * 120)
print("CELL 32A COMPLETE")
print("=" * 120)

print(
    "\nNo rows were removed and no historical artifact was modified."
)

In [ ]:
# CELL 32B — Controlled UK-DALE and AMPds2 Three-Model Boundary Correction with Split-Label Normalization

import numpy as np
import pandas as pd
from scipy.stats import spearmanr


print("=" * 125)
print("CONTROLLED THREE-MODEL CORRECTION — NORMALIZED SPLIT LABELS")
print("=" * 125)


# ============================================================
# A. Fresh output containers
# ============================================================

all_threshold_rows = []

all_split_historical = []
all_split_corrected = []

all_pairwise_historical = []
all_pairwise_corrected = []

all_consensus_historical = []
all_consensus_corrected = []

all_category_corrected = {}

corrected_scored_tables = {}


# ============================================================
# B. Correct each dataset
# ============================================================

for dataset_name, spec in DATASETS.items():

    print("\n")
    print("=" * 125)
    print(dataset_name)
    print("=" * 125)


    # --------------------------------------------------------
    # Load final scored evaluation table
    # --------------------------------------------------------

    final_df = pd.read_csv(
        spec["final_scored"],
        low_memory=False
    )


    final_df["window_start"] = pd.to_datetime(
        final_df["window_start"],
        utc=True,
        errors="raise"
    )

    final_df["window_end"] = pd.to_datetime(
        final_df["window_end"],
        utc=True,
        errors="raise"
    )


    # --------------------------------------------------------
    # Load audited conflicts
    # --------------------------------------------------------

    conflict_df = pd.read_csv(
        spec["conflicts"],
        low_memory=False
    )


    conflict_df["window_start"] = pd.to_datetime(
        conflict_df["window_start"],
        utc=True,
        errors="raise"
    )

    conflict_df["window_end"] = pd.to_datetime(
        conflict_df["window_end"],
        utc=True,
        errors="raise"
    )


    # --------------------------------------------------------
    # Critical fix:
    # audit terminology "validation" -> experiment "val"
    # --------------------------------------------------------

    conflict_df[
        "evaluation_split"
    ] = (
        conflict_df["split"]
        .replace({
            "validation": "val"
        })
    )


    print(
        "\nNormalized conflict split counts:"
    )

    print(
        conflict_df[
            "evaluation_split"
        ].value_counts()
    )


    # --------------------------------------------------------
    # Original split checks
    # --------------------------------------------------------

    original_counts = (
        final_df[
            "evaluation_split"
        ]
        .value_counts()
        .reindex(
            [
                "train",
                "val",
                "test",
            ],
            fill_value=0
        )
    )


    print(
        "\nOriginal evaluation split counts:"
    )

    print(
        original_counts
    )


    for split_name, expected in (
        spec[
            "expected_original"
        ].items()
    ):

        assert int(
            original_counts[
                split_name
            ]
        ) == expected


    # --------------------------------------------------------
    # Exact temporal identity matching
    # --------------------------------------------------------

    MATCH_COLUMNS = [
        "series_key",
        "evaluation_split",
        "window_start",
        "window_end",
    ]


    assert not (
        conflict_df[
            MATCH_COLUMNS
        ]
        .duplicated()
        .any()
    )


    assert not (
        final_df[
            MATCH_COLUMNS
        ]
        .duplicated()
        .any()
    )


    final_index = (
        pd.MultiIndex.from_frame(
            final_df[
                MATCH_COLUMNS
            ]
        )
    )


    conflict_index = (
        pd.MultiIndex.from_frame(
            conflict_df[
                MATCH_COLUMNS
            ]
        )
    )


    removal_mask = (
        final_index.isin(
            conflict_index
        )
    )


    matched_count = int(
        removal_mask.sum()
    )


    print(
        "\nConflict rows requested:",
        len(
            conflict_df
        )
    )

    print(
        "Conflict rows matched:",
        matched_count
    )


    assert (
        matched_count
        ==
        len(
            conflict_df
        )
    )


    print(
        "PASS: every audited conflict "
        "matched exactly one scored evaluation row."
    )


    # --------------------------------------------------------
    # Remove conflicts
    # --------------------------------------------------------

    removed_df = (
        final_df.loc[
            removal_mask
        ]
        .copy()
    )


    corrected_df = (
        final_df.loc[
            ~removal_mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    removed_counts = (
        removed_df[
            "evaluation_split"
        ]
        .value_counts()
        .reindex(
            [
                "train",
                "val",
                "test",
            ],
            fill_value=0
        )
    )


    corrected_counts = (
        corrected_df[
            "evaluation_split"
        ]
        .value_counts()
        .reindex(
            [
                "train",
                "val",
                "test",
            ],
            fill_value=0
        )
    )


    print(
        "\nRemoved rows:"
    )

    print(
        removed_counts
    )


    print(
        "\nCorrected evaluation counts:"
    )

    print(
        corrected_counts
    )


    for split_name, expected in (
        spec[
            "expected_removed"
        ].items()
    ):

        assert int(
            removed_counts[
                split_name
            ]
        ) == expected


    for split_name, expected in (
        spec[
            "expected_corrected"
        ].items()
    ):

        assert int(
            corrected_counts[
                split_name
            ]
        ) == expected


    # --------------------------------------------------------
    # Thresholds and corrected flags
    # --------------------------------------------------------

    historical_flag_columns = {}

    corrected_flag_columns = {}

    score_columns = {}


    print("\nThreshold recalculation:")


    for model_name, model_spec in MODELS.items():

        score_col = (
            model_spec["score"]
        )

        old_flag_col = (
            model_spec["old_flag"]
        )

        old_threshold_col = (
            model_spec[
                "old_threshold"
            ]
        )

        new_flag_col = (
            model_spec["new_flag"]
        )

        new_threshold_col = (
            model_spec[
                "new_threshold"
            ]
        )


        historical_flag_columns[
            model_name
        ] = old_flag_col

        corrected_flag_columns[
            model_name
        ] = new_flag_col

        score_columns[
            model_name
        ] = score_col


        # Stored historical threshold
        historical_threshold_values = (
            final_df[
                old_threshold_col
            ]
            .dropna()
            .unique()
        )


        assert len(
            historical_threshold_values
        ) == 1


        historical_threshold = float(
            historical_threshold_values[0]
        )


        # Recomputed original threshold
        original_val_scores = (
            final_df.loc[
                final_df[
                    "evaluation_split"
                ]
                == "val",
                score_col
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        recomputed_original_threshold = float(
            np.percentile(
                original_val_scores,
                95
            )
        )


        # Corrected validation threshold
        corrected_val_scores = (
            corrected_df.loc[
                corrected_df[
                    "evaluation_split"
                ]
                == "val",
                score_col
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        corrected_threshold = float(
            np.percentile(
                corrected_val_scores,
                95
            )
        )


        # Original controlled pipeline explicitly uses >
        corrected_flags = (
            corrected_df[
                score_col
            ]
            .to_numpy(
                dtype=np.float64
            )
            >
            corrected_threshold
        ).astype(
            np.int8
        )


        corrected_df[
            new_flag_col
        ] = corrected_flags


        corrected_df[
            new_threshold_col
        ] = corrected_threshold


        # Diagnostic only
        corrected_flags_ge = (
            corrected_df[
                score_col
            ]
            .to_numpy(
                dtype=np.float64
            )
            >=
            corrected_threshold
        ).astype(
            np.int8
        )


        operator_difference = int(
            (
                corrected_flags
                != corrected_flags_ge
            ).sum()
        )


        print(
            f"\n{model_name}"
        )

        print(
            f"  stored historical:   "
            f"{historical_threshold:.12f}"
        )

        print(
            f"  recomputed original: "
            f"{recomputed_original_threshold:.12f}"
        )

        print(
            f"  corrected:           "
            f"{corrected_threshold:.12f}"
        )

        print(
            f"  corrected delta:     "
            f"{corrected_threshold - historical_threshold:.12e}"
        )

        print(
            f"  > vs >= differences: "
            f"{operator_difference}"
        )


        all_threshold_rows.append({

            "dataset":
                dataset_name,

            "model":
                model_name,

            "stored_historical_threshold":
                historical_threshold,

            "recomputed_original_threshold":
                recomputed_original_threshold,

            "historical_threshold_recompute_delta":
                (
                    recomputed_original_threshold
                    -
                    historical_threshold
                ),

            "corrected_threshold":
                corrected_threshold,

            "corrected_minus_historical_threshold":
                (
                    corrected_threshold
                    -
                    historical_threshold
                ),

            "corrected_gt_vs_ge_differences":
                operator_difference,

            "original_validation_rows":
                int(
                    original_counts["val"]
                ),

            "corrected_validation_rows":
                int(
                    corrected_counts["val"]
                ),
        })


    # --------------------------------------------------------
    # Split summaries
    # --------------------------------------------------------

    historical_split = (
        model_split_summary(
            dataset_name,
            final_df,
            historical_flag_columns,
            score_columns
        )
    )


    corrected_split = (
        model_split_summary(
            dataset_name,
            corrected_df,
            corrected_flag_columns,
            score_columns
        )
    )


    all_split_historical.append(
        historical_split
    )

    all_split_corrected.append(
        corrected_split
    )


    # --------------------------------------------------------
    # Pairwise test agreement
    # --------------------------------------------------------

    historical_pairwise = (
        pairwise_test_summary(
            dataset_name,
            final_df,
            historical_flag_columns,
            score_columns
        )
    )


    corrected_pairwise = (
        pairwise_test_summary(
            dataset_name,
            corrected_df,
            corrected_flag_columns,
            score_columns
        )
    )


    all_pairwise_historical.append(
        historical_pairwise
    )

    all_pairwise_corrected.append(
        corrected_pairwise
    )


    # --------------------------------------------------------
    # Consensus
    # --------------------------------------------------------

    historical_consensus = (
        consensus_test_summary(
            dataset_name,
            final_df,
            historical_flag_columns
        )
    )


    corrected_consensus = (
        consensus_test_summary(
            dataset_name,
            corrected_df,
            corrected_flag_columns
        )
    )


    all_consensus_historical.append(
        historical_consensus
    )

    all_consensus_corrected.append(
        corrected_consensus
    )


    # --------------------------------------------------------
    # Category / channel summaries
    # --------------------------------------------------------

    corrected_category = (
        category_test_summary(
            dataset_name,
            corrected_df,
            spec[
                "category_columns"
            ],
            corrected_flag_columns
        )
    )


    all_category_corrected[
        dataset_name
    ] = corrected_category


    corrected_scored_tables[
        dataset_name
    ] = corrected_df


    # --------------------------------------------------------
    # Save dataset-specific files
    # --------------------------------------------------------

    dataset_dir = (
        CONTROLLED_CORRECTION_DIR
        /
        spec["slug"]
    )


    dataset_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    corrected_df.to_csv(
        dataset_dir
        /
        (
            f"{spec['slug']}_"
            "controlled_three_model_scored_corrected.csv"
        ),
        index=False
    )


    removed_df.to_csv(
        dataset_dir
        /
        (
            f"{spec['slug']}_"
            "controlled_removed_evaluation_rows.csv"
        ),
        index=False
    )


# ============================================================
# C. Consolidate outputs
# ============================================================

threshold_df = pd.DataFrame(
    all_threshold_rows
)


historical_split_df = pd.concat(
    all_split_historical,
    ignore_index=True
)

corrected_split_df = pd.concat(
    all_split_corrected,
    ignore_index=True
)


historical_pairwise_df = pd.concat(
    all_pairwise_historical,
    ignore_index=True
)

corrected_pairwise_df = pd.concat(
    all_pairwise_corrected,
    ignore_index=True
)


historical_consensus_df = pd.concat(
    all_consensus_historical,
    ignore_index=True
)

corrected_consensus_df = pd.concat(
    all_consensus_corrected,
    ignore_index=True
)


# ============================================================
# D. Split comparison
# ============================================================

split_comparison_df = (
    historical_split_df[
        [
            "dataset",
            "model",
            "split",
            "windows",
            "flagged_windows",
            "flagged_ratio_percent",
        ]
    ]
    .merge(
        corrected_split_df[
            [
                "dataset",
                "model",
                "split",
                "windows",
                "flagged_windows",
                "flagged_ratio_percent",
            ]
        ],
        on=[
            "dataset",
            "model",
            "split",
        ],
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


split_comparison_df[
    "flagged_ratio_percent_delta"
] = (
    split_comparison_df[
        "flagged_ratio_percent_corrected"
    ]
    -
    split_comparison_df[
        "flagged_ratio_percent_historical"
    ]
)


# ============================================================
# E. Pairwise comparison
# ============================================================

pairwise_comparison_df = (
    historical_pairwise_df.merge(
        corrected_pairwise_df,
        on=[
            "dataset",
            "model_a",
            "model_b",
        ],
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


# ============================================================
# F. Consensus comparison
# ============================================================

consensus_comparison_df = (
    historical_consensus_df.merge(
        corrected_consensus_df,
        on="dataset",
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


# ============================================================
# G. Print final correction outputs
# ============================================================

print("\n")
print("=" * 125)
print("CONTROLLED MODEL THRESHOLD CORRECTION")
print("=" * 125)


print(
    threshold_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.12f}"
    )
)


print("\n")
print("=" * 125)
print("CONTROLLED TEST FLAGGED-RATIO SUMMARY — HISTORICAL VS CORRECTED")
print("=" * 125)


print(
    split_comparison_df[
        split_comparison_df[
            "split"
        ]
        == "test"
    ][
        [
            "dataset",
            "model",

            "windows_historical",
            "windows_corrected",

            "flagged_windows_historical",
            "flagged_windows_corrected",

            "flagged_ratio_percent_historical",
            "flagged_ratio_percent_corrected",
            "flagged_ratio_percent_delta",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n")
print("=" * 125)
print("CONTROLLED TEST PAIRWISE AGREEMENT — HISTORICAL VS CORRECTED")
print("=" * 125)


print(
    pairwise_comparison_df[
        [
            "dataset",
            "model_a",
            "model_b",

            "test_windows_historical",
            "test_windows_corrected",

            "flag_jaccard_index_historical",
            "flag_jaccard_index_corrected",

            "score_spearman_correlation_historical",
            "score_spearman_correlation_corrected",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n")
print("=" * 125)
print("CONTROLLED TEST CONSENSUS — HISTORICAL VS CORRECTED")
print("=" * 125)


print(
    consensus_comparison_df[
        [
            "dataset",

            "test_windows_historical",
            "test_windows_corrected",

            "all_three_models_flagged_historical",
            "all_three_models_flagged_corrected",

            "all_three_ratio_percent_historical",
            "all_three_ratio_percent_corrected",

            "at_least_two_models_flagged_historical",
            "at_least_two_models_flagged_corrected",

            "at_least_two_ratio_percent_historical",
            "at_least_two_ratio_percent_corrected",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n")
print("=" * 125)
print("UK-DALE CORRECTED CONTROLLED TEST CATEGORY SUMMARY")
print("=" * 125)


print(
    all_category_corrected[
        "UK-DALE"
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n")
print("=" * 125)
print("AMPds2 CORRECTED CONTROLLED TEST CHANNEL SUMMARY — TOP 10")
print("=" * 125)


print(
    all_category_corrected[
        "AMPds2"
    ]
    .head(10)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# H. Save consolidated outputs
# ============================================================

threshold_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_corrected_thresholds.csv",
    index=False
)


corrected_split_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_corrected_split_summary.csv",
    index=False
)


split_comparison_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_historical_vs_corrected_split_summary.csv",
    index=False
)


corrected_pairwise_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_corrected_test_pairwise_agreement.csv",
    index=False
)


pairwise_comparison_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_historical_vs_corrected_test_pairwise_agreement.csv",
    index=False
)


corrected_consensus_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_corrected_test_consensus_summary.csv",
    index=False
)


consensus_comparison_df.to_csv(
    CONTROLLED_CORRECTION_DIR
    / "controlled_historical_vs_corrected_test_consensus_summary.csv",
    index=False
)


all_category_corrected[
    "UK-DALE"
].to_csv(
    CONTROLLED_CORRECTION_DIR
    / "ukdale_controlled_corrected_test_category_summary.csv",
    index=False
)


all_category_corrected[
    "AMPds2"
].to_csv(
    CONTROLLED_CORRECTION_DIR
    / "ampds2_controlled_corrected_test_channel_summary.csv",
    index=False
)


print(
    "\nSaved corrected outputs under:"
)

print(
    CONTROLLED_CORRECTION_DIR
)


print("\n")
print("=" * 125)
print("CELL 32B COMPLETE")
print("=" * 125)

print(
    "\nAll audited conflict rows were removed exactly once. "
    "No IF, LOF, or Autoencoder model was retrained, "
    "and no historical artifact was overwritten."
)

In [ ]:
# CELL 33 — UK-DALE and AMPds2 XAI Representative-Window Boundary-Correction Safety Audit

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 125)
print("XAI REPRESENTATIVE-WINDOW SPLIT-BOUNDARY SAFETY AUDIT")
print("=" * 125)


# ============================================================
# A. Dataset configuration
# ============================================================

XAI_AUDIT_CONFIG = {

    "UK-DALE": {

        "experiment_dir":
            BASE
            / "experiments"
            / "EXP_UKDALE_Selected_Pipeline",

        "corrected_scored":
            CORRECTION
            / "UKDALE"
            / "ukdale_if_scored_boundary_corrected.csv",

        "removed_windows":
            CORRECTION
            / "UKDALE"
            / "ukdale_removed_boundary_windows.csv",

        "identity_columns":
            [
                "building",
                "meter",
                "primary_category",
                "window_start",
                "window_end",
            ],

        "score_column":
            "anomaly_score",

        "corrected_flag_column":
            "predicted_anomaly_corrected",

        "corrected_threshold_column":
            "if_threshold_corrected",
    },

    "AMPds2": {

        "experiment_dir":
            BASE
            / "experiments"
            / "EXP_AMPDS2_Selected_Pipeline",

        "corrected_scored":
            CORRECTION
            / "AMPDS2"
            / "ampds2_if_scored_boundary_corrected.csv",

        "removed_windows":
            CORRECTION
            / "AMPDS2"
            / "ampds2_removed_boundary_windows.csv",

        "identity_columns":
            [
                "channel_code",
                "channel_name",
                "primary_category",
                "window_start",
                "window_end",
            ],

        "score_column":
            "anomaly_score",

        "corrected_flag_column":
            "predicted_anomaly_corrected",

        "corrected_threshold_column":
            "if_threshold_corrected",
    },
}


# ============================================================
# B. Helpers
# ============================================================

def normalize_time_columns(df):

    out = df.copy()

    for col in [
        "window_start",
        "window_end",
    ]:

        if col in out.columns:

            out[col] = pd.to_datetime(
                out[col],
                utc=True,
                errors="raise"
            )

    return out


def find_xai_csvs(
    experiment_dir
):

    candidates = []

    for path in experiment_dir.rglob(
        "*.csv"
    ):

        lower_name = (
            path.name.lower()
        )

        lower_full = (
            str(path).lower()
        )

        if (
            "xai" in lower_full
            or
            "shap" in lower_name
            or
            "lime" in lower_name
            or
            "candidate" in lower_name
        ):

            candidates.append(
                path
            )

    return sorted(
        set(
            candidates
        )
    )


# ============================================================
# C. Audit each dataset
# ============================================================

all_window_audit_rows = []

artifact_inventory_rows = []


for dataset_name, spec in (
    XAI_AUDIT_CONFIG.items()
):

    print("\n")
    print("=" * 125)
    print(dataset_name)
    print("=" * 125)


    experiment_dir = (
        spec[
            "experiment_dir"
        ]
    )

    corrected_scored_path = (
        spec[
            "corrected_scored"
        ]
    )

    removed_path = (
        spec[
            "removed_windows"
        ]
    )


    print(
        "\nExperiment directory:",
        experiment_dir
    )

    print(
        "Exists:",
        experiment_dir.exists()
    )

    print(
        "Corrected scored file exists:",
        corrected_scored_path.exists()
    )

    print(
        "Removed-window file exists:",
        removed_path.exists()
    )


    assert experiment_dir.exists()
    assert corrected_scored_path.exists()
    assert removed_path.exists()


    # --------------------------------------------------------
    # Load corrected and removed populations
    # --------------------------------------------------------

    corrected_df = normalize_time_columns(
        pd.read_csv(
            corrected_scored_path,
            low_memory=False
        )
    )


    removed_df = normalize_time_columns(
        pd.read_csv(
            removed_path,
            low_memory=False
        )
    )


    identity_columns = (
        spec[
            "identity_columns"
        ]
    )


    for col in identity_columns:

        assert col in corrected_df.columns

        assert col in removed_df.columns


    # --------------------------------------------------------
    # Discover XAI-related CSV artifacts
    # --------------------------------------------------------

    xai_files = (
        find_xai_csvs(
            experiment_dir
        )
    )


    print(
        "\nXAI-related CSV files discovered:",
        len(
            xai_files
        )
    )


    for path in xai_files:

        print(
            " -",
            path.relative_to(
                experiment_dir
            )
        )


    assert len(
        xai_files
    ) > 0


    # --------------------------------------------------------
    # Extract unique referenced windows from every XAI CSV
    # that contains the full identity.
    # --------------------------------------------------------

    referenced_windows = []


    for path in xai_files:

        try:

            temp = pd.read_csv(
                path,
                low_memory=False
            )

        except Exception as exc:

            print(
                f"\nSkipping unreadable CSV "
                f"{path.name}: {exc}"
            )

            continue


        available_identity = [
            col
            for col
            in identity_columns
            if col
            in temp.columns
        ]


        artifact_inventory_rows.append({

            "dataset":
                dataset_name,

            "artifact":
                str(
                    path
                ),

            "rows":
                len(
                    temp
                ),

            "identity_columns_present":
                "|".join(
                    available_identity
                ),

            "contains_full_window_identity":
                (
                    len(
                        available_identity
                    )
                    ==
                    len(
                        identity_columns
                    )
                ),
        })


        if (
            len(
                available_identity
            )
            !=
            len(
                identity_columns
            )
        ):

            continue


        temp = normalize_time_columns(
            temp
        )


        window_table = (
            temp[
                identity_columns
            ]
            .drop_duplicates()
            .copy()
        )


        window_table[
            "source_artifact"
        ] = str(
            path
        )


        referenced_windows.append(
            window_table
        )


    assert len(
        referenced_windows
    ) > 0, (
        f"No XAI artifact for {dataset_name} "
        "contained the full window identity."
    )


    referenced_df = pd.concat(
        referenced_windows,
        ignore_index=True
    )


    # One row per unique window, while retaining the list
    # of artifacts that reference it.
    artifact_map = (
        referenced_df
        .groupby(
            identity_columns,
            dropna=False
        )[
            "source_artifact"
        ]
        .agg(
            lambda x:
                "|".join(
                    sorted(
                        set(x)
                    )
                )
        )
        .reset_index()
        .rename(
            columns={
                "source_artifact":
                    "source_artifacts"
            }
        )
    )


    print(
        "\nUnique XAI-referenced windows:",
        len(
            artifact_map
        )
    )


    # --------------------------------------------------------
    # Build exact temporal identity indexes
    # --------------------------------------------------------

    corrected_index = (
        pd.MultiIndex.from_frame(
            corrected_df[
                identity_columns
            ]
        )
    )


    removed_index = (
        pd.MultiIndex.from_frame(
            removed_df[
                identity_columns
            ]
        )
    )


    reference_index = (
        pd.MultiIndex.from_frame(
            artifact_map[
                identity_columns
            ]
        )
    )


    in_corrected = (
        reference_index.isin(
            corrected_index
        )
    )


    in_removed = (
        reference_index.isin(
            removed_index
        )
    )


    # --------------------------------------------------------
    # Match each surviving XAI window to corrected score/flag
    # --------------------------------------------------------

    corrected_lookup_columns = (
        identity_columns
        +
        [
            "split",
            spec[
                "score_column"
            ],
            spec[
                "corrected_flag_column"
            ],
            spec[
                "corrected_threshold_column"
            ],
        ]
    )


    corrected_lookup = (
        corrected_df[
            corrected_lookup_columns
        ]
        .drop_duplicates(
            subset=identity_columns
        )
        .copy()
    )


    audit_df = (
        artifact_map.merge(
            corrected_lookup,
            on=identity_columns,
            how="left"
        )
    )


    audit_df[
        "dataset"
    ] = dataset_name


    audit_df[
        "survives_boundary_purge"
    ] = in_corrected


    audit_df[
        "was_removed_by_boundary_purge"
    ] = in_removed


    audit_df[
        "is_test_split_after_correction"
    ] = (
        audit_df[
            "split"
        ] == "test"
    )


    audit_df[
        "remains_flagged_after_correction"
    ] = (
        audit_df[
            spec[
                "corrected_flag_column"
            ]
        ]
        .fillna(0)
        .astype(int)
        == 1
    )


    audit_df[
        "score_minus_corrected_threshold"
    ] = (
        audit_df[
            spec[
                "score_column"
            ]
        ]
        -
        audit_df[
            spec[
                "corrected_threshold_column"
            ]
        ]
    )


    # --------------------------------------------------------
    # Print safety results
    # --------------------------------------------------------

    print("\n")
    print(
        "Boundary-purge removals among XAI windows:",
        int(
            audit_df[
                "was_removed_by_boundary_purge"
            ].sum()
        )
    )


    print(
        "XAI windows surviving corrected population:",
        int(
            audit_df[
                "survives_boundary_purge"
            ].sum()
        ),
        "/",
        len(
            audit_df
        )
    )


    print(
        "XAI windows still in TEST:",
        int(
            audit_df[
                "is_test_split_after_correction"
            ].sum()
        ),
        "/",
        len(
            audit_df
        )
    )


    print(
        "XAI windows still flagged:",
        int(
            audit_df[
                "remains_flagged_after_correction"
            ].sum()
        ),
        "/",
        len(
            audit_df
        )
    )


    print(
        "\nXAI window audit:"
    )


    display_columns = (
        identity_columns
        +
        [
            "split",
            spec[
                "score_column"
            ],
            spec[
                "corrected_threshold_column"
            ],
            "score_minus_corrected_threshold",
            "survives_boundary_purge",
            "was_removed_by_boundary_purge",
            "is_test_split_after_correction",
            "remains_flagged_after_correction",
        ]
    )


    print(
        audit_df[
            display_columns
        ]
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.9f}"
        )
    )


    # --------------------------------------------------------
    # Hard safety checks
    # --------------------------------------------------------

    assert int(
        audit_df[
            "was_removed_by_boundary_purge"
        ].sum()
    ) == 0, (
        f"{dataset_name}: at least one XAI window "
        "was removed by the corrected boundary purge."
    )


    assert bool(
        audit_df[
            "survives_boundary_purge"
        ].all()
    ), (
        f"{dataset_name}: at least one XAI window "
        "could not be found in the corrected population."
    )


    assert bool(
        audit_df[
            "is_test_split_after_correction"
        ].all()
    ), (
        f"{dataset_name}: an XAI window is no longer "
        "a retained test window."
    )


    assert bool(
        audit_df[
            "remains_flagged_after_correction"
        ].all()
    ), (
        f"{dataset_name}: at least one XAI representative "
        "window is no longer flagged after threshold correction."
    )


    print(
        "\nPASS:",
        dataset_name,
        "XAI representative windows remain valid."
    )


    all_window_audit_rows.append(
        audit_df
    )


# ============================================================
# D. Consolidated XAI audit
# ============================================================

combined_xai_audit_df = pd.concat(
    all_window_audit_rows,
    ignore_index=True,
    sort=False
)


artifact_inventory_df = pd.DataFrame(
    artifact_inventory_rows
)


summary_rows = []


for dataset_name, group in (
    combined_xai_audit_df.groupby(
        "dataset",
        sort=False
    )
):

    summary_rows.append({

        "dataset":
            dataset_name,

        "unique_xai_windows":
            len(
                group
            ),

        "removed_by_boundary_purge":
            int(
                group[
                    "was_removed_by_boundary_purge"
                ].sum()
            ),

        "surviving_corrected_windows":
            int(
                group[
                    "survives_boundary_purge"
                ].sum()
            ),

        "retained_as_test_windows":
            int(
                group[
                    "is_test_split_after_correction"
                ].sum()
            ),

        "remaining_flagged_windows":
            int(
                group[
                    "remains_flagged_after_correction"
                ].sum()
            ),

        "minimum_score_margin_above_threshold":
            float(
                group[
                    "score_minus_corrected_threshold"
                ].min()
            ),
    })


xai_summary_df = pd.DataFrame(
    summary_rows
)


print("\n")
print("=" * 125)
print("FINAL XAI SAFETY SUMMARY")
print("=" * 125)


print(
    xai_summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# E. Save audit
# ============================================================

XAI_AUDIT_DIR = (
    CORRECTION
    / "XAI_AUDIT"
)

XAI_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


combined_xai_audit_df.to_csv(
    XAI_AUDIT_DIR
    / "xai_representative_window_boundary_safety_audit.csv",
    index=False
)


artifact_inventory_df.to_csv(
    XAI_AUDIT_DIR
    / "xai_artifact_inventory.csv",
    index=False
)


xai_summary_df.to_csv(
    XAI_AUDIT_DIR
    / "xai_representative_window_safety_summary.csv",
    index=False
)


print(
    "\nSaved XAI audit under:"
)

print(
    XAI_AUDIT_DIR
)


print("\n")
print("=" * 125)
print("CELL 33 COMPLETE")
print("=" * 125)

print(
    "\nNo SHAP or LIME explanation was regenerated, "
    "no model was retrained, and no historical XAI artifact "
    "was modified."
)

In [ ]:
# CELL 34 — Master Split-Boundary Correction Coverage and Final-Artifact Audit

from pathlib import Path
import pandas as pd
import numpy as np


print("=" * 130)
print("MASTER SPLIT-BOUNDARY CORRECTION COVERAGE AND FINAL-ARTIFACT AUDIT")
print("=" * 130)


# ============================================================
# A. Final correction artifact paths
# ============================================================

FINAL_PATHS = {

    # NAB corrected seven-model branch
    "NAB final seven-model metrics":
        CORRECTION
        / "NAB"
        / "FINAL_nab_corrected_7model_metrics.csv",

    "NAB corrected split populations":
        CORRECTION
        / "NAB"
        / "FINAL_nab_corrected_split_populations.csv",

    "NAB corrected 60/15 vs 60/30":
        CORRECTION
        / "NAB"
        / "FINAL_nab_corrected_60_15_vs_60_30.csv",

    # Full UK-DALE IF selected pipeline
    "UK-DALE corrected split summary":
        CORRECTION
        / "UKDALE"
        / "ukdale_if_corrected_split_summary.csv",

    "UK-DALE corrected category summary":
        CORRECTION
        / "UKDALE"
        / "ukdale_if_corrected_test_category_summary.csv",

    # Full AMPds2 IF selected pipeline
    "AMPds2 corrected split summary":
        CORRECTION
        / "AMPDS2"
        / "ampds2_if_corrected_split_summary.csv",

    "AMPds2 corrected channel summary":
        CORRECTION
        / "AMPDS2"
        / "ampds2_if_corrected_test_channel_summary.csv",

    # Controlled residential comparison
    "Controlled corrected thresholds":
        CORRECTION
        / "CONTROLLED_RESIDENTIAL"
        / "controlled_corrected_thresholds.csv",

    "Controlled corrected split summary":
        CORRECTION
        / "CONTROLLED_RESIDENTIAL"
        / "controlled_corrected_split_summary.csv",

    "Controlled corrected pairwise agreement":
        CORRECTION
        / "CONTROLLED_RESIDENTIAL"
        / "controlled_corrected_test_pairwise_agreement.csv",

    "Controlled corrected consensus":
        CORRECTION
        / "CONTROLLED_RESIDENTIAL"
        / "controlled_corrected_test_consensus_summary.csv",

    # XAI safety
    "XAI safety summary":
        CORRECTION
        / "XAI_AUDIT"
        / "xai_representative_window_safety_summary.csv",
}


artifact_rows = []


for label, path in FINAL_PATHS.items():

    exists = path.exists()

    rows = None
    columns = None

    if exists:

        temp = pd.read_csv(
            path,
            low_memory=False
        )

        rows = len(temp)
        columns = len(temp.columns)

    artifact_rows.append({

        "artifact":
            label,

        "path":
            str(path),

        "exists":
            exists,

        "rows":
            rows,

        "columns":
            columns,
    })


artifact_audit_df = pd.DataFrame(
    artifact_rows
)


print("\n")
print("=" * 130)
print("FINAL CORRECTION ARTIFACT INVENTORY")
print("=" * 130)


print(
    artifact_audit_df[
        [
            "artifact",
            "exists",
            "rows",
            "columns",
        ]
    ].to_string(
        index=False
    )
)


assert bool(
    artifact_audit_df[
        "exists"
    ].all()
)


print(
    "\nPASS: all required final correction artifacts exist."
)


# ============================================================
# B. Final corrected populations
# ============================================================

nab_splits = pd.read_csv(
    FINAL_PATHS[
        "NAB corrected split populations"
    ]
)

uk_splits = pd.read_csv(
    FINAL_PATHS[
        "UK-DALE corrected split summary"
    ]
)

amp_splits = pd.read_csv(
    FINAL_PATHS[
        "AMPds2 corrected split summary"
    ]
)


population_rows = []


for _, row in nab_splits.iterrows():

    population_rows.append({

        "dataset":
            "NAB",

        "configuration":
            row[
                "configuration"
            ],

        "split":
            row[
                "split"
            ],

        "windows":
            int(
                row[
                    "total_windows"
                ]
            ),
    })


for _, row in uk_splits.iterrows():

    population_rows.append({

        "dataset":
            "UK-DALE",

        "configuration":
            "60/30",

        "split":
            row[
                "split"
            ],

        "windows":
            int(
                row[
                    "windows"
                ]
            ),
    })


for _, row in amp_splits.iterrows():

    population_rows.append({

        "dataset":
            "AMPds2",

        "configuration":
            "60/30",

        "split":
            row[
                "split"
            ],

        "windows":
            int(
                row[
                    "windows"
                ]
            ),
    })


population_df = pd.DataFrame(
    population_rows
)


print("\n")
print("=" * 130)
print("FINAL CORRECTED FULL-DATASET POPULATIONS")
print("=" * 130)


print(
    population_df.to_string(
        index=False
    )
)


# ============================================================
# C. Key thesis-result checkpoint
# ============================================================

nab_metrics = pd.read_csv(
    FINAL_PATHS[
        "NAB final seven-model metrics"
    ]
)


uk_test = (
    uk_splits[
        uk_splits[
            "split"
        ] == "test"
    ]
    .iloc[0]
)


amp_test = (
    amp_splits[
        amp_splits[
            "split"
        ] == "test"
    ]
    .iloc[0]
)


controlled_split = pd.read_csv(
    FINAL_PATHS[
        "Controlled corrected split summary"
    ]
)


controlled_test = (
    controlled_split[
        controlled_split[
            "split"
        ] == "test"
    ]
    .copy()
)


controlled_pairwise = pd.read_csv(
    FINAL_PATHS[
        "Controlled corrected pairwise agreement"
    ]
)


controlled_consensus = pd.read_csv(
    FINAL_PATHS[
        "Controlled corrected consensus"
    ]
)


xai_summary = pd.read_csv(
    FINAL_PATHS[
        "XAI safety summary"
    ]
)


print("\n")
print("=" * 130)
print("FINAL CORRECTED RESULT CHECKPOINT")
print("=" * 130)


print(
    "\nNAB best model by test F1:"
)


for config in [
    "60/30",
    "60/15",
]:

    part = (
        nab_metrics[
            nab_metrics[
                "configuration"
            ] == config
        ]
        .sort_values(
            "test_f1",
            ascending=False
        )
        .iloc[0]
    )

    print(
        f"{config}: "
        f"{part['model']} | "
        f"F1={part['test_f1']:.6f} | "
        f"FPR={part['test_fpr']:.6f} | "
        f"ROC-AUC={part['test_roc_auc']:.6f} | "
        f"PR-AUC={part['test_pr_auc']:.6f}"
    )


print(
    "\nFull residential IF test flagged-window rates:"
)

print(
    f"UK-DALE: "
    f"{int(uk_test['flagged_windows']):,}/"
    f"{int(uk_test['windows']):,} "
    f"({uk_test['flagged_ratio_percent']:.6f}%)"
)

print(
    f"AMPds2: "
    f"{int(amp_test['flagged_windows']):,}/"
    f"{int(amp_test['windows']):,} "
    f"({amp_test['flagged_ratio_percent']:.6f}%)"
)


print(
    "\nControlled residential test flagged-window rates:"
)

print(
    controlled_test[
        [
            "dataset",
            "model",
            "windows",
            "flagged_windows",
            "flagged_ratio_percent",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print(
    "\nControlled residential test consensus:"
)

print(
    controlled_consensus.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print(
    "\nXAI safety:"
)

print(
    xai_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# D. Search for distinct NAB result families still present
# ============================================================

SEARCH_ROOTS = [

    BASE
    / "results",

    BASE
    / "experiments",
]


NAB_KEYWORDS = [

    "nab",
]


INTERESTING_TERMS = [

    "representation",
    "stability",
    "five_model",
    "5model",
    "five-model",
    "ocsvm",
    "lstm",
    "dae",
    "label_blind",
    "label-blind",
    "per_series",
    "per-series",
]


nab_candidate_files = []


for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob(
        "*"
    ):

        if not path.is_file():
            continue

        lower_path = str(
            path
        ).lower()

        if "nab" not in lower_path:
            continue

        if path.suffix.lower() not in [
            ".csv",
            ".json",
            ".txt",
            ".md",
        ]:
            continue

        if any(
            term
            in lower_path
            for term
            in INTERESTING_TERMS
        ):

            nab_candidate_files.append(
                path
            )


nab_candidate_files = sorted(
    set(
        nab_candidate_files
    )
)


print("\n")
print("=" * 130)
print("DISTINCT NAB RESULT-FAMILY CANDIDATES")
print("=" * 130)


print(
    "Candidate files found:",
    len(
        nab_candidate_files
    )
)


for path in nab_candidate_files[:100]:

    try:

        relative = path.relative_to(
            BASE
        )

    except Exception:

        relative = path

    print(
        " -",
        relative
    )


# ============================================================
# E. Inspect candidate CSV schemas and row counts
# ============================================================

nab_family_rows = []


for path in nab_candidate_files:

    if path.suffix.lower() != ".csv":
        continue

    try:

        temp = pd.read_csv(
            path,
            low_memory=False
        )

    except Exception:

        continue


    cols = [
        str(c)
        for c
        in temp.columns
    ]


    lower_cols = [
        c.lower()
        for c
        in cols
    ]


    model_like = [
        c
        for c
        in cols
        if (
            "model"
            in c.lower()
        )
    ]


    metric_like = [
        c
        for c
        in cols
        if any(
            term
            in c.lower()
            for term
            in [
                "f1",
                "precision",
                "recall",
                "fpr",
                "roc",
                "pr_auc",
                "jaccard",
                "spearman",
            ]
        )
    ]


    nab_family_rows.append({

        "path":
            str(
                path
            ),

        "rows":
            len(
                temp
            ),

        "columns":
            len(
                cols
            ),

        "model_columns":
            "|".join(
                model_like
            ),

        "metric_columns":
            "|".join(
                metric_like
            ),

        "first_columns":
            "|".join(
                cols[:15]
            ),
    })


nab_family_df = pd.DataFrame(
    nab_family_rows
)


print("\n")
print("=" * 130)
print("NAB CANDIDATE CSV INVENTORY")
print("=" * 130)


if len(
    nab_family_df
):

    print(
        nab_family_df[
            [
                "path",
                "rows",
                "columns",
                "model_columns",
                "metric_columns",
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "No candidate NAB CSV files were identified."
    )


# ============================================================
# F. Master status
# ============================================================

master_status_rows = [

    {
        "result_family":
            "NAB 60/30 vs 60/15 seven-model comparison",

        "boundary_audit":
            "completed",

        "correction_status":
            "corrected",

        "retraining_needed":
            "AE/MLP refitted; classical reconstruction where required",

        "final_artifact_status":
            "available",
    },

    {
        "result_family":
            "UK-DALE full selected IF pipeline",

        "boundary_audit":
            "completed",

        "correction_status":
            "corrected",

        "retraining_needed":
            "no",

        "final_artifact_status":
            "available",
    },

    {
        "result_family":
            "AMPds2 full selected IF pipeline",

        "boundary_audit":
            "completed",

        "correction_status":
            "corrected",

        "retraining_needed":
            "no",

        "final_artifact_status":
            "available",
    },

    {
        "result_family":
            "Controlled UK-DALE/AMPds2 IF-LOF-AE",

        "boundary_audit":
            "completed",

        "correction_status":
            "corrected",

        "retraining_needed":
            "no",

        "final_artifact_status":
            "available",
    },

    {
        "result_family":
            "UK-DALE/AMPds2 SHAP-LIME representative windows",

        "boundary_audit":
            "completed",

        "correction_status":
            "no regeneration required",

        "retraining_needed":
            "no",

        "final_artifact_status":
            "validated",
    },

    {
        "result_family":
            "Other distinct NAB representation/stability or label-blind result families",

        "boundary_audit":
            "inventory pending interpretation",

        "correction_status":
            "to determine from candidate inventory",

        "retraining_needed":
            "not assumed",

        "final_artifact_status":
            "inspect Cell 34 candidate list",
    },
]


master_status_df = pd.DataFrame(
    master_status_rows
)


print("\n")
print("=" * 130)
print("MASTER CORRECTION STATUS")
print("=" * 130)


print(
    master_status_df.to_string(
        index=False
    )
)


# ============================================================
# G. Save master audit
# ============================================================

MASTER_AUDIT_DIR = (
    CORRECTION
    / "MASTER_AUDIT"
)

MASTER_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


artifact_audit_df.to_csv(
    MASTER_AUDIT_DIR
    / "master_final_artifact_inventory.csv",
    index=False
)


population_df.to_csv(
    MASTER_AUDIT_DIR
    / "master_corrected_population_summary.csv",
    index=False
)


nab_family_df.to_csv(
    MASTER_AUDIT_DIR
    / "master_distinct_nab_candidate_inventory.csv",
    index=False
)


master_status_df.to_csv(
    MASTER_AUDIT_DIR
    / "master_correction_status.csv",
    index=False
)


print(
    "\nSaved master audit under:"
)

print(
    MASTER_AUDIT_DIR
)


print("\n")
print("=" * 130)
print("CELL 34 COMPLETE")
print("=" * 130)

print(
    "\nNo model was trained and no historical artifact was modified. "
    "This cell determines whether any distinct NAB result family "
    "still requires boundary-correction handling before thesis updates."
)

In [ ]:
# CELL 35 — Distinct NAB Branch Dependency, Score-Artifact, and Training-Logic Audit

from pathlib import Path
import json
import pandas as pd
import numpy as np


print("=" * 130)
print("DISTINCT NAB BRANCH DEPENDENCY AND CORRECTION-FEASIBILITY AUDIT")
print("=" * 130)


# ============================================================
# A. Paths
# ============================================================

FIVE_MODEL_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_OCSVM_LSTM_Autoencoder"
)

FIVE_CONTROLLED = (
    FIVE_MODEL_ROOT
    / "Controlled_Five_Model_Comparison"
)

FIVE_FINAL = (
    FIVE_MODEL_ROOT
    / "FINAL_NAB_FIVE_MODEL_RESULTS"
)

FIVE_COMMON = (
    FIVE_MODEL_ROOT
    / "common"
)


REP_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_Representation_Improvement"
)

STABILITY_ROOT = (
    BASE
    / "results"
    / "EXP_NAB_interpretation_stability"
)

THRESHOLD_STABILITY_ROOT = (
    BASE
    / "results"
    / "EXP_NAB_threshold_stability"
)


# ============================================================
# B. Verify split manifest against the already-audited
#    original 60/30 NAB population
# ============================================================

MANIFEST_PATH = (
    FIVE_COMMON
    / "nab_split_manifest.csv"
)


print("\n")
print("=" * 130)
print("FIVE-MODEL SPLIT MANIFEST")
print("=" * 130)

print(
    "Path:",
    MANIFEST_PATH
)

print(
    "Exists:",
    MANIFEST_PATH.exists()
)

assert MANIFEST_PATH.exists()


manifest = pd.read_csv(
    MANIFEST_PATH,
    low_memory=False
)


print(
    "\nShape:",
    manifest.shape
)

print(
    "\nColumns:"
)

print(
    manifest.columns.tolist()
)


split_col_candidates = [
    c
    for c in manifest.columns
    if "split" in c.lower()
]


print(
    "\nSplit-column candidates:",
    split_col_candidates
)


assert len(
    split_col_candidates
) >= 1


manifest_split_col = (
    split_col_candidates[0]
)


print(
    "\nManifest split counts:"
)

print(
    manifest[
        manifest_split_col
    ].value_counts(
        dropna=False
    )
)


print(
    "\nManifest preview:"
)

print(
    manifest.head(8).to_string(
        index=False
    )
)


# ============================================================
# C. Inspect five-model score and metric artifacts
# ============================================================

FIVE_MODEL_FILES = {

    "OCSVM validation":
        FIVE_MODEL_ROOT
        / "OneClassSVM"
        / "validation_scores.csv",

    "OCSVM test":
        FIVE_MODEL_ROOT
        / "OneClassSVM"
        / "test_scores.csv",

    "LSTM AE validation":
        FIVE_MODEL_ROOT
        / "LSTM_Autoencoder"
        / "validation_scores.csv",

    "LSTM AE test":
        FIVE_MODEL_ROOT
        / "LSTM_Autoencoder"
        / "test_scores.csv",

    "Controlled IF test":
        FIVE_CONTROLLED
        / "isolation_forest_test_scores.csv",

    "Controlled LOF test":
        FIVE_CONTROLLED
        / "local_outlier_factor_test_scores.csv",

    "Controlled OCSVM test":
        FIVE_CONTROLLED
        / "oneclass_svm_test_scores.csv",

    "Five-model comparison":
        FIVE_CONTROLLED
        / "nab_label_blind_five_model_comparison.csv",

    "Final publication table":
        FIVE_FINAL
        / "tables"
        / "nab_five_model_publication_table.csv",
}


five_inventory_rows = []


for label, path in FIVE_MODEL_FILES.items():

    exists = path.exists()

    print("\n")
    print("-" * 110)
    print(label)
    print("-" * 110)

    print(
        "Exists:",
        exists
    )

    print(
        "Path:",
        path
    )

    assert exists


    df = pd.read_csv(
        path,
        low_memory=False
    )


    print(
        "Shape:",
        df.shape
    )

    print(
        "Columns:",
        df.columns.tolist()
    )


    # Compact preview only.
    print(
        "Preview:"
    )

    print(
        df.head(3).to_string(
            index=False
        )
    )


    identity_columns = [
        col
        for col in [
            "series",
            "series_id",
            "series_key",
            "window_start",
            "window_end",
            "start_time",
            "end_time",
            "timestamp",
            "source_row_id",
            "row_id",
            "split",
            "label",
            "y_true",
        ]
        if col in df.columns
    ]


    score_columns = [
        col
        for col in df.columns
        if any(
            term in col.lower()
            for term in [
                "score",
                "reconstruction",
                "decision",
                "prob",
            ]
        )
    ]


    threshold_columns = [
        col
        for col in df.columns
        if "threshold" in col.lower()
    ]


    five_inventory_rows.append({

        "artifact":
            label,

        "rows":
            len(df),

        "columns":
            len(df.columns),

        "identity_columns":
            "|".join(
                identity_columns
            ),

        "score_columns":
            "|".join(
                score_columns
            ),

        "threshold_columns":
            "|".join(
                threshold_columns
            ),
    })


five_inventory_df = pd.DataFrame(
    five_inventory_rows
)


print("\n")
print("=" * 130)
print("FIVE-MODEL ARTIFACT CAPABILITY SUMMARY")
print("=" * 130)


print(
    five_inventory_df.to_string(
        index=False
    )
)


# ============================================================
# D. Inspect all metrics/config files for training dependencies
# ============================================================

CONFIG_AND_METRIC_FILES = [

    FIVE_MODEL_ROOT
    / "OneClassSVM"
    / "config.json",

    FIVE_MODEL_ROOT
    / "OneClassSVM"
    / "metrics.csv",

    FIVE_MODEL_ROOT
    / "LSTM_Autoencoder"
    / "config.json",

    FIVE_MODEL_ROOT
    / "LSTM_Autoencoder"
    / "metrics.csv",

    FIVE_CONTROLLED
    / "dense_autoencoder_metrics.csv",

    FIVE_CONTROLLED
    / "dense_autoencoder_training_history.csv",

    FIVE_CONTROLLED
    / "isolation_forest_metrics.csv",

    FIVE_CONTROLLED
    / "local_outlier_factor_metrics.csv",

    FIVE_CONTROLLED
    / "oneclass_svm_metrics.csv",
]


print("\n")
print("=" * 130)
print("FIVE-MODEL CONFIGURATION / TRAINING DEPENDENCY ARTIFACTS")
print("=" * 130)


for path in CONFIG_AND_METRIC_FILES:

    print("\n")
    print("-" * 110)

    print(
        path.relative_to(
            BASE
        )
    )

    print("-" * 110)

    print(
        "Exists:",
        path.exists()
    )

    assert path.exists()


    if (
        path.suffix.lower()
        == ".json"
    ):

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            obj = json.load(f)


        # Print compact JSON.
        print(
            json.dumps(
                obj,
                indent=2
            )[:12000]
        )

    else:

        temp = pd.read_csv(
            path,
            low_memory=False
        )

        print(
            temp.to_string(
                index=False
            )
        )


# ============================================================
# E. Representation/stability artifacts:
#    identify whether they are descriptive population files,
#    threshold tables, or actual independent model results.
# ============================================================

REP_FILES = [

    REP_ROOT
    / "summary_best_f1_results_clean.csv",

    REP_ROOT
    / "summary_fpr010_results_clean.csv",

    REP_ROOT
    / "summary_if6030_improvement_over_ch4_baseline.csv",

    REP_ROOT
    / "summary_if_representation_comparison_fpr010.csv",

    STABILITY_ROOT
    / "window_representation_summary.csv",

    STABILITY_ROOT
    / "split_level_representation_summary.csv",

    STABILITY_ROOT
    / "threshold_sensitivity_60_15_unsupervised.csv",

    THRESHOLD_STABILITY_ROOT
    / "threshold_stability_summary_60_30_vs_60_15.csv",

    THRESHOLD_STABILITY_ROOT
    / "fpr_constrained_operating_points.csv",

    THRESHOLD_STABILITY_ROOT
    / "threshold_sensitivity_60_30_vs_60_15_unsupervised.csv",
]


rep_inventory_rows = []


print("\n")
print("=" * 130)
print("REPRESENTATION / STABILITY ARTIFACT AUDIT")
print("=" * 130)


for path in REP_FILES:

    print("\n")
    print("-" * 110)

    print(
        path.relative_to(
            BASE
        )
    )

    print("-" * 110)

    print(
        "Exists:",
        path.exists()
    )

    assert path.exists()


    temp = pd.read_csv(
        path,
        low_memory=False
    )


    print(
        "Shape:",
        temp.shape
    )

    print(
        "Columns:",
        temp.columns.tolist()
    )

    print(
        "Preview:"
    )

    print(
        temp.head(12).to_string(
            index=False
        )
    )


    rep_inventory_rows.append({

        "artifact":
            str(
                path.relative_to(
                    BASE
                )
            ),

        "rows":
            len(temp),

        "columns":
            len(temp.columns),

        "contains_model":
            any(
                "model"
                in c.lower()
                for c
                in temp.columns
            ),

        "contains_threshold":
            any(
                "threshold"
                in c.lower()
                or "percentile"
                in c.lower()
                for c
                in temp.columns
            ),

        "contains_split_counts":
            any(
                term
                in c.lower()
                for c
                in temp.columns
                for term
                in [
                    "total_windows",
                    "normal_windows",
                    "anomaly_windows",
                    "windows",
                    "n_windows",
                ]
            ),

        "contains_test_metrics":
            any(
                term
                in c.lower()
                for c
                in temp.columns
                for term
                in [
                    "test_f1",
                    "test_fpr",
                    "roc_auc",
                    "pr_auc",
                ]
            ),
    })


rep_inventory_df = pd.DataFrame(
    rep_inventory_rows
)


# ============================================================
# F. Search notebook source for exact distinct-branch logic
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)


assert NOTEBOOK_PATH.exists()


with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    notebook = json.load(f)


SOURCE_ANCHORS = [

    "EXP_NAB_OCSVM_LSTM_Autoencoder",
    "Controlled_Five_Model_Comparison",
    "OneClassSVM",
    "LSTM_Autoencoder",
    "EXP_NAB_Representation_Improvement",
    "per-series z-score",
    "per_series",
]


source_matches = []


for cell_no, cell in enumerate(
    notebook.get(
        "cells",
        []
    ),
    start=1
):

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(
            source,
            list
        )
        else str(source)
    )


    matched = [
        anchor
        for anchor
        in SOURCE_ANCHORS
        if anchor.lower()
        in text.lower()
    ]


    if matched:

        # Capture only lines carrying training/split/threshold logic.
        lines = text.splitlines()

        important_lines = []

        for idx, line in enumerate(
            lines,
            start=1
        ):

            lower = line.lower()

            if any(
                term in lower
                for term
                in [
                    "fit(",
                    ".fit(",
                    "validation",
                    "val_",
                    "earlystopping",
                    "threshold",
                    "percentile",
                    "split",
                    "train_df",
                    "val_df",
                    "test_df",
                    "isolationforest",
                    "localoutlierfactor",
                    "oneclasssvm",
                    "autoencoder",
                    "lstm",
                    "to_csv",
                ]
            ):

                important_lines.append(
                    (
                        idx,
                        line
                    )
                )


        source_matches.append({

            "cell":
                cell_no,

            "anchors":
                matched,

            "important_lines":
                important_lines,
        })


print("\n")
print("=" * 130)
print("DISTINCT NAB NOTEBOOK SOURCE LOGIC")
print("=" * 130)


print(
    "Matching notebook cells:",
    len(
        source_matches
    )
)


for match in source_matches:

    print("\n")
    print("-" * 110)

    print(
        f"Cell {match['cell']} | "
        f"anchors={match['anchors']}"
    )

    print("-" * 110)


    for line_no, line in (
        match[
            "important_lines"
        ][:120]
    ):

        print(
            f"{line_no:04d}: {line}"
        )


# ============================================================
# G. Save audit summaries
# ============================================================

DISTINCT_NAB_AUDIT_DIR = (
    CORRECTION
    / "NAB"
    / "DISTINCT_BRANCH_AUDIT"
)

DISTINCT_NAB_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


five_inventory_df.to_csv(
    DISTINCT_NAB_AUDIT_DIR
    / "five_model_artifact_capability_summary.csv",
    index=False
)


rep_inventory_df.to_csv(
    DISTINCT_NAB_AUDIT_DIR
    / "representation_stability_artifact_summary.csv",
    index=False
)


print("\n")
print("=" * 130)
print("CELL 35 CHECKPOINT")
print("=" * 130)


print(
    "\nFive-model split manifest rows:",
    len(
        manifest
    )
)


print(
    "Five-model validation/test artifact sizes should "
    "be compared with original 60/30 counts:"
)

print(
    "Original 60/30 validation = 20,705"
)

print(
    "Original 60/30 test       = 20,725"
)

print(
    "Corrected 60/30 validation = 20,648"
)

print(
    "Corrected 60/30 test       = 20,668"
)


print(
    "\nSaved audit under:"
)

print(
    DISTINCT_NAB_AUDIT_DIR
)


print("\n")
print("=" * 130)
print("CELL 35 COMPLETE")
print("=" * 130)

print(
    "\nThis cell made no corrections, trained no model, "
    "and modified no historical artifact."
)

In [ ]:
# CELL 36 — Final NAB Five-Model and Representation-Branch Correction Dependency Audit

from pathlib import Path
import json
import pandas as pd
import numpy as np


print("=" * 132)
print("FINAL NAB FIVE-MODEL / REPRESENTATION CORRECTION DEPENDENCY AUDIT")
print("=" * 132)


# ============================================================
# A. Core paths
# ============================================================

FIVE_MODEL_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_OCSVM_LSTM_Autoencoder"
)

FIVE_CONTROLLED = (
    FIVE_MODEL_ROOT
    / "Controlled_Five_Model_Comparison"
)

FIVE_COMMON = (
    FIVE_MODEL_ROOT
    / "common"
)

REP_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_Representation_Improvement"
)

NAB_CORRECTION_ROOT = (
    CORRECTION
    / "NAB"
)


# ============================================================
# B. Discover all saved model objects in five-model branch
# ============================================================

MODEL_SUFFIXES = {
    ".joblib",
    ".pkl",
    ".pickle",
    ".keras",
    ".h5",
    ".hdf5",
}


saved_model_files = []


for path in FIVE_MODEL_ROOT.rglob("*"):

    if (
        path.is_file()
        and
        path.suffix.lower()
        in MODEL_SUFFIXES
    ):

        saved_model_files.append(
            path
        )


saved_model_files = sorted(
    saved_model_files
)


print("\n")
print("=" * 132)
print("SAVED FIVE-MODEL MODEL OBJECTS")
print("=" * 132)


print(
    "Saved model files discovered:",
    len(
        saved_model_files
    )
)


for path in saved_model_files:

    print(
        " -",
        path.relative_to(
            BASE
        )
    )


# ============================================================
# C. Search all five-model files for validation score artifacts
# ============================================================

print("\n")
print("=" * 132)
print("FIVE-MODEL ROW-LEVEL SCORE ARTIFACT INVENTORY")
print("=" * 132)


score_artifact_rows = []


for path in FIVE_MODEL_ROOT.rglob("*.csv"):

    try:

        temp = pd.read_csv(
            path,
            low_memory=False
        )

    except Exception:

        continue


    lower_cols = {
        str(c).lower():
            c
        for c in temp.columns
    }


    has_score = any(
        (
            "score"
            in str(c).lower()
            or
            "reconstruction"
            in str(c).lower()
        )
        for c in temp.columns
    )


    has_identity = (
        "window_start"
        in lower_cols
        and
        "window_end"
        in lower_cols
    )


    if (
        has_score
        and
        has_identity
    ):

        split_values = ""

        if (
            "split"
            in temp.columns
        ):

            split_values = "|".join(
                sorted(
                    temp[
                        "split"
                    ]
                    .dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )
            )


        score_artifact_rows.append({

            "path":
                str(
                    path.relative_to(
                        BASE
                    )
                ),

            "rows":
                len(temp),

            "columns":
                len(
                    temp.columns
                ),

            "split_values":
                split_values,

            "has_window_id":
                "window_id"
                in temp.columns,

            "score_columns":
                "|".join(
                    [
                        str(c)
                        for c
                        in temp.columns
                        if (
                            "score"
                            in str(c).lower()
                            or
                            "reconstruction"
                            in str(c).lower()
                        )
                    ]
                ),
        })


score_artifact_df = pd.DataFrame(
    score_artifact_rows
)


if len(
    score_artifact_df
):

    print(
        score_artifact_df
        .sort_values(
            [
                "path"
            ]
        )
        .to_string(
            index=False
        )
    )

else:

    print(
        "No row-level score artifact found."
    )


# ============================================================
# D. Locate corrected 60/30 NAB row tables created previously
# ============================================================

print("\n")
print("=" * 132)
print("CORRECTED NAB 60/30 ROW-IDENTITY ARTIFACT DISCOVERY")
print("=" * 132)


corrected_candidate_rows = []


for path in NAB_CORRECTION_ROOT.rglob(
    "*.csv"
):

    try:

        temp = pd.read_csv(
            path,
            low_memory=False
        )

    except Exception:

        continue


    if len(
        temp
    ) not in [
        103412,
        62096,
        20648,
        20668,
    ]:

        continue


    useful_identity = [
        col
        for col
        in [
            "window_id",
            "series",
            "series_key",
            "window_start",
            "window_end",
            "split",
            "_split",
            "label",
        ]
        if col
        in temp.columns
    ]


    corrected_candidate_rows.append({

        "path":
            str(
                path
            ),

        "rows":
            len(
                temp
            ),

        "columns":
            len(
                temp.columns
            ),

        "identity_columns":
            "|".join(
                useful_identity
            ),
    })


corrected_candidate_df = pd.DataFrame(
    corrected_candidate_rows
)


if len(
    corrected_candidate_df
):

    print(
        corrected_candidate_df.to_string(
            index=False
        )
    )

else:

    print(
        "No corrected 60/30 row table found "
        "by expected row count."
    )


# ============================================================
# E. Inspect five-model training-reference artifact
# ============================================================

REFERENCE_PATH = (
    FIVE_COMMON
    / "ocsvm_training_reference_windows.csv"
)


print("\n")
print("=" * 132)
print("FIVE-MODEL TRAINING REFERENCE")
print("=" * 132)


print(
    "Path:",
    REFERENCE_PATH
)

print(
    "Exists:",
    REFERENCE_PATH.exists()
)

assert REFERENCE_PATH.exists()


reference_df = pd.read_csv(
    REFERENCE_PATH,
    low_memory=False
)


print(
    "Shape:",
    reference_df.shape
)

print(
    "Columns:",
    reference_df.columns.tolist()
)

print(
    "Preview:"
)

print(
    reference_df.head(8).to_string(
        index=False
    )
)


# ============================================================
# F. Inspect representation branch for saved models/scores
# ============================================================

print("\n")
print("=" * 132)
print("REPRESENTATION-BRANCH SAVED MODEL / SCORE INVENTORY")
print("=" * 132)


rep_files = []


for path in REP_ROOT.rglob("*"):

    if not path.is_file():
        continue


    interesting = (
        path.suffix.lower()
        in MODEL_SUFFIXES
        or
        "score"
        in path.name.lower()
        or
        "prediction"
        in path.name.lower()
    )


    if interesting:

        rep_files.append(
            path
        )


rep_files = sorted(
    rep_files
)


print(
    "Relevant files discovered:",
    len(
        rep_files
    )
)


for path in rep_files:

    print(
        " -",
        path.relative_to(
            BASE
        )
    )


# ============================================================
# G. Extract exact Dense Autoencoder notebook training logic
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)


assert NOTEBOOK_PATH.exists()


with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    notebook = json.load(
        f
    )


DENSE_SEARCH_TERMS = [

    "dense_autoencoder_metrics.csv",
    "dense_autoencoder_training_history.csv",
    "Controlled_Five_Model_Comparison",
]


dense_matches = []


for cell_no, cell in enumerate(
    notebook.get(
        "cells",
        []
    ),
    start=1
):

    source = cell.get(
        "source",
        []
    )


    text = (
        "".join(
            source
        )
        if isinstance(
            source,
            list
        )
        else str(
            source
        )
    )


    if not any(
        term.lower()
        in text.lower()
        for term
        in DENSE_SEARCH_TERMS
    ):

        continue


    lines = (
        text.splitlines()
    )


    relevant_lines = []


    for line_no, line in enumerate(
        lines,
        start=1
    ):

        lower = (
            line.lower()
        )


        if any(
            term
            in lower
            for term
            in [
                "autoencoder",
                "x_train",
                "x_validation",
                "x_val",
                "validation_data",
                "validation_split",
                "earlystopping",
                "modelcheckpoint",
                "fit(",
                ".fit(",
                "val_loss",
                "threshold",
                "percentile",
                "predict(",
                "reconstruction",
                "to_csv",
                "save(",
                ".keras",
            ]
        ):

            relevant_lines.append(
                (
                    line_no,
                    line
                )
            )


    dense_matches.append({

        "cell":
            cell_no,

        "relevant_lines":
            relevant_lines,
    })


print("\n")
print("=" * 132)
print("CONTROLLED DENSE AUTOENCODER NOTEBOOK LOGIC")
print("=" * 132)


print(
    "Matching cells:",
    len(
        dense_matches
    )
)


for match in dense_matches:

    print("\n")
    print("-" * 112)

    print(
        f"Notebook cell {match['cell']}"
    )

    print("-" * 112)


    for line_no, line in (
        match[
            "relevant_lines"
        ]
    ):

        print(
            f"{line_no:04d}: {line}"
        )


# ============================================================
# H. Extract exact five-model matrix construction logic
# ============================================================

MATRIX_SEARCH_TERMS = [

    "X_validation",
    "X_lstm_validation",
    "nab_split_manifest.csv",
    "ocsvm_training_reference_windows.csv",
]


matrix_matches = []


for cell_no, cell in enumerate(
    notebook.get(
        "cells",
        []
    ),
    start=1
):

    source = cell.get(
        "source",
        []
    )


    text = (
        "".join(
            source
        )
        if isinstance(
            source,
            list
        )
        else str(
            source
        )
    )


    matched_terms = [
        term
        for term
        in MATRIX_SEARCH_TERMS
        if term.lower()
        in text.lower()
    ]


    if not matched_terms:

        continue


    lines = (
        text.splitlines()
    )


    relevant_lines = []


    for line_no, line in enumerate(
        lines,
        start=1
    ):

        lower = (
            line.lower()
        )


        if any(
            term
            in lower
            for term
            in [
                "train_df",
                "validation_df",
                "test_df",
                "x_train",
                "x_validation",
                "x_test",
                "x_lstm_train",
                "x_lstm_validation",
                "x_lstm_test",
                "validation_indices",
                "test_indices",
                "feature",
                "scaled",
                "normalization",
                "sequence",
                "window_id",
                "split",
            ]
        ):

            relevant_lines.append(
                (
                    line_no,
                    line
                )
            )


    matrix_matches.append({

        "cell":
            cell_no,

        "terms":
            matched_terms,

        "relevant_lines":
            relevant_lines,
    })


print("\n")
print("=" * 132)
print("FIVE-MODEL MATRIX / ROW-CONSTRUCTION LOGIC")
print("=" * 132)


for match in matrix_matches:

    print("\n")
    print("-" * 112)

    print(
        f"Notebook cell {match['cell']} "
        f"| terms={match['terms']}"
    )

    print("-" * 112)


    for line_no, line in (
        match[
            "relevant_lines"
        ][:150]
    ):

        print(
            f"{line_no:04d}: {line}"
        )


# ============================================================
# I. Save compact inventories
# ============================================================

FINAL_DEP_AUDIT_DIR = (
    NAB_CORRECTION_ROOT
    / "FINAL_DEPENDENCY_AUDIT"
)

FINAL_DEP_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


score_artifact_df.to_csv(
    FINAL_DEP_AUDIT_DIR
    / "five_model_row_level_score_inventory.csv",
    index=False
)


corrected_candidate_df.to_csv(
    FINAL_DEP_AUDIT_DIR
    / "corrected_6030_row_identity_inventory.csv",
    index=False
)


# ============================================================
# J. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 36 CHECKPOINT")
print("=" * 132)


print(
    f"Saved five-model objects found: "
    f"{len(saved_model_files)}"
)

print(
    f"Five-model row-level score artifacts found: "
    f"{len(score_artifact_df)}"
)

print(
    f"Corrected 60/30 row-identity candidates found: "
    f"{len(corrected_candidate_df)}"
)

print(
    f"Representation saved model/score files found: "
    f"{len(rep_files)}"
)


print(
    "\nSaved audit under:"
)

print(
    FINAL_DEP_AUDIT_DIR
)


print("\n")
print("=" * 132)
print("CELL 36 COMPLETE")
print("=" * 132)

print(
    "\nRead-only audit complete. No model was trained "
    "and no historical artifact was modified."
)

In [ ]:
# CELL 37 — Final NAB Correction-Ready Input, Model, Tensor, and Representation-Configuration Audit

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


print("=" * 132)
print("FINAL NAB CORRECTION-READY CONFIGURATION AUDIT")
print("=" * 132)


# ============================================================
# A. Core paths
# ============================================================

FIVE_MODEL_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_OCSVM_LSTM_Autoencoder"
)

FIVE_CONTROLLED = (
    FIVE_MODEL_ROOT
    / "Controlled_Five_Model_Comparison"
)

FIVE_COMMON = (
    FIVE_MODEL_ROOT
    / "common"
)

LSTM_ROOT = (
    FIVE_MODEL_ROOT
    / "LSTM_Autoencoder"
)

REP_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_Representation_Improvement"
)

NAB_6030_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "60_30"
)


# ============================================================
# B. Five-model common artifact inventory
# ============================================================

print("\n")
print("=" * 132)
print("FIVE-MODEL COMMON ARTIFACT INVENTORY")
print("=" * 132)


for path in sorted(
    FIVE_COMMON.iterdir()
):

    if not path.is_file():
        continue

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{path.name:<55} "
        f"{size_mb:>10.3f} MB"
    )


# ============================================================
# C. Normalization statistics
# ============================================================

NORM_PATH = (
    FIVE_COMMON
    / "normalization_statistics.csv"
)


assert NORM_PATH.exists()


norm_df = pd.read_csv(
    NORM_PATH,
    low_memory=False
)


print("\n")
print("=" * 132)
print("FIVE-MODEL FEATURE NORMALIZATION STATISTICS")
print("=" * 132)


print(
    "Shape:",
    norm_df.shape
)

print(
    "Columns:",
    norm_df.columns.tolist()
)


series_col_candidates = [
    c
    for c in norm_df.columns
    if "series" in c.lower()
]


feature_col_candidates = [
    c
    for c in norm_df.columns
    if "feature" in c.lower()
]


print(
    "Series-column candidates:",
    series_col_candidates
)

print(
    "Feature-column candidates:",
    feature_col_candidates
)


print(
    "\nPreview:"
)

print(
    norm_df.head(12).to_string(
        index=False
    )
)


if series_col_candidates:

    print(
        "\nUnique series:",
        norm_df[
            series_col_candidates[0]
        ].nunique()
    )


if feature_col_candidates:

    feature_names_from_stats = (
        norm_df[
            feature_col_candidates[0]
        ]
        .drop_duplicates()
        .tolist()
    )

    print(
        "\nFeatures in normalization artifact:",
        len(
            feature_names_from_stats
        )
    )

    print(
        feature_names_from_stats
    )


# ============================================================
# D. Load saved classical five-model objects
# ============================================================

CLASSICAL_MODELS = {

    "Isolation Forest":
        FIVE_CONTROLLED
        / "isolation_forest.joblib",

    "Local Outlier Factor":
        FIVE_CONTROLLED
        / "local_outlier_factor.joblib",

    "One-Class SVM":
        FIVE_CONTROLLED
        / "oneclass_svm.joblib",
}


print("\n")
print("=" * 132)
print("SAVED CLASSICAL MODEL LOAD / PARAMETER AUDIT")
print("=" * 132)


classical_model_rows = []


for model_name, path in (
    CLASSICAL_MODELS.items()
):

    print("\n")
    print(
        "-" * 110
    )

    print(
        model_name
    )

    print(
        "-" * 110
    )

    print(
        "Path:",
        path
    )

    print(
        "Exists:",
        path.exists()
    )

    assert path.exists()


    model = joblib.load(
        path
    )


    print(
        "Loaded class:",
        type(
            model
        ).__name__
    )


    params = (
        model.get_params(
            deep=False
        )
        if hasattr(
            model,
            "get_params"
        )
        else {}
    )


    print(
        "Parameters:"
    )

    print(
        params
    )


    n_features = getattr(
        model,
        "n_features_in_",
        None
    )


    print(
        "n_features_in_:",
        n_features
    )


    classical_model_rows.append({

        "model":
            model_name,

        "path":
            str(
                path
            ),

        "loaded_class":
            type(
                model
            ).__name__,

        "n_features_in":
            n_features,

        "params":
            repr(
                params
            ),
    })


classical_model_audit_df = pd.DataFrame(
    classical_model_rows
)


# ============================================================
# E. LSTM tensor bundle
# ============================================================

LSTM_TENSOR_PATH = (
    FIVE_COMMON
    / "nab_lstm_sequence_tensors.npz"
)


print("\n")
print("=" * 132)
print("LSTM SEQUENCE TENSOR BUNDLE")
print("=" * 132)


print(
    "Path:",
    LSTM_TENSOR_PATH
)

print(
    "Exists:",
    LSTM_TENSOR_PATH.exists()
)

assert LSTM_TENSOR_PATH.exists()


with np.load(
    LSTM_TENSOR_PATH,
    allow_pickle=False
) as tensor_bundle:

    print(
        "\nKeys:"
    )

    print(
        tensor_bundle.files
    )


    tensor_summary_rows = []


    for key in tensor_bundle.files:

        arr = tensor_bundle[
            key
        ]

        print(
            f"{key:<30} "
            f"shape={arr.shape} "
            f"dtype={arr.dtype}"
        )


        tensor_summary_rows.append({

            "key":
                key,

            "shape":
                str(
                    arr.shape
                ),

            "dtype":
                str(
                    arr.dtype
                ),
        })


    # Explicit expected keys.
    required_tensor_keys = [
        "X_train",
        "X_validation",
        "X_test",
        "train_window_ids",
        "validation_window_ids",
        "test_window_ids",
    ]


    for key in required_tensor_keys:

        assert key in tensor_bundle.files


    print(
        "\nWindow-ID counts:"
    )

    print(
        "Train:",
        len(
            tensor_bundle[
                "train_window_ids"
            ]
        )
    )

    print(
        "Validation:",
        len(
            tensor_bundle[
                "validation_window_ids"
            ]
        )
    )

    print(
        "Test:",
        len(
            tensor_bundle[
                "test_window_ids"
            ]
        )
    )


tensor_summary_df = pd.DataFrame(
    tensor_summary_rows
)


# ============================================================
# F. Corrected 60/30 partition identity
# ============================================================

corrected_paths = {

    "train":
        NAB_6030_CORRECTION_DIR
        / "train_corrected.csv",

    "validation":
        NAB_6030_CORRECTION_DIR
        / "validation_corrected.csv",

    "test":
        NAB_6030_CORRECTION_DIR
        / "test_corrected.csv",
}


print("\n")
print("=" * 132)
print("CORRECTED 60/30 PARTITION IDENTITY")
print("=" * 132)


corrected_identity_rows = []


for split_name, path in (
    corrected_paths.items()
):

    assert path.exists()


    temp = pd.read_csv(
        path,
        low_memory=False
    )


    print(
        f"\n{split_name}:"
    )

    print(
        "Shape:",
        temp.shape
    )

    print(
        "Columns:",
        temp.columns.tolist()
    )


    identity_columns = [
        col
        for col in [
            "window_id",
            "series",
            "domain",
            "window_start",
            "window_end",
            "label",
            "split",
        ]
        if col
        in temp.columns
    ]


    print(
        "Identity columns:",
        identity_columns
    )


    corrected_identity_rows.append({

        "split":
            split_name,

        "rows":
            len(
                temp
            ),

        "identity_columns":
            "|".join(
                identity_columns
            ),
    })


corrected_identity_df = pd.DataFrame(
    corrected_identity_rows
)


# ============================================================
# G. Exact representation-improvement source extraction
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)


assert NOTEBOOK_PATH.exists()


with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    notebook = json.load(
        f
    )


def get_cell_text(
    notebook_cell_number
):

    cell = notebook[
        "cells"
    ][
        notebook_cell_number - 1
    ]

    source = cell.get(
        "source",
        []
    )

    return (
        "".join(
            source
        )
        if isinstance(
            source,
            list
        )
        else str(
            source
        )
    )


def print_selected_context(
    cell_number,
    patterns,
    context=5
):

    text = get_cell_text(
        cell_number
    )

    lines = (
        text.splitlines()
    )


    selected = set()


    for idx, line in enumerate(
        lines
    ):

        lower = (
            line.lower()
        )


        if any(
            pattern.lower()
            in lower
            for pattern
            in patterns
        ):

            start = max(
                0,
                idx - context
            )

            end = min(
                len(
                    lines
                ),
                idx + context + 1
            )


            selected.update(
                range(
                    start,
                    end
                )
            )


    if not selected:

        print(
            "No matching lines found."
        )

        return


    previous = None


    for idx in sorted(
        selected
    ):

        if (
            previous
            is not None
            and
            idx > previous + 1
        ):

            print(
                "     ..."
            )


        print(
            f"{idx + 1:04d}: "
            f"{lines[idx]}"
        )


        previous = idx


# ------------------------------------------------------------
# Cell 512 — representation scaling experiment
# ------------------------------------------------------------

print("\n")
print("=" * 132)
print("REPRESENTATION CELL 512 — EXACT CONFIGURATION CONTEXT")
print("=" * 132)


print_selected_context(
    512,
    patterns=[
        "BASE",
        "PATH",
        "FEATURE",
        "VARIANT",
        "SCAL",
        "RobustScaler",
        "StandardScaler",
        "IsolationForest(",
        "LocalOutlierFactor(",
        "n_estimators",
        "n_neighbors",
        "contamination",
        "random_state",
        "add_chronological_split",
        "percentiles",
        "np.arange",
        "fit(",
        "X_fit",
        "series_col",
        "label_col",
        "window_config",
    ],
    context=7
)


# ------------------------------------------------------------
# Cell 514 — feature-selection experiment
# ------------------------------------------------------------

print("\n")
print("=" * 132)
print("REPRESENTATION CELL 514 — FEATURE-SELECTION CONFIGURATION CONTEXT")
print("=" * 132)


print_selected_context(
    514,
    patterns=[
        "thresholds",
        "correlation_prune",
        "feature",
        "baseline",
        "expanded",
        "window_config",
        "IsolationForest",
        "LocalOutlierFactor",
        "evaluate_threshold",
        "fit(",
        "X_train",
        "X_val",
        "X_test",
    ],
    context=7
)


# ------------------------------------------------------------
# Cell 522 — robust-feature experiment
# ------------------------------------------------------------

print("\n")
print("=" * 132)
print("REPRESENTATION CELL 522 — ROBUST-FEATURE CONFIGURATION CONTEXT")
print("=" * 132)


print_selected_context(
    522,
    patterns=[
        "FEATURE",
        "ROBUST",
        "window",
        "add_chronological_split",
        "IsolationForest(",
        "LocalOutlierFactor(",
        "n_estimators",
        "n_neighbors",
        "contamination",
        "random_state",
        "percentiles",
        "np.arange",
        "fit(",
        "fpr",
    ],
    context=7
)


# ============================================================
# H. Exact Dense-AE architecture/training configuration
# ============================================================

print("\n")
print("=" * 132)
print("CONTROLLED DENSE AUTOENCODER — EXACT ARCHITECTURE / TRAINING CONTEXT")
print("=" * 132)


print_selected_context(
    572,
    patterns=[
        "Input(",
        "Dense(",
        "learning_rate",
        "EarlyStopping",
        "patience",
        "restore_best_weights",
        "epochs",
        "batch_size",
        "validation_data",
        "fit(",
        "threshold",
        "percentile",
    ],
    context=6
)


# ============================================================
# I. LSTM historical training checkpoint
# ============================================================

LSTM_HISTORY_PATH = (
    LSTM_ROOT
    / "training_history.csv"
)


assert LSTM_HISTORY_PATH.exists()


lstm_history_df = pd.read_csv(
    LSTM_HISTORY_PATH,
    low_memory=False
)


print("\n")
print("=" * 132)
print("LSTM HISTORICAL TRAINING CHECKPOINT")
print("=" * 132)


print(
    "History rows:",
    len(
        lstm_history_df
    )
)

print(
    "Columns:",
    lstm_history_df.columns.tolist()
)


print(
    "\nLast 15 rows:"
)

print(
    lstm_history_df
    .tail(15)
    .to_string(
        index=False
    )
)


if (
    "val_loss"
    in lstm_history_df.columns
):

    best_pos = int(
        np.argmin(
            lstm_history_df[
                "val_loss"
            ].to_numpy()
        )
    )

    print(
        "\nHistorical best validation position:",
        best_pos
    )

    print(
        "Historical best validation row:"
    )

    print(
        lstm_history_df.iloc[
            best_pos
        ].to_string()
    )


# ============================================================
# J. Save compact audits
# ============================================================

AUDIT_DIR = (
    CORRECTION
    / "NAB"
    / "FINAL_CORRECTION_READY_AUDIT"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


classical_model_audit_df.to_csv(
    AUDIT_DIR
    / "saved_classical_model_parameter_audit.csv",
    index=False
)


tensor_summary_df.to_csv(
    AUDIT_DIR
    / "lstm_tensor_bundle_audit.csv",
    index=False
)


corrected_identity_df.to_csv(
    AUDIT_DIR
    / "corrected_6030_partition_identity_audit.csv",
    index=False
)


# ============================================================
# K. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 37 CHECKPOINT")
print("=" * 132)


print(
    "Classical saved models successfully loaded:",
    len(
        classical_model_audit_df
    )
)


print(
    "LSTM tensor bundle keys:",
    len(
        tensor_summary_df
    )
)


print(
    "Corrected 60/30 partitions found:",
    len(
        corrected_identity_df
    )
)


print(
    "Dense AE external validation dependency:",
    "CONFIRMED"
)


print(
    "LSTM AE external validation dependency:",
    "CONFIRMED"
)


print(
    "\nSaved audit under:"
)

print(
    AUDIT_DIR
)


print("\n")
print("=" * 132)
print("CELL 37 COMPLETE")
print("=" * 132)

print(
    "\nRead-only audit complete. No model was trained, "
    "no threshold was changed, and no historical artifact "
    "was modified."
)

In [ ]:
# CELL 38 — Final NAB Five-Model Classical Correction and Neural Refit Input Preparation

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)


print("=" * 132)
print("FINAL NAB FIVE-MODEL CLASSICAL CORRECTION AND NEURAL REFIT INPUT PREPARATION")
print("=" * 132)


# ============================================================
# A. Paths
# ============================================================

FIVE_MODEL_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_OCSVM_LSTM_Autoencoder"
)

FIVE_CONTROLLED = (
    FIVE_MODEL_ROOT
    / "Controlled_Five_Model_Comparison"
)

FIVE_COMMON = (
    FIVE_MODEL_ROOT
    / "common"
)

NAB_6030_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "60_30"
)

FIVE_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "FIVE_MODEL_CORRECTION"
)

FIVE_CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MANIFEST_PATH = (
    FIVE_COMMON
    / "nab_split_manifest.csv"
)

NORMALIZATION_PATH = (
    FIVE_COMMON
    / "normalization_statistics.csv"
)

LSTM_TENSOR_PATH = (
    FIVE_COMMON
    / "nab_lstm_sequence_tensors.npz"
)


CORRECTED_PARTITION_PATHS = {

    "train":
        NAB_6030_CORRECTION_DIR
        / "train_corrected.csv",

    "validation":
        NAB_6030_CORRECTION_DIR
        / "validation_corrected.csv",

    "test":
        NAB_6030_CORRECTION_DIR
        / "test_corrected.csv",
}


CLASSICAL_MODEL_PATHS = {

    "Isolation Forest":
        FIVE_CONTROLLED
        / "isolation_forest.joblib",

    "Local Outlier Factor":
        FIVE_CONTROLLED
        / "local_outlier_factor.joblib",

    "One-Class SVM":
        FIVE_CONTROLLED
        / "oneclass_svm.joblib",
}


HISTORICAL_TEST_SCORE_PATHS = {

    "Isolation Forest":
        FIVE_CONTROLLED
        / "isolation_forest_test_scores.csv",

    "Local Outlier Factor":
        FIVE_CONTROLLED
        / "local_outlier_factor_test_scores.csv",

    "One-Class SVM":
        FIVE_CONTROLLED
        / "oneclass_svm_test_scores.csv",
}


HISTORICAL_COMPARISON_PATH = (
    FIVE_CONTROLLED
    / "nab_label_blind_five_model_comparison.csv"
)


# ============================================================
# B. Exact feature order from the historical branch
# ============================================================

FEATURE_COLUMNS = [

    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var",
]


assert len(
    FEATURE_COLUMNS
) == 23


# ============================================================
# C. Helpers
# ============================================================

def normalize_timestamps(
    df
):

    out = df.copy()

    for col in [
        "window_start",
        "window_end",
    ]:

        out[col] = pd.to_datetime(
            out[col],
            utc=True,
            errors="raise"
        )

    return out


def calculate_metrics(
    y_true,
    predictions,
    scores
):

    y_true = np.asarray(
        y_true,
        dtype=np.int8
    )

    predictions = np.asarray(
        predictions,
        dtype=np.int8
    )

    scores = np.asarray(
        scores,
        dtype=np.float64
    )


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[
            0,
            1,
        ]
    ).ravel()


    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )


    fpr = (
        fp
        /
        (
            fp
            + tn
        )
        if (
            fp
            + tn
        ) > 0
        else 0.0
    )


    roc_auc = roc_auc_score(
        y_true,
        scores
    )

    pr_auc = average_precision_score(
        y_true,
        scores
    )


    return {

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1_score":
            float(
                f1
            ),

        "false_positive_rate":
            float(
                fpr
            ),

        "roc_auc":
            float(
                roc_auc
            ),

        "pr_auc":
            float(
                pr_auc
            ),

        "flagged_windows":
            int(
                predictions.sum()
            ),

        "flagged_ratio_percent":
            float(
                100
                * predictions.mean()
            ),
    }


# ============================================================
# D. Load original split manifest
# ============================================================

assert MANIFEST_PATH.exists()


manifest = normalize_timestamps(
    pd.read_csv(
        MANIFEST_PATH,
        low_memory=False
    )
)


print("\n")
print("=" * 132)
print("ORIGINAL FIVE-MODEL MANIFEST")
print("=" * 132)


print(
    "Rows:",
    len(
        manifest
    )
)

print(
    manifest[
        "split"
    ].value_counts()
)


assert len(
    manifest
) == 103526

assert int(
    (
        manifest[
            "split"
        ] == "train"
    ).sum()
) == 62096

assert int(
    (
        manifest[
            "split"
        ] == "val"
    ).sum()
) == 20705

assert int(
    (
        manifest[
            "split"
        ] == "test"
    ).sum()
) == 20725


# ============================================================
# E. Load corrected partitions
# ============================================================

corrected_parts = {}


for split_name, path in (
    CORRECTED_PARTITION_PATHS.items()
):

    assert path.exists()

    part = normalize_timestamps(
        pd.read_csv(
            path,
            low_memory=False
        )
    )

    part = part.reset_index(
        drop=True
    )

    corrected_parts[
        split_name
    ] = part


print("\n")
print("=" * 132)
print("CORRECTED 60/30 PARTITIONS")
print("=" * 132)


for split_name, part in (
    corrected_parts.items()
):

    print(
        f"{split_name:<12}: "
        f"{len(part):,}"
    )


assert len(
    corrected_parts[
        "train"
    ]
) == 62096

assert len(
    corrected_parts[
        "validation"
    ]
) == 20648

assert len(
    corrected_parts[
        "test"
    ]
) == 20668


# ============================================================
# F. Recover original window_id for every corrected row
#
# We deliberately match by immutable temporal identity rather
# than by row order.
# ============================================================

IDENTITY_COLUMNS = [

    "series",
    "domain",
    "window_start",
    "window_end",
    "label",
]


manifest_identity = (
    manifest[
        [
            "window_id",
            *IDENTITY_COLUMNS,
        ]
    ]
    .copy()
)


assert not (
    manifest_identity[
        IDENTITY_COLUMNS
    ]
    .duplicated()
    .any()
)


for split_name in [
    "train",
    "validation",
    "test",
]:

    part = (
        corrected_parts[
            split_name
        ]
        .copy()
    )


    part = part.merge(
        manifest_identity,
        on=IDENTITY_COLUMNS,
        how="left",
        validate="one_to_one"
    )


    missing_window_ids = int(
        part[
            "window_id"
        ].isna()
        .sum()
    )


    print(
        f"\n{split_name}: "
        f"missing recovered window_id = "
        f"{missing_window_ids}"
    )


    assert (
        missing_window_ids
        == 0
    )


    part[
        "window_id"
    ] = (
        part[
            "window_id"
        ]
        .astype(
            np.int64
        )
    )


    assert (
        part[
            "window_id"
        ].is_unique
    )


    corrected_parts[
        split_name
    ] = part


print(
    "\nPASS: every corrected NAB 60/30 row "
    "was mapped uniquely to its original window_id."
)


# ============================================================
# G. Verify exactly which original validation/test IDs vanished
# ============================================================

original_val_ids = set(
    manifest.loc[
        manifest[
            "split"
        ] == "val",
        "window_id"
    ]
    .astype(
        int
    )
    .tolist()
)


original_test_ids = set(
    manifest.loc[
        manifest[
            "split"
        ] == "test",
        "window_id"
    ]
    .astype(
        int
    )
    .tolist()
)


corrected_val_ids = set(
    corrected_parts[
        "validation"
    ][
        "window_id"
    ]
    .astype(
        int
    )
    .tolist()
)


corrected_test_ids = set(
    corrected_parts[
        "test"
    ][
        "window_id"
    ]
    .astype(
        int
    )
    .tolist()
)


removed_val_ids = (
    original_val_ids
    -
    corrected_val_ids
)

removed_test_ids = (
    original_test_ids
    -
    corrected_test_ids
)


print("\n")
print("=" * 132)
print("PURGED FIVE-MODEL WINDOW-ID CHECK")
print("=" * 132)


print(
    "Removed validation IDs:",
    len(
        removed_val_ids
    )
)

print(
    "Removed test IDs:",
    len(
        removed_test_ids
    )
)


assert len(
    removed_val_ids
) == 57

assert len(
    removed_test_ids
) == 57


print(
    "PASS: corrected five-model population "
    "matches the previously audited 57 + 57 boundary purge."
)


# ============================================================
# H. Load archived per-series training normalization
# ============================================================

assert NORMALIZATION_PATH.exists()


normalization_df = pd.read_csv(
    NORMALIZATION_PATH,
    low_memory=False
)


assert (
    normalization_df[
        "series"
    ].nunique()
    == 58
)


feature_set_from_stats = set(
    normalization_df[
        "feature"
    ].unique()
)


assert set(
    FEATURE_COLUMNS
) == feature_set_from_stats


mean_table = (
    normalization_df
    .pivot(
        index="series",
        columns="feature",
        values="train_mean"
    )
    .reindex(
        columns=FEATURE_COLUMNS
    )
)


std_table = (
    normalization_df
    .pivot(
        index="series",
        columns="feature",
        values="train_std_used"
    )
    .reindex(
        columns=FEATURE_COLUMNS
    )
)


assert not (
    mean_table[
        FEATURE_COLUMNS
    ]
    .isna()
    .any()
    .any()
)

assert not (
    std_table[
        FEATURE_COLUMNS
    ]
    .isna()
    .any()
    .any()
)


# ============================================================
# I. Apply exact archived normalization to corrected rows
# ============================================================

def build_normalized_matrix(
    df
):

    df = df.reset_index(
        drop=True
    )


    X = np.empty(
        (
            len(df),
            len(
                FEATURE_COLUMNS
            ),
        ),
        dtype=np.float64
    )


    for series_name, idx in (
        df.groupby(
            "series",
            sort=False
        ).groups.items()
    ):

        idx = np.asarray(
            list(
                idx
            ),
            dtype=np.int64
        )


        assert (
            series_name
            in mean_table.index
        )

        assert (
            series_name
            in std_table.index
        )


        raw = (
            df.loc[
                idx,
                FEATURE_COLUMNS
            ]
            .apply(
                pd.to_numeric,
                errors="raise"
            )
            .to_numpy(
                dtype=np.float64
            )
        )


        means = (
            mean_table.loc[
                series_name,
                FEATURE_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        stds = (
            std_table.loc[
                series_name,
                FEATURE_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        X[
            idx
        ] = (
            raw
            - means
        ) / stds


    assert np.isfinite(
        X
    ).all()


    return X


X_train = build_normalized_matrix(
    corrected_parts[
        "train"
    ]
)

X_validation = build_normalized_matrix(
    corrected_parts[
        "validation"
    ]
)

X_test = build_normalized_matrix(
    corrected_parts[
        "test"
    ]
)


y_train = (
    corrected_parts[
        "train"
    ][
        "label"
    ]
    .to_numpy(
        dtype=np.int8
    )
)

y_validation = (
    corrected_parts[
        "validation"
    ][
        "label"
    ]
    .to_numpy(
        dtype=np.int8
    )
)

y_test = (
    corrected_parts[
        "test"
    ][
        "label"
    ]
    .to_numpy(
        dtype=np.int8
    )
)


print("\n")
print("=" * 132)
print("CORRECTED FEATURE MATRICES")
print("=" * 132)


print(
    "Train:",
    X_train.shape
)

print(
    "Validation:",
    X_validation.shape
)

print(
    "Test:",
    X_test.shape
)


print(
    "\nLabels:"
)

print(
    "Train anomaly windows:",
    int(
        y_train.sum()
    )
)

print(
    "Validation anomaly windows:",
    int(
        y_validation.sum()
    )
)

print(
    "Test anomaly windows:",
    int(
        y_test.sum()
    )
)


assert X_train.shape == (
    62096,
    23,
)

assert X_validation.shape == (
    20648,
    23,
)

assert X_test.shape == (
    20668,
    23,
)


assert int(
    y_train.sum()
) == 5279

assert int(
    y_validation.sum()
) == 2150

assert int(
    y_test.sum()
) == 2541


print(
    "PASS: corrected classical and Dense-AE "
    "feature matrices are ready."
)


# ============================================================
# J. Load saved classical models and score corrected rows
# ============================================================

classical_score_outputs = {}

score_reproduction_rows = []

corrected_classical_metric_rows = []


for model_name, model_path in (
    CLASSICAL_MODEL_PATHS.items()
):

    print("\n")
    print("=" * 110)
    print(model_name.upper())
    print("=" * 110)


    model = joblib.load(
        model_path
    )


    assert int(
        model.n_features_in_
    ) == 23


    # Historical controlled comparison used:
    # higher score = more anomalous = -score_samples(...)
    validation_scores = (
        -model.score_samples(
            X_validation
        )
    )

    test_scores = (
        -model.score_samples(
            X_test
        )
    )


    assert np.isfinite(
        validation_scores
    ).all()

    assert np.isfinite(
        test_scores
    ).all()


    # --------------------------------------------------------
    # Historical-score reproduction on retained TEST IDs
    # --------------------------------------------------------

    historical_score_path = (
        HISTORICAL_TEST_SCORE_PATHS[
            model_name
        ]
    )


    historical_scores_df = (
        normalize_timestamps(
            pd.read_csv(
                historical_score_path,
                low_memory=False
            )
        )
    )


    historical_retained = (
        historical_scores_df[
            historical_scores_df[
                "window_id"
            ].isin(
                corrected_parts[
                    "test"
                ][
                    "window_id"
                ]
            )
        ][
            [
                "window_id",
                "anomaly_score",
            ]
        ]
        .copy()
    )


    current_test_scores_df = pd.DataFrame({

        "window_id":
            corrected_parts[
                "test"
            ][
                "window_id"
            ]
            .to_numpy(
                dtype=np.int64
            ),

        "current_anomaly_score":
            test_scores,
    })


    reproduction = (
        current_test_scores_df.merge(
            historical_retained,
            on="window_id",
            how="inner",
            validate="one_to_one"
        )
    )


    assert len(
        reproduction
    ) == 20668


    max_abs_score_difference = float(
        np.max(
            np.abs(
                reproduction[
                    "current_anomaly_score"
                ]
                -
                reproduction[
                    "anomaly_score"
                ]
            )
        )
    )


    mean_abs_score_difference = float(
        np.mean(
            np.abs(
                reproduction[
                    "current_anomaly_score"
                ]
                -
                reproduction[
                    "anomaly_score"
                ]
            )
        )
    )


    print(
        "Retained historical-test score reproduction:"
    )

    print(
        f"  max abs difference  = "
        f"{max_abs_score_difference:.12e}"
    )

    print(
        f"  mean abs difference = "
        f"{mean_abs_score_difference:.12e}"
    )


    assert (
        max_abs_score_difference
        < 1e-8
    ), (
        f"{model_name}: saved-model score reproduction "
        "does not match historical retained test scores."
    )


    print(
        "PASS: saved model reproduces historical "
        "retained test scores."
    )


    # --------------------------------------------------------
    # Corrected validation-derived threshold
    # --------------------------------------------------------

    threshold = float(
        np.percentile(
            validation_scores,
            95.0
        )
    )


    validation_predictions = (
        validation_scores
        > threshold
    ).astype(
        np.int8
    )


    test_predictions = (
        test_scores
        > threshold
    ).astype(
        np.int8
    )


    validation_metrics = calculate_metrics(
        y_validation,
        validation_predictions,
        validation_scores
    )


    test_metrics = calculate_metrics(
        y_test,
        test_predictions,
        test_scores
    )


    print(
        f"\nCorrected threshold: "
        f"{threshold:.12f}"
    )


    print(
        "\nCorrected TEST:"
    )

    print(
        f"  Precision = "
        f"{test_metrics['precision']:.6f}"
    )

    print(
        f"  Recall    = "
        f"{test_metrics['recall']:.6f}"
    )

    print(
        f"  F1        = "
        f"{test_metrics['f1_score']:.6f}"
    )

    print(
        f"  FPR       = "
        f"{test_metrics['false_positive_rate']:.6f}"
    )

    print(
        f"  ROC-AUC   = "
        f"{test_metrics['roc_auc']:.6f}"
    )

    print(
        f"  PR-AUC    = "
        f"{test_metrics['pr_auc']:.6f}"
    )


    for split_name, metrics in [

        (
            "validation",
            validation_metrics,
        ),

        (
            "test",
            test_metrics,
        ),
    ]:

        corrected_classical_metric_rows.append({

            "dataset":
                "NAB",

            "model":
                model_name,

            "split":
                split_name,

            "threshold_percentile":
                95.0,

            "threshold":
                threshold,

            **metrics,
        })


    classical_score_outputs[
        model_name
    ] = {

        "threshold":
            threshold,

        "validation_scores":
            validation_scores,

        "validation_predictions":
            validation_predictions,

        "test_scores":
            test_scores,

        "test_predictions":
            test_predictions,
    }


    score_reproduction_rows.append({

        "model":
            model_name,

        "retained_test_windows":
            len(
                reproduction
            ),

        "max_abs_score_difference":
            max_abs_score_difference,

        "mean_abs_score_difference":
            mean_abs_score_difference,

        "reproduction_pass":
            (
                max_abs_score_difference
                < 1e-8
            ),
    })


# ============================================================
# K. Corrected classical metrics table
# ============================================================

corrected_classical_metrics_df = pd.DataFrame(
    corrected_classical_metric_rows
)


score_reproduction_df = pd.DataFrame(
    score_reproduction_rows
)


print("\n")
print("=" * 132)
print("CORRECTED FIVE-MODEL CLASSICAL TEST RESULTS")
print("=" * 132)


print(
    corrected_classical_metrics_df[
        corrected_classical_metrics_df[
            "split"
        ] == "test"
    ][
        [
            "model",
            "threshold",
            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",
            "roc_auc",
            "pr_auc",
            "flagged_windows",
            "flagged_ratio_percent",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# L. Historical vs corrected classical comparison
# ============================================================

historical_comparison = pd.read_csv(
    HISTORICAL_COMPARISON_PATH,
    low_memory=False
)


historical_classical = (
    historical_comparison[
        historical_comparison[
            "model"
        ].isin(
            [
                "Isolation Forest",
                "Local Outlier Factor",
                "One-Class SVM",
            ]
        )
    ]
    .copy()
)


corrected_test_classical = (
    corrected_classical_metrics_df[
        corrected_classical_metrics_df[
            "split"
        ] == "test"
    ]
    .copy()
)


classical_comparison = (
    historical_classical.merge(
        corrected_test_classical,
        on="model",
        suffixes=(
            "_historical",
            "_corrected",
        )
    )
)


classical_comparison[
    "f1_delta"
] = (
    classical_comparison[
        "f1_score_corrected"
    ]
    -
    classical_comparison[
        "f1_score_historical"
    ]
)


classical_comparison[
    "fpr_delta"
] = (
    classical_comparison[
        "false_positive_rate_corrected"
    ]
    -
    classical_comparison[
        "false_positive_rate_historical"
    ]
)


classical_comparison[
    "roc_auc_delta"
] = (
    classical_comparison[
        "roc_auc_corrected"
    ]
    -
    classical_comparison[
        "roc_auc_historical"
    ]
)


classical_comparison[
    "pr_auc_delta"
] = (
    classical_comparison[
        "pr_auc_corrected"
    ]
    -
    classical_comparison[
        "pr_auc_historical"
    ]
)


print("\n")
print("=" * 132)
print("HISTORICAL VS CORRECTED CLASSICAL TEST RESULTS")
print("=" * 132)


print(
    classical_comparison[
        [
            "model",

            "f1_score_historical",
            "f1_score_corrected",
            "f1_delta",

            "false_positive_rate_historical",
            "false_positive_rate_corrected",
            "fpr_delta",

            "roc_auc_historical",
            "roc_auc_corrected",
            "roc_auc_delta",

            "pr_auc_historical",
            "pr_auc_corrected",
            "pr_auc_delta",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# M. Prepare corrected LSTM tensors
#
# Training tensor remains exactly unchanged.
# Only the audited 57 validation and 57 test sequences are
# removed.
# ============================================================

assert LSTM_TENSOR_PATH.exists()


with np.load(
    LSTM_TENSOR_PATH,
    allow_pickle=False
) as bundle:

    X_lstm_train_original = (
        bundle[
            "X_train"
        ]
        .copy()
    )

    X_lstm_validation_original = (
        bundle[
            "X_validation"
        ]
        .copy()
    )

    X_lstm_test_original = (
        bundle[
            "X_test"
        ]
        .copy()
    )

    train_window_ids_original = (
        bundle[
            "train_window_ids"
        ]
        .copy()
    )

    validation_window_ids_original = (
        bundle[
            "validation_window_ids"
        ]
        .copy()
    )

    test_window_ids_original = (
        bundle[
            "test_window_ids"
        ]
        .copy()
    )


corrected_train_window_ids = (
    corrected_parts[
        "train"
    ][
        "window_id"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

corrected_validation_window_ids = (
    corrected_parts[
        "validation"
    ][
        "window_id"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

corrected_test_window_ids = (
    corrected_parts[
        "test"
    ][
        "window_id"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


assert set(
    train_window_ids_original.tolist()
) == set(
    corrected_train_window_ids.tolist()
)


validation_keep_mask = np.isin(
    validation_window_ids_original,
    corrected_validation_window_ids
)


test_keep_mask = np.isin(
    test_window_ids_original,
    corrected_test_window_ids
)


X_lstm_validation_corrected = (
    X_lstm_validation_original[
        validation_keep_mask
    ]
)

X_lstm_test_corrected = (
    X_lstm_test_original[
        test_keep_mask
    ]
)


validation_ids_corrected_from_bundle = (
    validation_window_ids_original[
        validation_keep_mask
    ]
)

test_ids_corrected_from_bundle = (
    test_window_ids_original[
        test_keep_mask
    ]
)


assert len(
    X_lstm_validation_corrected
) == 20648

assert len(
    X_lstm_test_corrected
) == 20668


assert set(
    validation_ids_corrected_from_bundle.tolist()
) == set(
    corrected_validation_window_ids.tolist()
)

assert set(
    test_ids_corrected_from_bundle.tolist()
) == set(
    corrected_test_window_ids.tolist()
)


print("\n")
print("=" * 132)
print("CORRECTED LSTM TENSOR CHECK")
print("=" * 132)


print(
    "Train:",
    X_lstm_train_original.shape
)

print(
    "Validation:",
    X_lstm_validation_corrected.shape
)

print(
    "Test:",
    X_lstm_test_corrected.shape
)


print(
    "Removed validation sequences:",
    int(
        (
            ~validation_keep_mask
        ).sum()
    )
)

print(
    "Removed test sequences:",
    int(
        (
            ~test_keep_mask
        ).sum()
    )
)


assert int(
    (
        ~validation_keep_mask
    ).sum()
) == 57

assert int(
    (
        ~test_keep_mask
    ).sum()
) == 57


print(
    "PASS: corrected LSTM tensors preserve "
    "the unchanged training tensor and purge exactly "
    "the audited boundary sequences."
)


# ============================================================
# N. Align LSTM labels to tensor order
# ============================================================

label_by_window_id = (
    manifest[
        [
            "window_id",
            "label",
        ]
    ]
    .set_index(
        "window_id"
    )[
        "label"
    ]
)


y_lstm_train = (
    label_by_window_id
    .loc[
        train_window_ids_original
    ]
    .to_numpy(
        dtype=np.int8
    )
)


y_lstm_validation = (
    label_by_window_id
    .loc[
        validation_ids_corrected_from_bundle
    ]
    .to_numpy(
        dtype=np.int8
    )
)


y_lstm_test = (
    label_by_window_id
    .loc[
        test_ids_corrected_from_bundle
    ]
    .to_numpy(
        dtype=np.int8
    )
)


assert int(
    y_lstm_train.sum()
) == 5279

assert int(
    y_lstm_validation.sum()
) == 2150

assert int(
    y_lstm_test.sum()
) == 2541


# ============================================================
# O. Save neural-refit inputs
# ============================================================

DENSE_INPUT_PATH = (
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_corrected_inputs.npz"
)


np.savez_compressed(

    DENSE_INPUT_PATH,

    X_train=X_train.astype(
        np.float32
    ),

    X_validation=X_validation.astype(
        np.float32
    ),

    X_test=X_test.astype(
        np.float32
    ),

    y_train=y_train,

    y_validation=y_validation,

    y_test=y_test,

    train_window_ids=corrected_train_window_ids,

    validation_window_ids=corrected_validation_window_ids,

    test_window_ids=corrected_test_window_ids,
)


LSTM_INPUT_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_corrected_inputs.npz"
)


np.savez_compressed(

    LSTM_INPUT_PATH,

    X_train=X_lstm_train_original,

    X_validation=X_lstm_validation_corrected,

    X_test=X_lstm_test_corrected,

    y_train=y_lstm_train,

    y_validation=y_lstm_validation,

    y_test=y_lstm_test,

    train_window_ids=train_window_ids_original,

    validation_window_ids=validation_ids_corrected_from_bundle,

    test_window_ids=test_ids_corrected_from_bundle,
)


# ============================================================
# P. Save corrected classical score artifacts
# ============================================================

for model_name, outputs in (
    classical_score_outputs.items()
):

    slug = (
        model_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            ""
        )
    )


    validation_output = (
        corrected_parts[
            "validation"
        ][
            [
                "window_id",
                "series",
                "domain",
                "window_start",
                "window_end",
                "label",
            ]
        ]
        .copy()
    )


    validation_output[
        "anomaly_score"
    ] = outputs[
        "validation_scores"
    ]


    validation_output[
        "threshold"
    ] = outputs[
        "threshold"
    ]


    validation_output[
        "prediction"
    ] = outputs[
        "validation_predictions"
    ]


    test_output = (
        corrected_parts[
            "test"
        ][
            [
                "window_id",
                "series",
                "domain",
                "window_start",
                "window_end",
                "label",
            ]
        ]
        .copy()
    )


    test_output[
        "anomaly_score"
    ] = outputs[
        "test_scores"
    ]


    test_output[
        "threshold"
    ] = outputs[
        "threshold"
    ]


    test_output[
        "prediction"
    ] = outputs[
        "test_predictions"
    ]


    validation_output.to_csv(
        FIVE_CORRECTION_DIR
        / (
            f"{slug}_validation_scores_corrected.csv"
        ),
        index=False
    )


    test_output.to_csv(
        FIVE_CORRECTION_DIR
        / (
            f"{slug}_test_scores_corrected.csv"
        ),
        index=False
    )


corrected_classical_metrics_df.to_csv(
    FIVE_CORRECTION_DIR
    / "classical_three_model_metrics_corrected.csv",
    index=False
)


classical_comparison.to_csv(
    FIVE_CORRECTION_DIR
    / "classical_three_model_historical_vs_corrected.csv",
    index=False
)


score_reproduction_df.to_csv(
    FIVE_CORRECTION_DIR
    / "classical_saved_model_score_reproduction_audit.csv",
    index=False
)


# Save recovered corrected partition IDs too.
for split_name, part in (
    corrected_parts.items()
):

    part.to_csv(
        FIVE_CORRECTION_DIR
        / (
            f"nab_6030_{split_name}_with_window_ids_corrected.csv"
        ),
        index=False
    )


# ============================================================
# Q. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 38 CHECKPOINT")
print("=" * 132)


print(
    "Classical models corrected:",
    len(
        classical_score_outputs
    )
)


print(
    "All saved-model score reproductions passed:",
    bool(
        score_reproduction_df[
            "reproduction_pass"
        ].all()
    )
)


print(
    "\nDense-AE corrected input:"
)

print(
    DENSE_INPUT_PATH
)


print(
    "\nLSTM-AE corrected input:"
)

print(
    LSTM_INPUT_PATH
)


print(
    "\nSaved all Cell 38 correction artifacts under:"
)

print(
    FIVE_CORRECTION_DIR
)


print("\n")
print("=" * 132)
print("CELL 38 COMPLETE")
print("=" * 132)

print(
    "\nNo model was retrained in this cell. "
    "The saved IF, LOF, and One-Class SVM models were retained, "
    "rescored on the purged validation/test populations, and "
    "rethresholded using the corrected validation partition. "
    "Dense-AE and LSTM-AE refit inputs are now frozen."
)

In [ ]:
# CELL 38A — Diagnose LOF Artifact Mismatch and Verify OCSVM Historical Score Provenance

import joblib
import numpy as np
import pandas as pd


print("=" * 132)
print("LOF / OCSVM SAVED-MODEL HISTORICAL SCORE PROVENANCE DIAGNOSTIC")
print("=" * 132)


# ============================================================
# A. Helper: compare one candidate score vector against the
#    historical retained TEST scores by original window_id
# ============================================================

def compare_score_candidate(
    model_name,
    candidate_name,
    candidate_scores,
    historical_df
):

    current_df = pd.DataFrame({

        "window_id":
            corrected_parts[
                "test"
            ][
                "window_id"
            ]
            .to_numpy(
                dtype=np.int64
            ),

        "candidate_score":
            np.asarray(
                candidate_scores,
                dtype=np.float64
            ),
    })


    retained_historical = (
        historical_df[
            historical_df[
                "window_id"
            ].isin(
                current_df[
                    "window_id"
                ]
            )
        ][
            [
                "window_id",
                "anomaly_score",
            ]
        ]
        .copy()
    )


    merged = (
        current_df.merge(
            retained_historical,
            on="window_id",
            how="inner",
            validate="one_to_one"
        )
    )


    assert len(
        merged
    ) == 20668


    diff = (
        merged[
            "candidate_score"
        ].to_numpy()
        -
        merged[
            "anomaly_score"
        ].to_numpy()
    )


    abs_diff = np.abs(
        diff
    )


    pearson = (
        merged[
            "candidate_score"
        ]
        .corr(
            merged[
                "anomaly_score"
            ],
            method="pearson"
        )
    )


    spearman = (
        merged[
            "candidate_score"
        ]
        .corr(
            merged[
                "anomaly_score"
            ],
            method="spearman"
        )
    )


    return {

        "model":
            model_name,

        "candidate":
            candidate_name,

        "rows":
            len(
                merged
            ),

        "max_abs_difference":
            float(
                abs_diff.max()
            ),

        "mean_abs_difference":
            float(
                abs_diff.mean()
            ),

        "median_difference":
            float(
                np.median(
                    diff
                )
            ),

        "std_difference":
            float(
                np.std(
                    diff
                )
            ),

        "pearson":
            float(
                pearson
            ),

        "spearman":
            float(
                spearman
            ),
    }


# ============================================================
# B. Diagnose LOF and OCSVM
# ============================================================

DIAGNOSTIC_MODELS = [

    "Local Outlier Factor",
    "One-Class SVM",
]


diagnostic_rows = []

fit_matrix_rows = []


for model_name in DIAGNOSTIC_MODELS:

    print("\n")
    print("=" * 118)
    print(model_name.upper())
    print("=" * 118)


    model_path = (
        CLASSICAL_MODEL_PATHS[
            model_name
        ]
    )


    historical_path = (
        HISTORICAL_TEST_SCORE_PATHS[
            model_name
        ]
    )


    model = joblib.load(
        model_path
    )


    historical_df = pd.read_csv(
        historical_path,
        low_memory=False
    )


    print(
        "Saved model:",
        model_path
    )

    print(
        "Historical test scores:",
        historical_path
    )

    print(
        "Model class:",
        type(
            model
        ).__name__
    )

    print(
        "n_features_in_:",
        getattr(
            model,
            "n_features_in_",
            None
        )
    )


    if hasattr(
        model,
        "offset_"
    ):

        print(
            "offset_:",
            model.offset_
        )


    # --------------------------------------------------------
    # C. Inspect archived fitted training matrix when available
    # --------------------------------------------------------

    if hasattr(
        model,
        "_fit_X"
    ):

        fitted_X = np.asarray(
            model._fit_X,
            dtype=np.float64
        )


        print(
            "\nSaved fitted matrix shape:",
            fitted_X.shape
        )

        print(
            "Current reconstructed X_train shape:",
            X_train.shape
        )


        same_shape = (
            fitted_X.shape
            ==
            X_train.shape
        )


        if same_shape:

            train_diff = np.abs(
                fitted_X
                -
                X_train
            )


            fit_max_diff = float(
                train_diff.max()
            )

            fit_mean_diff = float(
                train_diff.mean()
            )


            print(
                f"Saved fit-X vs reconstructed X_train "
                f"max abs difference:  "
                f"{fit_max_diff:.12e}"
            )

            print(
                f"Saved fit-X vs reconstructed X_train "
                f"mean abs difference: "
                f"{fit_mean_diff:.12e}"
            )


        else:

            fit_max_diff = np.nan
            fit_mean_diff = np.nan


        fit_matrix_rows.append({

            "model":
                model_name,

            "saved_fit_rows":
                fitted_X.shape[0],

            "saved_fit_features":
                fitted_X.shape[1],

            "current_train_rows":
                X_train.shape[0],

            "current_train_features":
                X_train.shape[1],

            "same_shape":
                same_shape,

            "max_abs_fit_matrix_difference":
                fit_max_diff,

            "mean_abs_fit_matrix_difference":
                fit_mean_diff,
        })


    else:

        print(
            "\nModel does not expose _fit_X."
        )


    # --------------------------------------------------------
    # D. Candidate 1:
    # historical controlled code says -score_samples(...)
    # --------------------------------------------------------

    candidate_score_samples = (
        -model.score_samples(
            X_test
        )
    )


    result_score_samples = (
        compare_score_candidate(
            model_name,
            "-score_samples",
            candidate_score_samples,
            historical_df
        )
    )


    diagnostic_rows.append(
        result_score_samples
    )


    # --------------------------------------------------------
    # E. Candidate 2:
    # check -decision_function(...) to rule out a scoring-
    # convention mismatch.
    # --------------------------------------------------------

    if hasattr(
        model,
        "decision_function"
    ):

        candidate_decision = (
            -model.decision_function(
                X_test
            )
        )


        result_decision = (
            compare_score_candidate(
                model_name,
                "-decision_function",
                candidate_decision,
                historical_df
            )
        )


        diagnostic_rows.append(
            result_decision
        )


    # --------------------------------------------------------
    # F. Historical artifact characteristics
    # --------------------------------------------------------

    historical_thresholds = (
        historical_df[
            "threshold"
        ]
        .dropna()
        .unique()
    )


    print(
        "\nHistorical threshold values:",
        historical_thresholds
    )


    print(
        "Historical score range:"
    )

    print(
        "  min:",
        float(
            historical_df[
                "anomaly_score"
            ].min()
        )
    )

    print(
        "  median:",
        float(
            historical_df[
                "anomaly_score"
            ].median()
        )
    )

    print(
        "  max:",
        float(
            historical_df[
                "anomaly_score"
            ].max()
        )
    )


# ============================================================
# G. Consolidated scoring comparison
# ============================================================

diagnostic_df = pd.DataFrame(
    diagnostic_rows
)


fit_matrix_df = pd.DataFrame(
    fit_matrix_rows
)


print("\n")
print("=" * 132)
print("HISTORICAL SCORE REPRODUCTION CANDIDATE SUMMARY")
print("=" * 132)


print(
    diagnostic_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.12e}"
    )
)


print("\n")
print("=" * 132)
print("SAVED FITTED-MATRIX PROVENANCE SUMMARY")
print("=" * 132)


if len(
    fit_matrix_df
):

    print(
        fit_matrix_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.12e}"
        )
    )

else:

    print(
        "No model exposed a directly comparable fitted matrix."
    )


# ============================================================
# H. Determine exact diagnostic verdicts
# ============================================================

print("\n")
print("=" * 132)
print("MODEL-BY-MODEL PROVENANCE VERDICT")
print("=" * 132)


for model_name in DIAGNOSTIC_MODELS:

    model_rows = (
        diagnostic_df[
            diagnostic_df[
                "model"
            ]
            ==
            model_name
        ]
        .sort_values(
            "max_abs_difference"
        )
    )


    best = (
        model_rows.iloc[
            0
        ]
    )


    print(
        f"\n{model_name}:"
    )

    print(
        f"  best candidate = "
        f"{best['candidate']}"
    )

    print(
        f"  max abs diff   = "
        f"{best['max_abs_difference']:.12e}"
    )

    print(
        f"  mean abs diff  = "
        f"{best['mean_abs_difference']:.12e}"
    )

    print(
        f"  Pearson        = "
        f"{best['pearson']:.9f}"
    )

    print(
        f"  Spearman       = "
        f"{best['spearman']:.9f}"
    )


    if (
        best[
            "max_abs_difference"
        ]
        < 1e-8
    ):

        print(
            "  VERDICT: historical scores are exactly "
            "reproducible from the saved model."
        )

    elif (
        abs(
            best[
                "std_difference"
            ]
        )
        < 1e-8
    ):

        print(
            "  VERDICT: mismatch is effectively a constant "
            "score offset / scoring-convention difference."
        )

    else:

        print(
            "  VERDICT: saved model does NOT reproduce the "
            "historical score artifact; model-state or fitted-"
            "reference provenance requires correction."
        )


# ============================================================
# I. Save diagnostic
# ============================================================

LOF_DIAG_DIR = (
    FIVE_CORRECTION_DIR
    / "ARTIFACT_PROVENANCE_DIAGNOSTIC"
)

LOF_DIAG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


diagnostic_df.to_csv(
    LOF_DIAG_DIR
    / "lof_ocsvm_historical_score_candidate_audit.csv",
    index=False
)


fit_matrix_df.to_csv(
    LOF_DIAG_DIR
    / "lof_ocsvm_saved_fit_matrix_audit.csv",
    index=False
)


print("\n")
print("=" * 132)
print("CELL 38A COMPLETE")
print("=" * 132)

print(
    "\nNo model was fitted or retrained. "
    "No threshold or historical artifact was modified."
)

In [ ]:
# CELL 38B — Verify LOF and OCSVM Numerical Score Differences at the Historical Decision Boundary

import joblib
import numpy as np
import pandas as pd


print("=" * 132)
print("LOF / OCSVM NUMERICAL-STABILITY AND HISTORICAL-PREDICTION REPRODUCTION AUDIT")
print("=" * 132)


MODELS_TO_VERIFY = [
    "Local Outlier Factor",
    "One-Class SVM",
]


verification_rows = []


for model_name in MODELS_TO_VERIFY:

    print("\n")
    print("=" * 118)
    print(model_name.upper())
    print("=" * 118)


    # --------------------------------------------------------
    # Load saved fitted model and historical test artifact
    # --------------------------------------------------------

    model = joblib.load(
        CLASSICAL_MODEL_PATHS[
            model_name
        ]
    )


    historical_df = pd.read_csv(
        HISTORICAL_TEST_SCORE_PATHS[
            model_name
        ],
        low_memory=False
    )


    historical_retained = (
        historical_df[
            historical_df[
                "window_id"
            ].isin(
                corrected_parts[
                    "test"
                ][
                    "window_id"
                ]
            )
        ][
            [
                "window_id",
                "label",
                "anomaly_score",
                "threshold",
                "prediction",
            ]
        ]
        .copy()
    )


    assert len(
        historical_retained
    ) == 20668


    # --------------------------------------------------------
    # Score retained corrected test rows using saved model
    # --------------------------------------------------------

    current_scores = (
        -model.score_samples(
            X_test
        )
    )


    current_df = pd.DataFrame({

        "window_id":
            corrected_parts[
                "test"
            ][
                "window_id"
            ]
            .to_numpy(
                dtype=np.int64
            ),

        "current_score":
            current_scores,
    })


    comparison = (
        historical_retained.merge(
            current_df,
            on="window_id",
            how="inner",
            validate="one_to_one"
        )
    )


    assert len(
        comparison
    ) == 20668


    # --------------------------------------------------------
    # Score-difference diagnostics
    # --------------------------------------------------------

    historical_scores = (
        comparison[
            "anomaly_score"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    current_scores_aligned = (
        comparison[
            "current_score"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    abs_diff = np.abs(
        current_scores_aligned
        -
        historical_scores
    )


    relative_diff = (
        abs_diff
        /
        np.maximum(
            np.abs(
                historical_scores
            ),
            1e-12
        )
    )


    print(
        "\nAbsolute score difference:"
    )

    print(
        f"  median = "
        f"{np.median(abs_diff):.12e}"
    )

    print(
        f"  95th   = "
        f"{np.quantile(abs_diff, 0.95):.12e}"
    )

    print(
        f"  99th   = "
        f"{np.quantile(abs_diff, 0.99):.12e}"
    )

    print(
        f"  max    = "
        f"{np.max(abs_diff):.12e}"
    )


    print(
        "\nRelative score difference:"
    )

    print(
        f"  median = "
        f"{np.median(relative_diff):.12e}"
    )

    print(
        f"  95th   = "
        f"{np.quantile(relative_diff, 0.95):.12e}"
    )

    print(
        f"  99th   = "
        f"{np.quantile(relative_diff, 0.99):.12e}"
    )

    print(
        f"  max    = "
        f"{np.max(relative_diff):.12e}"
    )


    # --------------------------------------------------------
    # Historical threshold
    # --------------------------------------------------------

    threshold_values = (
        comparison[
            "threshold"
        ]
        .dropna()
        .unique()
    )


    assert len(
        threshold_values
    ) == 1


    historical_threshold = float(
        threshold_values[
            0
        ]
    )


    print(
        f"\nHistorical threshold = "
        f"{historical_threshold:.12f}"
    )


    # Historical source uses strict >
    current_predictions_at_historical_threshold = (
        current_scores_aligned
        >
        historical_threshold
    ).astype(
        np.int8
    )


    stored_predictions = (
        comparison[
            "prediction"
        ]
        .to_numpy(
            dtype=np.int8
        )
    )


    prediction_disagreements = int(
        np.sum(
            current_predictions_at_historical_threshold
            !=
            stored_predictions
        )
    )


    print(
        "\nPrediction reproduction at historical threshold:"
    )

    print(
        "  Stored historical flagged:",
        int(
            stored_predictions.sum()
        )
    )

    print(
        "  Saved-model flagged:",
        int(
            current_predictions_at_historical_threshold.sum()
        )
    )

    print(
        "  Prediction disagreements:",
        prediction_disagreements
    )


    # --------------------------------------------------------
    # Examine any classification disagreements
    # --------------------------------------------------------

    disagreement_mask = (
        current_predictions_at_historical_threshold
        !=
        stored_predictions
    )


    if prediction_disagreements > 0:

        disagreements = (
            comparison.loc[
                disagreement_mask,
                [
                    "window_id",
                    "label",
                    "anomaly_score",
                    "current_score",
                    "threshold",
                    "prediction",
                ]
            ]
            .copy()
        )


        disagreements[
            "current_prediction"
        ] = (
            current_predictions_at_historical_threshold[
                disagreement_mask
            ]
        )


        disagreements[
            "historical_margin"
        ] = (
            disagreements[
                "anomaly_score"
            ]
            -
            historical_threshold
        )


        disagreements[
            "current_margin"
        ] = (
            disagreements[
                "current_score"
            ]
            -
            historical_threshold
        )


        print(
            "\nRows whose threshold classification changed:"
        )

        print(
            disagreements.to_string(
                index=False,
                float_format=lambda x: f"{x:.12e}"
            )
        )

    else:

        print(
            "  PASS: every retained test-window classification "
            "is reproduced exactly."
        )


    # --------------------------------------------------------
    # Compare historical metrics on retained population using:
    # A) stored historical predictions
    # B) scores from the saved model + same old threshold
    # --------------------------------------------------------

    y_retained = (
        comparison[
            "label"
        ]
        .to_numpy(
            dtype=np.int8
        )
    )


    stored_metrics = calculate_metrics(
        y_retained,
        stored_predictions,
        historical_scores
    )


    reproduced_metrics = calculate_metrics(
        y_retained,
        current_predictions_at_historical_threshold,
        current_scores_aligned
    )


    print(
        "\nRetained-test metrics using stored historical artifact:"
    )

    print(
        f"  Precision = {stored_metrics['precision']:.9f}"
    )

    print(
        f"  Recall    = {stored_metrics['recall']:.9f}"
    )

    print(
        f"  F1        = {stored_metrics['f1_score']:.9f}"
    )

    print(
        f"  FPR       = {stored_metrics['false_positive_rate']:.9f}"
    )

    print(
        f"  ROC-AUC   = {stored_metrics['roc_auc']:.9f}"
    )

    print(
        f"  PR-AUC    = {stored_metrics['pr_auc']:.9f}"
    )


    print(
        "\nRetained-test metrics using saved model + historical threshold:"
    )

    print(
        f"  Precision = {reproduced_metrics['precision']:.9f}"
    )

    print(
        f"  Recall    = {reproduced_metrics['recall']:.9f}"
    )

    print(
        f"  F1        = {reproduced_metrics['f1_score']:.9f}"
    )

    print(
        f"  FPR       = {reproduced_metrics['false_positive_rate']:.9f}"
    )

    print(
        f"  ROC-AUC   = {reproduced_metrics['roc_auc']:.9f}"
    )

    print(
        f"  PR-AUC    = {reproduced_metrics['pr_auc']:.9f}"
    )


    # --------------------------------------------------------
    # Quantify metric differences
    # --------------------------------------------------------

    f1_difference = (
        reproduced_metrics[
            "f1_score"
        ]
        -
        stored_metrics[
            "f1_score"
        ]
    )


    fpr_difference = (
        reproduced_metrics[
            "false_positive_rate"
        ]
        -
        stored_metrics[
            "false_positive_rate"
        ]
    )


    roc_difference = (
        reproduced_metrics[
            "roc_auc"
        ]
        -
        stored_metrics[
            "roc_auc"
        ]
    )


    pr_difference = (
        reproduced_metrics[
            "pr_auc"
        ]
        -
        stored_metrics[
            "pr_auc"
        ]
    )


    print(
        "\nMetric differences caused only by numerical score reproduction:"
    )

    print(
        f"  ΔF1      = {f1_difference:.12e}"
    )

    print(
        f"  ΔFPR     = {fpr_difference:.12e}"
    )

    print(
        f"  ΔROC-AUC = {roc_difference:.12e}"
    )

    print(
        f"  ΔPR-AUC  = {pr_difference:.12e}"
    )


    verification_rows.append({

        "model":
            model_name,

        "retained_test_rows":
            len(
                comparison
            ),

        "historical_threshold":
            historical_threshold,

        "median_abs_score_difference":
            float(
                np.median(
                    abs_diff
                )
            ),

        "max_abs_score_difference":
            float(
                np.max(
                    abs_diff
                )
            ),

        "median_relative_score_difference":
            float(
                np.median(
                    relative_diff
                )
            ),

        "p99_relative_score_difference":
            float(
                np.quantile(
                    relative_diff,
                    0.99
                )
            ),

        "max_relative_score_difference":
            float(
                np.max(
                    relative_diff
                )
            ),

        "prediction_disagreements":
            prediction_disagreements,

        "stored_flagged":
            int(
                stored_predictions.sum()
            ),

        "reproduced_flagged":
            int(
                current_predictions_at_historical_threshold.sum()
            ),

        "f1_difference":
            f1_difference,

        "fpr_difference":
            fpr_difference,

        "roc_auc_difference":
            roc_difference,

        "pr_auc_difference":
            pr_difference,
    })


# ============================================================
# C. Final provenance summary
# ============================================================

verification_df = pd.DataFrame(
    verification_rows
)


print("\n")
print("=" * 132)
print("FINAL NUMERICAL-PROVENANCE SUMMARY")
print("=" * 132)


print(
    verification_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.12e}"
    )
)


print("\n")
print("=" * 132)
print("DECISION-BOUNDARY VERDICT")
print("=" * 132)


for _, row in verification_df.iterrows():

    print(
        f"\n{row['model']}:"
    )

    print(
        f"  prediction disagreements = "
        f"{int(row['prediction_disagreements'])}"
    )

    print(
        f"  |ΔF1| = "
        f"{abs(row['f1_difference']):.12e}"
    )

    print(
        f"  |ΔFPR| = "
        f"{abs(row['fpr_difference']):.12e}"
    )


    if (
        int(
            row[
                "prediction_disagreements"
            ]
        ) == 0
        and
        abs(
            row[
                "f1_difference"
            ]
        ) < 1e-12
        and
        abs(
            row[
                "fpr_difference"
            ]
        ) < 1e-12
    ):

        print(
            "  PASS: saved fitted model is decision-equivalent "
            "to the historical artifact on all retained test rows."
        )

    else:

        print(
            "  REVIEW: at least one historical decision changed; "
            "inspect disagreement rows before proceeding."
        )


# ============================================================
# D. Save diagnostic
# ============================================================

verification_df.to_csv(
    LOF_DIAG_DIR
    / "lof_ocsvm_decision_boundary_reproduction_audit.csv",
    index=False
)


print("\n")
print("=" * 132)
print("CELL 38B COMPLETE")
print("=" * 132)

print(
    "\nNo model was fitted or retrained, no corrected threshold "
    "was selected, and no historical artifact was modified."
)

In [ ]:
# CELL 38C — Final Coherent Classical Five-Model Correction and Neural Refit Input Freeze

import time
import joblib
import numpy as np
import pandas as pd

from sklearn.neighbors import LocalOutlierFactor


print("=" * 132)
print("FINAL COHERENT NAB FIVE-MODEL CLASSICAL CORRECTION")
print("=" * 132)


# ============================================================
# A. Confirm required Cell 38 state is available
# ============================================================

required_objects = [

    "corrected_parts",

    "X_train",
    "X_validation",
    "X_test",

    "y_train",
    "y_validation",
    "y_test",

    "calculate_metrics",

    "CLASSICAL_MODEL_PATHS",

    "HISTORICAL_COMPARISON_PATH",

    "FIVE_CORRECTION_DIR",

    "LSTM_TENSOR_PATH",
]


for object_name in required_objects:

    assert object_name in globals(), (
        f"Required object '{object_name}' is missing."
    )


assert X_train.shape == (62096, 23)
assert X_validation.shape == (20648, 23)
assert X_test.shape == (20668, 23)

assert int(y_train.sum()) == 5279
assert int(y_validation.sum()) == 2150
assert int(y_test.sum()) == 2541


print(
    "PASS: corrected feature matrices and labels "
    "from Cell 38 are available."
)


# ============================================================
# B. Load safe saved models
# ============================================================

if_model = joblib.load(
    CLASSICAL_MODEL_PATHS[
        "Isolation Forest"
    ]
)


ocsvm_model = joblib.load(
    CLASSICAL_MODEL_PATHS[
        "One-Class SVM"
    ]
)


assert int(
    if_model.n_features_in_
) == 23

assert int(
    ocsvm_model.n_features_in_
) == 23


# ============================================================
# C. Reconstruct LOF from unchanged training population
#
# Historical frozen configuration:
# n_neighbors=35
# novelty=True
# contamination="auto"
# n_jobs=-1
#
# No label is used for fitting.
# ============================================================

print("\n")
print("=" * 132)
print("LOF CLEAN CURRENT-RUNTIME RECONSTRUCTION")
print("=" * 132)


lof_training_start = (
    time.perf_counter()
)


lof_model_corrected = (
    LocalOutlierFactor(

        n_neighbors=35,

        novelty=True,

        contamination="auto",

        n_jobs=-1,
    )
)


lof_model_corrected.fit(
    X_train
)


lof_training_seconds = (
    time.perf_counter()
    -
    lof_training_start
)


print(
    f"Training rows: {len(X_train):,}"
)

print(
    "Features:",
    X_train.shape[1]
)

print(
    f"Current-runtime LOF reconstruction time: "
    f"{lof_training_seconds:.3f} s"
)


assert int(
    lof_model_corrected.n_features_in_
) == 23


LOF_CORRECTED_MODEL_PATH = (
    FIVE_CORRECTION_DIR
    /
    "local_outlier_factor_reconstructed_current_runtime.joblib"
)


joblib.dump(
    lof_model_corrected,
    LOF_CORRECTED_MODEL_PATH
)


print(
    "\nSaved reconstructed LOF only to correction directory:"
)

print(
    LOF_CORRECTED_MODEL_PATH
)


# ============================================================
# D. Final coherent model set
# ============================================================

final_classical_models = {

    "Isolation Forest": {
        "model":
            if_model,

        "provenance":
            "saved_fitted_model_exactly_reproduced_historical_scores",

        "historical_delta_interpretation":
            "boundary-purge correction; historical score reproduction exact",
    },

    "Local Outlier Factor": {
        "model":
            lof_model_corrected,

        "provenance":
            "reconstructed_current_runtime_from_unchanged_training_population",

        "historical_delta_interpretation":
            (
                "historical-to-corrected numerical change includes "
                "runtime-sensitive LOF rescoring and is not a pure "
                "boundary-purge-only delta"
            ),
    },

    "One-Class SVM": {
        "model":
            ocsvm_model,

        "provenance":
            "saved_fitted_model_decision_equivalent_to_historical_artifact",

        "historical_delta_interpretation":
            (
                "boundary-purge correction with negligible numerical "
                "score drift and exact historical decision reproduction"
            ),
    },
}


# ============================================================
# E. Score corrected validation/test partitions
#
# Historical controlled pipeline:
# higher value = more anomalous = -score_samples(...)
# threshold = 95th percentile of validation anomaly scores
# flag iff score > threshold
# ============================================================

final_metric_rows = []

final_score_outputs = {}


for model_name, spec in (
    final_classical_models.items()
):

    print("\n")
    print("=" * 118)
    print(model_name.upper())
    print("=" * 118)


    model = (
        spec[
            "model"
        ]
    )


    validation_scores = (
        -model.score_samples(
            X_validation
        )
    )


    test_scores = (
        -model.score_samples(
            X_test
        )
    )


    assert np.isfinite(
        validation_scores
    ).all()

    assert np.isfinite(
        test_scores
    ).all()


    threshold = float(
        np.percentile(
            validation_scores,
            95.0
        )
    )


    validation_predictions = (
        validation_scores
        >
        threshold
    ).astype(
        np.int8
    )


    test_predictions = (
        test_scores
        >
        threshold
    ).astype(
        np.int8
    )


    validation_metrics = (
        calculate_metrics(
            y_validation,
            validation_predictions,
            validation_scores
        )
    )


    test_metrics = (
        calculate_metrics(
            y_test,
            test_predictions,
            test_scores
        )
    )


    print(
        f"Corrected threshold = "
        f"{threshold:.12f}"
    )


    print(
        "\nCorrected TEST:"
    )

    print(
        f"  Precision = "
        f"{test_metrics['precision']:.6f}"
    )

    print(
        f"  Recall    = "
        f"{test_metrics['recall']:.6f}"
    )

    print(
        f"  F1        = "
        f"{test_metrics['f1_score']:.6f}"
    )

    print(
        f"  FPR       = "
        f"{test_metrics['false_positive_rate']:.6f}"
    )

    print(
        f"  ROC-AUC   = "
        f"{test_metrics['roc_auc']:.6f}"
    )

    print(
        f"  PR-AUC    = "
        f"{test_metrics['pr_auc']:.6f}"
    )

    print(
        f"  Flagged   = "
        f"{test_metrics['flagged_windows']:,}/"
        f"{len(y_test):,} "
        f"({test_metrics['flagged_ratio_percent']:.6f}%)"
    )


    for split_name, metrics in [

        (
            "validation",
            validation_metrics
        ),

        (
            "test",
            test_metrics
        ),
    ]:

        final_metric_rows.append({

            "dataset":
                "NAB",

            "model":
                model_name,

            "split":
                split_name,

            "threshold_percentile":
                95.0,

            "threshold":
                threshold,

            "model_provenance":
                spec[
                    "provenance"
                ],

            "historical_delta_interpretation":
                spec[
                    "historical_delta_interpretation"
                ],

            **metrics,
        })


    final_score_outputs[
        model_name
    ] = {

        "threshold":
            threshold,

        "validation_scores":
            validation_scores,

        "validation_predictions":
            validation_predictions,

        "test_scores":
            test_scores,

        "test_predictions":
            test_predictions,
    }


final_classical_metrics_df = (
    pd.DataFrame(
        final_metric_rows
    )
)


# ============================================================
# F. Print final corrected classical results
# ============================================================

final_classical_test_df = (
    final_classical_metrics_df[
        final_classical_metrics_df[
            "split"
        ]
        == "test"
    ]
    .copy()
)


print("\n")
print("=" * 132)
print("FINAL CORRECTED FIVE-MODEL CLASSICAL TEST RESULTS")
print("=" * 132)


print(
    final_classical_test_df[
        [
            "model",
            "threshold",

            "precision",
            "recall",
            "f1_score",
            "false_positive_rate",

            "roc_auc",
            "pr_auc",

            "flagged_windows",
            "flagged_ratio_percent",

            "model_provenance",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# G. Historical publication values vs corrected values
# ============================================================

historical_five_model_df = (
    pd.read_csv(
        HISTORICAL_COMPARISON_PATH,
        low_memory=False
    )
)


historical_classical_df = (
    historical_five_model_df[
        historical_five_model_df[
            "model"
        ].isin(
            final_classical_test_df[
                "model"
            ]
        )
    ]
    .copy()
)


comparison_df = (
    historical_classical_df.merge(

        final_classical_test_df,

        on="model",

        suffixes=(
            "_historical",
            "_corrected"
        )
    )
)


for metric in [

    "precision",
    "recall",
    "f1_score",
    "false_positive_rate",
    "roc_auc",
    "pr_auc",
    "flagged_ratio_percent",
]:

    comparison_df[
        f"{metric}_delta"
    ] = (
        comparison_df[
            f"{metric}_corrected"
        ]
        -
        comparison_df[
            f"{metric}_historical"
        ]
    )


print("\n")
print("=" * 132)
print("HISTORICAL VS FINAL CORRECTED CLASSICAL RESULTS")
print("=" * 132)


print(
    comparison_df[
        [
            "model",

            "f1_score_historical",
            "f1_score_corrected",
            "f1_score_delta",

            "false_positive_rate_historical",
            "false_positive_rate_corrected",
            "false_positive_rate_delta",

            "roc_auc_historical",
            "roc_auc_corrected",
            "roc_auc_delta",

            "pr_auc_historical",
            "pr_auc_corrected",
            "pr_auc_delta",

            "historical_delta_interpretation",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# H. Save corrected row-level score outputs
# ============================================================

identity_columns = [

    "window_id",
    "series",
    "domain",
    "window_start",
    "window_end",
    "label",
]


for model_name, outputs in (
    final_score_outputs.items()
):

    slug = (
        model_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "-",
            ""
        )
    )


    validation_output = (
        corrected_parts[
            "validation"
        ][
            identity_columns
        ]
        .copy()
    )


    validation_output[
        "anomaly_score"
    ] = (
        outputs[
            "validation_scores"
        ]
    )


    validation_output[
        "threshold"
    ] = (
        outputs[
            "threshold"
        ]
    )


    validation_output[
        "prediction"
    ] = (
        outputs[
            "validation_predictions"
        ]
    )


    test_output = (
        corrected_parts[
            "test"
        ][
            identity_columns
        ]
        .copy()
    )


    test_output[
        "anomaly_score"
    ] = (
        outputs[
            "test_scores"
        ]
    )


    test_output[
        "threshold"
    ] = (
        outputs[
            "threshold"
        ]
    )


    test_output[
        "prediction"
    ] = (
        outputs[
            "test_predictions"
        ]
    )


    validation_output.to_csv(

        FIVE_CORRECTION_DIR
        /
        (
            f"{slug}_validation_scores_"
            "final_corrected.csv"
        ),

        index=False
    )


    test_output.to_csv(

        FIVE_CORRECTION_DIR
        /
        (
            f"{slug}_test_scores_"
            "final_corrected.csv"
        ),

        index=False
    )


final_classical_metrics_df.to_csv(

    FIVE_CORRECTION_DIR
    /
    "FINAL_classical_three_model_metrics_corrected.csv",

    index=False
)


comparison_df.to_csv(

    FIVE_CORRECTION_DIR
    /
    "FINAL_classical_historical_vs_corrected.csv",

    index=False
)


# ============================================================
# I. Freeze Dense-AE corrected inputs
# ============================================================

corrected_train_window_ids = (
    corrected_parts[
        "train"
    ][
        "window_id"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


corrected_validation_window_ids = (
    corrected_parts[
        "validation"
    ][
        "window_id"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


corrected_test_window_ids = (
    corrected_parts[
        "test"
    ][
        "window_id"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


DENSE_INPUT_PATH = (
    FIVE_CORRECTION_DIR
    /
    "dense_autoencoder_corrected_inputs.npz"
)


np.savez_compressed(

    DENSE_INPUT_PATH,

    X_train=
        X_train.astype(
            np.float32
        ),

    X_validation=
        X_validation.astype(
            np.float32
        ),

    X_test=
        X_test.astype(
            np.float32
        ),

    y_train=
        y_train.astype(
            np.int8
        ),

    y_validation=
        y_validation.astype(
            np.int8
        ),

    y_test=
        y_test.astype(
            np.int8
        ),

    train_window_ids=
        corrected_train_window_ids,

    validation_window_ids=
        corrected_validation_window_ids,

    test_window_ids=
        corrected_test_window_ids,
)


# ============================================================
# J. Freeze purged LSTM tensors
#
# Original training tensor is retained exactly.
# Only the 57 audited validation and 57 audited test
# sequences are removed.
# ============================================================

with np.load(
    LSTM_TENSOR_PATH,
    allow_pickle=False
) as bundle:

    X_lstm_train = (
        bundle[
            "X_train"
        ]
        .copy()
    )

    X_lstm_validation_original = (
        bundle[
            "X_validation"
        ]
        .copy()
    )

    X_lstm_test_original = (
        bundle[
            "X_test"
        ]
        .copy()
    )

    train_lstm_ids = (
        bundle[
            "train_window_ids"
        ]
        .copy()
    )

    validation_lstm_ids_original = (
        bundle[
            "validation_window_ids"
        ]
        .copy()
    )

    test_lstm_ids_original = (
        bundle[
            "test_window_ids"
        ]
        .copy()
    )


assert set(
    train_lstm_ids.tolist()
) == set(
    corrected_train_window_ids.tolist()
)


validation_keep = np.isin(

    validation_lstm_ids_original,

    corrected_validation_window_ids
)


test_keep = np.isin(

    test_lstm_ids_original,

    corrected_test_window_ids
)


X_lstm_validation = (
    X_lstm_validation_original[
        validation_keep
    ]
)


X_lstm_test = (
    X_lstm_test_original[
        test_keep
    ]
)


validation_lstm_ids = (
    validation_lstm_ids_original[
        validation_keep
    ]
)


test_lstm_ids = (
    test_lstm_ids_original[
        test_keep
    ]
)


assert X_lstm_train.shape == (
    62096,
    60,
    1
)

assert X_lstm_validation.shape == (
    20648,
    60,
    1
)

assert X_lstm_test.shape == (
    20668,
    60,
    1
)


assert int(
    (
        ~validation_keep
    ).sum()
) == 57

assert int(
    (
        ~test_keep
    ).sum()
) == 57


manifest_label_lookup = (
    manifest[
        [
            "window_id",
            "label",
        ]
    ]
    .set_index(
        "window_id"
    )[
        "label"
    ]
)


y_lstm_train = (
    manifest_label_lookup
    .loc[
        train_lstm_ids
    ]
    .to_numpy(
        dtype=np.int8
    )
)


y_lstm_validation = (
    manifest_label_lookup
    .loc[
        validation_lstm_ids
    ]
    .to_numpy(
        dtype=np.int8
    )
)


y_lstm_test = (
    manifest_label_lookup
    .loc[
        test_lstm_ids
    ]
    .to_numpy(
        dtype=np.int8
    )
)


assert int(
    y_lstm_train.sum()
) == 5279

assert int(
    y_lstm_validation.sum()
) == 2150

assert int(
    y_lstm_test.sum()
) == 2541


LSTM_INPUT_PATH = (
    FIVE_CORRECTION_DIR
    /
    "lstm_autoencoder_corrected_inputs.npz"
)


np.savez_compressed(

    LSTM_INPUT_PATH,

    X_train=
        X_lstm_train,

    X_validation=
        X_lstm_validation,

    X_test=
        X_lstm_test,

    y_train=
        y_lstm_train,

    y_validation=
        y_lstm_validation,

    y_test=
        y_lstm_test,

    train_window_ids=
        train_lstm_ids,

    validation_window_ids=
        validation_lstm_ids,

    test_window_ids=
        test_lstm_ids,
)


print("\n")
print("=" * 132)
print("NEURAL REFIT INPUT FREEZE")
print("=" * 132)


print(
    "Dense AE:"
)

print(
    "  train      ",
    X_train.shape
)

print(
    "  validation ",
    X_validation.shape
)

print(
    "  test       ",
    X_test.shape
)


print(
    "\nLSTM AE:"
)

print(
    "  train      ",
    X_lstm_train.shape
)

print(
    "  validation ",
    X_lstm_validation.shape
)

print(
    "  test       ",
    X_lstm_test.shape
)


print(
    "\nDense input saved:"
)

print(
    DENSE_INPUT_PATH
)


print(
    "\nLSTM input saved:"
)

print(
    LSTM_INPUT_PATH
)


# ============================================================
# K. Final Cell 38C checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 38C CHECKPOINT")
print("=" * 132)


print(
    "Isolation Forest provenance:",
    final_classical_models[
        "Isolation Forest"
    ][
        "provenance"
    ]
)


print(
    "LOF provenance:",
    final_classical_models[
        "Local Outlier Factor"
    ][
        "provenance"
    ]
)


print(
    "OCSVM provenance:",
    final_classical_models[
        "One-Class SVM"
    ][
        "provenance"
    ]
)


print(
    "\nLOF reconstruction training seconds:",
    f"{lof_training_seconds:.3f}"
)


print(
    "\nSaved all corrected artifacts under:"
)

print(
    FIVE_CORRECTION_DIR
)


print("\n")
print("=" * 132)
print("CELL 38C COMPLETE")
print("=" * 132)

print(
    "\nThe final three classical NAB label-blind models are now "
    "internally coherent on the corrected partitions. "
    "No historical artifact was overwritten. "
    "Dense-AE and LSTM-AE corrected refit inputs are frozen."
)

In [ ]:
# CELL 39 — Refit Controlled NAB Dense Autoencoder on Purged Validation Partition

from pathlib import Path
import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)


print("=" * 132)
print("CONTROLLED NAB DENSE AUTOENCODER — PURGED-VALIDATION REFIT")
print("=" * 132)


# ============================================================
# A. Paths
# ============================================================

FIVE_MODEL_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_OCSVM_LSTM_Autoencoder"
)

FIVE_CONTROLLED = (
    FIVE_MODEL_ROOT
    / "Controlled_Five_Model_Comparison"
)

FIVE_COMMON = (
    FIVE_MODEL_ROOT
    / "common"
)

FIVE_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "FIVE_MODEL_CORRECTION"
)

FIVE_CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


DENSE_INPUT_PATH = (
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_corrected_inputs.npz"
)


HISTORICAL_DENSE_METRICS_PATH = (
    FIVE_CONTROLLED
    / "dense_autoencoder_metrics.csv"
)


HISTORICAL_DENSE_HISTORY_PATH = (
    FIVE_CONTROLLED
    / "dense_autoencoder_training_history.csv"
)


MANIFEST_PATH = (
    FIVE_COMMON
    / "nab_split_manifest.csv"
)


assert DENSE_INPUT_PATH.exists()
assert HISTORICAL_DENSE_METRICS_PATH.exists()
assert HISTORICAL_DENSE_HISTORY_PATH.exists()
assert MANIFEST_PATH.exists()


# ============================================================
# B. Load frozen corrected inputs
# ============================================================

with np.load(
    DENSE_INPUT_PATH,
    allow_pickle=False
) as bundle:

    X_train_dense = (
        bundle["X_train"]
        .astype(
            np.float32,
            copy=False
        )
    )

    X_validation_dense = (
        bundle["X_validation"]
        .astype(
            np.float32,
            copy=False
        )
    )

    X_test_dense = (
        bundle["X_test"]
        .astype(
            np.float32,
            copy=False
        )
    )

    y_train_dense = (
        bundle["y_train"]
        .astype(
            np.int8,
            copy=False
        )
    )

    y_validation_dense = (
        bundle["y_validation"]
        .astype(
            np.int8,
            copy=False
        )
    )

    y_test_dense = (
        bundle["y_test"]
        .astype(
            np.int8,
            copy=False
        )
    )

    train_window_ids_dense = (
        bundle["train_window_ids"]
        .astype(
            np.int64,
            copy=False
        )
    )

    validation_window_ids_dense = (
        bundle["validation_window_ids"]
        .astype(
            np.int64,
            copy=False
        )
    )

    test_window_ids_dense = (
        bundle["test_window_ids"]
        .astype(
            np.int64,
            copy=False
        )
    )


print("\n")
print("=" * 132)
print("CORRECTED DENSE-AE INPUT AUDIT")
print("=" * 132)


print(
    "Train      :",
    X_train_dense.shape
)

print(
    "Validation :",
    X_validation_dense.shape
)

print(
    "Test       :",
    X_test_dense.shape
)


print(
    "\nAnomaly-window labels used only for evaluation:"
)

print(
    "Train      :",
    int(
        y_train_dense.sum()
    )
)

print(
    "Validation :",
    int(
        y_validation_dense.sum()
    )
)

print(
    "Test       :",
    int(
        y_test_dense.sum()
    )
)


assert X_train_dense.shape == (
    62096,
    23
)

assert X_validation_dense.shape == (
    20648,
    23
)

assert X_test_dense.shape == (
    20668,
    23
)


assert len(
    train_window_ids_dense
) == 62096

assert len(
    validation_window_ids_dense
) == 20648

assert len(
    test_window_ids_dense
) == 20668


assert int(
    y_train_dense.sum()
) == 5279

assert int(
    y_validation_dense.sum()
) == 2150

assert int(
    y_test_dense.sum()
) == 2541


assert np.isfinite(
    X_train_dense
).all()

assert np.isfinite(
    X_validation_dense
).all()

assert np.isfinite(
    X_test_dense
).all()


print(
    "\nPASS: frozen Dense-AE correction inputs are valid."
)


# ============================================================
# C. Metric helper
# ============================================================

def dense_calculate_metrics(
    y_true,
    predictions,
    scores
):

    y_true = np.asarray(
        y_true,
        dtype=np.int8
    )

    predictions = np.asarray(
        predictions,
        dtype=np.int8
    )

    scores = np.asarray(
        scores,
        dtype=np.float64
    )


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[
            0,
            1
        ]
    ).ravel()


    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )


    fpr = (
        fp
        /
        (
            fp
            + tn
        )
        if (
            fp
            + tn
        ) > 0
        else 0.0
    )


    roc_auc = roc_auc_score(
        y_true,
        scores
    )


    pr_auc = average_precision_score(
        y_true,
        scores
    )


    return {

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1_score":
            float(
                f1
            ),

        "false_positive_rate":
            float(
                fpr
            ),

        "roc_auc":
            float(
                roc_auc
            ),

        "pr_auc":
            float(
                pr_auc
            ),

        "flagged_windows":
            int(
                predictions.sum()
            ),

        "flagged_ratio_percent":
            float(
                100.0
                * predictions.mean()
            ),
    }


# ============================================================
# D. Historical checkpoint for comparison
# ============================================================

historical_dense_metrics = pd.read_csv(
    HISTORICAL_DENSE_METRICS_PATH,
    low_memory=False
)


historical_dense_history = pd.read_csv(
    HISTORICAL_DENSE_HISTORY_PATH,
    low_memory=False
)


historical_test_row = (
    historical_dense_metrics[
        historical_dense_metrics[
            "split"
        ] == "test"
    ]
    .iloc[0]
)


historical_validation_row = (
    historical_dense_metrics[
        historical_dense_metrics[
            "split"
        ] == "validation"
    ]
    .iloc[0]
)


historical_best_epoch_position = int(
    np.argmin(
        historical_dense_history[
            "val_loss"
        ].to_numpy()
    )
)


historical_best_val_loss = float(
    historical_dense_history.iloc[
        historical_best_epoch_position
    ][
        "val_loss"
    ]
)


print("\n")
print("=" * 132)
print("HISTORICAL DENSE-AE CHECKPOINT")
print("=" * 132)


print(
    "Historical epochs run:",
    len(
        historical_dense_history
    )
)

print(
    "Historical best epoch position:",
    historical_best_epoch_position
)

print(
    f"Historical best val_loss: "
    f"{historical_best_val_loss:.9f}"
)

print(
    f"Historical threshold: "
    f"{float(historical_test_row['threshold']):.12f}"
)

print(
    f"Historical test F1: "
    f"{float(historical_test_row['f1_score']):.9f}"
)

print(
    f"Historical test FPR: "
    f"{float(historical_test_row['false_positive_rate']):.9f}"
)

print(
    f"Historical test ROC-AUC: "
    f"{float(historical_test_row['roc_auc']):.9f}"
)

print(
    f"Historical test PR-AUC: "
    f"{float(historical_test_row['pr_auc']):.9f}"
)


# ============================================================
# E. Exact historical Dense-AE architecture
#
# 23 -> 16 -> 8 -> 4 -> 8 -> 16 -> 23
# Adam(lr=0.001), MSE
# max 150 epochs
# batch size 256
# early stopping patience 10
# min_delta 1e-5
# restore_best_weights=True
# ============================================================

RANDOM_STATE = 42
THRESHOLD_PERCENTILE = 95.0


tf.keras.backend.clear_session()

tf.keras.utils.set_random_seed(
    RANDOM_STATE
)


dense_input = Input(
    shape=(
        23,
    ),
    name="feature_input"
)


x = Dense(
    16,
    activation="relu",
    name="encoder_16"
)(
    dense_input
)


x = Dense(
    8,
    activation="relu",
    name="encoder_8"
)(
    x
)


latent = Dense(
    4,
    activation="relu",
    name="latent_4"
)(
    x
)


x = Dense(
    8,
    activation="relu",
    name="decoder_8"
)(
    latent
)


x = Dense(
    16,
    activation="relu",
    name="decoder_16"
)(
    x
)


dense_output = Dense(
    23,
    activation="linear",
    name="reconstruction"
)(
    x
)


dense_autoencoder_corrected = Model(
    inputs=dense_input,
    outputs=dense_output,
    name="NAB_Dense_Autoencoder_BoundaryCorrected"
)


dense_autoencoder_corrected.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="mean_squared_error"
)


dense_early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    min_delta=1e-5,

    restore_best_weights=True,

    verbose=1,
)


print("\n")
print("=" * 132)
print("CORRECTED DENSE-AE TRAINING")
print("=" * 132)


print(
    "Architecture: 23-16-8-4-8-16-23"
)

print(
    "Optimizer: Adam, learning_rate=0.001"
)

print(
    "Loss: mean_squared_error"
)

print(
    "Maximum epochs: 150"
)

print(
    "Batch size: 256"
)

print(
    "Early stopping: patience=10, min_delta=1e-5, restore_best_weights=True"
)

print(
    "External validation rows after purge:",
    len(
        X_validation_dense
    )
)


# ============================================================
# F. Refit using unchanged training rows and purged validation
# ============================================================

training_start = (
    time.perf_counter()
)


dense_history_corrected = (
    dense_autoencoder_corrected.fit(

        X_train_dense,
        X_train_dense,

        validation_data=(
            X_validation_dense,
            X_validation_dense
        ),

        epochs=150,

        batch_size=256,

        shuffle=True,

        callbacks=[
            dense_early_stopping
        ],

        verbose=1,
    )
)


dense_training_seconds_corrected = (
    time.perf_counter()
    -
    training_start
)


history_corrected_df = pd.DataFrame(
    dense_history_corrected.history
)


history_corrected_df.insert(
    0,
    "epoch",
    np.arange(
        len(
            history_corrected_df
        )
    )
)


corrected_best_epoch_position = int(
    np.argmin(
        history_corrected_df[
            "val_loss"
        ].to_numpy()
    )
)


corrected_best_val_loss = float(
    history_corrected_df.iloc[
        corrected_best_epoch_position
    ][
        "val_loss"
    ]
)


print("\n")
print("=" * 132)
print("CORRECTED DENSE-AE TRAINING CHECKPOINT")
print("=" * 132)


print(
    "Epochs run:",
    len(
        history_corrected_df
    )
)

print(
    "Best epoch position:",
    corrected_best_epoch_position
)

print(
    f"Best validation loss: "
    f"{corrected_best_val_loss:.9f}"
)

print(
    f"Training seconds: "
    f"{dense_training_seconds_corrected:.3f}"
)


# ============================================================
# G. Score corrected validation and test
# ============================================================

print("\n")
print("=" * 132)
print("CORRECTED DENSE-AE SCORING")
print("=" * 132)


validation_reconstruction = (
    dense_autoencoder_corrected.predict(

        X_validation_dense,

        batch_size=512,

        verbose=0,
    )
)


test_reconstruction = (
    dense_autoencoder_corrected.predict(

        X_test_dense,

        batch_size=512,

        verbose=0,
    )
)


dense_validation_scores_corrected = np.mean(

    np.square(
        X_validation_dense
        -
        validation_reconstruction
    ),

    axis=1
)


dense_test_scores_corrected = np.mean(

    np.square(
        X_test_dense
        -
        test_reconstruction
    ),

    axis=1
)


assert np.isfinite(
    dense_validation_scores_corrected
).all()

assert np.isfinite(
    dense_test_scores_corrected
).all()


dense_threshold_corrected = float(

    np.percentile(
        dense_validation_scores_corrected,
        THRESHOLD_PERCENTILE
    )
)


dense_validation_predictions_corrected = (

    dense_validation_scores_corrected
    >
    dense_threshold_corrected

).astype(
    np.int8
)


dense_test_predictions_corrected = (

    dense_test_scores_corrected
    >
    dense_threshold_corrected

).astype(
    np.int8
)


dense_validation_metrics_corrected = (
    dense_calculate_metrics(

        y_validation_dense,

        dense_validation_predictions_corrected,

        dense_validation_scores_corrected,
    )
)


dense_test_metrics_corrected = (
    dense_calculate_metrics(

        y_test_dense,

        dense_test_predictions_corrected,

        dense_test_scores_corrected,
    )
)


print(
    f"Corrected 95th-percentile threshold: "
    f"{dense_threshold_corrected:.12f}"
)


print("\nCorrected VALIDATION:")

print(
    f"  Precision = "
    f"{dense_validation_metrics_corrected['precision']:.6f}"
)

print(
    f"  Recall    = "
    f"{dense_validation_metrics_corrected['recall']:.6f}"
)

print(
    f"  F1        = "
    f"{dense_validation_metrics_corrected['f1_score']:.6f}"
)

print(
    f"  FPR       = "
    f"{dense_validation_metrics_corrected['false_positive_rate']:.6f}"
)

print(
    f"  Flagged   = "
    f"{dense_validation_metrics_corrected['flagged_windows']:,}/"
    f"{len(y_validation_dense):,}"
)


print("\nCorrected TEST:")

print(
    f"  Precision = "
    f"{dense_test_metrics_corrected['precision']:.6f}"
)

print(
    f"  Recall    = "
    f"{dense_test_metrics_corrected['recall']:.6f}"
)

print(
    f"  F1        = "
    f"{dense_test_metrics_corrected['f1_score']:.6f}"
)

print(
    f"  FPR       = "
    f"{dense_test_metrics_corrected['false_positive_rate']:.6f}"
)

print(
    f"  ROC-AUC   = "
    f"{dense_test_metrics_corrected['roc_auc']:.6f}"
)

print(
    f"  PR-AUC    = "
    f"{dense_test_metrics_corrected['pr_auc']:.6f}"
)

print(
    f"  Flagged   = "
    f"{dense_test_metrics_corrected['flagged_windows']:,}/"
    f"{len(y_test_dense):,} "
    f"({dense_test_metrics_corrected['flagged_ratio_percent']:.6f}%)"
)


# ============================================================
# H. Historical vs corrected comparison
#
# Important:
# This neural model was refitted because the purged external
# validation set affects early stopping/checkpoint selection.
# Therefore the historical-to-corrected numerical delta is not
# interpreted as a pure boundary-row-removal effect.
# ============================================================

historical_vs_corrected_dense = pd.DataFrame(
    [
        {
            "model":
                "Dense Autoencoder",

            "historical_validation_rows":
                20705,

            "corrected_validation_rows":
                20648,

            "historical_test_rows":
                20725,

            "corrected_test_rows":
                20668,

            "historical_epochs_run":
                len(
                    historical_dense_history
                ),

            "corrected_epochs_run":
                len(
                    history_corrected_df
                ),

            "historical_best_epoch_position":
                historical_best_epoch_position,

            "corrected_best_epoch_position":
                corrected_best_epoch_position,

            "historical_threshold":
                float(
                    historical_test_row[
                        "threshold"
                    ]
                ),

            "corrected_threshold":
                dense_threshold_corrected,

            "historical_precision":
                float(
                    historical_test_row[
                        "precision"
                    ]
                ),

            "corrected_precision":
                dense_test_metrics_corrected[
                    "precision"
                ],

            "historical_recall":
                float(
                    historical_test_row[
                        "recall"
                    ]
                ),

            "corrected_recall":
                dense_test_metrics_corrected[
                    "recall"
                ],

            "historical_f1":
                float(
                    historical_test_row[
                        "f1_score"
                    ]
                ),

            "corrected_f1":
                dense_test_metrics_corrected[
                    "f1_score"
                ],

            "f1_delta":
                (
                    dense_test_metrics_corrected[
                        "f1_score"
                    ]
                    -
                    float(
                        historical_test_row[
                            "f1_score"
                        ]
                    )
                ),

            "historical_fpr":
                float(
                    historical_test_row[
                        "false_positive_rate"
                    ]
                ),

            "corrected_fpr":
                dense_test_metrics_corrected[
                    "false_positive_rate"
                ],

            "fpr_delta":
                (
                    dense_test_metrics_corrected[
                        "false_positive_rate"
                    ]
                    -
                    float(
                        historical_test_row[
                            "false_positive_rate"
                        ]
                    )
                ),

            "historical_roc_auc":
                float(
                    historical_test_row[
                        "roc_auc"
                    ]
                ),

            "corrected_roc_auc":
                dense_test_metrics_corrected[
                    "roc_auc"
                ],

            "roc_auc_delta":
                (
                    dense_test_metrics_corrected[
                        "roc_auc"
                    ]
                    -
                    float(
                        historical_test_row[
                            "roc_auc"
                        ]
                    )
                ),

            "historical_pr_auc":
                float(
                    historical_test_row[
                        "pr_auc"
                    ]
                ),

            "corrected_pr_auc":
                dense_test_metrics_corrected[
                    "pr_auc"
                ],

            "pr_auc_delta":
                (
                    dense_test_metrics_corrected[
                        "pr_auc"
                    ]
                    -
                    float(
                        historical_test_row[
                            "pr_auc"
                        ]
                    )
                ),

            "interpretation":
                (
                    "Neural model refitted because the external "
                    "validation partition controlled early stopping; "
                    "historical-to-corrected differences therefore "
                    "include retraining/runtime variability and are "
                    "not attributable solely to boundary-row removal."
                ),
        }
    ]
)


print("\n")
print("=" * 132)
print("HISTORICAL VS CORRECTED DENSE-AE")
print("=" * 132)


print(
    historical_vs_corrected_dense[
        [
            "historical_epochs_run",
            "corrected_epochs_run",

            "historical_best_epoch_position",
            "corrected_best_epoch_position",

            "historical_threshold",
            "corrected_threshold",

            "historical_f1",
            "corrected_f1",
            "f1_delta",

            "historical_fpr",
            "corrected_fpr",
            "fpr_delta",

            "historical_roc_auc",
            "corrected_roc_auc",
            "roc_auc_delta",

            "historical_pr_auc",
            "corrected_pr_auc",
            "pr_auc_delta",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# I. Prepare corrected metrics table
# ============================================================

dense_corrected_metrics_df = pd.DataFrame(
    [
        {
            "dataset":
                "NAB",

            "model":
                "Dense Autoencoder",

            "split":
                "validation",

            "threshold_percentile":
                THRESHOLD_PERCENTILE,

            "threshold":
                dense_threshold_corrected,

            "training_seconds":
                dense_training_seconds_corrected,

            "epochs_run":
                len(
                    history_corrected_df
                ),

            "best_epoch_position":
                corrected_best_epoch_position,

            "model_provenance":
                (
                    "refitted_on_unchanged_training_population_"
                    "with_purged_external_validation"
                ),

            **dense_validation_metrics_corrected,
        },

        {
            "dataset":
                "NAB",

            "model":
                "Dense Autoencoder",

            "split":
                "test",

            "threshold_percentile":
                THRESHOLD_PERCENTILE,

            "threshold":
                dense_threshold_corrected,

            "training_seconds":
                dense_training_seconds_corrected,

            "epochs_run":
                len(
                    history_corrected_df
                ),

            "best_epoch_position":
                corrected_best_epoch_position,

            "model_provenance":
                (
                    "refitted_on_unchanged_training_population_"
                    "with_purged_external_validation"
                ),

            **dense_test_metrics_corrected,
        },
    ]
)


# ============================================================
# J. Save corrected row-level validation/test scores
# ============================================================

manifest = pd.read_csv(
    MANIFEST_PATH,
    low_memory=False
)


manifest_metadata = (
    manifest[
        [
            "window_id",
            "series",
            "domain",
            "window_start",
            "window_end",
            "label",
        ]
    ]
    .drop_duplicates(
        subset=[
            "window_id"
        ]
    )
    .set_index(
        "window_id"
    )
)


validation_dense_output = (
    manifest_metadata.loc[
        validation_window_ids_dense
    ]
    .reset_index()
)


test_dense_output = (
    manifest_metadata.loc[
        test_window_ids_dense
    ]
    .reset_index()
)


assert len(
    validation_dense_output
) == 20648

assert len(
    test_dense_output
) == 20668


validation_dense_output[
    "anomaly_score"
] = (
    dense_validation_scores_corrected
)


validation_dense_output[
    "threshold"
] = (
    dense_threshold_corrected
)


validation_dense_output[
    "prediction"
] = (
    dense_validation_predictions_corrected
)


test_dense_output[
    "anomaly_score"
] = (
    dense_test_scores_corrected
)


test_dense_output[
    "threshold"
] = (
    dense_threshold_corrected
)


test_dense_output[
    "prediction"
] = (
    dense_test_predictions_corrected
)


# ============================================================
# K. Save only into correction directory
# ============================================================

CORRECTED_DENSE_MODEL_PATH = (
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_refit_corrected.keras"
)


CORRECTED_DENSE_HISTORY_PATH = (
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_training_history_corrected.csv"
)


CORRECTED_DENSE_METRICS_PATH = (
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_metrics_corrected.csv"
)


CORRECTED_DENSE_VAL_SCORES_PATH = (
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_validation_scores_corrected.csv"
)


CORRECTED_DENSE_TEST_SCORES_PATH = (
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_test_scores_corrected.csv"
)


CORRECTED_DENSE_COMPARISON_PATH = (
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_historical_vs_corrected.csv"
)


dense_autoencoder_corrected.save(
    CORRECTED_DENSE_MODEL_PATH
)


history_corrected_df.to_csv(
    CORRECTED_DENSE_HISTORY_PATH,
    index=False
)


dense_corrected_metrics_df.to_csv(
    CORRECTED_DENSE_METRICS_PATH,
    index=False
)


validation_dense_output.to_csv(
    CORRECTED_DENSE_VAL_SCORES_PATH,
    index=False
)


test_dense_output.to_csv(
    CORRECTED_DENSE_TEST_SCORES_PATH,
    index=False
)


historical_vs_corrected_dense.to_csv(
    CORRECTED_DENSE_COMPARISON_PATH,
    index=False
)


# ============================================================
# L. Save concise reproducibility metadata
# ============================================================

dense_metadata = {

    "model":
        "Dense Autoencoder",

    "correction_reason":
        (
            "External validation partition was used for early "
            "stopping/checkpoint selection and therefore required "
            "refitting after split-boundary purging."
        ),

    "training_rows":
        62096,

    "validation_rows":
        20648,

    "test_rows":
        20668,

    "feature_count":
        23,

    "architecture":
        [
            23,
            16,
            8,
            4,
            8,
            16,
            23,
        ],

    "activation_hidden":
        "relu",

    "activation_output":
        "linear",

    "optimizer":
        "Adam",

    "learning_rate":
        0.001,

    "loss":
        "mean_squared_error",

    "maximum_epochs":
        150,

    "batch_size":
        256,

    "shuffle":
        True,

    "early_stopping_patience":
        10,

    "early_stopping_min_delta":
        1e-5,

    "restore_best_weights":
        True,

    "threshold_percentile":
        95.0,

    "threshold_operator":
        ">",

    "random_state":
        42,

    "epochs_run":
        int(
            len(
                history_corrected_df
            )
        ),

    "best_epoch_position":
        int(
            corrected_best_epoch_position
        ),

    "corrected_threshold":
        float(
            dense_threshold_corrected
        ),

    "training_seconds":
        float(
            dense_training_seconds_corrected
        ),
}


with open(
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_correction_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dense_metadata,
        f,
        indent=2
    )


# ============================================================
# M. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 39 CHECKPOINT")
print("=" * 132)


print(
    "Dense AE refit completed:",
    True
)

print(
    "Training population unchanged:",
    len(
        X_train_dense
    ) == 62096
)

print(
    "Purged validation rows:",
    len(
        X_validation_dense
    ) == 20648
)

print(
    "Purged test rows:",
    len(
        X_test_dense
    ) == 20668
)


print(
    f"\nCorrected test F1: "
    f"{dense_test_metrics_corrected['f1_score']:.9f}"
)

print(
    f"Corrected test FPR: "
    f"{dense_test_metrics_corrected['false_positive_rate']:.9f}"
)

print(
    f"Corrected test ROC-AUC: "
    f"{dense_test_metrics_corrected['roc_auc']:.9f}"
)

print(
    f"Corrected test PR-AUC: "
    f"{dense_test_metrics_corrected['pr_auc']:.9f}"
)


print(
    "\nSaved corrected Dense-AE artifacts under:"
)

print(
    FIVE_CORRECTION_DIR
)


print("\n")
print("=" * 132)
print("CELL 39 COMPLETE")
print("=" * 132)

print(
    "\nThe historical Dense Autoencoder artifact was not overwritten. "
    "The corrected model was refitted only because the external "
    "validation partition participated in early stopping."
)

In [ ]:
# CELL 40A — Exact Historical NAB LSTM Autoencoder Callback and Training-Protocol Audit

from pathlib import Path
import json
import pandas as pd
import numpy as np


print("=" * 132)
print("EXACT HISTORICAL NAB LSTM AUTOENCODER TRAINING-PROTOCOL AUDIT")
print("=" * 132)


# ============================================================
# A. Paths
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)

LSTM_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_OCSVM_LSTM_Autoencoder"
    / "LSTM_Autoencoder"
)

CONFIG_PATH = (
    LSTM_ROOT
    / "config.json"
)

HISTORY_PATH = (
    LSTM_ROOT
    / "training_history.csv"
)


assert NOTEBOOK_PATH.exists()
assert CONFIG_PATH.exists()
assert HISTORY_PATH.exists()


# ============================================================
# B. Historical configuration file
# ============================================================

print("\n")
print("=" * 132)
print("HISTORICAL LSTM CONFIG.JSON")
print("=" * 132)


with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    historical_config = json.load(
        f
    )


print(
    json.dumps(
        historical_config,
        indent=2
    )
)


# ============================================================
# C. Find exact historical LSTM training notebook cell
# ============================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    notebook = json.load(
        f
    )


matching_cells = []


for cell_index, cell in enumerate(
    notebook.get(
        "cells",
        []
    ),
    start=1
):

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(
            source,
            list
        )
        else str(source)
    )


    if (
        "LSTM AUTOENCODER TRAINING"
        in text.upper()
        and
        "lstm_autoencoder.fit"
        in text
    ):

        matching_cells.append(
            (
                cell_index,
                text
            )
        )


print("\n")
print("=" * 132)
print("MATCHING HISTORICAL LSTM TRAINING CELLS")
print("=" * 132)


print(
    "Matching cells:",
    len(
        matching_cells
    )
)


assert len(
    matching_cells
) >= 1


# Prefer the NAB cell containing the known model name.
nab_matches = [
    item
    for item
    in matching_cells
    if "NAB_LSTM_Autoencoder"
    in item[1]
]


assert len(
    nab_matches
) == 1, (
    "Could not uniquely identify the historical NAB "
    "LSTM Autoencoder training cell."
)


cell_number, cell_text = (
    nab_matches[0]
)


print(
    "Selected notebook cell:",
    cell_number
)


# ============================================================
# D. Print exact architecture / callback / fit context
# ============================================================

lines = cell_text.splitlines()


KEY_TERMS = [

    "from tensorflow",
    "LSTM_CONFIG",
    "sequence_length",
    "input_features",
    "encoder_units",
    "latent_units",
    "decoder_units",
    "learning_rate",
    "Input(",
    "LSTM(",
    "RepeatVector",
    "TimeDistributed",
    "Dense(",
    "compile(",
    "Adam(",
    "EarlyStopping(",
    "ReduceLROnPlateau(",
    "ModelCheckpoint(",
    "monitor=",
    "patience=",
    "factor=",
    "min_lr=",
    "min_delta=",
    "restore_best_weights=",
    "mode=",
    "callbacks",
    "fit(",
    "validation_data=",
    "epochs=",
    "batch_size=",
    "shuffle=",
]


selected_indices = set()


for idx, line in enumerate(
    lines
):

    if any(
        term.lower()
        in line.lower()
        for term
        in KEY_TERMS
    ):

        start = max(
            0,
            idx - 4
        )

        end = min(
            len(lines),
            idx + 7
        )

        selected_indices.update(
            range(
                start,
                end
            )
        )


print("\n")
print("=" * 132)
print("EXACT LSTM ARCHITECTURE / CALLBACK / FIT CONTEXT")
print("=" * 132)


previous = None


for idx in sorted(
    selected_indices
):

    if (
        previous is not None
        and
        idx > previous + 1
    ):

        print(
            "     ..."
        )


    print(
        f"{idx + 1:04d}: "
        f"{lines[idx]}"
    )

    previous = idx


# ============================================================
# E. Explicit callback-source lines
# ============================================================

print("\n")
print("=" * 132)
print("CALLBACK DEFINITIONS — EXACT SOURCE LINES")
print("=" * 132)


callback_terms = [

    "EarlyStopping",
    "ReduceLROnPlateau",
    "ModelCheckpoint",
    "monitor=",
    "patience=",
    "factor=",
    "min_lr=",
    "min_delta=",
    "restore_best_weights=",
]


for line_no, line in enumerate(
    lines,
    start=1
):

    if any(
        term.lower()
        in line.lower()
        for term
        in callback_terms
    ):

        print(
            f"{line_no:04d}: {line}"
        )


# ============================================================
# F. Historical learning-rate schedule from saved history
# ============================================================

history_df = pd.read_csv(
    HISTORY_PATH,
    low_memory=False
)


print("\n")
print("=" * 132)
print("HISTORICAL LEARNING-RATE CHANGE POINTS")
print("=" * 132)


print(
    "History rows:",
    len(
        history_df
    )
)

print(
    "Columns:",
    history_df.columns.tolist()
)


assert "learning_rate" in history_df.columns


lr_change_mask = (
    history_df[
        "learning_rate"
    ]
    .ne(
        history_df[
            "learning_rate"
        ].shift()
    )
)


lr_changes = (
    history_df.loc[
        lr_change_mask,
        [
            "epoch",
            "loss",
            "val_loss",
            "learning_rate",
        ]
    ]
    .copy()
)


print(
    lr_changes.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


print(
    "\nUnique learning rates:"
)

print(
    history_df[
        "learning_rate"
    ]
    .unique()
)


# ============================================================
# G. Early-stopping checkpoint
# ============================================================

best_position = int(
    np.argmin(
        history_df[
            "val_loss"
        ].to_numpy()
    )
)


best_row = (
    history_df.iloc[
        best_position
    ]
)


print("\n")
print("=" * 132)
print("HISTORICAL EARLY-STOPPING CHECKPOINT")
print("=" * 132)


print(
    "Epochs run:",
    len(
        history_df
    )
)

print(
    "Best epoch position:",
    best_position
)

print(
    f"Best recorded epoch value: "
    f"{best_row['epoch']}"
)

print(
    f"Best val_loss: "
    f"{best_row['val_loss']:.9f}"
)

print(
    f"Learning rate at best epoch: "
    f"{best_row['learning_rate']:.9f}"
)


# ============================================================
# H. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 40A CHECKPOINT")
print("=" * 132)


print(
    "Historical NAB LSTM training cell uniquely identified:",
    True
)

print(
    "Historical configuration loaded:",
    True
)

print(
    "Historical LR schedule recovered:",
    True
)

print(
    "Ready for exact purged-validation LSTM refit "
    "after reviewing callback parameters."
)


print("\n")
print("=" * 132)
print("CELL 40A COMPLETE")
print("=" * 132)

print(
    "\nRead-only audit complete. No model was trained, "
    "no threshold was recalculated, and no historical "
    "artifact was modified."
)

In [ ]:
# CELL 40 — Refit NAB LSTM Autoencoder on Purged Validation Partition and Save Final Corrected Results

from pathlib import Path
import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    RepeatVector,
    TimeDistributed,
    Dense,
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)


print("=" * 132)
print("NAB LSTM AUTOENCODER — PURGED-VALIDATION REFIT")
print("=" * 132)


# ============================================================
# A. Paths
# ============================================================

FIVE_MODEL_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_OCSVM_LSTM_Autoencoder"
)

HISTORICAL_LSTM_DIR = (
    FIVE_MODEL_ROOT
    / "LSTM_Autoencoder"
)

FIVE_COMMON_DIR = (
    FIVE_MODEL_ROOT
    / "common"
)

FIVE_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "FIVE_MODEL_CORRECTION"
)

FIVE_CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


LSTM_INPUT_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_corrected_inputs.npz"
)

HISTORICAL_METRICS_PATH = (
    HISTORICAL_LSTM_DIR
    / "metrics.csv"
)

HISTORICAL_HISTORY_PATH = (
    HISTORICAL_LSTM_DIR
    / "training_history.csv"
)

HISTORICAL_CONFIG_PATH = (
    HISTORICAL_LSTM_DIR
    / "config.json"
)

MANIFEST_PATH = (
    FIVE_COMMON_DIR
    / "nab_split_manifest.csv"
)


assert LSTM_INPUT_PATH.exists()
assert HISTORICAL_METRICS_PATH.exists()
assert HISTORICAL_HISTORY_PATH.exists()
assert HISTORICAL_CONFIG_PATH.exists()
assert MANIFEST_PATH.exists()


# ============================================================
# B. Load frozen corrected LSTM inputs
# ============================================================

with np.load(
    LSTM_INPUT_PATH,
    allow_pickle=False
) as bundle:

    X_lstm_train_corrected = (
        bundle[
            "X_train"
        ]
        .astype(
            np.float32,
            copy=False
        )
    )

    X_lstm_validation_corrected = (
        bundle[
            "X_validation"
        ]
        .astype(
            np.float32,
            copy=False
        )
    )

    X_lstm_test_corrected = (
        bundle[
            "X_test"
        ]
        .astype(
            np.float32,
            copy=False
        )
    )

    y_lstm_train_corrected = (
        bundle[
            "y_train"
        ]
        .astype(
            np.int8,
            copy=False
        )
    )

    y_lstm_validation_corrected = (
        bundle[
            "y_validation"
        ]
        .astype(
            np.int8,
            copy=False
        )
    )

    y_lstm_test_corrected = (
        bundle[
            "y_test"
        ]
        .astype(
            np.int8,
            copy=False
        )
    )

    train_window_ids_lstm = (
        bundle[
            "train_window_ids"
        ]
        .astype(
            np.int64,
            copy=False
        )
    )

    validation_window_ids_lstm = (
        bundle[
            "validation_window_ids"
        ]
        .astype(
            np.int64,
            copy=False
        )
    )

    test_window_ids_lstm = (
        bundle[
            "test_window_ids"
        ]
        .astype(
            np.int64,
            copy=False
        )
    )


print("\n")
print("=" * 132)
print("CORRECTED LSTM INPUT AUDIT")
print("=" * 132)


print(
    "Training tensor   :",
    X_lstm_train_corrected.shape
)

print(
    "Validation tensor :",
    X_lstm_validation_corrected.shape
)

print(
    "Test tensor       :",
    X_lstm_test_corrected.shape
)


print(
    "\nLabels used only for evaluation:"
)

print(
    "Train anomalies      :",
    int(
        y_lstm_train_corrected.sum()
    )
)

print(
    "Validation anomalies :",
    int(
        y_lstm_validation_corrected.sum()
    )
)

print(
    "Test anomalies       :",
    int(
        y_lstm_test_corrected.sum()
    )
)


assert X_lstm_train_corrected.shape == (
    62096,
    60,
    1
)

assert X_lstm_validation_corrected.shape == (
    20648,
    60,
    1
)

assert X_lstm_test_corrected.shape == (
    20668,
    60,
    1
)


assert len(
    train_window_ids_lstm
) == 62096

assert len(
    validation_window_ids_lstm
) == 20648

assert len(
    test_window_ids_lstm
) == 20668


assert int(
    y_lstm_train_corrected.sum()
) == 5279

assert int(
    y_lstm_validation_corrected.sum()
) == 2150

assert int(
    y_lstm_test_corrected.sum()
) == 2541


assert np.isfinite(
    X_lstm_train_corrected
).all()

assert np.isfinite(
    X_lstm_validation_corrected
).all()

assert np.isfinite(
    X_lstm_test_corrected
).all()


print(
    "\nPASS: frozen corrected LSTM inputs are valid."
)


# ============================================================
# C. Historical configuration and checkpoint
# ============================================================

with open(
    HISTORICAL_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    historical_lstm_config = json.load(
        f
    )


historical_lstm_metrics = pd.read_csv(
    HISTORICAL_METRICS_PATH,
    low_memory=False
)


historical_lstm_history = pd.read_csv(
    HISTORICAL_HISTORY_PATH,
    low_memory=False
)


historical_lstm_test = (
    historical_lstm_metrics[
        historical_lstm_metrics[
            "split"
        ] == "test"
    ]
    .iloc[0]
)


historical_lstm_validation = (
    historical_lstm_metrics[
        historical_lstm_metrics[
            "split"
        ] == "validation"
    ]
    .iloc[0]
)


historical_best_position = int(
    np.argmin(
        historical_lstm_history[
            "val_loss"
        ].to_numpy()
    )
)


historical_best_val_loss = float(
    historical_lstm_history.iloc[
        historical_best_position
    ][
        "val_loss"
    ]
)


print("\n")
print("=" * 132)
print("HISTORICAL LSTM CHECKPOINT")
print("=" * 132)


print(
    "Historical epochs run:",
    len(
        historical_lstm_history
    )
)

print(
    "Historical best epoch position:",
    historical_best_position
)

print(
    f"Historical best val_loss: "
    f"{historical_best_val_loss:.9f}"
)

print(
    f"Historical threshold: "
    f"{float(historical_lstm_test['threshold']):.12f}"
)

print(
    f"Historical TEST F1: "
    f"{float(historical_lstm_test['f1_score']):.9f}"
)

print(
    f"Historical TEST FPR: "
    f"{float(historical_lstm_test['false_positive_rate']):.9f}"
)

print(
    f"Historical TEST ROC-AUC: "
    f"{float(historical_lstm_test['roc_auc']):.9f}"
)

print(
    f"Historical TEST PR-AUC: "
    f"{float(historical_lstm_test['pr_auc']):.9f}"
)


# ============================================================
# D. Metric helper
# ============================================================

def lstm_calculate_metrics(
    y_true,
    predictions,
    scores
):

    y_true = np.asarray(
        y_true,
        dtype=np.int8
    )

    predictions = np.asarray(
        predictions,
        dtype=np.int8
    )

    scores = np.asarray(
        scores,
        dtype=np.float64
    )


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[
            0,
            1
        ]
    ).ravel()


    precision = precision_score(
        y_true,
        predictions,
        zero_division=0
    )


    recall = recall_score(
        y_true,
        predictions,
        zero_division=0
    )


    f1 = f1_score(
        y_true,
        predictions,
        zero_division=0
    )


    fpr = (
        fp
        /
        (
            fp
            + tn
        )
        if (
            fp
            + tn
        ) > 0
        else 0.0
    )


    roc_auc = roc_auc_score(
        y_true,
        scores
    )


    pr_auc = average_precision_score(
        y_true,
        scores
    )


    return {

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1_score":
            float(
                f1
            ),

        "false_positive_rate":
            float(
                fpr
            ),

        "roc_auc":
            float(
                roc_auc
            ),

        "pr_auc":
            float(
                pr_auc
            ),

        "flagged_windows":
            int(
                predictions.sum()
            ),

        "flagged_ratio_percent":
            float(
                predictions.mean()
                * 100.0
            ),
    }


# ============================================================
# E. Exact historical model configuration
# ============================================================

RANDOM_STATE = 42

SEQUENCE_LENGTH = 60

THRESHOLD_PERCENTILE = 95.0


LSTM_CONFIG_CORRECTED = {

    "model":
        "LSTM Autoencoder",

    "sequence_length":
        60,

    "input_features":
        1,

    "encoder_units_1":
        32,

    "latent_units":
        16,

    "decoder_units_1":
        16,

    "decoder_units_2":
        32,

    "optimizer":
        "Adam",

    "learning_rate":
        0.001,

    "loss":
        "mean_squared_error",

    "batch_size":
        256,

    "maximum_epochs":
        100,

    "early_stopping_patience":
        10,

    "early_stopping_min_delta":
        1e-5,

    "reduce_lr_factor":
        0.5,

    "reduce_lr_patience":
        5,

    "reduce_lr_min_lr":
        1e-6,

    "threshold_percentile":
        95.0,

    "threshold_operator":
        ">",

    "training_labels_used":
        False,

    "threshold_labels_used":
        False,

    "normalization":
        (
            "Per-series z-score using "
            "training sequence values only"
        ),

    "sequence_standardization":
        (
            "Each original 60-minute window "
            "interpolated to 60 ordered time steps"
        ),

    "random_state":
        42,
}


# ============================================================
# F. TensorFlow reproducibility setup
# ============================================================

tf.keras.backend.clear_session()


tf.keras.utils.set_random_seed(
    RANDOM_STATE
)


determinism_enabled = False


try:

    tf.config.experimental.enable_op_determinism()

    determinism_enabled = True

except Exception:

    determinism_enabled = False


print("\n")
print("=" * 132)
print("CORRECTED LSTM MODEL CONFIGURATION")
print("=" * 132)


print(
    "Architecture:"
)

print(
    "60x1 -> LSTM(32) -> LSTM(16) -> "
    "RepeatVector(60) -> LSTM(16) -> "
    "LSTM(32) -> TimeDistributed(Dense(1))"
)

print(
    "Optimizer: Adam"
)

print(
    "Initial learning rate: 0.001"
)

print(
    "Loss: mean_squared_error"
)

print(
    "Batch size: 256"
)

print(
    "Maximum epochs: 100"
)

print(
    "EarlyStopping: patience=10, min_delta=1e-5, "
    "restore_best_weights=True"
)

print(
    "ReduceLROnPlateau: factor=0.5, patience=5, "
    "min_lr=1e-6"
)

print(
    "TensorFlow op determinism enabled:",
    determinism_enabled
)


# ============================================================
# G. Build exact historical architecture
# ============================================================

inputs = Input(
    shape=(
        SEQUENCE_LENGTH,
        1
    ),
    name="sequence_input"
)


encoded_sequence = LSTM(
    32,
    return_sequences=True,
    name="encoder_lstm_1"
)(
    inputs
)


latent_vector = LSTM(
    16,
    return_sequences=False,
    name="latent_lstm"
)(
    encoded_sequence
)


repeated_vector = RepeatVector(
    SEQUENCE_LENGTH,
    name="repeat_latent_vector"
)(
    latent_vector
)


decoded_sequence = LSTM(
    16,
    return_sequences=True,
    name="decoder_lstm_1"
)(
    repeated_vector
)


decoded_sequence = LSTM(
    32,
    return_sequences=True,
    name="decoder_lstm_2"
)(
    decoded_sequence
)


outputs = TimeDistributed(
    Dense(
        1
    ),
    name="reconstructed_sequence"
)(
    decoded_sequence
)


lstm_autoencoder_corrected = Model(
    inputs=inputs,
    outputs=outputs,
    name="NAB_LSTM_Autoencoder_BoundaryCorrected"
)


lstm_autoencoder_corrected.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="mean_squared_error"
)


# ============================================================
# H. Exact historical callbacks
# ============================================================

early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    min_delta=1e-5,

    restore_best_weights=True,

    verbose=1,
)


reduce_learning_rate = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5,

    min_lr=1e-6,

    verbose=1,
)


# ============================================================
# I. Refit
# ============================================================

print("\n")
print("=" * 132)
print("CORRECTED LSTM TRAINING")
print("=" * 132)


print(
    "Training rows unchanged:",
    len(
        X_lstm_train_corrected
    )
)

print(
    "Purged external validation rows:",
    len(
        X_lstm_validation_corrected
    )
)


training_start = time.perf_counter()


lstm_history_corrected = (
    lstm_autoencoder_corrected.fit(

        X_lstm_train_corrected,
        X_lstm_train_corrected,

        validation_data=(
            X_lstm_validation_corrected,
            X_lstm_validation_corrected
        ),

        epochs=100,

        batch_size=256,

        shuffle=True,

        callbacks=[
            early_stopping,
            reduce_learning_rate
        ],

        verbose=1
    )
)


lstm_training_seconds_corrected = (
    time.perf_counter()
    -
    training_start
)


history_corrected_df = pd.DataFrame(
    lstm_history_corrected.history
)


history_corrected_df.insert(
    0,
    "epoch",
    np.arange(
        len(
            history_corrected_df
        )
    )
)


# Keras may expose LR under either key depending on version.
if (
    "learning_rate"
    not in history_corrected_df.columns
):

    if (
        "lr"
        in history_corrected_df.columns
    ):

        history_corrected_df[
            "learning_rate"
        ] = (
            history_corrected_df[
                "lr"
            ]
        )


corrected_best_position = int(
    np.argmin(
        history_corrected_df[
            "val_loss"
        ].to_numpy()
    )
)


corrected_best_val_loss = float(
    history_corrected_df.iloc[
        corrected_best_position
    ][
        "val_loss"
    ]
)


print("\n")
print("=" * 132)
print("CORRECTED LSTM TRAINING CHECKPOINT")
print("=" * 132)


print(
    "Epochs run:",
    len(
        history_corrected_df
    )
)

print(
    "Best epoch position:",
    corrected_best_position
)

print(
    f"Best validation loss: "
    f"{corrected_best_val_loss:.9f}"
)

print(
    f"Training seconds: "
    f"{lstm_training_seconds_corrected:.3f}"
)


if (
    "learning_rate"
    in history_corrected_df.columns
):

    lr_change_mask = (
        history_corrected_df[
            "learning_rate"
        ]
        .ne(
            history_corrected_df[
                "learning_rate"
            ].shift()
        )
    )


    lr_changes_corrected = (
        history_corrected_df.loc[
            lr_change_mask,
            [
                "epoch",
                "loss",
                "val_loss",
                "learning_rate",
            ]
        ]
        .copy()
    )


    print(
        "\nCorrected learning-rate change points:"
    )

    print(
        lr_changes_corrected.to_string(
            index=False,
            float_format=lambda x: f"{x:.9f}"
        )
    )


# ============================================================
# J. Reconstruction scoring
# ============================================================

print("\n")
print("=" * 132)
print("CORRECTED LSTM SCORING")
print("=" * 132)


validation_reconstruction = (
    lstm_autoencoder_corrected.predict(

        X_lstm_validation_corrected,

        batch_size=256,

        verbose=0
    )
)


test_reconstruction = (
    lstm_autoencoder_corrected.predict(

        X_lstm_test_corrected,

        batch_size=256,

        verbose=0
    )
)


lstm_validation_scores_corrected = np.mean(

    np.square(

        X_lstm_validation_corrected
        -
        validation_reconstruction
    ),

    axis=(
        1,
        2
    )
)


lstm_test_scores_corrected = np.mean(

    np.square(

        X_lstm_test_corrected
        -
        test_reconstruction
    ),

    axis=(
        1,
        2
    )
)


assert np.isfinite(
    lstm_validation_scores_corrected
).all()

assert np.isfinite(
    lstm_test_scores_corrected
).all()


# ============================================================
# K. Corrected 95th-percentile threshold
# ============================================================

lstm_threshold_corrected = float(

    np.percentile(

        lstm_validation_scores_corrected,

        THRESHOLD_PERCENTILE
    )
)


lstm_validation_predictions_corrected = (

    lstm_validation_scores_corrected
    >
    lstm_threshold_corrected

).astype(
    np.int8
)


lstm_test_predictions_corrected = (

    lstm_test_scores_corrected
    >
    lstm_threshold_corrected

).astype(
    np.int8
)


lstm_validation_metrics_corrected = (
    lstm_calculate_metrics(

        y_lstm_validation_corrected,

        lstm_validation_predictions_corrected,

        lstm_validation_scores_corrected
    )
)


lstm_test_metrics_corrected = (
    lstm_calculate_metrics(

        y_lstm_test_corrected,

        lstm_test_predictions_corrected,

        lstm_test_scores_corrected
    )
)


print(
    f"Corrected 95th-percentile threshold: "
    f"{lstm_threshold_corrected:.12f}"
)


print(
    "\nCorrected VALIDATION:"
)

print(
    f"  Precision = "
    f"{lstm_validation_metrics_corrected['precision']:.6f}"
)

print(
    f"  Recall    = "
    f"{lstm_validation_metrics_corrected['recall']:.6f}"
)

print(
    f"  F1        = "
    f"{lstm_validation_metrics_corrected['f1_score']:.6f}"
)

print(
    f"  FPR       = "
    f"{lstm_validation_metrics_corrected['false_positive_rate']:.6f}"
)

print(
    f"  Flagged   = "
    f"{lstm_validation_metrics_corrected['flagged_windows']:,}/"
    f"{len(y_lstm_validation_corrected):,}"
)


print(
    "\nCorrected TEST:"
)

print(
    f"  Precision = "
    f"{lstm_test_metrics_corrected['precision']:.6f}"
)

print(
    f"  Recall    = "
    f"{lstm_test_metrics_corrected['recall']:.6f}"
)

print(
    f"  F1        = "
    f"{lstm_test_metrics_corrected['f1_score']:.6f}"
)

print(
    f"  FPR       = "
    f"{lstm_test_metrics_corrected['false_positive_rate']:.6f}"
)

print(
    f"  ROC-AUC   = "
    f"{lstm_test_metrics_corrected['roc_auc']:.6f}"
)

print(
    f"  PR-AUC    = "
    f"{lstm_test_metrics_corrected['pr_auc']:.6f}"
)

print(
    f"  Flagged   = "
    f"{lstm_test_metrics_corrected['flagged_windows']:,}/"
    f"{len(y_lstm_test_corrected):,} "
    f"({lstm_test_metrics_corrected['flagged_ratio_percent']:.6f}%)"
)


# ============================================================
# L. Historical vs corrected comparison
# ============================================================

historical_vs_corrected_lstm = pd.DataFrame(
    [
        {
            "model":
                "LSTM Autoencoder",

            "historical_validation_rows":
                20705,

            "corrected_validation_rows":
                20648,

            "historical_test_rows":
                20725,

            "corrected_test_rows":
                20668,

            "historical_epochs_run":
                len(
                    historical_lstm_history
                ),

            "corrected_epochs_run":
                len(
                    history_corrected_df
                ),

            "historical_best_epoch_position":
                historical_best_position,

            "corrected_best_epoch_position":
                corrected_best_position,

            "historical_best_val_loss":
                historical_best_val_loss,

            "corrected_best_val_loss":
                corrected_best_val_loss,

            "historical_threshold":
                float(
                    historical_lstm_test[
                        "threshold"
                    ]
                ),

            "corrected_threshold":
                lstm_threshold_corrected,

            "historical_precision":
                float(
                    historical_lstm_test[
                        "precision"
                    ]
                ),

            "corrected_precision":
                lstm_test_metrics_corrected[
                    "precision"
                ],

            "historical_recall":
                float(
                    historical_lstm_test[
                        "recall"
                    ]
                ),

            "corrected_recall":
                lstm_test_metrics_corrected[
                    "recall"
                ],

            "historical_f1":
                float(
                    historical_lstm_test[
                        "f1_score"
                    ]
                ),

            "corrected_f1":
                lstm_test_metrics_corrected[
                    "f1_score"
                ],

            "f1_delta":
                (
                    lstm_test_metrics_corrected[
                        "f1_score"
                    ]
                    -
                    float(
                        historical_lstm_test[
                            "f1_score"
                        ]
                    )
                ),

            "historical_fpr":
                float(
                    historical_lstm_test[
                        "false_positive_rate"
                    ]
                ),

            "corrected_fpr":
                lstm_test_metrics_corrected[
                    "false_positive_rate"
                ],

            "fpr_delta":
                (
                    lstm_test_metrics_corrected[
                        "false_positive_rate"
                    ]
                    -
                    float(
                        historical_lstm_test[
                            "false_positive_rate"
                        ]
                    )
                ),

            "historical_roc_auc":
                float(
                    historical_lstm_test[
                        "roc_auc"
                    ]
                ),

            "corrected_roc_auc":
                lstm_test_metrics_corrected[
                    "roc_auc"
                ],

            "roc_auc_delta":
                (
                    lstm_test_metrics_corrected[
                        "roc_auc"
                    ]
                    -
                    float(
                        historical_lstm_test[
                            "roc_auc"
                        ]
                    )
                ),

            "historical_pr_auc":
                float(
                    historical_lstm_test[
                        "pr_auc"
                    ]
                ),

            "corrected_pr_auc":
                lstm_test_metrics_corrected[
                    "pr_auc"
                ],

            "pr_auc_delta":
                (
                    lstm_test_metrics_corrected[
                        "pr_auc"
                    ]
                    -
                    float(
                        historical_lstm_test[
                            "pr_auc"
                        ]
                    )
                ),

            "interpretation":
                (
                    "The LSTM Autoencoder was refitted because "
                    "the external validation partition participated "
                    "in early stopping and learning-rate scheduling. "
                    "Historical-to-corrected differences therefore "
                    "include neural retraining/runtime variability "
                    "and are not attributable solely to the "
                    "57+57 boundary-row purge."
                ),
        }
    ]
)


print("\n")
print("=" * 132)
print("HISTORICAL VS CORRECTED LSTM AUTOENCODER")
print("=" * 132)


print(
    historical_vs_corrected_lstm[
        [
            "historical_epochs_run",
            "corrected_epochs_run",

            "historical_best_epoch_position",
            "corrected_best_epoch_position",

            "historical_best_val_loss",
            "corrected_best_val_loss",

            "historical_threshold",
            "corrected_threshold",

            "historical_f1",
            "corrected_f1",
            "f1_delta",

            "historical_fpr",
            "corrected_fpr",
            "fpr_delta",

            "historical_roc_auc",
            "corrected_roc_auc",
            "roc_auc_delta",

            "historical_pr_auc",
            "corrected_pr_auc",
            "pr_auc_delta",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# M. Corrected metrics table
# ============================================================

lstm_corrected_metrics_df = pd.DataFrame(
    [
        {
            "dataset":
                "NAB",

            "model":
                "LSTM Autoencoder",

            "split":
                "validation",

            "threshold_percentile":
                THRESHOLD_PERCENTILE,

            "threshold":
                lstm_threshold_corrected,

            "training_seconds":
                lstm_training_seconds_corrected,

            "epochs_run":
                len(
                    history_corrected_df
                ),

            "best_epoch_position":
                corrected_best_position,

            "best_val_loss":
                corrected_best_val_loss,

            "model_provenance":
                (
                    "refitted_on_unchanged_training_population_"
                    "with_purged_external_validation"
                ),

            **lstm_validation_metrics_corrected,
        },

        {
            "dataset":
                "NAB",

            "model":
                "LSTM Autoencoder",

            "split":
                "test",

            "threshold_percentile":
                THRESHOLD_PERCENTILE,

            "threshold":
                lstm_threshold_corrected,

            "training_seconds":
                lstm_training_seconds_corrected,

            "epochs_run":
                len(
                    history_corrected_df
                ),

            "best_epoch_position":
                corrected_best_position,

            "best_val_loss":
                corrected_best_val_loss,

            "model_provenance":
                (
                    "refitted_on_unchanged_training_population_"
                    "with_purged_external_validation"
                ),

            **lstm_test_metrics_corrected,
        },
    ]
)


# ============================================================
# N. Build corrected row-level score outputs
# ============================================================

manifest = pd.read_csv(
    MANIFEST_PATH,
    low_memory=False
)


manifest_metadata = (
    manifest[
        [
            "window_id",
            "series",
            "domain",
            "window_start",
            "window_end",
            "label",
        ]
    ]
    .drop_duplicates(
        subset=[
            "window_id"
        ]
    )
    .set_index(
        "window_id"
    )
)


validation_lstm_output = (
    manifest_metadata.loc[
        validation_window_ids_lstm
    ]
    .reset_index()
)


test_lstm_output = (
    manifest_metadata.loc[
        test_window_ids_lstm
    ]
    .reset_index()
)


assert len(
    validation_lstm_output
) == 20648

assert len(
    test_lstm_output
) == 20668


validation_lstm_output[
    "anomaly_score"
] = (
    lstm_validation_scores_corrected
)


validation_lstm_output[
    "threshold"
] = (
    lstm_threshold_corrected
)


validation_lstm_output[
    "prediction"
] = (
    lstm_validation_predictions_corrected
)


test_lstm_output[
    "anomaly_score"
] = (
    lstm_test_scores_corrected
)


test_lstm_output[
    "threshold"
] = (
    lstm_threshold_corrected
)


test_lstm_output[
    "prediction"
] = (
    lstm_test_predictions_corrected
)


# ============================================================
# O. Save corrected artifacts only
# ============================================================

CORRECTED_LSTM_MODEL_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_refit_corrected.keras"
)

CORRECTED_LSTM_HISTORY_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_training_history_corrected.csv"
)

CORRECTED_LSTM_METRICS_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_metrics_corrected.csv"
)

CORRECTED_LSTM_VAL_SCORES_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_validation_scores_corrected.csv"
)

CORRECTED_LSTM_TEST_SCORES_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_test_scores_corrected.csv"
)

CORRECTED_LSTM_COMPARISON_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_historical_vs_corrected.csv"
)

CORRECTED_LSTM_METADATA_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_correction_metadata.json"
)


lstm_autoencoder_corrected.save(
    CORRECTED_LSTM_MODEL_PATH
)


history_corrected_df.to_csv(
    CORRECTED_LSTM_HISTORY_PATH,
    index=False
)


lstm_corrected_metrics_df.to_csv(
    CORRECTED_LSTM_METRICS_PATH,
    index=False
)


validation_lstm_output.to_csv(
    CORRECTED_LSTM_VAL_SCORES_PATH,
    index=False
)


test_lstm_output.to_csv(
    CORRECTED_LSTM_TEST_SCORES_PATH,
    index=False
)


historical_vs_corrected_lstm.to_csv(
    CORRECTED_LSTM_COMPARISON_PATH,
    index=False
)


LSTM_CONFIG_CORRECTED[
    "correction_reason"
] = (
    "External validation partition was used for "
    "early stopping and ReduceLROnPlateau; the model "
    "therefore required refitting after boundary purging."
)


LSTM_CONFIG_CORRECTED[
    "training_rows"
] = 62096

LSTM_CONFIG_CORRECTED[
    "validation_rows"
] = 20648

LSTM_CONFIG_CORRECTED[
    "test_rows"
] = 20668

LSTM_CONFIG_CORRECTED[
    "epochs_run"
] = int(
    len(
        history_corrected_df
    )
)

LSTM_CONFIG_CORRECTED[
    "best_epoch_position"
] = int(
    corrected_best_position
)

LSTM_CONFIG_CORRECTED[
    "best_validation_loss"
] = float(
    corrected_best_val_loss
)

LSTM_CONFIG_CORRECTED[
    "corrected_threshold"
] = float(
    lstm_threshold_corrected
)

LSTM_CONFIG_CORRECTED[
    "training_seconds"
] = float(
    lstm_training_seconds_corrected
)

LSTM_CONFIG_CORRECTED[
    "tensorflow_op_determinism_enabled"
] = bool(
    determinism_enabled
)


with open(
    CORRECTED_LSTM_METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        LSTM_CONFIG_CORRECTED,
        f,
        indent=2
    )


# ============================================================
# P. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 40 CHECKPOINT")
print("=" * 132)


print(
    "LSTM Autoencoder refit completed:",
    True
)

print(
    "Training population unchanged:",
    len(
        X_lstm_train_corrected
    ) == 62096
)

print(
    "Corrected validation population:",
    len(
        X_lstm_validation_corrected
    ) == 20648
)

print(
    "Corrected test population:",
    len(
        X_lstm_test_corrected
    ) == 20668
)


print(
    f"\nCorrected TEST F1: "
    f"{lstm_test_metrics_corrected['f1_score']:.9f}"
)

print(
    f"Corrected TEST FPR: "
    f"{lstm_test_metrics_corrected['false_positive_rate']:.9f}"
)

print(
    f"Corrected TEST ROC-AUC: "
    f"{lstm_test_metrics_corrected['roc_auc']:.9f}"
)

print(
    f"Corrected TEST PR-AUC: "
    f"{lstm_test_metrics_corrected['pr_auc']:.9f}"
)


print(
    "\nSaved corrected LSTM artifacts under:"
)

print(
    FIVE_CORRECTION_DIR
)


print("\n")
print("=" * 132)
print("CELL 40 COMPLETE")
print("=" * 132)

print(
    "\nThe historical LSTM Autoencoder artifact was not overwritten. "
    "The corrected LSTM was refitted because the purged external "
    "validation partition controlled both early stopping and the "
    "learning-rate schedule."
)

In [ ]:
# CELL 41 — Build Final Corrected NAB Five-Model Publication Table and Historical-Delta Audit

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 132)
print("FINAL CORRECTED NAB FIVE-MODEL PUBLICATION TABLE")
print("=" * 132)


# ============================================================
# A. Paths
# ============================================================

FIVE_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "FIVE_MODEL_CORRECTION"
)

HISTORICAL_FINAL_DIR = (
    BASE
    / "experiments"
    / "EXP_NAB_OCSVM_LSTM_Autoencoder"
    / "FINAL_NAB_FIVE_MODEL_RESULTS"
)

HISTORICAL_PUBLICATION_PATH = (
    HISTORICAL_FINAL_DIR
    / "tables"
    / "nab_five_model_publication_table.csv"
)


CLASSICAL_PATH = (
    FIVE_CORRECTION_DIR
    / "FINAL_classical_three_model_metrics_corrected.csv"
)

DENSE_PATH = (
    FIVE_CORRECTION_DIR
    / "dense_autoencoder_metrics_corrected.csv"
)

LSTM_PATH = (
    FIVE_CORRECTION_DIR
    / "lstm_autoencoder_metrics_corrected.csv"
)


for path in [
    CLASSICAL_PATH,
    DENSE_PATH,
    LSTM_PATH,
    HISTORICAL_PUBLICATION_PATH,
]:

    assert path.exists(), (
        f"Missing required artifact: {path}"
    )


print(
    "PASS: all corrected five-model input artifacts exist."
)


# ============================================================
# B. Load corrected TEST rows only
# ============================================================

classical_df = pd.read_csv(
    CLASSICAL_PATH,
    low_memory=False
)

dense_df = pd.read_csv(
    DENSE_PATH,
    low_memory=False
)

lstm_df = pd.read_csv(
    LSTM_PATH,
    low_memory=False
)


classical_test = (
    classical_df[
        classical_df["split"] == "test"
    ]
    .copy()
)

dense_test = (
    dense_df[
        dense_df["split"] == "test"
    ]
    .copy()
)

lstm_test = (
    lstm_df[
        lstm_df["split"] == "test"
    ]
    .copy()
)


assert len(classical_test) == 3
assert len(dense_test) == 1
assert len(lstm_test) == 1


corrected_all = pd.concat(
    [
        classical_test,
        dense_test,
        lstm_test,
    ],
    ignore_index=True,
    sort=False
)


assert len(corrected_all) == 5
assert corrected_all["model"].nunique() == 5


# ============================================================
# C. Canonical model names / provenance
# ============================================================

MODEL_ORDER = [

    "One-Class SVM",
    "Isolation Forest",
    "Dense Autoencoder",
    "Local Outlier Factor",
    "LSTM Autoencoder",
]


PROVENANCE = {

    "One-Class SVM":
        (
            "Saved fitted model retained; reproduced all historical "
            "retained test decisions exactly. Validation threshold "
            "recalculated on purged validation partition."
        ),

    "Isolation Forest":
        (
            "Saved fitted model retained; historical scores reproduced "
            "to numerical precision. Validation threshold recalculated "
            "on purged validation partition."
        ),

    "Dense Autoencoder":
        (
            "Refitted on unchanged training population because the "
            "external validation partition controlled early stopping. "
            "Purged validation partition used for checkpoint selection "
            "and threshold recalculation."
        ),

    "Local Outlier Factor":
        (
            "Reconstructed from the unchanged 62,096-window training "
            "population using the frozen hyperparameters because the "
            "serialized historical LOF object was numerically unstable "
            "under the current runtime."
        ),

    "LSTM Autoencoder":
        (
            "Refitted on unchanged training population because the "
            "external validation partition controlled early stopping "
            "and ReduceLROnPlateau. Purged validation partition used "
            "for checkpoint selection and threshold recalculation."
        ),
}


corrected_all[
    "correction_provenance"
] = (
    corrected_all[
        "model"
    ].map(
        PROVENANCE
    )
)


assert corrected_all[
    "correction_provenance"
].notna().all()


# ============================================================
# D. Build thesis-facing publication table
# ============================================================

publication_table = pd.DataFrame({

    "Model":
        corrected_all[
            "model"
        ],

    "Precision":
        corrected_all[
            "precision"
        ],

    "Recall":
        corrected_all[
            "recall"
        ],

    "F1-Score":
        corrected_all[
            "f1_score"
        ],

    "FPR":
        corrected_all[
            "false_positive_rate"
        ],

    "ROC-AUC":
        corrected_all[
            "roc_auc"
        ],

    "PR-AUC":
        corrected_all[
            "pr_auc"
        ],

    "Flagged Windows":
        corrected_all[
            "flagged_windows"
        ].astype(int),

    "Flagged Ratio (%)":
        corrected_all[
            "flagged_ratio_percent"
        ],

    "Threshold":
        corrected_all[
            "threshold"
        ],
})


order_lookup = {
    model: i
    for i, model
    in enumerate(
        MODEL_ORDER
    )
}


publication_table[
    "_order"
] = (
    publication_table[
        "Model"
    ].map(
        order_lookup
    )
)


publication_table = (
    publication_table
    .sort_values(
        "_order"
    )
    .drop(
        columns=[
            "_order"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# E. Validate expected final population
# ============================================================

assert (
    publication_table[
        "Flagged Windows"
    ]
    >= 0
).all()


assert len(
    publication_table
) == 5


EXPECTED_F1 = {

    "One-Class SVM":
        0.278158205,

    "Isolation Forest":
        0.232743583,

    "Dense Autoencoder":
        0.229755179,

    "Local Outlier Factor":
        0.189484127,

    "LSTM Autoencoder":
        0.156415695,
}


for model_name, expected in (
    EXPECTED_F1.items()
):

    observed = float(
        publication_table.loc[
            publication_table[
                "Model"
            ] == model_name,
            "F1-Score"
        ].iloc[0]
    )

    assert np.isclose(
        observed,
        expected,
        atol=5e-9,
        rtol=0.0
    ), (
        f"{model_name}: unexpected corrected F1 "
        f"{observed} != {expected}"
    )


print("\n")
print("=" * 132)
print("FINAL CORRECTED FIVE-MODEL TEST TABLE")
print("=" * 132)


print(
    publication_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# F. Ranking audit
# ============================================================

ranking = (
    publication_table[
        [
            "Model",
            "F1-Score",
            "FPR",
            "ROC-AUC",
            "PR-AUC",
        ]
    ]
    .sort_values(
        [
            "F1-Score",
            "FPR",
        ],
        ascending=[
            False,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


ranking.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(ranking) + 1
    )
)


print("\n")
print("=" * 132)
print("FINAL CORRECTED MODEL RANKING BY TEST F1")
print("=" * 132)


print(
    ranking.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


assert (
    ranking.iloc[0][
        "Model"
    ]
    == "One-Class SVM"
)


# ============================================================
# G. Historical publication table
# ============================================================

historical = pd.read_csv(
    HISTORICAL_PUBLICATION_PATH,
    low_memory=False
)


print("\n")
print("=" * 132)
print("HISTORICAL FIVE-MODEL PUBLICATION TABLE")
print("=" * 132)


print(
    historical.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# H. Historical vs corrected metric comparison
# ============================================================

historical_core = historical[
    [
        "Model",
        "Precision",
        "Recall",
        "F1-Score",
        "FPR",
        "ROC-AUC",
        "PR-AUC",
        "Flagged Ratio (%)",
    ]
].copy()


comparison = (
    historical_core.merge(
        publication_table,
        on="Model",
        how="inner",
        suffixes=(
            "_historical",
            "_corrected"
        ),
        validate="one_to_one"
    )
)


assert len(
    comparison
) == 5


for metric in [

    "Precision",
    "Recall",
    "F1-Score",
    "FPR",
    "ROC-AUC",
    "PR-AUC",
    "Flagged Ratio (%)",
]:

    comparison[
        f"{metric} Delta"
    ] = (
        comparison[
            f"{metric}_corrected"
        ]
        -
        comparison[
            f"{metric}_historical"
        ]
    )


comparison[
    "Correction Provenance"
] = (
    comparison[
        "Model"
    ].map(
        PROVENANCE
    )
)


print("\n")
print("=" * 132)
print("HISTORICAL VS CORRECTED FIVE-MODEL DELTAS")
print("=" * 132)


delta_view = comparison[
    [
        "Model",

        "F1-Score_historical",
        "F1-Score_corrected",
        "F1-Score Delta",

        "FPR_historical",
        "FPR_corrected",
        "FPR Delta",

        "ROC-AUC_historical",
        "ROC-AUC_corrected",
        "ROC-AUC Delta",

        "PR-AUC_historical",
        "PR-AUC_corrected",
        "PR-AUC Delta",
    ]
]


print(
    delta_view.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# I. Ranking stability audit
# ============================================================

historical_rank = (
    historical_core[
        [
            "Model",
            "F1-Score"
        ]
    ]
    .sort_values(
        "F1-Score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


corrected_rank = (
    publication_table[
        [
            "Model",
            "F1-Score"
        ]
    ]
    .sort_values(
        "F1-Score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


ranking_unchanged = (
    historical_rank[
        "Model"
    ].tolist()
    ==
    corrected_rank[
        "Model"
    ].tolist()
)


print("\n")
print("=" * 132)
print("RANKING STABILITY")
print("=" * 132)


print(
    "Historical ranking:"
)

for i, row in (
    historical_rank.iterrows()
):

    print(
        f"  {i + 1}. "
        f"{row['Model']} "
        f"(F1={row['F1-Score']:.6f})"
    )


print(
    "\nCorrected ranking:"
)

for i, row in (
    corrected_rank.iterrows()
):

    print(
        f"  {i + 1}. "
        f"{row['Model']} "
        f"(F1={row['F1-Score']:.6f})"
    )


print(
    "\nRanking unchanged:",
    ranking_unchanged
)


assert ranking_unchanged


# ============================================================
# J. Correction provenance table
# ============================================================

provenance_table = pd.DataFrame(
    [
        {
            "Model":
                model_name,

            "Correction Handling":
                PROVENANCE[
                    model_name
                ],
        }
        for model_name
        in MODEL_ORDER
    ]
)


print("\n")
print("=" * 132)
print("FINAL MODEL CORRECTION PROVENANCE")
print("=" * 132)


print(
    provenance_table.to_string(
        index=False
    )
)


# ============================================================
# K. Save authoritative correction artifacts
# ============================================================

FINAL_PUBLICATION_PATH = (
    FIVE_CORRECTION_DIR
    / "FINAL_NAB_FIVE_MODEL_PUBLICATION_TABLE_CORRECTED.csv"
)


FINAL_RANKING_PATH = (
    FIVE_CORRECTION_DIR
    / "FINAL_NAB_FIVE_MODEL_RANKING_CORRECTED.csv"
)


FINAL_COMPARISON_PATH = (
    FIVE_CORRECTION_DIR
    / "FINAL_NAB_FIVE_MODEL_HISTORICAL_VS_CORRECTED.csv"
)


FINAL_PROVENANCE_PATH = (
    FIVE_CORRECTION_DIR
    / "FINAL_NAB_FIVE_MODEL_CORRECTION_PROVENANCE.csv"
)


publication_table.to_csv(
    FINAL_PUBLICATION_PATH,
    index=False
)


ranking.to_csv(
    FINAL_RANKING_PATH,
    index=False
)


comparison.to_csv(
    FINAL_COMPARISON_PATH,
    index=False
)


provenance_table.to_csv(
    FINAL_PROVENANCE_PATH,
    index=False
)


# ============================================================
# L. Final branch checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 41 CHECKPOINT")
print("=" * 132)


print(
    "Final five-model corrected rows:",
    len(
        publication_table
    )
)

print(
    "Corrected test population:",
    20668
)

print(
    "Historical ranking preserved:",
    ranking_unchanged
)

print(
    "Corrected best model:",
    ranking.iloc[0][
        "Model"
    ]
)

print(
    f"Corrected best-model F1: "
    f"{ranking.iloc[0]['F1-Score']:.9f}"
)


print(
    "\nSaved:"
)

print(
    FINAL_PUBLICATION_PATH
)

print(
    FINAL_RANKING_PATH
)

print(
    FINAL_COMPARISON_PATH
)

print(
    FINAL_PROVENANCE_PATH
)


print("\n")
print("=" * 132)
print("CELL 41 COMPLETE")
print("=" * 132)

print(
    "\nThe distinct NAB five-model label-blind branch is now "
    "fully boundary-corrected and frozen. No additional model "
    "training is required for this result family."
)

In [ ]:
# CELL 42 — Correct NAB Representation-Scaling Branch on Purged 60/30 and 60/15 Partitions

from pathlib import Path
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)


print("=" * 132)
print("NAB REPRESENTATION-SCALING BRANCH — SPLIT-BOUNDARY CORRECTION")
print("=" * 132)


# ============================================================
# A. Paths
# ============================================================

REP_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_Representation_Improvement"
)

REP_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "REPRESENTATION_SCALING_CORRECTION"
)

REP_CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


CORRECTED_PATHS = {

    "60/30":
        CORRECTION
        / "NAB"
        / "60_30"
        / "all_corrected_partitions.csv",

    "60/15":
        CORRECTION
        / "NAB"
        / "60_15"
        / "all_corrected_partitions.csv",
}


HISTORICAL_BEST_PATH = (
    REP_ROOT
    / "summary_best_f1_results_clean.csv"
)

HISTORICAL_FPR010_PATH = (
    REP_ROOT
    / "summary_fpr010_results_clean.csv"
)


for path in [
    *CORRECTED_PATHS.values(),
    HISTORICAL_BEST_PATH,
    HISTORICAL_FPR010_PATH,
]:

    assert path.exists(), (
        f"Missing required artifact: {path}"
    )


# ============================================================
# B. Exact historical configuration
# ============================================================

RANDOM_STATE = 42

LOF_TRAIN_MAX = 60000


FEATURE_COLUMNS = [

    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var",
]


VARIANTS = [

    "global_standard",
    "global_robust",
    "per_series_zscore",
    "per_series_robust",
]


MODELS = [

    "IsolationForest",
    "LOF",
]


PERCENTILES = np.arange(
    50,
    98,
    2.5
)


FPR_LIMITS = [

    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
]


assert len(
    FEATURE_COLUMNS
) == 23


# ============================================================
# C. Helpers
# ============================================================

def safe_scale(
    scale
):

    scale = scale.copy()

    scale = scale.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )

    scale = scale.fillna(
        1.0
    )

    scale[
        scale == 0
    ] = 1.0

    return scale


def load_corrected_population(
    path,
    expected_counts,
    expected_anomalies
):

    df = pd.read_csv(
        path,
        low_memory=False
    )


    assert all(
        feature
        in df.columns
        for feature
        in FEATURE_COLUMNS
    )


    assert "series" in df.columns
    assert "window_start" in df.columns
    assert "label" in df.columns
    assert "split" in df.columns


    df[
        "window_start"
    ] = pd.to_datetime(
        df[
            "window_start"
        ],
        utc=True,
        errors="raise"
    )


    if (
        "window_end"
        in df.columns
    ):

        df[
            "window_end"
        ] = pd.to_datetime(
            df[
                "window_end"
            ],
            utc=True,
            errors="raise"
        )


    # Normalize split vocabulary.
    df[
        "_split"
    ] = (
        df[
            "split"
        ]
        .astype(str)
        .str.lower()
        .replace(
            {
                "validation":
                    "val",
            }
        )
    )


    assert set(
        df[
            "_split"
        ].unique()
    ) == {
        "train",
        "val",
        "test",
    }


    # Historical representation code sorted by series/time
    # before assigning the chronological partitions. Sorting
    # here preserves the same training-row order, which also
    # preserves deterministic LOF sampling.
    df = (
        df
        .sort_values(
            [
                "series",
                "window_start"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    observed_counts = (
        df[
            "_split"
        ]
        .value_counts()
        .to_dict()
    )


    observed_anomalies = (
        df.groupby(
            "_split"
        )[
            "label"
        ]
        .sum()
        .astype(int)
        .to_dict()
    )


    for split_name in [
        "train",
        "val",
        "test",
    ]:

        assert int(
            observed_counts[
                split_name
            ]
        ) == int(
            expected_counts[
                split_name
            ]
        )

        assert int(
            observed_anomalies[
                split_name
            ]
        ) == int(
            expected_anomalies[
                split_name
            ]
        )


    return df


def transform_features(
    df,
    variant
):

    train_df = (
        df[
            df[
                "_split"
            ] == "train"
        ]
        .copy()
    )


    X_raw = (
        df[
            FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )


    X_train_raw = (
        train_df[
            FEATURE_COLUMNS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
    )


    # Historical implementation:
    # training-derived global median used for missing values.
    global_median = (
        X_train_raw.median()
    )


    X_raw = (
        X_raw.fillna(
            global_median
        )
    )


    X_train_raw = (
        X_train_raw.fillna(
            global_median
        )
    )


    if (
        variant
        == "global_standard"
    ):

        center = (
            X_train_raw.mean()
        )

        scale = safe_scale(
            X_train_raw.std()
        )


        X = (
            X_raw
            - center
        ) / scale


        return X.astype(
            float
        )


    if (
        variant
        == "global_robust"
    ):

        center = (
            X_train_raw.median()
        )

        q75 = (
            X_train_raw.quantile(
                0.75
            )
        )

        q25 = (
            X_train_raw.quantile(
                0.25
            )
        )

        scale = safe_scale(
            q75
            - q25
        )


        X = (
            X_raw
            - center
        ) / scale


        return X.astype(
            float
        )


    if variant in [
        "per_series_zscore",
        "per_series_robust",
    ]:

        X_out = pd.DataFrame(
            index=df.index,
            columns=FEATURE_COLUMNS,
            dtype=float
        )


        global_mean = (
            X_train_raw.mean()
        )

        global_std = safe_scale(
            X_train_raw.std()
        )

        global_med = (
            X_train_raw.median()
        )

        global_iqr = safe_scale(
            X_train_raw.quantile(
                0.75
            )
            -
            X_train_raw.quantile(
                0.25
            )
        )


        for series_value, idx in (
            df.groupby(
                "series",
                sort=False
            ).groups.items()
        ):

            idx = list(
                idx
            )


            train_idx = (
                df[
                    (
                        df[
                            "series"
                        ]
                        ==
                        series_value
                    )
                    &
                    (
                        df[
                            "_split"
                        ]
                        ==
                        "train"
                    )
                ]
                .index
            )


            if len(
                train_idx
            ) == 0:

                if (
                    variant
                    == "per_series_zscore"
                ):

                    center = (
                        global_mean
                    )

                    scale = (
                        global_std
                    )

                else:

                    center = (
                        global_med
                    )

                    scale = (
                        global_iqr
                    )


            else:

                train_part = (
                    X_raw.loc[
                        train_idx,
                        FEATURE_COLUMNS
                    ]
                )


                if (
                    variant
                    == "per_series_zscore"
                ):

                    center = (
                        train_part.mean()
                    )

                    scale = safe_scale(
                        train_part.std()
                    )

                else:

                    center = (
                        train_part.median()
                    )

                    scale = safe_scale(
                        train_part.quantile(
                            0.75
                        )
                        -
                        train_part.quantile(
                            0.25
                        )
                    )


            X_out.loc[
                idx,
                FEATURE_COLUMNS
            ] = (
                X_raw.loc[
                    idx,
                    FEATURE_COLUMNS
                ]
                -
                center
            ) / scale


        return X_out.astype(
            float
        )


    raise ValueError(
        f"Unknown variant: {variant}"
    )


def metric_dict(
    y_true,
    scores,
    threshold
):

    # Historical representation branch uses >=.
    prediction = (
        scores
        >= threshold
    ).astype(
        np.int8
    )


    precision = precision_score(
        y_true,
        prediction,
        zero_division=0
    )


    recall = recall_score(
        y_true,
        prediction,
        zero_division=0
    )


    f1 = f1_score(
        y_true,
        prediction,
        zero_division=0
    )


    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            prediction,
            labels=[
                0,
                1
            ]
        )
        .ravel()
    )


    fpr = (
        fp
        /
        (
            fp
            + tn
        )
        if (
            fp
            + tn
        ) > 0
        else 0.0
    )


    return {

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "f1":
            float(
                f1
            ),

        "fpr":
            float(
                fpr
            ),

        "tp":
            int(tp),

        "fp":
            int(fp),

        "tn":
            int(tn),

        "fn":
            int(fn),

        "flagged_windows":
            int(
                prediction.sum()
            ),
    }


def fit_and_score_model(
    model_name,
    X_train,
    X_val,
    X_test
):

    start = (
        time.perf_counter()
    )


    if (
        model_name
        == "IsolationForest"
    ):

        model = IsolationForest(

            n_estimators=300,

            random_state=
                RANDOM_STATE,

            n_jobs=-1,

            contamination="auto"
        )


        model.fit(
            X_train
        )


    elif (
        model_name
        == "LOF"
    ):

        X_fit = (
            X_train
        )


        if (
            LOF_TRAIN_MAX
            is not None
            and
            len(
                X_fit
            )
            >
            LOF_TRAIN_MAX
        ):

            rng = (
                np.random.default_rng(
                    RANDOM_STATE
                )
            )


            sample_idx = (
                rng.choice(

                    len(
                        X_fit
                    ),

                    size=
                        LOF_TRAIN_MAX,

                    replace=False
                )
            )


            X_fit = (
                X_fit.iloc[
                    sample_idx
                ]
            )


        model = (
            LocalOutlierFactor(

                n_neighbors=35,

                novelty=True,

                contamination="auto",

                n_jobs=-1
            )
        )


        model.fit(
            X_fit
        )


    else:

        raise ValueError(
            model_name
        )


    fit_seconds = (
        time.perf_counter()
        -
        start
    )


    # Historical representation branch used
    # -decision_function(), not -score_samples().
    scores_val = (
        -model.decision_function(
            X_val
        )
    )


    scores_test = (
        -model.decision_function(
            X_test
        )
    )


    assert np.isfinite(
        scores_val
    ).all()

    assert np.isfinite(
        scores_test
    ).all()


    return (
        scores_val,
        scores_test,
        fit_seconds
    )


def evaluate_thresholds(
    y_val,
    scores_val,
    y_test,
    scores_test
):

    thresholds = np.unique(
        np.percentile(
            scores_val,
            PERCENTILES
        )
    )


    grid_rows = []


    for threshold in (
        thresholds
    ):

        val_metrics = (
            metric_dict(
                y_val,
                scores_val,
                threshold
            )
        )


        grid_rows.append({

            "threshold":
                float(
                    threshold
                ),

            "val_precision":
                val_metrics[
                    "precision"
                ],

            "val_recall":
                val_metrics[
                    "recall"
                ],

            "val_f1":
                val_metrics[
                    "f1"
                ],

            "val_fpr":
                val_metrics[
                    "fpr"
                ],
        })


    grid = pd.DataFrame(
        grid_rows
    )


    # Exact historical selection:
    # best validation F1, tie-break lower validation FPR.
    best_row = (
        grid
        .sort_values(
            [
                "val_f1",
                "val_fpr",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .iloc[0]
    )


    best_threshold = float(
        best_row[
            "threshold"
        ]
    )


    test_metrics = (
        metric_dict(
            y_test,
            scores_test,
            best_threshold
        )
    )


    test_roc_auc = (
        roc_auc_score(
            y_test,
            scores_test
        )
    )


    test_pr_auc = (
        average_precision_score(
            y_test,
            scores_test
        )
    )


    best_result = {

        "selected_threshold":
            best_threshold,

        "val_precision":
            float(
                best_row[
                    "val_precision"
                ]
            ),

        "val_recall":
            float(
                best_row[
                    "val_recall"
                ]
            ),

        "val_f1":
            float(
                best_row[
                    "val_f1"
                ]
            ),

        "val_fpr":
            float(
                best_row[
                    "val_fpr"
                ]
            ),

        "test_precision":
            test_metrics[
                "precision"
            ],

        "test_recall":
            test_metrics[
                "recall"
            ],

        "test_f1":
            test_metrics[
                "f1"
            ],

        "test_fpr":
            test_metrics[
                "fpr"
            ],

        "test_roc_auc":
            float(
                test_roc_auc
            ),

        "test_pr_auc":
            float(
                test_pr_auc
            ),

        "test_tp":
            test_metrics[
                "tp"
            ],

        "test_fp":
            test_metrics[
                "fp"
            ],

        "test_tn":
            test_metrics[
                "tn"
            ],

        "test_fn":
            test_metrics[
                "fn"
            ],

        "test_flagged_windows":
            test_metrics[
                "flagged_windows"
            ],
    }


    # --------------------------------------------------------
    # Historical FPR-constrained operating points
    # --------------------------------------------------------

    fpr_rows = []


    for limit in (
        FPR_LIMITS
    ):

        allowed = (
            grid[
                grid[
                    "val_fpr"
                ]
                <=
                limit
            ]
            .copy()
        )


        if len(
            allowed
        ) == 0:

            chosen = (
                grid
                .sort_values(
                    [
                        "val_fpr",
                        "val_f1"
                    ],
                    ascending=[
                        True,
                        False
                    ]
                )
                .iloc[0]
            )

            status = (
                "no_feasible_point"
            )


        else:

            chosen = (
                allowed
                .sort_values(
                    [
                        "val_f1",
                        "val_fpr"
                    ],
                    ascending=[
                        False,
                        True
                    ]
                )
                .iloc[0]
            )

            status = (
                "eligible"
            )


        threshold = float(
            chosen[
                "threshold"
            ]
        )


        test_m = (
            metric_dict(
                y_test,
                scores_test,
                threshold
            )
        )


        fpr_rows.append({

            "fpr_limit":
                float(
                    limit
                ),

            "status":
                status,

            "selected_threshold":
                threshold,

            "val_precision":
                float(
                    chosen[
                        "val_precision"
                    ]
                ),

            "val_recall":
                float(
                    chosen[
                        "val_recall"
                    ]
                ),

            "val_f1":
                float(
                    chosen[
                        "val_f1"
                    ]
                ),

            "val_fpr":
                float(
                    chosen[
                        "val_fpr"
                    ]
                ),

            "test_precision":
                test_m[
                    "precision"
                ],

            "test_recall":
                test_m[
                    "recall"
                ],

            "test_f1":
                test_m[
                    "f1"
                ],

            "test_fpr":
                test_m[
                    "fpr"
                ],

            "test_roc_auc":
                float(
                    test_roc_auc
                ),

            "test_pr_auc":
                float(
                    test_pr_auc
                ),
        })


    return (
        best_result,
        pd.DataFrame(
            fpr_rows
        ),
        grid
    )


# ============================================================
# D. Load corrected populations
# ============================================================

EXPECTED_COUNTS = {

    "60/30": {

        "train":
            62096,

        "val":
            20648,

        "test":
            20668,
    },

    "60/15": {

        "train":
            124204,

        "val":
            41235,

        "test":
            41254,
    },
}


EXPECTED_ANOMALIES = {

    "60/30": {

        "train":
            5279,

        "val":
            2150,

        "test":
            2541,
    },

    "60/15": {

        "train":
            10546,

        "val":
            4296,

        "test":
            5068,
    },
}


corrected_populations = {}


print("\n")
print("=" * 132)
print("CORRECTED REPRESENTATION POPULATION AUDIT")
print("=" * 132)


for config_name, path in (
    CORRECTED_PATHS.items()
):

    df = (
        load_corrected_population(

            path,

            EXPECTED_COUNTS[
                config_name
            ],

            EXPECTED_ANOMALIES[
                config_name
            ],
        )
    )


    corrected_populations[
        config_name
    ] = df


    print(
        f"\n{config_name}"
    )


    summary = (
        df.groupby(
            "_split"
        )[
            "label"
        ]
        .agg(
            [
                "count",
                "sum",
                "mean"
            ]
        )
    )


    print(
        summary.to_string()
    )


print(
    "\nPASS: corrected 60/30 and 60/15 "
    "populations match the audited purge."
)


# ============================================================
# E. Run corrected representation-scaling experiment
# ============================================================

best_rows = []

fpr_rows_all = []

threshold_grids = []

runtime_rows = []


for config_name in [
    "60/30",
    "60/15",
]:

    df = (
        corrected_populations[
            config_name
        ]
    )


    y_val = (
        df.loc[
            df[
                "_split"
            ] == "val",
            "label"
        ]
        .astype(int)
        .to_numpy()
    )


    y_test = (
        df.loc[
            df[
                "_split"
            ] == "test",
            "label"
        ]
        .astype(int)
        .to_numpy()
    )


    print("\n")
    print("=" * 132)
    print(
        f"RUNNING CORRECTED REPRESENTATION: "
        f"{config_name}"
    )
    print("=" * 132)


    for variant in (
        VARIANTS
    ):

        print(
            f"\n--- {config_name} | "
            f"{variant} ---"
        )


        X_all = (
            transform_features(
                df,
                variant
            )
        )


        X_all = (
            X_all
            .replace(
                [
                    np.inf,
                    -np.inf
                ],
                np.nan
            )
            .fillna(
                0.0
            )
        )


        X_train = (
            X_all.loc[
                df[
                    "_split"
                ] == "train"
            ]
        )

        X_val = (
            X_all.loc[
                df[
                    "_split"
                ] == "val"
            ]
        )

        X_test = (
            X_all.loc[
                df[
                    "_split"
                ] == "test"
            ]
        )


        assert len(
            X_train
        ) == (
            EXPECTED_COUNTS[
                config_name
            ][
                "train"
            ]
        )


        for model_name in (
            MODELS
        ):

            print(
                f"  {model_name}..."
            )


            (
                scores_val,
                scores_test,
                fit_seconds
            ) = (
                fit_and_score_model(

                    model_name,

                    X_train,

                    X_val,

                    X_test,
                )
            )


            (
                best_result,
                fpr_results,
                grid
            ) = (
                evaluate_thresholds(

                    y_val,

                    scores_val,

                    y_test,

                    scores_test,
                )
            )


            best_rows.append({

                "window_config":
                    config_name,

                "variant":
                    variant,

                "model":
                    model_name,

                **best_result,
            })


            fpr_results.insert(
                0,
                "model",
                model_name
            )

            fpr_results.insert(
                0,
                "variant",
                variant
            )

            fpr_results.insert(
                0,
                "window_config",
                config_name
            )


            fpr_rows_all.append(
                fpr_results
            )


            grid_export = (
                grid.copy()
            )

            grid_export.insert(
                0,
                "model",
                model_name
            )

            grid_export.insert(
                0,
                "variant",
                variant
            )

            grid_export.insert(
                0,
                "window_config",
                config_name
            )


            threshold_grids.append(
                grid_export
            )


            runtime_rows.append({

                "window_config":
                    config_name,

                "variant":
                    variant,

                "model":
                    model_name,

                "fit_seconds":
                    fit_seconds,

                "training_rows":
                    len(
                        X_train
                    ),

                "validation_rows":
                    len(
                        X_val
                    ),

                "test_rows":
                    len(
                        X_test
                    ),
            })


            print(
                f"    selected threshold "
                f"{best_result['selected_threshold']:.9f}"
            )

            print(
                f"    test F1="
                f"{best_result['test_f1']:.6f} "
                f"| FPR="
                f"{best_result['test_fpr']:.6f} "
                f"| ROC="
                f"{best_result['test_roc_auc']:.6f} "
                f"| PR="
                f"{best_result['test_pr_auc']:.6f}"
            )


corrected_best_df = (
    pd.DataFrame(
        best_rows
    )
)


corrected_fpr_df = (
    pd.concat(
        fpr_rows_all,
        ignore_index=True
    )
)


corrected_grid_df = (
    pd.concat(
        threshold_grids,
        ignore_index=True
    )
)


runtime_df = (
    pd.DataFrame(
        runtime_rows
    )
)


assert len(
    corrected_best_df
) == 16

assert len(
    corrected_fpr_df
) == 80


# ============================================================
# F. Final corrected best-F1 representation table
# ============================================================

print("\n")
print("=" * 132)
print("CORRECTED REPRESENTATION-SCALING BEST-F1 RESULTS")
print("=" * 132)


best_view = (
    corrected_best_df[
        [
            "window_config",
            "variant",
            "model",
            "selected_threshold",
            "val_f1",
            "val_fpr",
            "test_precision",
            "test_recall",
            "test_f1",
            "test_fpr",
            "test_roc_auc",
            "test_pr_auc",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "test_fpr"
        ],
        ascending=[
            False,
            True
        ]
    )
)


print(
    best_view.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# G. Corrected validation-FPR <= 0.10 table
# ============================================================

corrected_fpr010 = (
    corrected_fpr_df[
        np.isclose(
            corrected_fpr_df[
                "fpr_limit"
            ],
            0.10
        )
    ]
    .copy()
)


assert len(
    corrected_fpr010
) == 16


print("\n")
print("=" * 132)
print("CORRECTED REPRESENTATION RESULTS UNDER VALIDATION FPR <= 0.10")
print("=" * 132)


print(
    corrected_fpr010[
        [
            "window_config",
            "variant",
            "model",
            "status",
            "selected_threshold",
            "val_f1",
            "val_fpr",
            "test_precision",
            "test_recall",
            "test_f1",
            "test_fpr",
            "test_roc_auc",
            "test_pr_auc",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "test_fpr"
        ],
        ascending=[
            False,
            True
        ]
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# H. Historical vs corrected best-F1 comparison
# ============================================================

historical_best = pd.read_csv(
    HISTORICAL_BEST_PATH,
    low_memory=False
)


assert len(
    historical_best
) == 16


historical_best[
    "model"
] = (
    historical_best[
        "model"
    ]
    .replace(
        {
            "Isolation Forest":
                "IsolationForest",
        }
    )
)


best_comparison = (
    historical_best.merge(

        corrected_best_df,

        on=[
            "window_config",
            "variant",
            "model",
        ],

        how="inner",

        suffixes=(
            "_historical",
            "_corrected"
        ),

        validate="one_to_one"
    )
)


assert len(
    best_comparison
) == 16


for metric in [

    "test_precision",
    "test_recall",
    "test_f1",
    "test_fpr",
    "test_roc_auc",
    "test_pr_auc",
]:

    best_comparison[
        f"{metric}_delta"
    ] = (
        best_comparison[
            f"{metric}_corrected"
        ]
        -
        best_comparison[
            f"{metric}_historical"
        ]
    )


print("\n")
print("=" * 132)
print("HISTORICAL VS CORRECTED REPRESENTATION BEST-F1 DELTAS")
print("=" * 132)


comparison_view = (
    best_comparison[
        [
            "window_config",
            "variant",
            "model",

            "test_f1_historical",
            "test_f1_corrected",
            "test_f1_delta",

            "test_fpr_historical",
            "test_fpr_corrected",
            "test_fpr_delta",

            "test_roc_auc_historical",
            "test_roc_auc_corrected",
            "test_roc_auc_delta",

            "test_pr_auc_historical",
            "test_pr_auc_corrected",
            "test_pr_auc_delta",
        ]
    ]
    .sort_values(
        [
            "test_f1_corrected",
            "test_fpr_corrected"
        ],
        ascending=[
            False,
            True
        ]
    )
)


print(
    comparison_view.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# I. Historical vs corrected FPR<=0.10 comparison
# ============================================================

historical_fpr010 = pd.read_csv(
    HISTORICAL_FPR010_PATH,
    low_memory=False
)


assert len(
    historical_fpr010
) == 16


historical_fpr010[
    "model"
] = (
    historical_fpr010[
        "model"
    ]
    .replace(
        {
            "Isolation Forest":
                "IsolationForest",
        }
    )
)


fpr010_comparison = (
    historical_fpr010.merge(

        corrected_fpr010,

        on=[
            "window_config",
            "variant",
            "model",
        ],

        how="inner",

        suffixes=(
            "_historical",
            "_corrected"
        ),

        validate="one_to_one"
    )
)


assert len(
    fpr010_comparison
) == 16


for metric in [

    "test_precision",
    "test_recall",
    "test_f1",
    "test_fpr",
]:

    fpr010_comparison[
        f"{metric}_delta"
    ] = (
        fpr010_comparison[
            f"{metric}_corrected"
        ]
        -
        fpr010_comparison[
            f"{metric}_historical"
        ]
    )


# ============================================================
# J. Key thesis checkpoint: per-series z-score IF
# ============================================================

key_results = (
    corrected_best_df[
        (
            corrected_best_df[
                "variant"
            ]
            ==
            "per_series_zscore"
        )
        &
        (
            corrected_best_df[
                "model"
            ]
            ==
            "IsolationForest"
        )
    ]
    .copy()
    .sort_values(
        "window_config"
    )
)


assert len(
    key_results
) == 2


print("\n")
print("=" * 132)
print("KEY CORRECTED PER-SERIES Z-SCORE ISOLATION FOREST RESULTS")
print("=" * 132)


print(
    key_results[
        [
            "window_config",
            "selected_threshold",
            "val_f1",
            "val_fpr",
            "test_precision",
            "test_recall",
            "test_f1",
            "test_fpr",
            "test_roc_auc",
            "test_pr_auc",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# K. Ranking stability
# ============================================================

historical_rank = (
    historical_best[
        [
            "window_config",
            "variant",
            "model",
            "test_f1",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "window_config",
            "variant",
            "model",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


corrected_rank = (
    corrected_best_df[
        [
            "window_config",
            "variant",
            "model",
            "test_f1",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "window_config",
            "variant",
            "model",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


historical_top = (
    historical_rank.iloc[
        0
    ]
)


corrected_top = (
    corrected_rank.iloc[
        0
    ]
)


print("\n")
print("=" * 132)
print("REPRESENTATION-BRANCH TOP-CONFIGURATION STABILITY")
print("=" * 132)


print(
    "Historical top:"
)

print(
    f"  {historical_top['window_config']} | "
    f"{historical_top['variant']} | "
    f"{historical_top['model']} | "
    f"F1={historical_top['test_f1']:.9f}"
)


print(
    "\nCorrected top:"
)

print(
    f"  {corrected_top['window_config']} | "
    f"{corrected_top['variant']} | "
    f"{corrected_top['model']} | "
    f"F1={corrected_top['test_f1']:.9f}"
)


top_configuration_unchanged = (

    historical_top[
        "window_config"
    ]
    ==
    corrected_top[
        "window_config"
    ]

    and

    historical_top[
        "variant"
    ]
    ==
    corrected_top[
        "variant"
    ]

    and

    historical_top[
        "model"
    ]
    ==
    corrected_top[
        "model"
    ]
)


print(
    "\nTop configuration unchanged:",
    top_configuration_unchanged
)


# ============================================================
# L. Save correction artifacts
# ============================================================

BEST_PATH = (
    REP_CORRECTION_DIR
    / "representation_scaling_best_f1_results_corrected.csv"
)

FPR_PATH = (
    REP_CORRECTION_DIR
    / "representation_scaling_fpr_constrained_results_corrected.csv"
)

GRID_PATH = (
    REP_CORRECTION_DIR
    / "representation_scaling_threshold_grids_corrected.csv"
)

RUNTIME_PATH = (
    REP_CORRECTION_DIR
    / "representation_scaling_runtime_corrected.csv"
)

BEST_COMPARISON_PATH = (
    REP_CORRECTION_DIR
    / "representation_scaling_historical_vs_corrected.csv"
)

FPR010_COMPARISON_PATH = (
    REP_CORRECTION_DIR
    / "representation_scaling_fpr010_historical_vs_corrected.csv"
)

KEY_RESULTS_PATH = (
    REP_CORRECTION_DIR
    / "key_per_series_zscore_if_results_corrected.csv"
)


corrected_best_df.to_csv(
    BEST_PATH,
    index=False
)


corrected_fpr_df.to_csv(
    FPR_PATH,
    index=False
)


corrected_grid_df.to_csv(
    GRID_PATH,
    index=False
)


runtime_df.to_csv(
    RUNTIME_PATH,
    index=False
)


best_comparison.to_csv(
    BEST_COMPARISON_PATH,
    index=False
)


fpr010_comparison.to_csv(
    FPR010_COMPARISON_PATH,
    index=False
)


key_results.to_csv(
    KEY_RESULTS_PATH,
    index=False
)


# ============================================================
# M. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 42 CHECKPOINT")
print("=" * 132)


print(
    "Corrected representation combinations:",
    len(
        corrected_best_df
    )
)


print(
    "Corrected FPR-constrained rows:",
    len(
        corrected_fpr_df
    )
)


print(
    "60/30 corrected population:",
    len(
        corrected_populations[
            "60/30"
        ]
    )
)


print(
    "60/15 corrected population:",
    len(
        corrected_populations[
            "60/15"
        ]
    )
)


print(
    "Top representation configuration unchanged:",
    top_configuration_unchanged
)


print(
    "\nSaved correction artifacts under:"
)

print(
    REP_CORRECTION_DIR
)


print("\n")
print("=" * 132)
print("CELL 42 COMPLETE")
print("=" * 132)

print(
    "\nThe primary NAB representation-scaling branch has been "
    "reconstructed using the unchanged training populations and "
    "purged validation/test partitions. No neural model was trained "
    "and no historical artifact was overwritten."
)

In [ ]:
# CELL 43 — Correct NAB Feature-Selection Branch on Purged 60/30 and 60/15 Partitions

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 132)
print("NAB FEATURE-SELECTION BRANCH — SPLIT-BOUNDARY CORRECTION")
print("=" * 132)


# ============================================================
# A. Require successful Cell 42 state
# ============================================================

required_objects = [

    "corrected_populations",
    "transform_features",
    "fit_and_score_model",
    "evaluate_thresholds",
    "FEATURE_COLUMNS",
    "CORRECTION",
    "BASE",
]


for object_name in required_objects:

    assert object_name in globals(), (
        f"Missing required Cell 42 object: {object_name}"
    )


assert len(
    corrected_populations[
        "60/30"
    ]
) == 103412

assert len(
    corrected_populations[
        "60/15"
    ]
) == 206693


print(
    "PASS: corrected representation populations from Cell 42 are available."
)


# ============================================================
# B. Paths
# ============================================================

REP_ROOT = (
    BASE
    / "experiments"
    / "EXP_NAB_Representation_Improvement"
)


FEATURE_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "FEATURE_SELECTION_CORRECTION"
)


FEATURE_CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


HISTORICAL_FEATURE_BEST_PATH = (
    REP_ROOT
    / "feature_selection_best_f1_results.csv"
)


HISTORICAL_FEATURE_FPR_PATH = (
    REP_ROOT
    / "feature_selection_fpr_constrained_results.csv"
)


assert HISTORICAL_FEATURE_BEST_PATH.exists()
assert HISTORICAL_FEATURE_FPR_PATH.exists()


# ============================================================
# C. Exact historical feature-subset helpers
# ============================================================

def correlation_prune_features(
    X_train,
    threshold=0.95
):

    corr = (
        X_train.corr().abs()
    )


    upper = corr.where(
        np.triu(
            np.ones(
                corr.shape
            ),
            k=1
        ).astype(
            bool
        )
    )


    mean_corr = (
        corr.mean(
            axis=1
        )
    )


    to_drop = set()


    for col in (
        upper.columns
    ):

        high_corr_with = (
            upper.index[
                upper[col]
                >
                threshold
            ]
            .tolist()
        )


        for row in (
            high_corr_with
        ):

            if (
                row in to_drop
                or
                col in to_drop
            ):

                continue


            if (
                mean_corr[col]
                >=
                mean_corr[row]
            ):

                to_drop.add(
                    col
                )

            else:

                to_drop.add(
                    row
                )


    kept = [

        c
        for c
        in X_train.columns
        if c not in to_drop
    ]


    return (
        kept,
        sorted(
            list(
                to_drop
            )
        )
    )


def near_constant_filter(
    X_train,
    min_std=1e-8
):

    stds = (
        X_train.std()
    )


    kept = (
        stds[
            stds
            >
            min_std
        ]
        .index
        .tolist()
    )


    dropped = (
        stds[
            stds
            <=
            min_std
        ]
        .index
        .tolist()
    )


    return (
        kept,
        dropped
    )


def get_domain_feature_sets(
    all_features
):

    sets = {}


    sets[
        "all_features"
    ] = list(
        all_features
    )


    change_features = [

        "mean_abs_diff",
        "max_abs_diff",
        "std_diff",
        "first_last_diff",
        "slope",
        "peak_count",
    ]


    variability_features = [

        "std",
        "var",
        "iqr",
        "range",
        "mad",
        "skew",
        "kurtosis",
    ]


    level_energy_features = [

        "mean",
        "median",
        "min",
        "max",
        "q25",
        "q75",
        "sum",
        "energy",
        "rms",
    ]


    robust_mixed_features = [

        "mean_abs_diff",
        "max_abs_diff",
        "std_diff",
        "first_last_diff",
        "slope",
        "peak_count",

        "std",
        "iqr",
        "range",
        "mad",

        "energy",
        "rms",
        "skew",
        "kurtosis",
    ]


    compact_features = [

        "mean_abs_diff",
        "std_diff",
        "mad",
        "iqr",
        "range",

        "energy",
        "rms",
        "slope",
        "peak_count",
        "skew",
    ]


    sets[
        "change_only"
    ] = [

        f
        for f
        in change_features
        if f in all_features
    ]


    sets[
        "variability_only"
    ] = [

        f
        for f
        in variability_features
        if f in all_features
    ]


    sets[
        "level_energy_only"
    ] = [

        f
        for f
        in level_energy_features
        if f in all_features
    ]


    sets[
        "robust_mixed"
    ] = [

        f
        for f
        in robust_mixed_features
        if f in all_features
    ]


    sets[
        "compact_domain"
    ] = [

        f
        for f
        in compact_features
        if f in all_features
    ]


    return sets


# ============================================================
# D. Build exact corrected feature subsets separately for each
#    window configuration
# ============================================================

corrected_best_rows = []

corrected_fpr_rows = []

subset_audit_rows = []


for window_config in [

    "60/30",
    "60/15",
]:

    print("\n")
    print("=" * 132)
    print(
        f"CORRECTED FEATURE SELECTION: {window_config}"
    )
    print("=" * 132)


    df = (
        corrected_populations[
            window_config
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Historical winning representation:
    # per-series z-score
    # --------------------------------------------------------

    X_all = (
        transform_features(
            df,
            "per_series_zscore"
        )
    )


    X_all = (
        X_all
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .fillna(
            0.0
        )
    )


    X_train_full = (
        X_all.loc[
            df[
                "_split"
            ]
            ==
            "train"
        ]
    )


    X_val_full = (
        X_all.loc[
            df[
                "_split"
            ]
            ==
            "val"
        ]
    )


    X_test_full = (
        X_all.loc[
            df[
                "_split"
            ]
            ==
            "test"
        ]
    )


    y_val = (
        df.loc[
            df[
                "_split"
            ]
            ==
            "val",
            "label"
        ]
        .astype(int)
        .to_numpy()
    )


    y_test = (
        df.loc[
            df[
                "_split"
            ]
            ==
            "test",
            "label"
        ]
        .astype(int)
        .to_numpy()
    )


    # --------------------------------------------------------
    # Historical label-free candidate subsets
    # --------------------------------------------------------

    subset_dict = (
        get_domain_feature_sets(
            FEATURE_COLUMNS
        )
    )


    non_constant_features, dropped_constant = (
        near_constant_filter(
            X_train_full
        )
    )


    subset_dict[
        "non_constant"
    ] = (
        non_constant_features
    )


    for threshold in [

        0.95,
        0.90,
        0.85,
        0.80,
    ]:

        kept, dropped = (
            correlation_prune_features(

                X_train_full[
                    non_constant_features
                ],

                threshold=
                    threshold,
            )
        )


        subset_dict[
            f"corr_pruned_{threshold}"
        ] = kept


    # --------------------------------------------------------
    # Historical duplicate-subset removal
    # --------------------------------------------------------

    clean_subsets = {}

    seen = set()


    for subset_name, features in (
        subset_dict.items()
    ):

        features = [

            feature
            for feature
            in features
            if feature
            in X_train_full.columns
        ]


        key = tuple(
            features
        )


        if len(
            features
        ) == 0:

            continue


        if key in seen:

            continue


        seen.add(
            key
        )


        clean_subsets[
            subset_name
        ] = features


    print(
        "Unique feature subsets:",
        len(
            clean_subsets
        )
    )


    for subset_name, features in (
        clean_subsets.items()
    ):

        print(
            f"  {subset_name:<22} "
            f"{len(features):>2} features"
        )


        subset_audit_rows.append({

            "window_config":
                window_config,

            "feature_subset":
                subset_name,

            "n_features":
                len(
                    features
                ),

            "features":
                "|".join(
                    features
                ),
        })


    # Historical branch had 40 model/subset/config result rows,
    # i.e. 10 unique subsets × 2 models × 2 windows.
    assert len(
        clean_subsets
    ) == 10, (
        f"{window_config}: expected 10 unique subsets, "
        f"found {len(clean_subsets)}"
    )


    # --------------------------------------------------------
    # E. Fit/evaluate exact historical models
    # --------------------------------------------------------

    for subset_name, features in (
        clean_subsets.items()
    ):

        print(
            f"\n--- {subset_name} "
            f"({len(features)} features) ---"
        )


        X_train = (
            X_train_full[
                features
            ]
        )


        X_val = (
            X_val_full[
                features
            ]
        )


        X_test = (
            X_test_full[
                features
            ]
        )


        for model_name in [

            "IsolationForest",
            "LOF",
        ]:

            print(
                f"  {model_name}..."
            )


            (
                scores_val,
                scores_test,
                fit_seconds
            ) = (
                fit_and_score_model(

                    model_name,

                    X_train,

                    X_val,

                    X_test,
                )
            )


            (
                best_result,
                fpr_df,
                grid_df
            ) = (
                evaluate_thresholds(

                    y_val,

                    scores_val,

                    y_test,

                    scores_test,
                )
            )


            corrected_best_rows.append({

                "window_config":
                    window_config,

                "representation":
                    "per_series_zscore",

                "feature_subset":
                    subset_name,

                "n_features":
                    len(
                        features
                    ),

                "features":
                    "|".join(
                        features
                    ),

                "model":
                    model_name,

                "fit_seconds":
                    fit_seconds,

                **best_result,
            })


            fpr_df.insert(
                0,
                "model",
                model_name
            )


            fpr_df.insert(
                0,
                "features",
                "|".join(
                    features
                )
            )


            fpr_df.insert(
                0,
                "n_features",
                len(
                    features
                )
            )


            fpr_df.insert(
                0,
                "feature_subset",
                subset_name
            )


            fpr_df.insert(
                0,
                "representation",
                "per_series_zscore"
            )


            fpr_df.insert(
                0,
                "window_config",
                window_config
            )


            corrected_fpr_rows.append(
                fpr_df
            )


            print(
                f"    test F1="
                f"{best_result['test_f1']:.6f} "
                f"| FPR="
                f"{best_result['test_fpr']:.6f}"
            )


# ============================================================
# F. Assemble corrected outputs
# ============================================================

corrected_feature_best = (
    pd.DataFrame(
        corrected_best_rows
    )
)


corrected_feature_fpr = (
    pd.concat(
        corrected_fpr_rows,
        ignore_index=True
    )
)


subset_audit_df = (
    pd.DataFrame(
        subset_audit_rows
    )
)


assert len(
    corrected_feature_best
) == 40


assert len(
    corrected_feature_fpr
) == 200


print("\n")
print("=" * 132)
print("CORRECTED FEATURE-SELECTION BEST-F1 RESULTS")
print("=" * 132)


corrected_feature_best_view = (
    corrected_feature_best[
        [
            "window_config",
            "feature_subset",
            "n_features",
            "model",
            "selected_threshold",

            "val_f1",
            "val_fpr",

            "test_precision",
            "test_recall",
            "test_f1",
            "test_fpr",

            "test_roc_auc",
            "test_pr_auc",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "test_fpr"
        ],
        ascending=[
            False,
            True
        ]
    )
)


print(
    corrected_feature_best_view
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# G. Corrected FPR <= 0.10 results
# ============================================================

corrected_feature_fpr010 = (
    corrected_feature_fpr[
        np.isclose(
            corrected_feature_fpr[
                "fpr_limit"
            ],
            0.10
        )
    ]
    .copy()
)


assert len(
    corrected_feature_fpr010
) == 40


print("\n")
print("=" * 132)
print("CORRECTED FEATURE-SELECTION RESULTS UNDER VALIDATION FPR <= 0.10")
print("=" * 132)


print(
    corrected_feature_fpr010[
        [
            "window_config",
            "feature_subset",
            "n_features",
            "model",
            "status",
            "selected_threshold",

            "val_f1",
            "val_fpr",

            "test_precision",
            "test_recall",
            "test_f1",
            "test_fpr",

            "test_roc_auc",
            "test_pr_auc",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "test_fpr"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# H. Historical comparison
# ============================================================

historical_feature_best = (
    pd.read_csv(
        HISTORICAL_FEATURE_BEST_PATH,
        low_memory=False
    )
)


historical_feature_fpr = (
    pd.read_csv(
        HISTORICAL_FEATURE_FPR_PATH,
        low_memory=False
    )
)


print("\nHistorical feature-selection shapes:")

print(
    "  Best-F1:",
    historical_feature_best.shape
)

print(
    "  FPR constrained:",
    historical_feature_fpr.shape
)


assert len(
    historical_feature_best
) == 40


assert len(
    historical_feature_fpr
) == 200


# Normalize possible historical naming.
if (
    "model"
    in historical_feature_best.columns
):

    historical_feature_best[
        "model"
    ] = (
        historical_feature_best[
            "model"
        ]
        .replace(
            {
                "Isolation Forest":
                    "IsolationForest",
            }
        )
    )


if (
    "model"
    in historical_feature_fpr.columns
):

    historical_feature_fpr[
        "model"
    ] = (
        historical_feature_fpr[
            "model"
        ]
        .replace(
            {
                "Isolation Forest":
                    "IsolationForest",
            }
        )
    )


feature_best_comparison = (
    historical_feature_best.merge(

        corrected_feature_best,

        on=[
            "window_config",
            "representation",
            "feature_subset",
            "model",
        ],

        how="inner",

        suffixes=(
            "_historical",
            "_corrected"
        ),

        validate="one_to_one"
    )
)


assert len(
    feature_best_comparison
) == 40


for metric in [

    "test_precision",
    "test_recall",
    "test_f1",
    "test_fpr",
    "test_roc_auc",
    "test_pr_auc",
]:

    feature_best_comparison[
        f"{metric}_delta"
    ] = (
        feature_best_comparison[
            f"{metric}_corrected"
        ]
        -
        feature_best_comparison[
            f"{metric}_historical"
        ]
    )


print("\n")
print("=" * 132)
print("HISTORICAL VS CORRECTED FEATURE-SELECTION TOP DELTAS")
print("=" * 132)


comparison_view = (
    feature_best_comparison[
        [
            "window_config",
            "feature_subset",
            "model",

            "test_f1_historical",
            "test_f1_corrected",
            "test_f1_delta",

            "test_fpr_historical",
            "test_fpr_corrected",
            "test_fpr_delta",

            "test_roc_auc_historical",
            "test_roc_auc_corrected",
            "test_roc_auc_delta",

            "test_pr_auc_historical",
            "test_pr_auc_corrected",
            "test_pr_auc_delta",
        ]
    ]
    .sort_values(
        [
            "test_f1_corrected",
            "test_fpr_corrected"
        ],
        ascending=[
            False,
            True
        ]
    )
)


print(
    comparison_view
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# I. Top feature-subset stability
# ============================================================

historical_rank = (
    historical_feature_best[
        [
            "window_config",
            "feature_subset",
            "model",
            "test_f1",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "window_config",
            "feature_subset",
            "model",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


corrected_rank = (
    corrected_feature_best[
        [
            "window_config",
            "feature_subset",
            "model",
            "test_f1",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "window_config",
            "feature_subset",
            "model",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


historical_top = (
    historical_rank.iloc[
        0
    ]
)


corrected_top = (
    corrected_rank.iloc[
        0
    ]
)


top_unchanged = (

    historical_top[
        "window_config"
    ]
    ==
    corrected_top[
        "window_config"
    ]

    and

    historical_top[
        "feature_subset"
    ]
    ==
    corrected_top[
        "feature_subset"
    ]

    and

    historical_top[
        "model"
    ]
    ==
    corrected_top[
        "model"
    ]
)


print("\n")
print("=" * 132)
print("FEATURE-SELECTION TOP-CONFIGURATION STABILITY")
print("=" * 132)


print(
    "Historical top:"
)

print(
    f"  {historical_top['window_config']} | "
    f"{historical_top['feature_subset']} | "
    f"{historical_top['model']} | "
    f"F1={historical_top['test_f1']:.9f}"
)


print(
    "\nCorrected top:"
)

print(
    f"  {corrected_top['window_config']} | "
    f"{corrected_top['feature_subset']} | "
    f"{corrected_top['model']} | "
    f"F1={corrected_top['test_f1']:.9f}"
)


print(
    "\nTop configuration unchanged:",
    top_unchanged
)


# ============================================================
# J. Isolation-Forest-only thesis-facing checkpoint
# ============================================================

corrected_if_top = (
    corrected_feature_best[
        corrected_feature_best[
            "model"
        ]
        ==
        "IsolationForest"
    ]
    .sort_values(
        [
            "test_f1",
            "test_fpr"
        ],
        ascending=[
            False,
            True
        ]
    )
    .head(10)
)


print("\n")
print("=" * 132)
print("TOP CORRECTED FEATURE-SELECTION ISOLATION FOREST RESULTS")
print("=" * 132)


print(
    corrected_if_top[
        [
            "window_config",
            "feature_subset",
            "n_features",
            "selected_threshold",

            "test_precision",
            "test_recall",
            "test_f1",
            "test_fpr",

            "test_roc_auc",
            "test_pr_auc",
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# K. Save correction artifacts
# ============================================================

BEST_PATH = (
    FEATURE_CORRECTION_DIR
    / "feature_selection_best_f1_results_corrected.csv"
)


FPR_PATH = (
    FEATURE_CORRECTION_DIR
    / "feature_selection_fpr_constrained_results_corrected.csv"
)


SUBSET_PATH = (
    FEATURE_CORRECTION_DIR
    / "feature_selection_subset_audit_corrected.csv"
)


COMPARISON_PATH = (
    FEATURE_CORRECTION_DIR
    / "feature_selection_historical_vs_corrected.csv"
)


FPR010_PATH = (
    FEATURE_CORRECTION_DIR
    / "feature_selection_fpr010_corrected.csv"
)


corrected_feature_best.to_csv(
    BEST_PATH,
    index=False
)


corrected_feature_fpr.to_csv(
    FPR_PATH,
    index=False
)


subset_audit_df.to_csv(
    SUBSET_PATH,
    index=False
)


feature_best_comparison.to_csv(
    COMPARISON_PATH,
    index=False
)


corrected_feature_fpr010.to_csv(
    FPR010_PATH,
    index=False
)


# ============================================================
# L. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 43 CHECKPOINT")
print("=" * 132)


print(
    "Corrected feature-selection Best-F1 rows:",
    len(
        corrected_feature_best
    )
)


print(
    "Corrected FPR-constrained rows:",
    len(
        corrected_feature_fpr
    )
)


print(
    "60/30 unique feature subsets:",
    int(
        subset_audit_df[
            subset_audit_df[
                "window_config"
            ]
            ==
            "60/30"
        ][
            "feature_subset"
        ]
        .nunique()
    )
)


print(
    "60/15 unique feature subsets:",
    int(
        subset_audit_df[
            subset_audit_df[
                "window_config"
            ]
            ==
            "60/15"
        ][
            "feature_subset"
        ]
        .nunique()
    )
)


print(
    "Top feature-selection configuration unchanged:",
    top_unchanged
)


print(
    "\nSaved correction artifacts under:"
)

print(
    FEATURE_CORRECTION_DIR
)


print("\n")
print("=" * 132)
print("CELL 43 COMPLETE")
print("=" * 132)

print(
    "\nThe NAB feature-selection branch has been corrected "
    "using unchanged training populations and purged validation/test "
    "partitions. No neural model was trained and no historical "
    "artifact was overwritten."
)

In [ ]:
# CELL 44 — Correct NAB Robust-Feature Branch on Purged 60/30 and 60/15 Partitions

from pathlib import Path
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)


print("=" * 132)
print("NAB ROBUST-FEATURE BRANCH — SPLIT-BOUNDARY CORRECTION")
print("=" * 132)


# ============================================================
# A. Paths
# ============================================================

ROBUST_PROCESSED_DIR = (
    BASE
    / "processed_data"
    / "PH4"
    / "EXP_NAB_Robust_Window_Features"
)

ROBUST_EXP_DIR = (
    BASE
    / "experiments"
    / "EXP_NAB_Robust_Window_Features"
)

ROBUST_CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "ROBUST_FEATURE_CORRECTION"
)

ROBUST_CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ROBUST_INPUT_PATHS = {

    "60/30":
        ROBUST_PROCESSED_DIR
        / "nab_robust_template_windows_60min_stride30.csv",

    "60/15":
        ROBUST_PROCESSED_DIR
        / "nab_robust_template_windows_60min_stride15.csv",
}


HISTORICAL_BEST_PATH = (
    ROBUST_EXP_DIR
    / "robust_feature_best_f1_results.csv"
)

HISTORICAL_FPR_PATH = (
    ROBUST_EXP_DIR
    / "robust_feature_fpr_constrained_results.csv"
)


for path in [
    *ROBUST_INPUT_PATHS.values(),
    HISTORICAL_BEST_PATH,
    HISTORICAL_FPR_PATH,
]:

    assert path.exists(), (
        f"Missing required artifact: {path}"
    )


assert "corrected_populations" in globals()


# ============================================================
# B. Exact historical constants
# ============================================================

RANDOM_STATE = 42
LOF_TRAIN_MAX = 60000

PERCENTILES = np.arange(
    50,
    98,
    2.5
)

FPR_LIMITS = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
]


ORIGINAL_23 = [

    "energy",
    "first_last_diff",
    "iqr",
    "kurtosis",
    "mad",
    "max",
    "max_abs_diff",
    "mean",
    "mean_abs_diff",
    "median",
    "min",
    "n_points",
    "peak_count",
    "q25",
    "q75",
    "range",
    "rms",
    "skew",
    "slope",
    "std",
    "std_diff",
    "sum",
    "var",
]


ROBUST_COMPACT_REQUESTED = [

    "mean_abs_diff",
    "max_abs_diff",
    "std_diff",
    "first_last_diff",
    "slope",
    "peak_count",
    "mad",
    "iqr",
    "range",
    "energy",
    "rms",

    "robust_z_max",
    "robust_z_mean",
    "robust_z_q90",
    "robust_z_q95",
    "robust_outlier_count_z3",
    "robust_outlier_ratio_z3",

    "delta_abs_q90",
    "delta_abs_q95",
    "delta_abs_iqr",

    "positive_jump_count",
    "negative_jump_count",
    "sign_change_count",

    "local_dev_mean",
    "local_dev_max",

    "trend_strength",
    "normalized_range_mad",
    "entropy",
    "autocorr_lag1",
]


# ============================================================
# C. Attach already-audited corrected split identities to the
#    robust-feature source tables
# ============================================================

def load_and_purge_robust_population(
    window_config,
    robust_path
):

    robust_df = pd.read_csv(
        robust_path,
        low_memory=False
    )


    print(
        f"\n{window_config} original robust rows:",
        len(robust_df)
    )


    expected_original = (
        103526
        if window_config == "60/30"
        else 207033
    )

    expected_corrected = (
        103412
        if window_config == "60/30"
        else 206693
    )


    assert len(
        robust_df
    ) == expected_original


    for time_col in [
        "window_start",
        "window_end",
    ]:

        assert time_col in robust_df.columns

        robust_df[
            time_col
        ] = pd.to_datetime(
            robust_df[
                time_col
            ],
            utc=True,
            errors="raise"
        )


    # We do not trust/reuse a historical split column here.
    if "_split" in robust_df.columns:

        robust_df = robust_df.drop(
            columns=[
                "_split"
            ]
        )


    corrected_identity = (
        corrected_populations[
            window_config
        ][
            [
                "series",
                "window_start",
                "window_end",
                "_split",
            ]
        ]
        .copy()
    )


    assert not corrected_identity.duplicated(
        subset=[
            "series",
            "window_start",
            "window_end",
        ]
    ).any()


    assert not robust_df.duplicated(
        subset=[
            "series",
            "window_start",
            "window_end",
        ]
    ).any()


    corrected_robust = robust_df.merge(

        corrected_identity,

        on=[
            "series",
            "window_start",
            "window_end",
        ],

        how="inner",

        validate="one_to_one"
    )


    assert len(
        corrected_robust
    ) == expected_corrected


    # Preserve historical chronological ordering.
    corrected_robust = (
        corrected_robust
        .sort_values(
            [
                "series",
                "window_start"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    expected_counts = {

        "60/30": {
            "train": 62096,
            "val": 20648,
            "test": 20668,
        },

        "60/15": {
            "train": 124204,
            "val": 41235,
            "test": 41254,
        },
    }


    expected_anomalies = {

        "60/30": {
            "train": 5279,
            "val": 2150,
            "test": 2541,
        },

        "60/15": {
            "train": 10546,
            "val": 4296,
            "test": 5068,
        },
    }


    observed_counts = (
        corrected_robust[
            "_split"
        ]
        .value_counts()
        .to_dict()
    )


    observed_anomalies = (
        corrected_robust.groupby(
            "_split"
        )[
            "label"
        ]
        .sum()
        .astype(int)
        .to_dict()
    )


    for split_name in [
        "train",
        "val",
        "test",
    ]:

        assert int(
            observed_counts[
                split_name
            ]
        ) == expected_counts[
            window_config
        ][
            split_name
        ]

        assert int(
            observed_anomalies[
                split_name
            ]
        ) == expected_anomalies[
            window_config
        ][
            split_name
        ]


    removed = (
        expected_original
        -
        len(
            corrected_robust
        )
    )


    print(
        f"{window_config} removed by audited purge:",
        removed
    )


    return corrected_robust


# ============================================================
# D. Exact historical feature detection
# ============================================================

def detect_robust_feature_columns(
    df
):

    excluded = {

        "series",
        "domain",
        "window_start",
        "window_end",
        "label",
        "_split",
        "window_id",
    }


    return [

        c
        for c
        in df.columns

        if (
            c not in excluded
            and
            pd.api.types.is_numeric_dtype(
                df[c]
            )
        )
    ]


# ============================================================
# E. Exact historical per-series z-score transformation
# ============================================================

def robust_per_series_zscore(
    df,
    feature_cols
):

    X = (
        df[
            feature_cols
        ]
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .fillna(
            0.0
        )
        .copy()
    )


    X_out = pd.DataFrame(
        index=df.index,
        columns=feature_cols,
        dtype=float
    )


    for series_name, idx_all in (
        df.groupby(
            "series",
            sort=False
        ).groups.items()
    ):

        idx_all = list(
            idx_all
        )


        idx_train = (
            df.loc[
                idx_all
            ]
            .index[
                df.loc[
                    idx_all,
                    "_split"
                ]
                ==
                "train"
            ]
            .tolist()
        )


        train_values = (
            X.loc[
                idx_train,
                feature_cols
            ]
        )


        means = (
            train_values.mean()
        )


        stds = (
            train_values.std()
            .replace(
                0,
                1.0
            )
            .fillna(
                1.0
            )
        )


        X_out.loc[
            idx_all,
            feature_cols
        ] = (
            X.loc[
                idx_all,
                feature_cols
            ]
            -
            means
        ) / stds


    return (
        X_out
        .replace(
            [
                np.inf,
                -np.inf
            ],
            np.nan
        )
        .fillna(
            0.0
        )
    )


# ============================================================
# F. Exact historical model fitting/scoring
# ============================================================

def robust_fit_score_model(
    model_name,
    X_train,
    X_val,
    X_test
):

    start = time.perf_counter()


    if model_name == "IsolationForest":

        model = IsolationForest(

            n_estimators=300,

            contamination="auto",

            random_state=42,

            n_jobs=-1
        )


        model.fit(
            X_train
        )


    elif model_name == "LOF":

        if len(
            X_train
        ) > LOF_TRAIN_MAX:

            # Exact historical Cell 522 method:
            X_train_fit = (
                X_train.sample(
                    LOF_TRAIN_MAX,
                    random_state=42
                )
            )

        else:

            X_train_fit = (
                X_train
            )


        model = LocalOutlierFactor(

            n_neighbors=35,

            novelty=True,

            contamination="auto",

            n_jobs=-1
        )


        model.fit(
            X_train_fit
        )


    else:

        raise ValueError(
            model_name
        )


    fit_seconds = (
        time.perf_counter()
        -
        start
    )


    scores_val = (
        -model.decision_function(
            X_val
        )
    )


    scores_test = (
        -model.decision_function(
            X_test
        )
    )


    assert np.isfinite(
        scores_val
    ).all()

    assert np.isfinite(
        scores_test
    ).all()


    return (
        scores_val,
        scores_test,
        fit_seconds
    )


# ============================================================
# G. Exact metric helper
# ============================================================

def robust_compute_metrics(
    y_true,
    y_pred,
    scores
):

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )


    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )


    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1
        ]
    ).ravel()


    fpr = (
        fp
        /
        (
            fp + tn
        )
        if (
            fp + tn
        ) > 0
        else 0.0
    )


    return {

        "test_precision":
            float(
                precision
            ),

        "test_recall":
            float(
                recall
            ),

        "test_f1":
            float(
                f1
            ),

        "test_fpr":
            float(
                fpr
            ),

        "test_roc_auc":
            float(
                roc_auc_score(
                    y_true,
                    scores
                )
            ),

        "test_pr_auc":
            float(
                average_precision_score(
                    y_true,
                    scores
                )
            ),

        "test_tp":
            int(tp),

        "test_fp":
            int(fp),

        "test_tn":
            int(tn),

        "test_fn":
            int(fn),
    }


# ============================================================
# H. Exact historical threshold-grid selection
# ============================================================

def robust_evaluate_threshold_grid(
    y_val,
    scores_val,
    y_test,
    scores_test
):

    grid_rows = []

    best = None


    for percentile in (
        PERCENTILES
    ):

        threshold = float(
            np.percentile(
                scores_val,
                percentile
            )
        )


        pred_val = (
            scores_val
            >=
            threshold
        ).astype(
            np.int8
        )


        pred_test = (
            scores_test
            >=
            threshold
        ).astype(
            np.int8
        )


        val_precision = precision_score(
            y_val,
            pred_val,
            zero_division=0
        )


        val_recall = recall_score(
            y_val,
            pred_val,
            zero_division=0
        )


        val_f1 = f1_score(
            y_val,
            pred_val,
            zero_division=0
        )


        tn, fp, fn, tp = confusion_matrix(
            y_val,
            pred_val,
            labels=[
                0,
                1
            ]
        ).ravel()


        val_fpr = (
            fp
            /
            (
                fp + tn
            )
            if (
                fp + tn
            ) > 0
            else 0.0
        )


        test_metrics = (
            robust_compute_metrics(
                y_test,
                pred_test,
                scores_test
            )
        )


        row = {

            "percentile":
                float(
                    percentile
                ),

            "selected_threshold":
                threshold,

            "val_precision":
                float(
                    val_precision
                ),

            "val_recall":
                float(
                    val_recall
                ),

            "val_f1":
                float(
                    val_f1
                ),

            "val_fpr":
                float(
                    val_fpr
                ),

            **test_metrics,
        }


        grid_rows.append(
            row
        )


        # Exact historical behavior:
        # only strictly greater validation F1 replaces the best.
        if (
            best is None
            or
            val_f1
            >
            best[
                "val_f1"
            ]
        ):

            best = (
                row.copy()
            )


    return (
        best,
        pd.DataFrame(
            grid_rows
        )
    )


def robust_evaluate_fpr_constrained(
    y_val,
    scores_val,
    y_test,
    scores_test
):

    candidate_rows = []


    for percentile in (
        PERCENTILES
    ):

        threshold = float(
            np.percentile(
                scores_val,
                percentile
            )
        )


        pred_val = (
            scores_val
            >=
            threshold
        ).astype(
            np.int8
        )


        pred_test = (
            scores_test
            >=
            threshold
        ).astype(
            np.int8
        )


        val_precision = precision_score(
            y_val,
            pred_val,
            zero_division=0
        )


        val_recall = recall_score(
            y_val,
            pred_val,
            zero_division=0
        )


        val_f1 = f1_score(
            y_val,
            pred_val,
            zero_division=0
        )


        tn, fp, fn, tp = confusion_matrix(
            y_val,
            pred_val,
            labels=[
                0,
                1
            ]
        ).ravel()


        val_fpr = (
            fp
            /
            (
                fp + tn
            )
            if (
                fp + tn
            ) > 0
            else 0.0
        )


        test_metrics = (
            robust_compute_metrics(
                y_test,
                pred_test,
                scores_test
            )
        )


        candidate_rows.append({

            "percentile":
                float(
                    percentile
                ),

            "selected_threshold":
                threshold,

            "val_precision":
                float(
                    val_precision
                ),

            "val_recall":
                float(
                    val_recall
                ),

            "val_f1":
                float(
                    val_f1
                ),

            "val_fpr":
                float(
                    val_fpr
                ),

            **test_metrics,
        })


    candidates = pd.DataFrame(
        candidate_rows
    )


    selected_rows = []


    for limit in (
        FPR_LIMITS
    ):

        feasible = (
            candidates[
                candidates[
                    "val_fpr"
                ]
                <=
                limit
            ]
            .copy()
        )


        if len(
            feasible
        ) == 0:

            chosen = (
                candidates
                .sort_values(
                    "val_fpr"
                )
                .iloc[0]
                .copy()
            )

            feasible_flag = False

        else:

            chosen = (
                feasible
                .sort_values(
                    [
                        "val_f1",
                        "val_fpr"
                    ],
                    ascending=[
                        False,
                        True
                    ]
                )
                .iloc[0]
                .copy()
            )

            feasible_flag = True


        chosen[
            "fpr_limit"
        ] = float(
            limit
        )


        chosen[
            "feasible_on_val"
        ] = feasible_flag


        selected_rows.append(
            chosen
        )


    return pd.DataFrame(
        selected_rows
    )


# ============================================================
# I. Correct both robust populations
# ============================================================

robust_corrected = {}


print("\n")
print("=" * 132)
print("CORRECTED ROBUST POPULATION AUDIT")
print("=" * 132)


for window_config, path in (
    ROBUST_INPUT_PATHS.items()
):

    robust_corrected[
        window_config
    ] = (
        load_and_purge_robust_population(
            window_config,
            path
        )
    )


    print(
        robust_corrected[
            window_config
        ]
        .groupby(
            "_split"
        )[
            "label"
        ]
        .agg(
            [
                "count",
                "sum",
                "mean"
            ]
        )
        .to_string()
    )


# ============================================================
# J. Run exact robust-feature experiment
# ============================================================

best_rows = []

fpr_rows = []

feature_set_audit_rows = []


for window_config in [
    "60/30",
    "60/15",
]:

    df = (
        robust_corrected[
            window_config
        ]
    )


    all_features = (
        detect_robust_feature_columns(
            df
        )
    )


    original_23 = [

        f
        for f
        in ORIGINAL_23
        if f in all_features
    ]


    robust_new = [

        f
        for f
        in all_features
        if f not in original_23
    ]


    robust_compact = [

        f
        for f
        in ROBUST_COMPACT_REQUESTED
        if f in all_features
    ]


    feature_sets = {

        "original_23_recomputed":
            original_23,

        "robust_new_only":
            robust_new,

        "original_plus_robust_all":
            original_23
            +
            robust_new,

        "robust_compact":
            robust_compact,
    }


    print("\n")
    print("=" * 132)
    print(
        f"ROBUST FEATURE SETS: {window_config}"
    )
    print("=" * 132)


    print(
        "All numeric features:",
        len(
            all_features
        )
    )

    print(
        "Original 23:",
        len(
            original_23
        )
    )

    print(
        "Robust new only:",
        len(
            robust_new
        )
    )

    print(
        "Original + robust:",
        len(
            original_23
            +
            robust_new
        )
    )

    print(
        "Robust compact:",
        len(
            robust_compact
        )
    )


    for feature_set_name, features in (
        feature_sets.items()
    ):

        assert len(
            features
        ) > 0


        feature_set_audit_rows.append({

            "window_config":
                window_config,

            "feature_set":
                feature_set_name,

            "n_features":
                len(
                    features
                ),

            "features":
                "|".join(
                    features
                ),
        })


        print(
            f"\n--- {feature_set_name} "
            f"({len(features)} features) ---"
        )


        X_all = (
            robust_per_series_zscore(
                df,
                features
            )
        )


        X_train = (
            X_all.loc[
                df[
                    "_split"
                ]
                ==
                "train"
            ]
        )


        X_val = (
            X_all.loc[
                df[
                    "_split"
                ]
                ==
                "val"
            ]
        )


        X_test = (
            X_all.loc[
                df[
                    "_split"
                ]
                ==
                "test"
            ]
        )


        y_val = (
            df.loc[
                df[
                    "_split"
                ]
                ==
                "val",
                "label"
            ]
            .astype(int)
            .to_numpy()
        )


        y_test = (
            df.loc[
                df[
                    "_split"
                ]
                ==
                "test",
                "label"
            ]
            .astype(int)
            .to_numpy()
        )


        for model_name in [
            "IsolationForest",
            "LOF",
        ]:

            print(
                f"  {model_name}..."
            )


            (
                scores_val,
                scores_test,
                fit_seconds
            ) = (
                robust_fit_score_model(

                    model_name,

                    X_train,

                    X_val,

                    X_test,
                )
            )


            best, grid = (
                robust_evaluate_threshold_grid(

                    y_val,

                    scores_val,

                    y_test,

                    scores_test,
                )
            )


            fpr_df = (
                robust_evaluate_fpr_constrained(

                    y_val,

                    scores_val,

                    y_test,

                    scores_test,
                )
            )


            best_rows.append({

                "window_config":
                    window_config,

                "normalization":
                    "per_series_zscore",

                "feature_set":
                    feature_set_name,

                "n_features":
                    len(
                        features
                    ),

                "model":
                    model_name,

                "fit_seconds":
                    fit_seconds,

                **best,
            })


            fpr_df.insert(
                0,
                "model",
                model_name
            )


            fpr_df.insert(
                0,
                "n_features",
                len(
                    features
                )
            )


            fpr_df.insert(
                0,
                "feature_set",
                feature_set_name
            )


            fpr_df.insert(
                0,
                "normalization",
                "per_series_zscore"
            )


            fpr_df.insert(
                0,
                "window_config",
                window_config
            )


            fpr_rows.append(
                fpr_df
            )


            print(
                f"    test F1="
                f"{best['test_f1']:.6f} "
                f"| FPR="
                f"{best['test_fpr']:.6f} "
                f"| ROC="
                f"{best['test_roc_auc']:.6f} "
                f"| PR="
                f"{best['test_pr_auc']:.6f}"
            )


# ============================================================
# K. Assemble outputs
# ============================================================

corrected_robust_best = (
    pd.DataFrame(
        best_rows
    )
)


corrected_robust_fpr = (
    pd.concat(
        fpr_rows,
        ignore_index=True
    )
)


feature_set_audit = (
    pd.DataFrame(
        feature_set_audit_rows
    )
)


assert len(
    corrected_robust_best
) == 16


assert len(
    corrected_robust_fpr
) == 80


print("\n")
print("=" * 132)
print("CORRECTED ROBUST-FEATURE BEST-F1 RESULTS")
print("=" * 132)


robust_best_view = (
    corrected_robust_best[
        [
            "window_config",
            "feature_set",
            "n_features",
            "model",
            "percentile",
            "selected_threshold",

            "val_f1",
            "val_fpr",

            "test_precision",
            "test_recall",
            "test_f1",
            "test_fpr",

            "test_roc_auc",
            "test_pr_auc",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "test_fpr"
        ],
        ascending=[
            False,
            True
        ]
    )
)


print(
    robust_best_view.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# L. FPR <= 0.10 corrected view
# ============================================================

corrected_robust_fpr010 = (
    corrected_robust_fpr[
        np.isclose(
            corrected_robust_fpr[
                "fpr_limit"
            ],
            0.10
        )
    ]
    .copy()
)


assert len(
    corrected_robust_fpr010
) == 16


print("\n")
print("=" * 132)
print("CORRECTED ROBUST FEATURES UNDER VALIDATION FPR <= 0.10")
print("=" * 132)


print(
    corrected_robust_fpr010[
        [
            "window_config",
            "feature_set",
            "n_features",
            "model",
            "feasible_on_val",
            "percentile",
            "selected_threshold",

            "val_f1",
            "val_fpr",

            "test_precision",
            "test_recall",
            "test_f1",
            "test_fpr",

            "test_roc_auc",
            "test_pr_auc",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "test_fpr"
        ],
        ascending=[
            False,
            True
        ]
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# M. Historical comparison
# ============================================================

historical_best = pd.read_csv(
    HISTORICAL_BEST_PATH,
    low_memory=False
)


historical_fpr = pd.read_csv(
    HISTORICAL_FPR_PATH,
    low_memory=False
)


print("\nHistorical robust shapes:")

print(
    "  Best-F1:",
    historical_best.shape
)

print(
    "  FPR constrained:",
    historical_fpr.shape
)


assert len(
    historical_best
) == 16


assert len(
    historical_fpr
) == 80


for hist_df in [
    historical_best,
    historical_fpr,
]:

    if "model" in hist_df.columns:

        hist_df[
            "model"
        ] = (
            hist_df[
                "model"
            ]
            .replace(
                {
                    "Isolation Forest":
                        "IsolationForest",
                }
            )
        )


robust_comparison = (
    historical_best.merge(

        corrected_robust_best,

        on=[
            "window_config",
            "normalization",
            "feature_set",
            "model",
        ],

        how="inner",

        suffixes=(
            "_historical",
            "_corrected"
        ),

        validate="one_to_one"
    )
)


assert len(
    robust_comparison
) == 16


for metric in [

    "test_precision",
    "test_recall",
    "test_f1",
    "test_fpr",
    "test_roc_auc",
    "test_pr_auc",
]:

    robust_comparison[
        f"{metric}_delta"
    ] = (
        robust_comparison[
            f"{metric}_corrected"
        ]
        -
        robust_comparison[
            f"{metric}_historical"
        ]
    )


print("\n")
print("=" * 132)
print("HISTORICAL VS CORRECTED ROBUST-FEATURE DELTAS")
print("=" * 132)


print(
    robust_comparison[
        [
            "window_config",
            "feature_set",
            "model",

            "test_f1_historical",
            "test_f1_corrected",
            "test_f1_delta",

            "test_fpr_historical",
            "test_fpr_corrected",
            "test_fpr_delta",

            "test_roc_auc_historical",
            "test_roc_auc_corrected",
            "test_roc_auc_delta",

            "test_pr_auc_historical",
            "test_pr_auc_corrected",
            "test_pr_auc_delta",
        ]
    ]
    .sort_values(
        [
            "test_f1_corrected",
            "test_fpr_corrected"
        ],
        ascending=[
            False,
            True
        ]
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# N. Top-configuration stability
# ============================================================

historical_rank = (
    historical_best[
        [
            "window_config",
            "feature_set",
            "model",
            "test_f1",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "window_config",
            "feature_set",
            "model",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


corrected_rank = (
    corrected_robust_best[
        [
            "window_config",
            "feature_set",
            "model",
            "test_f1",
        ]
    ]
    .sort_values(
        [
            "test_f1",
            "window_config",
            "feature_set",
            "model",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


historical_top = (
    historical_rank.iloc[
        0
    ]
)


corrected_top = (
    corrected_rank.iloc[
        0
    ]
)


top_unchanged = (

    historical_top[
        "window_config"
    ]
    ==
    corrected_top[
        "window_config"
    ]

    and

    historical_top[
        "feature_set"
    ]
    ==
    corrected_top[
        "feature_set"
    ]

    and

    historical_top[
        "model"
    ]
    ==
    corrected_top[
        "model"
    ]
)


print("\n")
print("=" * 132)
print("ROBUST-FEATURE TOP-CONFIGURATION STABILITY")
print("=" * 132)


print(
    "Historical top:"
)

print(
    f"  {historical_top['window_config']} | "
    f"{historical_top['feature_set']} | "
    f"{historical_top['model']} | "
    f"F1={historical_top['test_f1']:.9f}"
)


print(
    "\nCorrected top:"
)

print(
    f"  {corrected_top['window_config']} | "
    f"{corrected_top['feature_set']} | "
    f"{corrected_top['model']} | "
    f"F1={corrected_top['test_f1']:.9f}"
)


print(
    "\nTop configuration unchanged:",
    top_unchanged
)


# ============================================================
# O. Save
# ============================================================

BEST_OUTPUT = (
    ROBUST_CORRECTION_DIR
    / "robust_feature_best_f1_results_corrected.csv"
)

FPR_OUTPUT = (
    ROBUST_CORRECTION_DIR
    / "robust_feature_fpr_constrained_results_corrected.csv"
)

FPR010_OUTPUT = (
    ROBUST_CORRECTION_DIR
    / "robust_feature_fpr010_corrected.csv"
)

COMPARISON_OUTPUT = (
    ROBUST_CORRECTION_DIR
    / "robust_feature_historical_vs_corrected.csv"
)

FEATURE_AUDIT_OUTPUT = (
    ROBUST_CORRECTION_DIR
    / "robust_feature_set_audit_corrected.csv"
)


corrected_robust_best.to_csv(
    BEST_OUTPUT,
    index=False
)


corrected_robust_fpr.to_csv(
    FPR_OUTPUT,
    index=False
)


corrected_robust_fpr010.to_csv(
    FPR010_OUTPUT,
    index=False
)


robust_comparison.to_csv(
    COMPARISON_OUTPUT,
    index=False
)


feature_set_audit.to_csv(
    FEATURE_AUDIT_OUTPUT,
    index=False
)


# ============================================================
# P. Checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 44 CHECKPOINT")
print("=" * 132)


print(
    "Corrected robust Best-F1 rows:",
    len(
        corrected_robust_best
    )
)


print(
    "Corrected robust FPR-constrained rows:",
    len(
        corrected_robust_fpr
    )
)


print(
    "60/30 corrected robust rows:",
    len(
        robust_corrected[
            "60/30"
        ]
    )
)


print(
    "60/15 corrected robust rows:",
    len(
        robust_corrected[
            "60/15"
        ]
    )
)


print(
    "Top robust configuration unchanged:",
    top_unchanged
)


print(
    "\nSaved correction artifacts under:"
)

print(
    ROBUST_CORRECTION_DIR
)


print("\n")
print("=" * 132)
print("CELL 44 COMPLETE")
print("=" * 132)

print(
    "\nThe NAB robust-feature branch is now boundary-corrected "
    "using unchanged training populations and purged validation/test "
    "partitions. No neural model was trained and no historical artifact "
    "was overwritten."
)

In [ ]:
# CELL 45A — Audit Exact Historical NAB Threshold-Stability Branch Before Final Correction

from pathlib import Path
import json
import pandas as pd
import os


print("=" * 132)
print("NAB THRESHOLD / INTERPRETATION-STABILITY HISTORICAL PROTOCOL AUDIT")
print("=" * 132)


# ============================================================
# A. Known locations
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)

CODE_ROOT = Path(
    "/content/drive/MyDrive/Master/Thesis/Code"
)

assert NOTEBOOK_PATH.exists()
assert CODE_ROOT.exists()


# ============================================================
# B. Search filesystem for interpretation / stability artifacts
# ============================================================

keywords = [
    "interpretation",
    "stability",
    "threshold",
    "representation_summary",
    "split_level",
]


candidate_files = []


for root, dirs, files in os.walk(
    CODE_ROOT
):

    # Avoid extremely irrelevant hidden/cache folders.
    dirs[:] = [
        d
        for d in dirs
        if not d.startswith(".")
    ]


    for filename in files:

        lower = filename.lower()

        if any(
            keyword in lower
            for keyword in keywords
        ):

            path = Path(root) / filename

            candidate_files.append(
                path
            )


candidate_files = sorted(
    candidate_files,
    key=lambda p: str(p)
)


print("\n")
print("=" * 132)
print("CANDIDATE STABILITY / THRESHOLD ARTIFACTS")
print("=" * 132)


for path in candidate_files:

    try:

        size_mb = (
            path.stat().st_size
            /
            (1024 ** 2)
        )

    except Exception:

        size_mb = float("nan")


    print(
        f"{size_mb:9.3f} MB | {path}"
    )


# ============================================================
# C. Inspect CSV schemas and concise contents
# ============================================================

csv_candidates = [
    path
    for path in candidate_files
    if path.suffix.lower() == ".csv"
]


print("\n")
print("=" * 132)
print("CANDIDATE CSV SCHEMAS")
print("=" * 132)


for path in csv_candidates:

    try:

        df = pd.read_csv(
            path,
            low_memory=False
        )

    except Exception as exc:

        print(
            f"\nFAILED TO READ: {path}"
        )

        print(
            type(exc).__name__,
            str(exc)
        )

        continue


    print("\n" + "-" * 118)

    print(
        path
    )

    print(
        "shape:",
        df.shape
    )

    print(
        "columns:",
        df.columns.tolist()
    )


    # Print compact data only for small summary-like files.
    if (
        len(df) <= 100
        and
        df.shape[1] <= 30
    ):

        print(
            df.to_string(
                index=False,
                max_rows=100
            )
        )


# ============================================================
# D. Locate exact notebook cells
# ============================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    notebook = json.load(
        f
    )


SEARCH_TERMS = [

    "threshold stability",
    "threshold_stability",
    "interpretation stability",
    "interpretation_stability",
    "window_representation_summary",
    "split_level_representation_summary",
    "percentile",
    "cohen",
]


matching_cells = []


for cell_index, cell in enumerate(
    notebook.get(
        "cells",
        []
    ),
    start=1
):

    source = cell.get(
        "source",
        []
    )

    text = (
        "".join(source)
        if isinstance(
            source,
            list
        )
        else str(source)
    )


    lowered = (
        text.lower()
    )


    matched_terms = [
        term
        for term in SEARCH_TERMS
        if term.lower()
        in lowered
    ]


    if matched_terms:

        matching_cells.append(
            {
                "cell_index":
                    cell_index,

                "matched_terms":
                    matched_terms,

                "text":
                    text,
            }
        )


print("\n")
print("=" * 132)
print("MATCHING NOTEBOOK CELLS")
print("=" * 132)


print(
    "Matching cells:",
    len(
        matching_cells
    )
)


for item in matching_cells:

    text = (
        item[
            "text"
        ]
    )

    lines = (
        text.splitlines()
    )


    print("\n" + "#" * 118)

    print(
        "Notebook cell:",
        item[
            "cell_index"
        ]
    )

    print(
        "Matched terms:",
        item[
            "matched_terms"
        ]
    )

    print("-" * 118)


    # Print only potentially important lines plus context.
    important_terms = [

        "read_csv",
        "to_csv",
        "percentile",
        "threshold",
        "IsolationForest",
        "LocalOutlierFactor",
        "n_estimators",
        "n_neighbors",
        "random_state",
        "decision_function",
        "score_samples",
        "precision",
        "recall",
        "f1",
        "fpr",
        "roc_auc",
        "average_precision",
        "window_config",
        "60/30",
        "60/15",
        "split",
        "train",
        "validation",
        "test",
        ">= threshold",
        "> threshold",
    ]


    selected = set()


    for line_idx, line in enumerate(
        lines
    ):

        if any(
            term.lower()
            in line.lower()
            for term in important_terms
        ):

            selected.update(
                range(
                    max(
                        0,
                        line_idx - 3
                    ),
                    min(
                        len(lines),
                        line_idx + 5
                    )
                )
            )


    previous = None


    for line_idx in sorted(
        selected
    ):

        if (
            previous is not None
            and
            line_idx > previous + 1
        ):

            print(
                "     ..."
            )


        print(
            f"{line_idx + 1:04d}: "
            f"{lines[line_idx]}"
        )


        previous = (
            line_idx
        )


# ============================================================
# E. Search explicitly for known historical headline values
# ============================================================

KNOWN_VALUES = [

    "0.248373",
    "0.290836",
    "0.287809",
    "103526",
    "207033",
]


print("\n")
print("=" * 132)
print("NOTEBOOK CELLS CONTAINING KNOWN HISTORICAL VALUES")
print("=" * 132)


for value in KNOWN_VALUES:

    hits = []


    for cell_index, cell in enumerate(
        notebook.get(
            "cells",
            []
        ),
        start=1
    ):

        source = cell.get(
            "source",
            []
        )

        text = (
            "".join(source)
            if isinstance(
                source,
                list
            )
            else str(source)
        )


        if value in text:

            hits.append(
                cell_index
            )


    print(
        f"{value}: {hits}"
    )


# ============================================================
# F. Find likely experiment directories
# ============================================================

print("\n")
print("=" * 132)
print("LIKELY HISTORICAL EXPERIMENT DIRECTORIES")
print("=" * 132)


directory_candidates = []


for path in (
    CODE_ROOT
    / "experiments"
).glob(
    "*"
):

    if not path.is_dir():

        continue


    lower = (
        path.name.lower()
    )


    if (
        "interpret" in lower
        or
        "stability" in lower
        or
        "represent" in lower
    ):

        directory_candidates.append(
            path
        )


for path in sorted(
    directory_candidates
):

    print(
        path
    )


# ============================================================
# G. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 45A CHECKPOINT")
print("=" * 132)


print(
    "Candidate files found:",
    len(
        candidate_files
    )
)

print(
    "Candidate CSV files:",
    len(
        csv_candidates
    )
)

print(
    "Matching notebook cells:",
    len(
        matching_cells
    )
)

print(
    "Read-only audit:",
    True
)


print("\n")
print("=" * 132)
print("CELL 45A COMPLETE")
print("=" * 132)

print(
    "\nNo model was fitted, no threshold was changed, "
    "and no historical artifact was modified."
)

In [ ]:
# CELL 45B — Narrow Audit of Exact NAB Threshold-Stability and Interpretation-Stability Artifacts and Source Cells

from pathlib import Path
import json
import os
import pandas as pd


print("=" * 132)
print("NARROW NAB THRESHOLD / INTERPRETATION-STABILITY AUDIT")
print("=" * 132)


# ============================================================
# A. Exact target directories only
# ============================================================

TARGET_DIRS = {

    "interpretation_stability":
        BASE
        / "experiments"
        / "EXP_NAB_interpretation_stability",

    "threshold_stability":
        BASE
        / "experiments"
        / "EXP_NAB_threshold_stability",
}


for name, path in TARGET_DIRS.items():

    assert path.exists(), (
        f"Missing target directory: {path}"
    )

    print(
        f"{name}: {path}"
    )


# ============================================================
# B. Exact file inventory
# ============================================================

print("\n")
print("=" * 132)
print("TARGET-DIRECTORY FILE INVENTORY")
print("=" * 132)


inventory_rows = []


for experiment_name, root in (
    TARGET_DIRS.items()
):

    for path in sorted(
        root.rglob("*")
    ):

        if not path.is_file():

            continue


        size_mb = (
            path.stat().st_size
            /
            (1024 ** 2)
        )


        inventory_rows.append({

            "experiment":
                experiment_name,

            "relative_path":
                str(
                    path.relative_to(
                        root
                    )
                ),

            "suffix":
                path.suffix.lower(),

            "size_mb":
                size_mb,

            "absolute_path":
                str(path),
        })


inventory_df = pd.DataFrame(
    inventory_rows
)


print(
    inventory_df[
        [
            "experiment",
            "relative_path",
            "suffix",
            "size_mb",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# C. Inspect every CSV in only these two directories
# ============================================================

print("\n")
print("=" * 132)
print("TARGET CSV SCHEMAS AND CONTENT")
print("=" * 132)


csv_inventory = (
    inventory_df[
        inventory_df[
            "suffix"
        ]
        ==
        ".csv"
    ]
    .copy()
)


csv_summaries = []


for _, row in (
    csv_inventory.iterrows()
):

    path = Path(
        row[
            "absolute_path"
        ]
    )


    df = pd.read_csv(
        path,
        low_memory=False
    )


    csv_summaries.append({

        "experiment":
            row[
                "experiment"
            ],

        "file":
            row[
                "relative_path"
            ],

        "rows":
            len(df),

        "columns":
            len(
                df.columns
            ),

        "column_names":
            " | ".join(
                df.columns.astype(str)
            ),
    })


    print("\n" + "-" * 118)

    print(
        f"{row['experiment']} / "
        f"{row['relative_path']}"
    )

    print(
        "shape:",
        df.shape
    )

    print(
        "columns:",
        df.columns.tolist()
    )


    # Print complete small summary tables.
    if (
        len(df) <= 100
        and
        len(df.columns) <= 30
    ):

        print(
            df.to_string(
                index=False,
                max_rows=100
            )
        )

    else:

        print(
            "\nHEAD:"
        )

        print(
            df.head(
                8
            ).to_string(
                index=False
            )
        )

        print(
            "\nTAIL:"
        )

        print(
            df.tail(
                5
            ).to_string(
                index=False
            )
        )


csv_summary_df = pd.DataFrame(
    csv_summaries
)


print("\n")
print("=" * 132)
print("CSV SUMMARY")
print("=" * 132)


print(
    csv_summary_df[
        [
            "experiment",
            "file",
            "rows",
            "columns",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# D. Search notebook ONLY for exact experiment-directory names
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)


assert NOTEBOOK_PATH.exists()


with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    notebook = json.load(
        f
    )


EXACT_SEARCH_TERMS = [

    "EXP_NAB_interpretation_stability",

    "EXP_NAB_threshold_stability",

    "window_representation_summary.csv",

    "split_level_representation_summary.csv",
]


exact_matches = []


for cell_index, cell in enumerate(
    notebook.get(
        "cells",
        []
    ),
    start=1
):

    source = cell.get(
        "source",
        []
    )


    text = (
        "".join(source)
        if isinstance(
            source,
            list
        )
        else str(source)
    )


    matched = [

        term
        for term
        in EXACT_SEARCH_TERMS

        if term in text
    ]


    if matched:

        exact_matches.append({

            "cell_index":
                cell_index,

            "matched":
                matched,

            "text":
                text,
        })


print("\n")
print("=" * 132)
print("EXACT MATCHING NOTEBOOK CELLS")
print("=" * 132)


print(
    "Exact matching cells:",
    len(
        exact_matches
    )
)


for item in exact_matches:

    print("\n" + "#" * 118)

    print(
        "Notebook cell:",
        item[
            "cell_index"
        ]
    )

    print(
        "Matched:",
        item[
            "matched"
        ]
    )

    print("-" * 118)


    lines = (
        item[
            "text"
        ]
        .splitlines()
    )


    # Exact branch cells should be manageable.
    # Hard cap prevents another 5000-line truncation.
    max_lines = min(
        len(lines),
        350
    )


    for line_number in range(
        max_lines
    ):

        print(
            f"{line_number + 1:04d}: "
            f"{lines[line_number]}"
        )


    if len(
        lines
    ) > max_lines:

        print(
            f"... TRUNCATED THIS CELL AFTER "
            f"{max_lines} OF {len(lines)} LINES ..."
        )


# ============================================================
# E. Search exact target directories for config / JSON / TXT
# ============================================================

print("\n")
print("=" * 132)
print("NON-CSV CONFIG / SUMMARY FILES")
print("=" * 132)


TEXT_SUFFIXES = {
    ".json",
    ".txt",
    ".md",
    ".yaml",
    ".yml",
}


for _, row in (
    inventory_df.iterrows()
):

    if row[
        "suffix"
    ] not in TEXT_SUFFIXES:

        continue


    path = Path(
        row[
            "absolute_path"
        ]
    )


    print("\n" + "-" * 118)

    print(
        f"{row['experiment']} / "
        f"{row['relative_path']}"
    )


    try:

        text = path.read_text(
            encoding="utf-8"
        )

    except Exception as exc:

        print(
            "Could not read:",
            type(exc).__name__,
            str(exc)
        )

        continue


    lines = text.splitlines()


    for line in lines[
        :200
    ]:

        print(line)


    if len(
        lines
    ) > 200:

        print(
            f"... truncated after 200 of "
            f"{len(lines)} lines ..."
        )


# ============================================================
# F. Detect historical threshold-result tables automatically
# ============================================================

print("\n")
print("=" * 132)
print("POTENTIAL THRESHOLD-STABILITY RESULT TABLES")
print("=" * 132)


candidate_result_rows = []


for _, row in (
    csv_inventory.iterrows()
):

    path = Path(
        row[
            "absolute_path"
        ]
    )


    df = pd.read_csv(
        path,
        low_memory=False
    )


    lower_columns = {
        str(c).lower()
        for c in df.columns
    }


    has_threshold = any(

        "threshold"
        in col

        for col in lower_columns
    )


    has_metric = any(

        metric in lower_columns

        for metric in [
            "f1",
            "f1_score",
            "test_f1",
            "precision",
            "recall",
            "fpr",
            "false_positive_rate",
        ]
    )


    if (
        has_threshold
        or
        has_metric
    ):

        candidate_result_rows.append({

            "experiment":
                row[
                    "experiment"
                ],

            "file":
                row[
                    "relative_path"
                ],

            "rows":
                len(df),

            "columns":
                len(
                    df.columns
                ),

            "has_threshold_column":
                has_threshold,

            "has_metric_column":
                has_metric,
        })


candidate_result_df = pd.DataFrame(
    candidate_result_rows
)


if not candidate_result_df.empty:

    print(
        candidate_result_df.to_string(
            index=False
        )
    )

else:

    print(
        "No threshold/metric CSV candidates detected."
    )


# ============================================================
# G. Checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 45B CHECKPOINT")
print("=" * 132)


print(
    "Target directories:",
    len(
        TARGET_DIRS
    )
)

print(
    "Files in target directories:",
    len(
        inventory_df
    )
)

print(
    "CSV files in target directories:",
    len(
        csv_inventory
    )
)

print(
    "Exact matching notebook cells:",
    len(
        exact_matches
    )
)

print(
    "Potential threshold-result tables:",
    len(
        candidate_result_df
    )
)


print("\n")
print("=" * 132)
print("CELL 45B COMPLETE")
print("=" * 132)

print(
    "\nThis was a read-only narrow audit. "
    "No model was fitted, no threshold was changed, "
    "and no historical artifact was modified."
)

In [ ]:
# CELL 45B — Locate Exact NAB Threshold / Interpretation-Stability Artifacts and Historical Source (FIXED)

from pathlib import Path
import json
import os
import pandas as pd


print("=" * 132)
print("NAB THRESHOLD / INTERPRETATION-STABILITY — NARROW ARTIFACT AUDIT (FIXED)")
print("=" * 132)


# ============================================================
# A. Core paths
# ============================================================

CODE_ROOT = Path(
    "/content/drive/MyDrive/Master/Thesis/Code"
)

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)


INTERPRETATION_DIR = (
    CODE_ROOT
    / "experiments"
    / "EXP_NAB_interpretation_stability"
)


THRESHOLD_DIR = (
    CODE_ROOT
    / "experiments"
    / "EXP_NAB_threshold_stability"
)


assert CODE_ROOT.exists()
assert NOTEBOOK_PATH.exists()
assert INTERPRETATION_DIR.exists()
assert THRESHOLD_DIR.exists()


print("\nTarget directories:")
print(INTERPRETATION_DIR)
print(THRESHOLD_DIR)


# ============================================================
# B. Read the two historical Markdown summaries completely
# ============================================================

print("\n")
print("=" * 132)
print("HISTORICAL INTERPRETATION / STABILITY MARKDOWN SUMMARIES")
print("=" * 132)


markdown_paths = sorted(
    list(
        INTERPRETATION_DIR.glob("*.md")
    )
    +
    list(
        THRESHOLD_DIR.glob("*.md")
    )
)


assert len(markdown_paths) == 2, (
    f"Expected 2 Markdown summaries, found {len(markdown_paths)}"
)


for path in markdown_paths:

    print("\n" + "#" * 118)
    print(path)
    print("#" * 118)

    text = path.read_text(
        encoding="utf-8"
    )

    print(text)


# ============================================================
# C. Search entire Code tree for only relevant artifact names
# ============================================================

SEARCH_NAME_TERMS = [

    "threshold_stability",

    "interpretation_stability",

    "window_representation",

    "split_level_representation",

    "representation_summary",

    "threshold_summary",

    "threshold_grid",

    "percentile_stability",
]


artifact_rows = []


for root, dirs, files in os.walk(
    CODE_ROOT
):

    dirs[:] = [
        d
        for d in dirs
        if not d.startswith(".")
    ]


    for filename in files:

        lower_name = (
            filename.lower()
        )


        if not any(
            term
            in lower_name
            for term in SEARCH_NAME_TERMS
        ):

            continue


        path = (
            Path(root)
            / filename
        )


        artifact_rows.append({

            "filename":
                filename,

            "suffix":
                path.suffix.lower(),

            "size_mb":
                path.stat().st_size
                /
                (1024 ** 2),

            "path":
                str(path),
        })


artifact_inventory = (
    pd.DataFrame(
        artifact_rows
    )
)


if not artifact_inventory.empty:

    artifact_inventory = (
        artifact_inventory
        .sort_values(
            [
                "suffix",
                "filename",
                "path"
            ]
        )
        .reset_index(
            drop=True
        )
    )


print("\n")
print("=" * 132)
print("RELEVANT ARTIFACTS FOUND ACROSS ENTIRE CODE TREE")
print("=" * 132)


if artifact_inventory.empty:

    print(
        "No filename-based stability artifacts were found."
    )

else:

    print(
        artifact_inventory.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


# ============================================================
# D. Explicit search for our two known historical CSV filenames
# ============================================================

KNOWN_EXACT_FILENAMES = [

    "window_representation_summary.csv",

    "split_level_representation_summary.csv",
]


print("\n")
print("=" * 132)
print("EXACT KNOWN REPRESENTATION SUMMARY FILE SEARCH")
print("=" * 132)


exact_known_paths = []


for target_name in (
    KNOWN_EXACT_FILENAMES
):

    matches = list(
        CODE_ROOT.rglob(
            target_name
        )
    )


    print(
        f"\n{target_name}: "
        f"{len(matches)} match(es)"
    )


    for path in matches:

        print(
            "  ",
            path
        )

        exact_known_paths.append(
            path
        )


# ============================================================
# E. Inspect only relevant CSV artifacts, if any
# ============================================================

candidate_csv_paths = []


if not artifact_inventory.empty:

    candidate_csv_paths.extend(

        Path(path)

        for path in artifact_inventory.loc[
            artifact_inventory[
                "suffix"
            ]
            ==
            ".csv",
            "path"
        ].tolist()
    )


candidate_csv_paths.extend(
    exact_known_paths
)


# Deduplicate.
candidate_csv_paths = list(
    dict.fromkeys(
        candidate_csv_paths
    )
)


print("\n")
print("=" * 132)
print("RELEVANT CSV CONTENT")
print("=" * 132)


csv_summary_rows = []


if len(
    candidate_csv_paths
) == 0:

    print(
        "No relevant CSV artifacts were located."
    )

else:

    for path in (
        candidate_csv_paths
    ):

        if not path.exists():

            continue


        try:

            df = pd.read_csv(
                path,
                low_memory=False
            )

        except Exception as exc:

            print("\nFAILED:")
            print(path)
            print(
                type(exc).__name__,
                str(exc)
            )

            continue


        csv_summary_rows.append({

            "file":
                path.name,

            "rows":
                len(df),

            "columns":
                len(
                    df.columns
                ),

            "path":
                str(path),
        })


        print("\n" + "-" * 118)

        print(path)

        print(
            "shape:",
            df.shape
        )

        print(
            "columns:",
            df.columns.tolist()
        )


        if (
            len(df) <= 120
            and
            len(df.columns) <= 35
        ):

            print(
                df.to_string(
                    index=False,
                    max_rows=120
                )
            )

        else:

            print("\nHEAD:")

            print(
                df.head(
                    10
                ).to_string(
                    index=False
                )
            )

            print("\nTAIL:")

            print(
                df.tail(
                    5
                ).to_string(
                    index=False
                )
            )


csv_summary_df = pd.DataFrame(
    csv_summary_rows,
    columns=[
        "file",
        "rows",
        "columns",
        "path",
    ]
)


print("\n")
print("=" * 132)
print("RELEVANT CSV SUMMARY")
print("=" * 132)


if csv_summary_df.empty:

    print(
        "Relevant CSV count: 0"
    )

else:

    print(
        csv_summary_df.to_string(
            index=False
        )
    )


# ============================================================
# F. Exact notebook search — no broad 'percentile' search
# ============================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as file:

    notebook = json.load(
        file
    )


EXACT_NOTEBOOK_TERMS = [

    "EXP_NAB_interpretation_stability",

    "EXP_NAB_threshold_stability",

    "window_representation_summary.csv",

    "split_level_representation_summary.csv",

    "threshold_stability_60_30_vs_60_15",
]


matching_cells = []


for cell_number, cell in enumerate(
    notebook.get(
        "cells",
        []
    ),
    start=1
):

    source = cell.get(
        "source",
        []
    )


    text = (
        "".join(source)
        if isinstance(
            source,
            list
        )
        else str(source)
    )


    matched_terms = [

        term
        for term
        in EXACT_NOTEBOOK_TERMS

        if term.lower()
        in text.lower()
    ]


    if matched_terms:

        matching_cells.append({

            "cell_number":
                cell_number,

            "matched_terms":
                matched_terms,

            "text":
                text,
        })


print("\n")
print("=" * 132)
print("EXACT HISTORICAL NOTEBOOK CELLS")
print("=" * 132)


print(
    "Matching cells:",
    len(
        matching_cells
    )
)


for item in (
    matching_cells
):

    print("\n" + "#" * 118)

    print(
        "Notebook cell:",
        item[
            "cell_number"
        ]
    )

    print(
        "Matched:",
        item[
            "matched_terms"
        ]
    )

    print("-" * 118)


    lines = (
        item[
            "text"
        ].splitlines()
    )


    # These are now exact branch matches,
    # so show up to 500 source lines.
    show_n = min(
        500,
        len(lines)
    )


    for line_idx in range(
        show_n
    ):

        print(
            f"{line_idx + 1:04d}: "
            f"{lines[line_idx]}"
        )


    if len(
        lines
    ) > show_n:

        print(
            f"... source truncated after "
            f"{show_n}/{len(lines)} lines ..."
        )


# ============================================================
# G. Search notebook for likely threshold-grid implementation
#    only around NAB + both stride configurations
# ============================================================

print("\n")
print("=" * 132)
print("NAB 60/30 VS 60/15 THRESHOLD-CODE CANDIDATES")
print("=" * 132)


threshold_code_candidates = []


for cell_number, cell in enumerate(
    notebook.get(
        "cells",
        []
    ),
    start=1
):

    source = cell.get(
        "source",
        []
    )


    text = (
        "".join(source)
        if isinstance(
            source,
            list
        )
        else str(source)
    )


    lower = text.lower()


    # Much narrower than Cell 45A:
    # require NAB, both stride configs, and threshold language.
    has_nab = (
        "nab"
        in lower
    )


    has_60_30 = (
        "60/30"
        in lower
        or
        "stride30"
        in lower
        or
        "stride_30"
        in lower
        or
        "60_30"
        in lower
    )


    has_60_15 = (
        "60/15"
        in lower
        or
        "stride15"
        in lower
        or
        "stride_15"
        in lower
        or
        "60_15"
        in lower
    )


    has_threshold = (
        "threshold"
        in lower
        and
        "percentile"
        in lower
    )


    if (
        has_nab
        and
        has_60_30
        and
        has_60_15
        and
        has_threshold
    ):

        threshold_code_candidates.append({

            "cell_number":
                cell_number,

            "text":
                text,
        })


print(
    "Candidate cells:",
    len(
        threshold_code_candidates
    )
)


for item in (
    threshold_code_candidates
):

    print("\n" + "#" * 118)

    print(
        "Notebook cell:",
        item[
            "cell_number"
        ]
    )

    print("-" * 118)


    lines = (
        item[
            "text"
        ].splitlines()
    )


    # Extract only relevant lines plus context.
    selected_lines = set()


    KEYWORDS = [

        "percentile",
        "threshold",
        "np.arange",
        "range(",
        "IsolationForest",
        "n_estimators",
        "random_state",
        "decision_function",
        "score_samples",
        "f1",
        "false_positive",
        "fpr",
        "roc_auc",
        "average_precision",
        "60/30",
        "60/15",
        "stride30",
        "stride15",
        "to_csv",
    ]


    for i, line in enumerate(
        lines
    ):

        if any(
            keyword.lower()
            in line.lower()
            for keyword in KEYWORDS
        ):

            selected_lines.update(
                range(
                    max(
                        0,
                        i - 3
                    ),
                    min(
                        len(lines),
                        i + 5
                    )
                )
            )


    previous = None


    for i in sorted(
        selected_lines
    ):

        if (
            previous is not None
            and
            i > previous + 1
        ):

            print("     ...")


        print(
            f"{i + 1:04d}: "
            f"{lines[i]}"
        )


        previous = i


# ============================================================
# H. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 45B CHECKPOINT")
print("=" * 132)


print(
    "Historical Markdown summaries:",
    len(
        markdown_paths
    )
)


print(
    "Relevant filesystem artifacts:",
    len(
        artifact_inventory
    )
    if not artifact_inventory.empty
    else 0
)


print(
    "Relevant CSV artifacts:",
    len(
        csv_summary_df
    )
)


print(
    "Exact known representation-summary matches:",
    len(
        exact_known_paths
    )
)


print(
    "Exact historical notebook cells:",
    len(
        matching_cells
    )
)


print(
    "NAB 60/30-vs-60/15 threshold-code candidates:",
    len(
        threshold_code_candidates
    )
)


print(
    "Read-only audit:",
    True
)


print("\n")
print("=" * 132)
print("CELL 45B COMPLETE")
print("=" * 132)

print(
    "\nNo model was fitted, no threshold was recalculated, "
    "and no historical artifact was modified."
)

In [ ]:
# CELL 45C — Recover Exact Historical NAB Threshold-Stability Score and Threshold Protocol

from pathlib import Path
import json
import pandas as pd


print("=" * 132)
print("EXACT NAB THRESHOLD-STABILITY COMPUTATION AUDIT")
print("=" * 132)


# ============================================================
# A. Paths
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/Thesis.ipynb"
)

THRESHOLD_RESULTS_DIR = Path(
    "/content/drive/MyDrive/Master/Thesis/Code/results/EXP_NAB_threshold_stability"
)


assert NOTEBOOK_PATH.exists()
assert THRESHOLD_RESULTS_DIR.exists()


# ============================================================
# B. Locate exact full threshold-curve artifact
# ============================================================

FULL_CURVE_PATH = (
    THRESHOLD_RESULTS_DIR
    / "threshold_sensitivity_60_30_vs_60_15_unsupervised.csv"
)

SUMMARY_PATH = (
    THRESHOLD_RESULTS_DIR
    / "threshold_stability_summary_60_30_vs_60_15.csv"
)


print("\n")
print("=" * 132)
print("HISTORICAL THRESHOLD ARTIFACT CHECK")
print("=" * 132)


print(
    "Full threshold curve exists:",
    FULL_CURVE_PATH.exists()
)

print(
    "Stability summary exists:",
    SUMMARY_PATH.exists()
)


if FULL_CURVE_PATH.exists():

    threshold_curve = pd.read_csv(
        FULL_CURVE_PATH,
        low_memory=False
    )


    print(
        "\nFull curve shape:",
        threshold_curve.shape
    )

    print(
        "Columns:",
        threshold_curve.columns.tolist()
    )


    print(
        "\nFirst 12 rows:"
    )

    print(
        threshold_curve.head(
            12
        ).to_string(
            index=False,
            float_format=lambda x: f"{x:.9f}"
        )
    )


    print(
        "\nRows per configuration/model:"
    )

    print(
        threshold_curve.groupby(
            [
                "configuration",
                "model",
            ]
        )
        .size()
        .to_string()
    )


# ============================================================
# C. Read notebook and isolate Cells 497–506 only
# ============================================================

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:

    notebook = json.load(
        f
    )


TARGET_CELL_NUMBERS = range(
    497,
    507
)


IMPORTANT_TERMS = [

    "PATH_6015_PRED",
    "PATH_6030_PRED",

    "read_csv",

    "Isolation Forest",
    "IsolationForest",

    "LOF",

    "Autoencoder",

    "anomaly_score",
    "score",

    "percentiles",
    "percentile",

    "threshold",

    "precision_score",
    "recall_score",
    "f1_score",
    "confusion_matrix",
    "roc_auc_score",
    "average_precision_score",

    "validation",
    "test",

    "threshold_rows",
    "threshold_all",
    "threshold_all_path",

    "stability_summary",
    "stability_summary_path",

    "stable",
    "0.95",

    "to_csv",
]


print("\n")
print("=" * 132)
print("EXACT HISTORICAL THRESHOLD-STABILITY SOURCE")
print("=" * 132)


matched_target_cells = []


for cell_number in TARGET_CELL_NUMBERS:

    # Human numbering used by the previous audits was 1-based.
    cell = notebook[
        "cells"
    ][
        cell_number - 1
    ]


    source = cell.get(
        "source",
        []
    )


    text = (
        "".join(source)
        if isinstance(
            source,
            list
        )
        else str(source)
    )


    if not any(
        term.lower()
        in text.lower()
        for term in IMPORTANT_TERMS
    ):

        continue


    matched_target_cells.append(
        cell_number
    )


    lines = text.splitlines()


    print("\n" + "#" * 118)

    print(
        "Notebook cell:",
        cell_number
    )

    print("-" * 118)


    selected = set()


    for i, line in enumerate(
        lines
    ):

        if any(
            term.lower()
            in line.lower()
            for term in IMPORTANT_TERMS
        ):

            selected.update(
                range(
                    max(
                        0,
                        i - 4
                    ),
                    min(
                        len(lines),
                        i + 7
                    )
                )
            )


    previous = None


    for i in sorted(
        selected
    ):

        if (
            previous is not None
            and
            i > previous + 1
        ):

            print(
                "     ..."
            )


        print(
            f"{i + 1:04d}: "
            f"{lines[i]}"
        )


        previous = i


# ============================================================
# D. Explicit protocol searches across Cells 497–506
# ============================================================

target_text = "\n".join(

    (
        "".join(
            notebook[
                "cells"
            ][
                cell_number - 1
            ].get(
                "source",
                []
            )
        )
    )

    for cell_number in TARGET_CELL_NUMBERS
)


protocol_checks = {

    "uses_validation_text":
        "validation" in target_text.lower(),

    "uses_test_text":
        "test" in target_text.lower(),

    "uses_np_percentile":
        "np.percentile" in target_text,

    "uses_score_samples":
        "score_samples" in target_text,

    "uses_decision_function":
        "decision_function" in target_text,

    "uses_existing_prediction_files":
        (
            "PATH_6015_PRED" in target_text
            or
            "PATH_6030_PRED" in target_text
        ),

    "defines_95_percent_stability":
        (
            "0.95" in target_text
            and
            "stable" in target_text.lower()
        ),
}


print("\n")
print("=" * 132)
print("PROTOCOL CHECK FLAGS")
print("=" * 132)


for key, value in (
    protocol_checks.items()
):

    print(
        f"{key}: {value}"
    )


# ============================================================
# E. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 45C CHECKPOINT")
print("=" * 132)


print(
    "Historical full threshold curve located:",
    FULL_CURVE_PATH.exists()
)

print(
    "Historical summary located:",
    SUMMARY_PATH.exists()
)

print(
    "Relevant source cells among 497–506:",
    matched_target_cells
)

print(
    "Read-only audit:",
    True
)


print("\n")
print("=" * 132)
print("CELL 45C COMPLETE")
print("=" * 132)

print(
    "\nNo model was fitted, no threshold was recalculated, "
    "and no historical artifact was modified."
)

In [ ]:
# CELL 45 — Finalize Corrected NAB Interpretation and Validation-Based Threshold-Stability Evidence

from pathlib import Path
import numpy as np
import pandas as pd


print("=" * 132)
print("NAB INTERPRETATION / THRESHOLD-STABILITY — FINAL CORRECTION")
print("=" * 132)


# ============================================================
# A. Required corrected state
# ============================================================

assert "corrected_populations" in globals(), (
    "Cell 42 corrected populations are not available."
)

assert "CORRECTION" in globals()
assert "BASE" in globals()


assert len(
    corrected_populations["60/30"]
) == 103412

assert len(
    corrected_populations["60/15"]
) == 206693


CORRECTION_DIR = (
    CORRECTION
    / "NAB"
    / "INTERPRETATION_STABILITY_CORRECTION"
)

CORRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


REP_GRID_PATH = (
    CORRECTION
    / "NAB"
    / "REPRESENTATION_SCALING_CORRECTION"
    / "representation_scaling_threshold_grids_corrected.csv"
)


HISTORICAL_THRESHOLD_CURVE_PATH = (
    BASE
    / "results"
    / "EXP_NAB_threshold_stability"
    / "threshold_sensitivity_60_30_vs_60_15_unsupervised.csv"
)


HISTORICAL_THRESHOLD_SUMMARY_PATH = (
    BASE
    / "results"
    / "EXP_NAB_threshold_stability"
    / "threshold_stability_summary_60_30_vs_60_15.csv"
)


for path in [
    REP_GRID_PATH,
    HISTORICAL_THRESHOLD_CURVE_PATH,
    HISTORICAL_THRESHOLD_SUMMARY_PATH,
]:

    assert path.exists(), (
        f"Missing required artifact: {path}"
    )


print(
    "PASS: all required corrected and historical artifacts exist."
)


# ============================================================
# B. Corrected global window-representation summary
# ============================================================

def make_representation_row(
    df,
    configuration
):

    total = int(
        len(df)
    )

    anomaly = int(
        df["label"]
        .astype(int)
        .sum()
    )

    normal = (
        total - anomaly
    )


    return {

        "configuration":
            configuration,

        "total_windows":
            total,

        "normal_windows":
            normal,

        "anomaly_windows":
            anomaly,

        "anomaly_ratio":
            (
                anomaly / total
                if total > 0
                else np.nan
            ),

        "unique_series":
            int(
                df["series"]
                .nunique()
            ),

        "unique_domains":
            (
                int(
                    df["domain"]
                    .nunique()
                )
                if "domain" in df.columns
                else np.nan
            ),

        "windows_per_anomaly":
            (
                total / anomaly
                if anomaly > 0
                else np.nan
            ),
    }


corrected_representation_summary = (
    pd.DataFrame(
        [
            make_representation_row(

                corrected_populations[
                    "60/30"
                ],

                "60min_window_30min_stride",
            ),

            make_representation_row(

                corrected_populations[
                    "60/15"
                ],

                "60min_window_15min_stride",
            ),
        ]
    )
)


DELTA_COLUMNS = [

    "total_windows",
    "normal_windows",
    "anomaly_windows",
    "anomaly_ratio",
    "windows_per_anomaly",
]


corrected_representation_summary[
    "delta_vs_60_30"
] = np.nan


for column in (
    DELTA_COLUMNS
):

    corrected_representation_summary.loc[
        1,
        f"{column}_delta"
    ] = (

        corrected_representation_summary.loc[
            1,
            column
        ]

        -

        corrected_representation_summary.loc[
            0,
            column
        ]
    )


print("\n")
print("=" * 132)
print("CORRECTED WINDOW-REPRESENTATION SUMMARY")
print("=" * 132)


print(
    corrected_representation_summary
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


assert (
    corrected_representation_summary.loc[
        0,
        "total_windows"
    ]
    == 103412
)

assert (
    corrected_representation_summary.loc[
        1,
        "total_windows"
    ]
    == 206693
)

assert (
    corrected_representation_summary.loc[
        0,
        "anomaly_windows"
    ]
    == 9970
)

assert (
    corrected_representation_summary.loc[
        1,
        "anomaly_windows"
    ]
    == 19910
)


# ============================================================
# C. Corrected split-level representation summary
# ============================================================

split_rows = []


split_name_map = {

    "train":
        "train",

    "val":
        "validation",

    "test":
        "test",
}


for window_config in [
    "60/30",
    "60/15",
]:

    df = (
        corrected_populations[
            window_config
        ]
    )


    for internal_split in [
        "train",
        "val",
        "test",
    ]:

        split_df = (
            df[
                df["_split"]
                ==
                internal_split
            ]
        )


        total = int(
            len(split_df)
        )

        anomaly = int(
            split_df[
                "label"
            ]
            .astype(int)
            .sum()
        )

        normal = (
            total - anomaly
        )


        split_rows.append({

            "configuration":
                window_config,

            "split":
                split_name_map[
                    internal_split
                ],

            "total_windows":
                total,

            "normal_windows":
                normal,

            "anomaly_windows":
                anomaly,

            "anomaly_ratio":
                anomaly / total,

            "series_count":
                int(
                    split_df[
                        "series"
                    ]
                    .nunique()
                ),
        })


corrected_split_summary = (
    pd.DataFrame(
        split_rows
    )
)


print("\n")
print("=" * 132)
print("CORRECTED SPLIT-LEVEL REPRESENTATION SUMMARY")
print("=" * 132)


print(
    corrected_split_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


expected_split_counts = {

    ("60/30", "train"):
        62096,

    ("60/30", "validation"):
        20648,

    ("60/30", "test"):
        20668,

    ("60/15", "train"):
        124204,

    ("60/15", "validation"):
        41235,

    ("60/15", "test"):
        41254,
}


for key, expected in (
    expected_split_counts.items()
):

    observed = int(
        corrected_split_summary.loc[
            (
                corrected_split_summary[
                    "configuration"
                ]
                ==
                key[0]
            )
            &
            (
                corrected_split_summary[
                    "split"
                ]
                ==
                key[1]
            ),
            "total_windows"
        ].iloc[0]
    )


    assert observed == expected


# ============================================================
# D. Confirm exact methodological status of historical
#    threshold-stability branch
# ============================================================

historical_curve = (
    pd.read_csv(
        HISTORICAL_THRESHOLD_CURVE_PATH,
        low_memory=False
    )
)


historical_summary = (
    pd.read_csv(
        HISTORICAL_THRESHOLD_SUMMARY_PATH,
        low_memory=False
    )
)


assert len(
    historical_curve
) == 120

assert len(
    historical_summary
) == 6


historical_group_counts = (
    historical_curve.groupby(
        [
            "configuration",
            "model",
        ]
    )
    .size()
)


assert (
    historical_group_counts
    ==
    20
).all()


print("\n")
print("=" * 132)
print("HISTORICAL THRESHOLD-STABILITY STATUS")
print("=" * 132)


print(
    "Historical threshold-curve rows:",
    len(
        historical_curve
    )
)

print(
    "Historical model/configuration groups:",
    len(
        historical_group_counts
    )
)

print(
    "Percentile points per group:",
    int(
        historical_group_counts.iloc[0]
    )
)


print(
    "\nAUDIT FINDING:"
)

print(
    "The historical EXP_NAB_threshold_stability branch "
    "computed each percentile threshold directly from the "
    "TEST anomaly-score distribution."
)

print(
    "Its best-F1 point and 95%-of-best stability region were "
    "therefore retrospective TEST-set diagnostics."
)

print(
    "They were not validation-selected operating thresholds "
    "and must not be presented as such in the final thesis."
)

print(
    "The historical Autoencoder in that branch also used the "
    "old unpurged external validation partition for early stopping."
)


# ============================================================
# E. Build corrected validation-based stability diagnostic
#
#    This does NOT attempt to reproduce the historical test-based
#    exploratory branch.
#
#    It uses the already-corrected Cell 42 threshold grids for the
#    selected per-series z-score representation, where thresholds
#    and stability are evaluated on PURGED VALIDATION data.
# ============================================================

corrected_grid = (
    pd.read_csv(
        REP_GRID_PATH,
        low_memory=False
    )
)


selected_grid = (
    corrected_grid[
        corrected_grid[
            "variant"
        ]
        ==
        "per_series_zscore"
    ]
    .copy()
)


selected_grid = (
    selected_grid[
        selected_grid[
            "model"
        ]
        .isin(
            [
                "IsolationForest",
                "LOF",
            ]
        )
    ]
    .copy()
)


assert len(
    selected_grid
) == 80


# Cell 42 generated exactly:
# np.arange(50, 98, 2.5)
PERCENTILES = np.arange(
    50,
    98,
    2.5
)


assert len(
    PERCENTILES
) == 20


selected_grid = (
    selected_grid
    .sort_values(
        [
            "window_config",
            "model",
        ],
        kind="stable"
    )
    .reset_index(
        drop=True
    )
)


# Assign percentile sequence independently inside each
# model/configuration group while preserving stored grid order.
selected_grid[
    "percentile"
] = (
    selected_grid.groupby(
        [
            "window_config",
            "model",
        ]
    )
    .cumcount()
    .map(
        lambda i:
            float(
                PERCENTILES[
                    i
                ]
            )
    )
)


group_sizes = (
    selected_grid.groupby(
        [
            "window_config",
            "model",
        ]
    )
    .size()
)


assert (
    group_sizes
    ==
    20
).all()


validation_stability_rows = []


for (
    window_config,
    model_name
), group in (
    selected_grid.groupby(
        [
            "window_config",
            "model",
        ],
        sort=False
    )
):

    group = (
        group
        .sort_values(
            "percentile"
        )
        .reset_index(
            drop=True
        )
    )


    best = (
        group
        .sort_values(
            [
                "val_f1",
                "val_fpr",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .iloc[0]
    )


    stability_cutoff = (
        0.95
        *
        float(
            best[
                "val_f1"
            ]
        )
    )


    stable = (
        group[
            group[
                "val_f1"
            ]
            >=
            stability_cutoff
        ]
        .copy()
    )


    assert len(
        stable
    ) > 0


    validation_stability_rows.append({

        "window_config":
            window_config,

        "variant":
            "per_series_zscore",

        "model":
            model_name,

        "best_validation_percentile":
            float(
                best[
                    "percentile"
                ]
            ),

        "best_validation_threshold":
            float(
                best[
                    "threshold"
                ]
            ),

        "best_validation_f1":
            float(
                best[
                    "val_f1"
                ]
            ),

        "validation_precision_at_best":
            float(
                best[
                    "val_precision"
                ]
            ),

        "validation_recall_at_best":
            float(
                best[
                    "val_recall"
                ]
            ),

        "validation_fpr_at_best":
            float(
                best[
                    "val_fpr"
                ]
            ),

        "stable_percentile_min":
            float(
                stable[
                    "percentile"
                ]
                .min()
            ),

        "stable_percentile_max":
            float(
                stable[
                    "percentile"
                ]
                .max()
            ),

        "stable_percentile_width":
            float(
                stable[
                    "percentile"
                ]
                .max()
                -
                stable[
                    "percentile"
                ]
                .min()
            ),

        "stable_points_count":
            int(
                len(
                    stable
                )
            ),

        "validation_f1_std_across_thresholds":
            float(
                group[
                    "val_f1"
                ]
                .std(
                    ddof=1
                )
            ),

        "validation_recall_std_across_thresholds":
            float(
                group[
                    "val_recall"
                ]
                .std(
                    ddof=1
                )
            ),

        "validation_fpr_std_across_thresholds":
            float(
                group[
                    "val_fpr"
                ]
                .std(
                    ddof=1
                )
            ),
    })


corrected_validation_stability = (
    pd.DataFrame(
        validation_stability_rows
    )
)


assert len(
    corrected_validation_stability
) == 4


print("\n")
print("=" * 132)
print("CORRECTED VALIDATION-BASED THRESHOLD-STABILITY SUMMARY")
print("=" * 132)


print(
    corrected_validation_stability
    .sort_values(
        [
            "best_validation_f1",
            "validation_fpr_at_best",
        ],
        ascending=[
            False,
            True,
        ]
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)


# ============================================================
# F. Formal branch-status record
# ============================================================

branch_status = pd.DataFrame(
    [
        {
            "result_family":
                (
                    "Historical NAB threshold-stability "
                    "experiment"
                ),

            "historical_location":
                (
                    "results/EXP_NAB_threshold_stability"
                ),

            "final_status":
                "RETIRED_FROM_FINAL_NUMERIC_EVIDENCE",

            "reason_1":
                (
                    "Percentile thresholds were computed "
                    "directly from test anomaly scores."
                ),

            "reason_2":
                (
                    "Best-F1 and stability regions were "
                    "evaluated retrospectively on test labels."
                ),

            "reason_3":
                (
                    "Historical Autoencoder early stopping "
                    "used the old unpurged external validation "
                    "partition."
                ),

            "final_handling":
                (
                    "Do not use historical threshold-stability "
                    "numbers as validation-selected evidence. "
                    "Use the corrected purged-validation "
                    "stability diagnostic from Cell 42/45 for "
                    "IF and LOF, and the corrected five-model "
                    "results for Autoencoder performance."
                ),
        }
    ]
)


print("\n")
print("=" * 132)
print("FINAL THRESHOLD-STABILITY BRANCH STATUS")
print("=" * 132)


print(
    branch_status.to_string(
        index=False
    )
)


# ============================================================
# G. Save authoritative correction artifacts
# ============================================================

REPRESENTATION_OUTPUT = (
    CORRECTION_DIR
    / "window_representation_summary_corrected.csv"
)


SPLIT_OUTPUT = (
    CORRECTION_DIR
    / "split_level_representation_summary_corrected.csv"
)


VALIDATION_STABILITY_OUTPUT = (
    CORRECTION_DIR
    / "validation_threshold_stability_per_series_zscore_corrected.csv"
)


VALIDATION_GRID_OUTPUT = (
    CORRECTION_DIR
    / "validation_threshold_grid_per_series_zscore_corrected.csv"
)


BRANCH_STATUS_OUTPUT = (
    CORRECTION_DIR
    / "historical_threshold_stability_branch_status.csv"
)


corrected_representation_summary.to_csv(
    REPRESENTATION_OUTPUT,
    index=False
)


corrected_split_summary.to_csv(
    SPLIT_OUTPUT,
    index=False
)


corrected_validation_stability.to_csv(
    VALIDATION_STABILITY_OUTPUT,
    index=False
)


selected_grid.to_csv(
    VALIDATION_GRID_OUTPUT,
    index=False
)


branch_status.to_csv(
    BRANCH_STATUS_OUTPUT,
    index=False
)


# ============================================================
# H. Final checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 45 CHECKPOINT")
print("=" * 132)


print(
    "Corrected 60/30 total windows:",
    int(
        corrected_representation_summary.loc[
            corrected_representation_summary[
                "configuration"
            ]
            ==
            "60min_window_30min_stride",
            "total_windows"
        ].iloc[0]
    )
)


print(
    "Corrected 60/15 total windows:",
    int(
        corrected_representation_summary.loc[
            corrected_representation_summary[
                "configuration"
            ]
            ==
            "60min_window_15min_stride",
            "total_windows"
        ].iloc[0]
    )
)


print(
    "Corrected split-summary rows:",
    len(
        corrected_split_summary
    )
)


print(
    "Validation-based stability rows:",
    len(
        corrected_validation_stability
    )
)


print(
    "Historical test-derived threshold branch retired:",
    True
)


print(
    "Additional model training performed:",
    False
)


print(
    "\nSaved correction artifacts under:"
)

print(
    CORRECTION_DIR
)


print("\n")
print("=" * 132)
print("CELL 45 COMPLETE")
print("=" * 132)

print(
    "\nThe old test-derived NAB threshold-stability experiment "
    "has been formally separated from final thesis-facing "
    "evidence. Corrected population summaries and a purged-"
    "validation threshold-stability diagnostic are now frozen."
)

In [ ]:
# CELL 46 — Final Master Audit and Freeze of Split-Boundary Leakage Correction

from pathlib import Path
from datetime import datetime
import hashlib
import json
import numpy as np
import pandas as pd


print("=" * 132)
print("FINAL MASTER AUDIT — SPLIT-BOUNDARY LEAKAGE CORRECTION")
print("=" * 132)


# ============================================================
# A. Paths
# ============================================================

BASE = Path(
    "/content/drive/MyDrive/Master/Thesis/Code"
)

CORRECTION_ROOT = (
    BASE
    / "experiments"
    / "EXP_SPLIT_BOUNDARY_PURGE_CORRECTION"
)

FREEZE_DIR = (
    CORRECTION_ROOT
    / "FINAL_FREEZE"
)

FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


assert BASE.exists()
assert CORRECTION_ROOT.exists()


NAB_ROOT = (
    CORRECTION_ROOT
    / "NAB"
)

FIVE_MODEL_DIR = (
    NAB_ROOT
    / "FIVE_MODEL_CORRECTION"
)

REPRESENTATION_DIR = (
    NAB_ROOT
    / "REPRESENTATION_SCALING_CORRECTION"
)

FEATURE_SELECTION_DIR = (
    NAB_ROOT
    / "FEATURE_SELECTION_CORRECTION"
)

ROBUST_FEATURE_DIR = (
    NAB_ROOT
    / "ROBUST_FEATURE_CORRECTION"
)

INTERPRETATION_DIR = (
    NAB_ROOT
    / "INTERPRETATION_STABILITY_CORRECTION"
)

UKDALE_DIR = (
    CORRECTION_ROOT
    / "UKDALE"
)

AMPDS2_DIR = (
    CORRECTION_ROOT
    / "AMPDS2"
)


CRITICAL_DIRECTORIES = {

    "NAB five-model correction":
        FIVE_MODEL_DIR,

    "NAB representation scaling":
        REPRESENTATION_DIR,

    "NAB feature selection":
        FEATURE_SELECTION_DIR,

    "NAB robust features":
        ROBUST_FEATURE_DIR,

    "NAB interpretation/stability":
        INTERPRETATION_DIR,

    "UK-DALE correction":
        UKDALE_DIR,

    "AMPds2 correction":
        AMPDS2_DIR,
}


print("\n")
print("=" * 132)
print("CRITICAL CORRECTION DIRECTORY AUDIT")
print("=" * 132)


for name, path in (
    CRITICAL_DIRECTORIES.items()
):

    print(
        f"{name:<35} "
        f"exists={path.exists()} | "
        f"{path}"
    )

    assert path.exists(), (
        f"Missing critical correction directory: {path}"
    )


# ============================================================
# B. Helper — locate corrected CSV by required columns
# ============================================================

def find_csv_with_columns(
    directory,
    required_columns,
):

    candidates = []


    for path in sorted(
        directory.rglob("*.csv")
    ):

        try:

            sample = pd.read_csv(
                path,
                nrows=5,
                low_memory=False
            )

        except Exception:

            continue


        if set(
            required_columns
        ).issubset(
            sample.columns
        ):

            candidates.append(
                path
            )


    return candidates


# ============================================================
# C. NAB five-model authoritative result
# ============================================================

FIVE_MODEL_PATH = (
    FIVE_MODEL_DIR
    / "FINAL_NAB_FIVE_MODEL_PUBLICATION_TABLE_CORRECTED.csv"
)

FIVE_MODEL_RANKING_PATH = (
    FIVE_MODEL_DIR
    / "FINAL_NAB_FIVE_MODEL_RANKING_CORRECTED.csv"
)

FIVE_MODEL_PROVENANCE_PATH = (
    FIVE_MODEL_DIR
    / "FINAL_NAB_FIVE_MODEL_CORRECTION_PROVENANCE.csv"
)


for path in [
    FIVE_MODEL_PATH,
    FIVE_MODEL_RANKING_PATH,
    FIVE_MODEL_PROVENANCE_PATH,
]:

    assert path.exists(), (
        f"Missing final five-model artifact: {path}"
    )


five_model = pd.read_csv(
    FIVE_MODEL_PATH,
    low_memory=False
)

five_model_ranking = pd.read_csv(
    FIVE_MODEL_RANKING_PATH,
    low_memory=False
)

five_model_provenance = pd.read_csv(
    FIVE_MODEL_PROVENANCE_PATH,
    low_memory=False
)


assert len(
    five_model
) == 5

assert len(
    five_model_ranking
) == 5

assert len(
    five_model_provenance
) == 5


expected_five_model = {

    "One-Class SVM": {
        "F1-Score": 0.278158205,
        "FPR": 0.060958791,
        "ROC-AUC": 0.674766662,
        "PR-AUC": 0.238738745,
    },

    "Isolation Forest": {
        "F1-Score": 0.232743583,
        "FPR": 0.055442158,
        "ROC-AUC": 0.686086201,
        "PR-AUC": 0.243243549,
    },

    "Dense Autoencoder": {
        "F1-Score": 0.229755179,
        "FPR": 0.067247752,
        "ROC-AUC": 0.576805486,
        "PR-AUC": 0.196588594,
    },

    "Local Outlier Factor": {
        "F1-Score": 0.189484127,
        "FPR": 0.061179456,
        "ROC-AUC": 0.596568470,
        "PR-AUC": 0.181413585,
    },

    "LSTM Autoencoder": {
        "F1-Score": 0.156415695,
        "FPR": 0.051635682,
        "ROC-AUC": 0.566270650,
        "PR-AUC": 0.166804151,
    },
}


for model_name, metrics in (
    expected_five_model.items()
):

    row = (
        five_model[
            five_model[
                "Model"
            ]
            ==
            model_name
        ]
    )


    assert len(
        row
    ) == 1


    row = row.iloc[
        0
    ]


    for metric_name, expected_value in (
        metrics.items()
    ):

        assert np.isclose(
            float(
                row[
                    metric_name
                ]
            ),
            expected_value,
            atol=5e-9,
            rtol=0,
        ), (
            f"{model_name} {metric_name} mismatch"
        )


assert (
    five_model_ranking.iloc[
        0
    ][
        "Model"
    ]
    ==
    "One-Class SVM"
)


# ============================================================
# D. Representation-scaling authoritative result
# ============================================================

representation_candidates = (
    find_csv_with_columns(
        REPRESENTATION_DIR,
        [
            "window_config",
            "variant",
            "model",
            "test_f1",
            "test_fpr",
        ],
    )
)


assert len(
    representation_candidates
) > 0


representation_tables = []


for path in (
    representation_candidates
):

    df = pd.read_csv(
        path,
        low_memory=False
    )


    # We want the 16-row Best-F1 table,
    # not the larger threshold grid.
    if (
        len(df) == 16
        and
        "variant" in df.columns
        and
        "test_f1" in df.columns
    ):

        representation_tables.append(
            (
                path,
                df
            )
        )


assert len(
    representation_tables
) >= 1


representation_path, representation_best = (
    representation_tables[
        0
    ]
)


representation_top = (
    representation_best
    .sort_values(
        [
            "test_f1",
            "test_fpr",
        ],
        ascending=[
            False,
            True,
        ]
    )
    .iloc[0]
)


assert (
    representation_top[
        "window_config"
    ]
    ==
    "60/30"
)

assert (
    representation_top[
        "variant"
    ]
    ==
    "per_series_zscore"
)

assert (
    representation_top[
        "model"
    ]
    ==
    "IsolationForest"
)

assert np.isclose(
    float(
        representation_top[
            "test_f1"
        ]
    ),
    0.290618762,
    atol=5e-9,
    rtol=0,
)


# ============================================================
# E. Feature-selection authoritative result
# ============================================================

feature_candidates = (
    find_csv_with_columns(
        FEATURE_SELECTION_DIR,
        [
            "window_config",
            "feature_subset",
            "model",
            "test_f1",
            "test_fpr",
        ],
    )
)


assert len(
    feature_candidates
) > 0


feature_tables = []


for path in (
    feature_candidates
):

    df = pd.read_csv(
        path,
        low_memory=False
    )


    if (
        len(df) == 40
        and
        "feature_subset" in df.columns
    ):

        feature_tables.append(
            (
                path,
                df
            )
        )


assert len(
    feature_tables
) >= 1


feature_path, feature_best = (
    feature_tables[
        0
    ]
)


feature_top = (
    feature_best
    .sort_values(
        [
            "test_f1",
            "test_fpr",
        ],
        ascending=[
            False,
            True,
        ]
    )
    .iloc[0]
)


assert (
    feature_top[
        "window_config"
    ]
    ==
    "60/15"
)

assert (
    feature_top[
        "feature_subset"
    ]
    ==
    "corr_pruned_0.95"
)

assert (
    feature_top[
        "model"
    ]
    ==
    "IsolationForest"
)

assert np.isclose(
    float(
        feature_top[
            "test_f1"
        ]
    ),
    0.293259074,
    atol=5e-9,
    rtol=0,
)


# ============================================================
# F. Robust-feature authoritative result
# ============================================================

robust_candidates = (
    find_csv_with_columns(
        ROBUST_FEATURE_DIR,
        [
            "window_config",
            "feature_set",
            "model",
            "test_f1",
            "test_fpr",
        ],
    )
)


assert len(
    robust_candidates
) > 0


robust_tables = []


for path in (
    robust_candidates
):

    df = pd.read_csv(
        path,
        low_memory=False
    )


    if (
        len(df) == 16
        and
        "feature_set" in df.columns
    ):

        robust_tables.append(
            (
                path,
                df
            )
        )


assert len(
    robust_tables
) >= 1


robust_path, robust_best = (
    robust_tables[
        0
    ]
)


robust_top = (
    robust_best
    .sort_values(
        [
            "test_f1",
            "test_fpr",
        ],
        ascending=[
            False,
            True,
        ]
    )
    .iloc[0]
)


assert (
    robust_top[
        "window_config"
    ]
    ==
    "60/30"
)

assert (
    robust_top[
        "feature_set"
    ]
    ==
    "original_23_recomputed"
)

assert (
    robust_top[
        "model"
    ]
    ==
    "IsolationForest"
)

assert np.isclose(
    float(
        robust_top[
            "test_f1"
        ]
    ),
    0.262594739,
    atol=5e-9,
    rtol=0,
)


# ============================================================
# G. Corrected NAB population and stability evidence
# ============================================================

REPRESENTATION_SUMMARY_PATH = (
    INTERPRETATION_DIR
    / "window_representation_summary_corrected.csv"
)

SPLIT_SUMMARY_PATH = (
    INTERPRETATION_DIR
    / "split_level_representation_summary_corrected.csv"
)

VALIDATION_STABILITY_PATH = (
    INTERPRETATION_DIR
    / "validation_threshold_stability_per_series_zscore_corrected.csv"
)

THRESHOLD_BRANCH_STATUS_PATH = (
    INTERPRETATION_DIR
    / "historical_threshold_stability_branch_status.csv"
)


for path in [
    REPRESENTATION_SUMMARY_PATH,
    SPLIT_SUMMARY_PATH,
    VALIDATION_STABILITY_PATH,
    THRESHOLD_BRANCH_STATUS_PATH,
]:

    assert path.exists()


corrected_representation = pd.read_csv(
    REPRESENTATION_SUMMARY_PATH
)

corrected_splits = pd.read_csv(
    SPLIT_SUMMARY_PATH
)

validation_stability = pd.read_csv(
    VALIDATION_STABILITY_PATH
)

threshold_branch_status = pd.read_csv(
    THRESHOLD_BRANCH_STATUS_PATH
)


assert len(
    corrected_representation
) == 2

assert len(
    corrected_splits
) == 6

assert len(
    validation_stability
) == 4

assert len(
    threshold_branch_status
) == 1


assert int(
    corrected_representation.loc[
        corrected_representation[
            "configuration"
        ]
        ==
        "60min_window_30min_stride",
        "total_windows"
    ].iloc[0]
) == 103412


assert int(
    corrected_representation.loc[
        corrected_representation[
            "configuration"
        ]
        ==
        "60min_window_15min_stride",
        "total_windows"
    ].iloc[0]
) == 206693


assert (
    threshold_branch_status[
        "final_status"
    ].iloc[0]
    ==
    "RETIRED_FROM_FINAL_NUMERIC_EVIDENCE"
)


# ============================================================
# H. Dataset-wide correction status
# ============================================================

correction_status = pd.DataFrame(
    [
        {
            "result_family":
                "NAB supervised V1/V2",

            "window_configuration":
                "60 min / 30 min",

            "boundary_handling":
                "Already boundary-purged",

            "original_rows":
                np.nan,

            "removed_rows":
                np.nan,

            "corrected_rows":
                67737,

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "BLUED five-model event-alignment",

            "window_configuration":
                "1 min / 30 sec",

            "boundary_handling":
                "Purged chronological split",

            "original_rows":
                20414,

            "removed_rows":
                8,

            "corrected_rows":
                20406,

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "NAB 60/30 later benchmark population",

            "window_configuration":
                "60 min / 30 min",

            "boundary_handling":
                "Timestamp-level boundary purge",

            "original_rows":
                103526,

            "removed_rows":
                114,

            "corrected_rows":
                103412,

            "training_population_changed":
                False,

            "model_refit_required":
                "Only where external validation controlled training",

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "NAB 60/15 later benchmark population",

            "window_configuration":
                "60 min / 15 min",

            "boundary_handling":
                "Timestamp-level boundary purge",

            "original_rows":
                207033,

            "removed_rows":
                340,

            "corrected_rows":
                206693,

            "training_population_changed":
                False,

            "model_refit_required":
                "Only where external validation controlled training",

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "NAB five-model label-blind branch",

            "window_configuration":
                "60 min / 30 min",

            "boundary_handling":
                "Purged validation/test partitions",

            "original_rows":
                103526,

            "removed_rows":
                114,

            "corrected_rows":
                103412,

            "training_population_changed":
                False,

            "model_refit_required":
                (
                    "Dense AE and LSTM AE refitted; "
                    "LOF reconstructed for runtime stability"
                ),

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "NAB representation scaling",

            "window_configuration":
                "60/30 and 60/15",

            "boundary_handling":
                "Purged validation/test partitions",

            "original_rows":
                "103526 / 207033",

            "removed_rows":
                "114 / 340",

            "corrected_rows":
                "103412 / 206693",

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "NAB feature selection",

            "window_configuration":
                "60/30 and 60/15",

            "boundary_handling":
                "Purged validation/test partitions",

            "original_rows":
                "103526 / 207033",

            "removed_rows":
                "114 / 340",

            "corrected_rows":
                "103412 / 206693",

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "NAB robust-feature experiment",

            "window_configuration":
                "60/30 and 60/15",

            "boundary_handling":
                "Purged validation/test partitions",

            "original_rows":
                "103526 / 207033",

            "removed_rows":
                "114 / 340",

            "corrected_rows":
                "103412 / 206693",

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "NAB historical threshold-stability branch",

            "window_configuration":
                "60/30 and 60/15",

            "boundary_handling":
                "Historical branch not promoted",

            "original_rows":
                "103526 / 207033",

            "removed_rows":
                np.nan,

            "corrected_rows":
                np.nan,

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "RETIRED_FROM_FINAL_NUMERIC_EVIDENCE",
        },

        {
            "result_family":
                "UK-DALE selected Isolation Forest",

            "window_configuration":
                "60 min / 30 min",

            "boundary_handling":
                "Timestamp-level boundary purge",

            "original_rows":
                363940,

            "removed_rows":
                31,

            "corrected_rows":
                363909,

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "AMPds2 selected Isolation Forest",

            "window_configuration":
                "60 min / 30 min",

            "boundary_handling":
                "Timestamp-level boundary purge",

            "original_rows":
                735819,

            "removed_rows":
                42,

            "corrected_rows":
                735777,

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "Controlled UK-DALE cross-model comparison",

            "window_configuration":
                "60 min / 30 min",

            "boundary_handling":
                "Reference/evaluation timestamp-overlap purge",

            "original_rows":
                19694,

            "removed_rows":
                1303,

            "corrected_rows":
                18391,

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "CLOSED",
        },

        {
            "result_family":
                "Controlled AMPds2 cross-model comparison",

            "window_configuration":
                "60 min / 30 min",

            "boundary_handling":
                "Reference/evaluation timestamp-overlap purge",

            "original_rows":
                26250,

            "removed_rows":
                374,

            "corrected_rows":
                25876,

            "training_population_changed":
                False,

            "model_refit_required":
                False,

            "final_status":
                "CLOSED",
        },
    ]
)


print("\n")
print("=" * 132)
print("FINAL CORRECTION STATUS TABLE")
print("=" * 132)


print(
    correction_status.to_string(
        index=False
    )
)


# ============================================================
# I. Authoritative corrected NAB headline results
# ============================================================

authoritative_nab = pd.DataFrame(
    [
        {
            "result_family":
                "Five-model label-blind",

            "selected_configuration":
                "One-Class SVM",

            "headline_metric":
                "Test F1",

            "value":
                0.278158205,

            "interpretation":
                (
                    "Highest corrected five-model test F1; "
                    "ranking unchanged after purge."
                ),
        },

        {
            "result_family":
                "Representation scaling",

            "selected_configuration":
                (
                    "60/30 | per_series_zscore | "
                    "IsolationForest"
                ),

            "headline_metric":
                "Test F1",

            "value":
                0.290618762,

            "interpretation":
                (
                    "Top representation configuration "
                    "unchanged after purge."
                ),
        },

        {
            "result_family":
                "Feature selection",

            "selected_configuration":
                (
                    "60/15 | corr_pruned_0.95 | "
                    "IsolationForest"
                ),

            "headline_metric":
                "Test F1",

            "value":
                0.293259074,

            "interpretation":
                (
                    "Top feature-selection configuration "
                    "unchanged after purge."
                ),
        },

        {
            "result_family":
                "Robust features",

            "selected_configuration":
                (
                    "60/30 | original_23_recomputed | "
                    "IsolationForest"
                ),

            "headline_metric":
                "Test F1",

            "value":
                0.262594739,

            "interpretation":
                (
                    "Top robust-feature configuration "
                    "unchanged after purge."
                ),
        },

        {
            "result_family":
                "Validation threshold stability",

            "selected_configuration":
                (
                    "Purged validation diagnostic; "
                    "per_series_zscore IF/LOF"
                ),

            "headline_metric":
                "Status",

            "value":
                np.nan,

            "interpretation":
                (
                    "Uses purged validation data. "
                    "Historical test-derived stability "
                    "branch retired."
                ),
        },
    ]
)


print("\n")
print("=" * 132)
print("AUTHORITATIVE NAB RESULTS AFTER PURGE")
print("=" * 132)


print(
    authoritative_nab.to_string(
        index=False
    )
)


# ============================================================
# J. Core master audit checks
# ============================================================

uk_csv_count = len(
    list(
        UKDALE_DIR.rglob(
            "*.csv"
        )
    )
)

amp_csv_count = len(
    list(
        AMPDS2_DIR.rglob(
            "*.csv"
        )
    )
)


correction_files_before_freeze = [

    path
    for path in CORRECTION_ROOT.rglob("*")

    if (
        path.is_file()
        and
        FREEZE_DIR not in path.parents
    )
]


nonzero_artifacts = all(

    path.stat().st_size > 0

    for path in correction_files_before_freeze
)


audit_checks = pd.DataFrame(
    [
        {
            "check":
                "Correction root exists",

            "passed":
                CORRECTION_ROOT.exists(),

            "detail":
                str(
                    CORRECTION_ROOT
                ),
        },

        {
            "check":
                "All critical branch directories exist",

            "passed":
                all(
                    path.exists()
                    for path in
                    CRITICAL_DIRECTORIES.values()
                ),

            "detail":
                f"{len(CRITICAL_DIRECTORIES)} directories",
        },

        {
            "check":
                "NAB five-model corrected table validated",

            "passed":
                True,

            "detail":
                (
                    "5 models; OCSVM remains rank 1 "
                    "with F1=0.278158205"
                ),
        },

        {
            "check":
                "Representation top configuration validated",

            "passed":
                True,

            "detail":
                (
                    "60/30 per_series_zscore "
                    "IsolationForest F1=0.290618762"
                ),
        },

        {
            "check":
                "Feature-selection top configuration validated",

            "passed":
                True,

            "detail":
                (
                    "60/15 corr_pruned_0.95 "
                    "IsolationForest F1=0.293259074"
                ),
        },

        {
            "check":
                "Robust-feature top configuration validated",

            "passed":
                True,

            "detail":
                (
                    "60/30 original_23_recomputed "
                    "IsolationForest F1=0.262594739"
                ),
        },

        {
            "check":
                "Corrected NAB 60/30 population validated",

            "passed":
                (
                    int(
                        corrected_representation.loc[
                            corrected_representation[
                                "configuration"
                            ]
                            ==
                            "60min_window_30min_stride",
                            "total_windows"
                        ].iloc[0]
                    )
                    ==
                    103412
                ),

            "detail":
                "103,412 windows after 114-row purge",
        },

        {
            "check":
                "Corrected NAB 60/15 population validated",

            "passed":
                (
                    int(
                        corrected_representation.loc[
                            corrected_representation[
                                "configuration"
                            ]
                            ==
                            "60min_window_15min_stride",
                            "total_windows"
                        ].iloc[0]
                    )
                    ==
                    206693
                ),

            "detail":
                "206,693 windows after 340-row purge",
        },

        {
            "check":
                "Historical test-derived stability branch retired",

            "passed":
                (
                    threshold_branch_status[
                        "final_status"
                    ].iloc[0]
                    ==
                    "RETIRED_FROM_FINAL_NUMERIC_EVIDENCE"
                ),

            "detail":
                (
                    "Replaced by purged-validation "
                    "diagnostic for final evidence"
                ),
        },

        {
            "check":
                "UK-DALE correction artifacts present",

            "passed":
                uk_csv_count > 0,

            "detail":
                f"{uk_csv_count} CSV artifact(s)",
        },

        {
            "check":
                "AMPds2 correction artifacts present",

            "passed":
                amp_csv_count > 0,

            "detail":
                f"{amp_csv_count} CSV artifact(s)",
        },

        {
            "check":
                "No zero-byte correction artifacts",

            "passed":
                nonzero_artifacts,

            "detail":
                (
                    f"{len(correction_files_before_freeze)} "
                    "pre-freeze correction files checked"
                ),
        },
    ]
)


all_core_checks_passed = bool(
    audit_checks[
        "passed"
    ].all()
)


print("\n")
print("=" * 132)
print("MASTER AUDIT CHECKS")
print("=" * 132)


print(
    audit_checks.to_string(
        index=False
    )
)


assert all_core_checks_passed, (
    "At least one final master-audit check failed."
)


# ============================================================
# K. Create cryptographic artifact manifest
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


manifest_rows = []


for path in sorted(
    correction_files_before_freeze
):

    relative_path = (
        path.relative_to(
            CORRECTION_ROOT
        )
    )


    manifest_rows.append(
        {
            "relative_path":
                str(
                    relative_path
                ),

            "suffix":
                path.suffix.lower(),

            "size_bytes":
                int(
                    path.stat().st_size
                ),

            "modified_time":
                datetime.fromtimestamp(
                    path.stat().st_mtime
                ).isoformat(),

            "sha256":
                sha256_file(
                    path
                ),
        }
    )


artifact_manifest = pd.DataFrame(
    manifest_rows
)


assert len(
    artifact_manifest
) == len(
    correction_files_before_freeze
)


assert artifact_manifest[
    "sha256"
].str.len().eq(
    64
).all()


print("\n")
print("=" * 132)
print("ARTIFACT FREEZE MANIFEST SUMMARY")
print("=" * 132)


print(
    "Correction artifacts hashed:",
    len(
        artifact_manifest
    )
)

print(
    "Total frozen size (MB):",
    f"{artifact_manifest['size_bytes'].sum() / (1024 ** 2):.3f}"
)

print(
    "Unique SHA-256 values:",
    artifact_manifest[
        "sha256"
    ].nunique()
)


# ============================================================
# L. Save final audit/freeze artifacts
# ============================================================

FREEZE_TIMESTAMP = (
    datetime.now()
    .astimezone()
    .isoformat()
)


STATUS_PATH = (
    FREEZE_DIR
    / "FINAL_SPLIT_BOUNDARY_CORRECTION_STATUS.csv"
)

NAB_HEADLINE_PATH = (
    FREEZE_DIR
    / "FINAL_AUTHORITATIVE_NAB_RESULTS.csv"
)

AUDIT_PATH = (
    FREEZE_DIR
    / "FINAL_SPLIT_BOUNDARY_MASTER_AUDIT.csv"
)

MANIFEST_PATH = (
    FREEZE_DIR
    / "FINAL_SPLIT_BOUNDARY_ARTIFACT_MANIFEST.csv"
)

README_PATH = (
    FREEZE_DIR
    / "README_SPLIT_BOUNDARY_CORRECTION_FREEZE.md"
)

JSON_PATH = (
    FREEZE_DIR
    / "FINAL_SPLIT_BOUNDARY_FREEZE_METADATA.json"
)


correction_status.to_csv(
    STATUS_PATH,
    index=False
)

authoritative_nab.to_csv(
    NAB_HEADLINE_PATH,
    index=False
)

audit_checks.to_csv(
    AUDIT_PATH,
    index=False
)

artifact_manifest.to_csv(
    MANIFEST_PATH,
    index=False
)


freeze_metadata = {

    "freeze_timestamp":
        FREEZE_TIMESTAMP,

    "correction_root":
        str(
            CORRECTION_ROOT
        ),

    "core_audit_passed":
        all_core_checks_passed,

    "artifact_count":
        int(
            len(
                artifact_manifest
            )
        ),

    "nab_60_30_corrected_windows":
        103412,

    "nab_60_15_corrected_windows":
        206693,

    "ukdale_corrected_windows":
        363909,

    "ampds2_corrected_windows":
        735777,

    "blued_retained_windows":
        20406,

    "controlled_ukdale_evaluation_rows":
        18391,

    "controlled_ampds2_evaluation_rows":
        25876,

    "historical_threshold_stability_status":
        "RETIRED_FROM_FINAL_NUMERIC_EVIDENCE",

    "additional_training_after_cell_45":
        False,

    "historical_artifacts_overwritten":
        False,
}


with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        freeze_metadata,
        file,
        indent=2
    )


readme_text = f"""# Split-Boundary Leakage Correction — Final Freeze

Freeze timestamp: {FREEZE_TIMESTAMP}

## Final audit status

PASS.

All principal experimental result families have now been audited for
raw temporal overlap across chronological partition boundaries.

## Leakage-control rule

Chronological ordering alone was not treated as sufficient when
overlapping windows had already been constructed. A timestamp-level
purge was applied within the relevant series, meter, channel, or
recording block so that retained windows from adjacent partitions did
not share raw temporal observations.

Training populations and training-only normalization statistics were
preserved wherever possible.

Neural models were refitted only when the external validation
partition directly controlled training through early stopping,
checkpoint selection, or learning-rate scheduling.

## Corrected primary populations

- NAB 60-minute / 30-minute stride:
  103,526 original windows -> 103,412 retained windows.
  114 boundary-overlap windows removed.

- NAB 60-minute / 15-minute stride:
  207,033 original windows -> 206,693 retained windows.
  340 boundary-overlap windows removed.

- UK-DALE selected Isolation Forest:
  363,940 original windows -> 363,909 retained windows.
  31 boundary-overlap windows removed.

- AMPds2 selected Isolation Forest:
  735,819 original windows -> 735,777 retained windows.
  42 boundary-overlap windows removed.

- BLUED:
  20,414 original valid windows -> 20,406 retained modeling windows.
  8 boundary-crossing windows removed.

## Controlled residential comparison

- UK-DALE external evaluation:
  19,694 original evaluation rows -> 18,391 retained rows.

- AMPds2 external evaluation:
  26,250 original evaluation rows -> 25,876 retained rows.

The dense Autoencoders in the controlled residential comparison used
internal validation drawn from the balanced reference pool rather than
the external evaluation partitions, so no retraining was required.

## Corrected NAB result families

### Five-model label-blind comparison

Corrected F1 ranking:

1. One-Class SVM: 0.278158205
2. Isolation Forest: 0.232743583
3. Dense Autoencoder: 0.229755179
4. Local Outlier Factor: 0.189484127
5. LSTM Autoencoder: 0.156415695

The historical ranking was preserved.

### Representation scaling

Top corrected configuration:

60/30 + per-series z-score + Isolation Forest

Test F1: 0.290618762

The top configuration was unchanged.

### Feature selection

Top corrected configuration:

60/15 + correlation pruning at 0.95 + Isolation Forest

Test F1: 0.293259074

The top configuration was unchanged.

### Robust-feature experiment

Top corrected configuration:

60/30 + original 23 recomputed features + Isolation Forest

Test F1: 0.262594739

The top configuration was unchanged.

## Threshold-stability handling

The historical EXP_NAB_threshold_stability experiment is not used as
final numeric evidence because percentile thresholds were calculated
directly from the test anomaly-score distribution and the reported
best/stable regions were evaluated retrospectively with test labels.

Its historical Autoencoder also used the old unpurged external
validation partition for early stopping.

For final thesis-facing stability evidence, use the purged-validation
diagnostic created from the corrected representation-scaling branch.
The historical threshold-stability artifacts remain preserved for
provenance only.

## Historical artifact policy

Historical experiment artifacts were not overwritten.

Corrected outputs are stored separately under:

{CORRECTION_ROOT}

This FINAL_FREEZE directory contains the master audit, authoritative
status tables, and SHA-256 artifact manifest.

## Interpretation rule

Differences between historical and corrected results are generally
small and the principal rankings/configuration conclusions were
preserved. These corrections establish leakage-controlled evaluation;
they should not be described as model improvements.
"""


README_PATH.write_text(
    readme_text,
    encoding="utf-8"
)


# ============================================================
# M. Verify files written
# ============================================================

FINAL_OUTPUTS = [
    STATUS_PATH,
    NAB_HEADLINE_PATH,
    AUDIT_PATH,
    MANIFEST_PATH,
    README_PATH,
    JSON_PATH,
]


for path in (
    FINAL_OUTPUTS
):

    assert path.exists()

    assert (
        path.stat().st_size
        >
        0
    )


# ============================================================
# N. Final freeze checkpoint
# ============================================================

print("\n")
print("=" * 132)
print("CELL 46 CHECKPOINT")
print("=" * 132)


print(
    "Core master audit passed:",
    all_core_checks_passed
)

print(
    "Critical correction directories:",
    len(
        CRITICAL_DIRECTORIES
    )
)

print(
    "Correction artifacts hashed:",
    len(
        artifact_manifest
    )
)

print(
    "NAB 60/30 final windows:",
    103412
)

print(
    "NAB 60/15 final windows:",
    206693
)

print(
    "UK-DALE final windows:",
    363909
)

print(
    "AMPds2 final windows:",
    735777
)

print(
    "BLUED final windows:",
    20406
)

print(
    "Historical threshold-stability branch retired:",
    True
)

print(
    "Historical experiment artifacts overwritten:",
    False
)

print(
    "Additional model training required:",
    False
)

print(
    "\nFinal freeze directory:"
)

print(
    FREEZE_DIR
)

print(
    "\nFinal files:"
)

for path in (
    FINAL_OUTPUTS
):

    print(
        " -",
        path.name
    )


print("\n")
print("=" * 132)
print("CELL 46 COMPLETE — SPLIT-BOUNDARY CORRECTION CLOSED")
print("=" * 132)

print(
    "\nAll identified overlapping-window split-boundary issues "
    "have now been audited, corrected, or formally retired from "
    "final numeric evidence. No further model execution is required "
    "for this correction problem."
)

In [ ]:
BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")
CORRECTION = BASE / "experiments" / "EXP_SPLIT_BOUNDARY_PURGE_CORRECTION"

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")
CORRECTION = BASE / "experiments" / "EXP_SPLIT_BOUNDARY_PURGE_CORRECTION"

print(BASE)
print(CORRECTION)
print("BASE exists:", BASE.exists())
print("CORRECTION exists:", CORRECTION.exists())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")
CORRECTION = BASE / "experiments" / "EXP_SPLIT_BOUNDARY_PURGE_CORRECTION"

print("BASE:", BASE)
print("BASE exists:", BASE.exists())
print("CORRECTION:", CORRECTION)
print("CORRECTION exists:", CORRECTION.exists())

In [ ]:
# ============================================================
# CELL 33B — AMPds2 LIME correction after split-boundary purge
# Regenerates ONLY the 8 AMPds2 LIME explanations.
# No hyperparameter tuning, no model-selection change, no SHAP rerun.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

try:
    from lime.lime_tabular import LimeTabularExplainer
except Exception:
    !pip -q install lime
    from lime.lime_tabular import LimeTabularExplainer

print("=" * 120)
print("AMPds2 LIME CORRECTION — CORRECTED VALIDATION THRESHOLD")
print("=" * 120)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

AMP_EXP = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"
AMP_XAI = AMP_EXP / "xai"
AMP_TABLES = AMP_EXP / "tables"

CORRECTED_SCORED = (
    CORRECTION / "AMPDS2" / "ampds2_if_scored_boundary_corrected.csv"
)

CANDIDATES = (
    AMP_TABLES / "ampds2_xai_representative_top8_channels_test_windows.csv"
)

HISTORICAL_LIME = (
    AMP_XAI / "ampds2_lime_explanations_selected_test_windows.csv"
)

OUT = CORRECTION / "AMPDS2" / "XAI_LIME_CORRECTION"
OUT.mkdir(parents=True, exist_ok=True)

CORRECTED_LIME = (
    OUT / "ampds2_lime_explanations_selected_test_windows_CORRECTED.csv"
)

CORRECTED_TOP5 = (
    OUT / "ampds2_lime_top5_per_candidate_CORRECTED.csv"
)

COMPARISON = (
    OUT / "ampds2_lime_historical_vs_corrected_top5.csv"
)

for p in [CORRECTED_SCORED, CANDIDATES, HISTORICAL_LIME]:
    print(f"{p.name}: {p.exists()}")
    assert p.exists(), f"Missing required file: {p}"

# ------------------------------------------------------------
# 2. Load authoritative corrected population and historical XAI
# ------------------------------------------------------------

df = pd.read_csv(CORRECTED_SCORED, low_memory=False)
cand = pd.read_csv(CANDIDATES, low_memory=False)
old_lime = pd.read_csv(HISTORICAL_LIME, low_memory=False)

for frame in [df, cand, old_lime]:
    for col in ["window_start", "window_end"]:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], utc=True, errors="raise")

assert len(df) == 735777
assert len(cand) == 8
assert len(old_lime) == 80

FEATURES = [
    "energy", "first_last_diff", "iqr", "kurtosis", "mad", "max",
    "max_abs_diff", "mean", "mean_abs_diff", "median", "min",
    "n_points", "peak_count", "q25", "q75", "range", "rms",
    "skew", "slope", "std", "std_diff", "sum", "var"
]

SCALED = [f"{f}_z" for f in FEATURES]

missing = [c for c in SCALED if c not in df.columns]
assert not missing, f"Missing scaled features: {missing}"

# ------------------------------------------------------------
# 3. Match the SAME 8 selected windows to corrected TEST rows
# ------------------------------------------------------------

KEYS = [
    "channel_code",
    "primary_category",
    "window_start",
    "window_end",
]

cand_keys = cand[KEYS].drop_duplicates().copy()
assert len(cand_keys) == 8

cand_keys["candidate_id"] = np.arange(8)

explain_df = (
    cand_keys
    .merge(df, on=KEYS, how="left", validate="one_to_one")
    .sort_values("candidate_id")
    .reset_index(drop=True)
)

assert explain_df["split"].notna().all()
assert (explain_df["split"] == "test").all()

assert (
    explain_df["predicted_anomaly_corrected"].astype(int) == 1
).all()

print("\nSelected corrected AMPds2 XAI windows:")

display(
    explain_df[
        [
            "candidate_id",
            "channel_code",
            "primary_category",
            "window_start",
            "window_end",
            "anomaly_score",
            "if_threshold_corrected",
            "predicted_anomaly_corrected"
        ]
    ]
)

# ------------------------------------------------------------
# 4. Deterministically reconstruct the fixed IF model
#
# Training population was unchanged by the boundary purge.
# This is reconstruction of the already-fixed model setup,
# not tuning or a new experiment.
# ------------------------------------------------------------

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()

X_train = train_df[SCALED].to_numpy(dtype=np.float64)
X_val = val_df[SCALED].to_numpy(dtype=np.float64)
X_explain = explain_df[SCALED].to_numpy(dtype=np.float64)

assert len(X_train) == 441483
assert len(X_val) == 147147
assert len(X_explain) == 8

if_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("\nReconstructing fixed Isolation Forest...")

if_model.fit(X_train)

# ------------------------------------------------------------
# 5. Verify exact score provenance BEFORE regenerating LIME
# ------------------------------------------------------------

val_scores = -if_model.decision_function(X_val)

stored_val_scores = (
    val_df["anomaly_score"]
    .to_numpy(dtype=np.float64)
)

val_max_diff = float(
    np.max(
        np.abs(
            val_scores - stored_val_scores
        )
    )
)

explain_scores = -if_model.decision_function(X_explain)

stored_explain_scores = (
    explain_df["anomaly_score"]
    .to_numpy(dtype=np.float64)
)

explain_max_diff = float(
    np.max(
        np.abs(
            explain_scores - stored_explain_scores
        )
    )
)

print(
    f"Max validation-score difference: "
    f"{val_max_diff:.18e}"
)

print(
    f"Max selected-window score difference: "
    f"{explain_max_diff:.18e}"
)

if val_max_diff > 1e-10 or explain_max_diff > 1e-10:
    raise RuntimeError(
        "STOP: reconstructed IF does not reproduce the stored scores. "
        "Do not generate new LIME outputs; send me this output."
    )

# ------------------------------------------------------------
# 6. Verify corrected threshold
# ------------------------------------------------------------

threshold_from_scores = float(
    np.percentile(
        val_scores,
        95
    )
)

saved_thresholds = (
    val_df["if_threshold_corrected"]
    .dropna()
    .astype(float)
    .unique()
)

assert len(saved_thresholds) == 1

corrected_threshold = float(
    saved_thresholds[0]
)

EXPECTED = 0.029990580900317053

print(
    f"\nThreshold from purged validation: "
    f"{threshold_from_scores:.18f}"
)

print(
    f"Threshold in corrected artifact:  "
    f"{corrected_threshold:.18f}"
)

print(
    f"Expected audited threshold:       "
    f"{EXPECTED:.18f}"
)

assert np.isclose(
    threshold_from_scores,
    corrected_threshold,
    rtol=0.0,
    atol=1e-12
)

assert np.isclose(
    corrected_threshold,
    EXPECTED,
    rtol=0.0,
    atol=1e-12
)

old_thresholds = (
    old_lime["if_threshold"]
    .dropna()
    .astype(float)
    .unique()
)

assert len(old_thresholds) == 1

historical_threshold = float(
    old_thresholds[0]
)

print(
    f"Historical LIME threshold:        "
    f"{historical_threshold:.18f}"
)

print(
    "Threshold delta:                 "
    f"{corrected_threshold - historical_threshold:.18e}"
)

# ------------------------------------------------------------
# 7. Corrected classifier-style LIME interface
#
# Same historical design, but threshold AND validation-score
# temperature now use the PURGED validation population.
# ------------------------------------------------------------

temperature = float(
    np.std(
        val_scores
    )
)

if temperature <= 1e-9:
    temperature = 1.0

print(
    f"Purged-validation temperature:    "
    f"{temperature:.18f}"
)

def lime_predict_corrected(X):

    X = np.asarray(
        X,
        dtype=np.float64
    )

    scores = -if_model.decision_function(X)

    z = (
        scores - corrected_threshold
    ) / temperature

    z = np.clip(
        z,
        -60.0,
        60.0
    )

    p_flagged = (
        1.0 /
        (
            1.0 + np.exp(-z)
        )
    )

    p_normal = 1.0 - p_flagged

    return np.column_stack(
        [
            p_normal,
            p_flagged
        ]
    )

# ------------------------------------------------------------
# 8. Regenerate ONLY the 8 LIME explanations
# ------------------------------------------------------------

lime_explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=SCALED,
    class_names=[
        "normal_like",
        "flagged_like"
    ],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

rows = []

for i in range(8):

    meta = explain_df.iloc[i]

    exp = lime_explainer.explain_instance(
        data_row=X_explain[i],
        predict_fn=lime_predict_corrected,
        num_features=10,
        labels=[1]
    )

    local = exp.as_list(
        label=1
    )

    assert len(local) == 10

    for rank, (condition, weight) in enumerate(
        local,
        start=1
    ):

        rows.append(
            {
                "candidate_id": i,
                "dataset": "AMPds2",
                "channel_code": meta["channel_code"],
                "primary_category": meta["primary_category"],
                "window_start": meta["window_start"],
                "window_end": meta["window_end"],
                "anomaly_score": meta["anomaly_score"],
                "if_threshold": corrected_threshold,
                "historical_if_threshold": historical_threshold,
                "threshold_provenance":
                    "purged_validation_95th_percentile",
                "lime_mode": "classification",
                "lime_label": 1,
                "lime_num_features": 10,
                "lime_random_state": 42,
                "rank": rank,
                "lime_condition": condition,
                "lime_weight": float(weight),
                "abs_lime_weight": abs(float(weight)),
            }
        )

lime_new = pd.DataFrame(rows)

assert len(lime_new) == 80

# ------------------------------------------------------------
# 9. Corrected top-5 summary
# ------------------------------------------------------------

top5_new = (
    lime_new
    .sort_values(
        [
            "candidate_id",
            "abs_lime_weight"
        ],
        ascending=[
            True,
            False
        ]
    )
    .groupby("candidate_id")
    .head(5)
    .groupby(
        [
            "candidate_id",
            "channel_code",
            "primary_category"
        ]
    )
    .agg(
        top5_lime_conditions=(
            "lime_condition",
            lambda x: " | ".join(x)
        ),
        top5_abs_lime=(
            "abs_lime_weight",
            lambda x:
                ", ".join(
                    f"{v:.6f}"
                    for v in x
                )
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 10. Historical-vs-corrected top-5 comparison
# ------------------------------------------------------------

summary_rows = []

for i in range(8):

    old_i = (
        old_lime[
            old_lime["candidate_id"] == i
        ]
        .sort_values(
            "abs_lime_weight",
            ascending=False
        )
        .head(5)
    )

    new_i = (
        lime_new[
            lime_new["candidate_id"] == i
        ]
        .sort_values(
            "abs_lime_weight",
            ascending=False
        )
        .head(5)
    )

    old_set = set(
        old_i["lime_condition"]
    )

    new_set = set(
        new_i["lime_condition"]
    )

    inter = old_set & new_set
    union = old_set | new_set

    summary_rows.append(
        {
            "candidate_id": i,
            "channel_code":
                new_i["channel_code"].iloc[0],
            "primary_category":
                new_i["primary_category"].iloc[0],
            "historical_top5":
                " | ".join(
                    old_i["lime_condition"]
                ),
            "corrected_top5":
                " | ".join(
                    new_i["lime_condition"]
                ),
            "top5_overlap_count":
                len(inter),
            "top5_jaccard":
                (
                    len(inter) / len(union)
                    if union
                    else 1.0
                ),
        }
    )

comparison_df = pd.DataFrame(
    summary_rows
)

# ------------------------------------------------------------
# 11. Save corrected artifacts WITHOUT overwriting history
# ------------------------------------------------------------

lime_new.to_csv(
    CORRECTED_LIME,
    index=False
)

top5_new.to_csv(
    CORRECTED_TOP5,
    index=False
)

comparison_df.to_csv(
    COMPARISON,
    index=False
)

print("\nCorrected LIME top-5 summary:")
display(top5_new)

print("\nHistorical vs corrected top-5 comparison:")
display(comparison_df)

print("\nSaved:")
print(CORRECTED_LIME)
print(CORRECTED_TOP5)
print(COMPARISON)

print("\n" + "=" * 120)
print("AMPds2 LIME CORRECTION COMPLETE")
print("=" * 120)

print(
    "No hyperparameter tuning, no model-selection change, "
    "no SHAP rerun, and no new experiment was performed."
)

In [ ]:
# ============================================================
# CELL 33C — FINAL FIGURE-ASSET AUDIT
# No experiments, no training, no files modified.
# ============================================================

from pathlib import Path

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

print("=" * 110)
print("FINAL THESIS FIGURE-ASSET AUDIT")
print("=" * 110)

# ------------------------------------------------------------
# 1. Known relevant directories
# ------------------------------------------------------------

V2_FIG_DIR = (
    BASE
    / "processed_data"
    / "PH4"
    / "figures"
    / "NAB"
    / "feature_set_v2"
)

V2_METRIC_DIR = (
    BASE
    / "processed_data"
    / "PH4"
    / "supervised"
    / "NAB"
    / "feature_set_v2"
    / "metrics"
)

V1_METRIC_DIR = (
    BASE
    / "processed_data"
    / "PH4"
    / "supervised"
    / "NAB"
    / "metrics"
)

NAB_INTERP_DIR = (
    BASE
    / "results"
    / "EXP_NAB_interpretation_stability"
)

BLUED_FIG_DIR = (
    BASE
    / "experiments"
    / "EXP_BLUED_FIVE_MODEL_LABEL_BLIND"
    / "final_five_model_comparison"
    / "publication_outputs"
    / "figures"
)

BLUED_TABLE_DIR = (
    BASE
    / "experiments"
    / "EXP_BLUED_FIVE_MODEL_LABEL_BLIND"
    / "final_five_model_comparison"
    / "publication_outputs"
    / "tables"
)

dirs = {
    "NAB V2 figures": V2_FIG_DIR,
    "NAB V2 metrics": V2_METRIC_DIR,
    "NAB V1 metrics": V1_METRIC_DIR,
    "NAB interpretation": NAB_INTERP_DIR,
    "BLUED figures": BLUED_FIG_DIR,
    "BLUED tables": BLUED_TABLE_DIR,
}

print("\nDIRECTORY CHECK")
print("-" * 110)

for name, path in dirs.items():
    print(f"{name:25s} exists={path.exists()} | {path}")

# ------------------------------------------------------------
# 2. Exact assets we expect
# ------------------------------------------------------------

expected = {
    "Figure 4.1 current V2":
        V2_FIG_DIR
        / "nab_supervised_feature_set_v2_model_comparison_deduplicated.png",

    "Figure 4.2 genuine V1-vs-V2":
        V2_FIG_DIR
        / "nab_feature_set_v1_vs_v2_comparison_deduplicated.png",

    "Figure 4.3 intended mean_abs_diff":
        NAB_INTERP_DIR
        / "feature_boxplot_6015_mean_abs_diff.png",

    "Figure 4.5 current BLUED ranking":
        BLUED_FIG_DIR
        / "blued_auc_metrics.png",

    "NAB V2 metrics":
        V2_METRIC_DIR
        / "nab_supervised_feature_set_v2_metrics_deduplicated.csv",

    "NAB V1 metrics":
        V1_METRIC_DIR
        / "nab_supervised_4model_metrics_train_val_test.csv",

    "BLUED publication results":
        BLUED_TABLE_DIR
        / "blued_five_model_complete_results.csv",
}

print("\nEXACT ASSET CHECK")
print("-" * 110)

for name, path in expected.items():
    print(
        f"{name:38s} "
        f"exists={path.exists()} | {path}"
    )

# ------------------------------------------------------------
# 3. Search for alternative candidates if exact names differ
# ------------------------------------------------------------

print("\nNAB FIGURE CANDIDATES")
print("-" * 110)

nab_candidates = []

for root in [
    V2_FIG_DIR,
    NAB_INTERP_DIR,
]:
    if root.exists():
        nab_candidates.extend(
            sorted(root.glob("*.png"))
        )

for p in nab_candidates:
    name = p.name.lower()

    if (
        "feature_set" in name
        or "mean_abs_diff" in name
        or "boxplot" in name
        or "supervised" in name
    ):
        print(p)

print("\nBLUED FIGURE CANDIDATES")
print("-" * 110)

if BLUED_FIG_DIR.exists():
    for p in sorted(BLUED_FIG_DIR.glob("*")):
        print(p)

print("\n" + "=" * 110)
print("FIGURE-ASSET AUDIT COMPLETE")
print("=" * 110)

In [ ]:
# ============================================================
# CELL 33D — FINAL THESIS FIGURE CORRECTIONS
#
# Corrects presentation/label issues only.
# No model training.
# No threshold selection.
# No experiment rerun.
# Historical figure assets are NOT overwritten.
# ============================================================

from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 120)
print("FINAL THESIS FIGURE CORRECTIONS")
print("=" * 120)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

V2_FIG_DIR = (
    BASE
    / "processed_data"
    / "PH4"
    / "figures"
    / "NAB"
    / "feature_set_v2"
)

V2_METRICS_PATH = (
    BASE
    / "processed_data"
    / "PH4"
    / "supervised"
    / "NAB"
    / "feature_set_v2"
    / "metrics"
    / "nab_supervised_feature_set_v2_metrics_deduplicated.csv"
)

V1_METRICS_PATH = (
    BASE
    / "processed_data"
    / "PH4"
    / "supervised"
    / "NAB"
    / "metrics"
    / "nab_supervised_4model_metrics_train_val_test.csv"
)

MEAN_ABS_DIFF_SOURCE = (
    BASE
    / "results"
    / "EXP_NAB_interpretation_stability"
    / "feature_boxplot_6015_mean_abs_diff.png"
)

BLUED_RESULTS_PATH = (
    BASE
    / "experiments"
    / "EXP_BLUED_FIVE_MODEL_LABEL_BLIND"
    / "final_five_model_comparison"
    / "publication_outputs"
    / "tables"
    / "blued_five_model_complete_results.csv"
)

OUT = (
    BASE
    / "experiments"
    / "EXP_SPLIT_BOUNDARY_PURGE_CORRECTION"
    / "FINAL_THESIS_FIGURE_CORRECTIONS"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

for p in [
    V2_METRICS_PATH,
    V1_METRICS_PATH,
    MEAN_ABS_DIFF_SOURCE,
    BLUED_RESULTS_PATH,
]:
    print(f"{p.name}: {p.exists()}")
    assert p.exists(), f"Required source missing: {p}"

# ------------------------------------------------------------
# 2. Load NAB supervised result tables
# ------------------------------------------------------------

v2 = pd.read_csv(
    V2_METRICS_PATH
)

v1 = pd.read_csv(
    V1_METRICS_PATH
)

print("\nV2 columns:")
print(v2.columns.tolist())

print("\nV1 columns:")
print(v1.columns.tolist())

# Use only the validation-selected operating threshold rows.
v2_plot = (
    v2[
        v2["threshold_source"]
        == "validation_f1_optimized"
    ]
    .copy()
)

v1_plot = (
    v1[
        v1["threshold_source"]
        == "validation_f1_optimized"
    ]
    .copy()
)

assert len(v2_plot) == 4
assert len(v1_plot) == 4

MODEL_ORDER = [
    "RandomForest",
    "XGBoost",
    "MLP",
    "LogisticRegression",
]

# ------------------------------------------------------------
# 3. Figure 4.1
#    Corrected Feature Set V2 FINAL-TEST chart
# ------------------------------------------------------------

fig41 = v2_plot.copy()

fig41["model"] = pd.Categorical(
    fig41["model"],
    categories=MODEL_ORDER,
    ordered=True
)

fig41 = (
    fig41
    .sort_values("model")
    .reset_index(drop=True)
)

x = np.arange(
    len(fig41)
)

width = 0.18

figure, axis = plt.subplots(
    figsize=(11, 6)
)

axis.bar(
    x - 1.5 * width,
    fig41["test_precision"],
    width,
    label="Precision",
)

axis.bar(
    x - 0.5 * width,
    fig41["test_recall"],
    width,
    label="Recall",
)

axis.bar(
    x + 0.5 * width,
    fig41["test_f1"],
    width,
    label="F1-Score",
)

axis.bar(
    x + 1.5 * width,
    fig41["test_roc_auc"],
    width,
    label="ROC-AUC",
)

axis.set_xticks(x)

axis.set_xticklabels(
    fig41["model"],
    rotation=15
)

axis.set_ylabel("Metric value")

axis.set_title(
    "NAB Supervised Final-Test Results: Feature Set V2\n"
    "Validation-Selected Operating Thresholds"
)

axis.set_ylim(
    0,
    1
)

axis.legend()

axis.grid(
    axis="y",
    alpha=0.25
)

figure.tight_layout()

FIG41_PNG = (
    OUT
    / "Figure_4_1_NAB_Feature_Set_V2_FINAL_TEST_CORRECTED.png"
)

FIG41_PDF = (
    OUT
    / "Figure_4_1_NAB_Feature_Set_V2_FINAL_TEST_CORRECTED.pdf"
)

figure.savefig(
    FIG41_PNG,
    dpi=300,
    bbox_inches="tight"
)

figure.savefig(
    FIG41_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

# ------------------------------------------------------------
# 4. Figure 4.2
#    Genuine V1 vs V2 FINAL-TEST comparison
# ------------------------------------------------------------

comparison = pd.merge(
    v1_plot[
        [
            "model",
            "test_f1",
            "test_roc_auc",
        ]
    ].rename(
        columns={
            "test_f1": "v1_f1",
            "test_roc_auc": "v1_roc_auc",
        }
    ),
    v2_plot[
        [
            "model",
            "test_f1",
            "test_roc_auc",
        ]
    ].rename(
        columns={
            "test_f1": "v2_f1",
            "test_roc_auc": "v2_roc_auc",
        }
    ),
    on="model",
    how="inner",
    validate="one_to_one",
)

assert len(comparison) == 4

comparison["model"] = pd.Categorical(
    comparison["model"],
    categories=MODEL_ORDER,
    ordered=True
)

comparison = (
    comparison
    .sort_values("model")
    .reset_index(drop=True)
)

x = np.arange(
    len(comparison)
)

width = 0.20

figure, axis = plt.subplots(
    figsize=(11, 6)
)

axis.bar(
    x - 1.5 * width,
    comparison["v1_f1"],
    width,
    label="V1 F1-Score",
)

axis.bar(
    x - 0.5 * width,
    comparison["v2_f1"],
    width,
    label="V2 F1-Score",
)

axis.bar(
    x + 0.5 * width,
    comparison["v1_roc_auc"],
    width,
    label="V1 ROC-AUC",
)

axis.bar(
    x + 1.5 * width,
    comparison["v2_roc_auc"],
    width,
    label="V2 ROC-AUC",
)

axis.set_xticks(x)

axis.set_xticklabels(
    comparison["model"],
    rotation=15
)

axis.set_ylabel("Metric value")

axis.set_title(
    "NAB Supervised Final-Test Comparison: Feature Set V1 vs V2\n"
    "Validation-Selected Operating Thresholds"
)

axis.set_ylim(
    0,
    1
)

axis.legend()

axis.grid(
    axis="y",
    alpha=0.25
)

figure.tight_layout()

FIG42_PNG = (
    OUT
    / "Figure_4_2_NAB_V1_vs_V2_FINAL_TEST_CORRECTED.png"
)

FIG42_PDF = (
    OUT
    / "Figure_4_2_NAB_V1_vs_V2_FINAL_TEST_CORRECTED.pdf"
)

figure.savefig(
    FIG42_PNG,
    dpi=300,
    bbox_inches="tight"
)

figure.savefig(
    FIG42_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

# ------------------------------------------------------------
# 5. Figure 4.3
#    Use the EXISTING genuine mean_abs_diff separation plot.
#
#    No regeneration is necessary because the intended asset
#    already exists and was verified by the figure audit.
# ------------------------------------------------------------

FIG43_PNG = (
    OUT
    / "Figure_4_3_mean_abs_diff_feature_separation_CORRECTED.png"
)

shutil.copy2(
    MEAN_ABS_DIFF_SOURCE,
    FIG43_PNG
)

print(
    "\nFigure 4.3 copied from genuine source:"
)

print(
    MEAN_ABS_DIFF_SOURCE
)

# ------------------------------------------------------------
# 6. Figure 4.5
#    BLUED ROC-AUC + Average Precision
#
#    Existing `pr_auc` field is AP calculated using
#    average_precision_score; values are unchanged.
# ------------------------------------------------------------

blued = pd.read_csv(
    BLUED_RESULTS_PATH
)

required = [
    "model",
    "roc_auc",
    "pr_auc",
]

missing = [
    c for c in required
    if c not in blued.columns
]

assert not missing, (
    f"Missing BLUED columns: {missing}"
)

ranking = (
    blued
    .set_index("model")[
        [
            "roc_auc",
            "pr_auc",
        ]
    ]
    .rename(
        columns={
            "roc_auc": "ROC-AUC",
            "pr_auc": "Average Precision (AP)",
        }
    )
)

# Correct purged BLUED test prevalence:
# 450 event windows / 4079 total test windows.
event_prevalence = (
    450 / 4079
)

print(
    "\nBLUED positive-class prevalence:",
    event_prevalence
)

figure, axis = plt.subplots(
    figsize=(11, 6)
)

ranking.plot(
    kind="bar",
    ax=axis,
)

axis.axhline(
    y=event_prevalence,
    linestyle="--",
    linewidth=1.5,
    label=(
        "Positive-class prevalence / "
        "no-skill AP reference "
        f"({event_prevalence:.3f})"
    ),
)

axis.set_title(
    "BLUED Continuous Anomaly-Score Ranking"
)

axis.set_xlabel("Model")

axis.set_ylabel(
    "Metric value"
)

axis.set_ylim(
    0,
    0.85
)

axis.tick_params(
    axis="x",
    rotation=20
)

axis.grid(
    axis="y",
    alpha=0.3
)

axis.legend()

figure.tight_layout()

FIG45_PNG = (
    OUT
    / "Figure_4_5_BLUED_ROC_AUC_and_AP_CORRECTED.png"
)

FIG45_PDF = (
    OUT
    / "Figure_4_5_BLUED_ROC_AUC_and_AP_CORRECTED.pdf"
)

figure.savefig(
    FIG45_PNG,
    dpi=300,
    bbox_inches="tight"
)

figure.savefig(
    FIG45_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

# ------------------------------------------------------------
# 7. Save figure-correction manifest
# ------------------------------------------------------------

manifest = pd.DataFrame(
    [
        {
            "figure": "4.1",
            "file": FIG41_PNG.name,
            "action":
                "Regenerated from existing V2 final-test metrics; "
                "corrected validation/final-test wording.",
        },
        {
            "figure": "4.2",
            "file": FIG42_PNG.name,
            "action":
                "Regenerated genuine V1-vs-V2 comparison from "
                "existing final-test metrics at validation-selected thresholds.",
        },
        {
            "figure": "4.3",
            "file": FIG43_PNG.name,
            "action":
                "Copied verified genuine 60/15 mean_abs_diff "
                "feature-separation asset.",
        },
        {
            "figure": "4.5",
            "file": FIG45_PNG.name,
            "action":
                "Regenerated from existing BLUED results; "
                "PR-AUC relabeled correctly as Average Precision (AP).",
        },
    ]
)

MANIFEST_PATH = (
    OUT
    / "FINAL_THESIS_FIGURE_CORRECTION_MANIFEST.csv"
)

manifest.to_csv(
    MANIFEST_PATH,
    index=False
)

print("\n" + "=" * 120)
print("FINAL FIGURE CORRECTIONS COMPLETE")
print("=" * 120)

display(manifest)

print("\nOutput directory:")
print(OUT)

print("\nCorrected assets:")
for p in [
    FIG41_PNG,
    FIG42_PNG,
    FIG43_PNG,
    FIG45_PNG,
]:
    print(
        f"{p.name}: "
        f"exists={p.exists()} "
        f"size={p.stat().st_size if p.exists() else 0:,} bytes"
    )

print("\nNo model training or new experiment was performed.")

In [ ]:
# ============================================================
# FINAL VISUAL CHECK — Figures 4.1, 4.2, 4.3, 4.5
# ============================================================

from PIL import Image
from IPython.display import display

figure_files = [
    FIG41_PNG,
    FIG42_PNG,
    FIG43_PNG,
    FIG45_PNG,
]

for p in figure_files:
    print("\n" + "=" * 100)
    print(p.name)
    print("Exists:", p.exists())
    print("Size:", f"{p.stat().st_size:,} bytes")

    img = Image.open(p)
    print("Dimensions:", img.size)
    display(img)